# Paper B1 v5, GPU notebook, ONE HOUR (Skala-1.1 and four standard functionals, six reactions)

**What this notebook does.** The GPU side of the paper in a deadline of 50 minutes on a T4 (L4 and A100 reach further down the same list): Skala-1.1 (with its D3 term, headline) and its "without D3" energy from the same SCF, wB97M-V, B3LYP-D3(BJ), r2SCAN and PBE at every stationary point and fragment of six reactions (HCN, NH3, H2CO, CH3NC, F- + CH3F, Cl- + CH3Cl), Skala, wB97M-V and B3LYP-D3(BJ) along the relaxed paths of HCN, NH3, H2CO and the CH4 stretch (40 points), the same three at the GFN2-xTB scout geometries, a basis check with def2-QZVPP for HCN, NH3, H2CO (Skala and wB97M-V), and finally the figures, the tables, the frozen-criteria outcome table and the automated check on whatever exists. Everything that does not fit is listed as NOT RUN (never FAIL).

**How to run it.** Run the **CPU notebook first**, then this one (order CPU, GPU; no third pass). *Runtime > Change runtime type > GPU* (T4 is enough; L4 or A100 recommended), then *Runtime > Run all*. The only click is the Google Drive permission popup in the first cell. The geometries come from the same shipped, hashed **geometry pack** as the CPU notebook (no geometry work here). At the end the runtime is released (set `AUTO_DISCONNECT = False` to keep it).

**The deadline queue.** One ordered list. Before every calculation: *will it finish before minute 50?* (elapsed + 1.5 x the running mean of that kind of calculation below 50 minutes, and the phase inside its cap). If not, it is not started and is listed as "not run (time)" in `RUN_SUMMARY.md`. After the core comes a tail list (T4: expected to be empty; L4: reactions 7 and 8; A100: items 1 to 4, perhaps 5). Five minutes are reserved for figures, check and summary, which always run. A second *Run all* continues with what was not run.

**Skala with D3.** The headline method `Skala-1.1` is the published functional with its D3 term (package default); `Skala-1.1-noD3` is the same SCF minus the stored D3 energy. D3 is switched off only in the G0 reproduction of the stored v4 energies, which were computed without it.

**If something is not available.** No GPU: only the G0 diagnostics run and the notebook says so. Skala weights cannot be downloaded within the cap: the Skala rows are missing, the other functionals run, the check says NOT RUN. `STANDIN = True` (settings cell) is the dry run on the CPU with cheap functionals in tiny bases (numbers are not results).

| Phase | Computes | Est. min (T4) | Cap |
|---|---|---|---|
| install, weights | pinned pyscf, Skala wheel, gpu4pyscf, Skala weights (Drive cache) | 8 | 15 |
| pack | hash check, geometries to Drive | under 1 | |
| G0 | functional probe, Skala check, stored v4 HCN point 0 (gate, 1e-5 hartree) | 2 | 4 |
| G1a | 20 species x 5 functionals, def2-TZVP(D) | 13 | 18 |
| G1b | 40 path points x 3 functionals | 11 | 12 |
| G3 | scout check: 36 energies | 5 | 6 |
| G2 | def2-QZVPP, 9 species, Skala and wB97M-V | 8 | 8 |
| tail | by GPU class (see above) | T4 0 | to minute 50 |
| G5, check | figures, tables, outcome table, automated check | 3 | 5 |

Extension sessions (one click each, any order, see the README): `EXTENSION = 'GPU1'` (tail 1 to 3: reactions 7 and 8, PBE and r2SCAN on paths, QZVPP(D) for CH3NC and the SN2 systems) and `EXTENSION = 'GPU2'` (tail 4 and 5: PBE and r2SCAN at the scout geometries, Skala-relaxed HCN, CH3NC, F- + CH3F). `EXTENSION = 'GEOM'` is a flag of the CPU notebook (rebuild of the geometry pack).

**Cell 1: the clock, install (cap 15 minutes), shared helpers, Drive.** The clock of the 50-minute deadline starts in the first line. Installs the pinned pyscf 2.12.1, pyscf-dispersion, Skala 2026.9 and gpu4pyscf 1.8.1 (if a GPU is present), defines the shared helper functions and mounts Google Drive (one permission click).

In [ ]:
# ---- Cell 1: the clock starts here (install time counts against the 50 minutes), install with a 15-minute cap, shared helpers, Drive ----
import time as _t
_T0 = _t.time()
import sys, subprocess, importlib.util, shutil, os
try:   # Colab: ask for Google Drive FIRST, so that the Allow popup is the first thing you see. The 50-minute clock starts after the mount.
    from google.colab import drive as _drive
    print("=" * 90); print("CLICK 'Connect to Google Drive' and then Allow in the popup NOW. The 50-minute clock starts after you have clicked."); print("=" * 90)
    for _a in range(3):
        try:
            _drive.mount("/content/drive", force_remount=(_a > 0))
            if os.path.isdir("/content/drive/MyDrive"):
                break
        except Exception as _ex:
            print(f"Drive mount attempt {_a + 1} of 3 failed ({repr(_ex)[:80]}); trying again in 20 s: please click Allow.")
            _t.sleep(20)
    _T0 = _t.time()
except ImportError:
    pass
SKALA_VERSION = "2026.9"      # the version verified on Colab (2026.4 gave wrong forces)
PYSCF_VERSION = "2.12.1"      # PINNED: skala 2026.9 needs pyscf >= 2.11 and < 2.15 (wheel metadata), gpu4pyscf 1.8.1 needs >= 2.8.  Not yet run on Colab.
DISPERSION_VERSION = "1.7.0"  # pyscf-dispersion (the D3 library Skala and B3LYP-D3(BJ) use); the CPU notebook pins the same
INSTALL_CAP_MIN = 15          # install + weights together may not use more than this
def _left_s():
    return max(20.0, 60.0 * INSTALL_CAP_MIN - (_t.time() - _T0))
def _pip(*a):
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True, timeout=_left_s())
    except Exception as ex:
        print("FAIL (cap or error) " + " ".join(a), repr(ex)[:80]); return False
    print(("ok   " if r.returncode == 0 else "FAIL ") + " ".join(a) + f"  ({_t.time() - _T0:.0f} s since start)")
    if r.returncode:
        print(r.stderr[-500:])
    return r.returncode == 0
def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
def _gpu_present():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, timeout=15).stdout.strip()
        return bool(out)
    except Exception:
        return False
GPU_PRESENT = _gpu_present()
try:
    if GPU_PRESENT and not (_have("skala") and _have("gpu4pyscf")):
        _pip(f"pyscf=={PYSCF_VERSION}", f"pyscf-dispersion=={DISPERSION_VERSION}")                  # pyscf first, pinned, so that no later package pulls another version
        if not _pip(f"skala-cuda12x=={SKALA_VERSION}"):                                             # GPU wheel for CUDA 12 (the Colab default); needs gpu4pyscf-cuda12x >= 1.8, < 1.9
            _pip(f"skala=={SKALA_VERSION}")
        if not _have("gpu4pyscf"):
            _pip("gpu4pyscf-cuda12x==1.8.1")                                                         # pinned; NO unpinned fallback.  If this fails, the GPU route is off and the notebook says so
    elif not GPU_PRESENT:
        print("NO GPU found in this session: Skala and gpu4pyscf are not installed. Only the G0 diagnostics will run (Runtime > Change runtime type > GPU).")
    need = [spec for mod, spec in {"pyscf": f"pyscf=={PYSCF_VERSION}", "pyscf.dispersion": f"pyscf-dispersion=={DISPERSION_VERSION}", "openpyxl": "openpyxl", "matplotlib": "matplotlib"}.items() if not _have(mod)]
    if need:
        print("installing:", need)
        _pip(*need)
except Exception as ex:
    print("install step problem (the notebook carries on):", repr(ex)[:200])

# ===== b1v5_common.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_common.py  --  shared helpers for the Paper B1 v5 notebooks (CPU and GPU).
#
# This file is pasted unchanged into the first code cell of BOTH notebooks, and
# it can also be imported as a normal module.  It only needs numpy.
#
# What is in here, in plain words:
#   1. where the files live (Google Drive folder or a local folder)
#   2. one results table per phase (CSV), written after EVERY calculation,
#      so a disconnect never loses work and a rerun skips what is done
#   3. DONE_phaseXX marker files
#   4. geometry files (geoms/*.json and geoms/*.xyz)
#   5. the system database: the eight reactions plus the CH4 stretch, with
#      start geometries, reference tiers and the species each reaction needs
#   6. unit constants, hardware and package-version recorders
# ==============================================================================
import os, sys, json, csv, io, math, time, platform, datetime, importlib.metadata
import numpy as np

# ------------------------------------------------------------------ constants
HARTREE_KCAL = 627.5094740631          # 1 hartree in kcal/mol
BOHR = 0.529177210903                  # 1 bohr in Angstrom
HARTREE_CM1 = 219474.6313705           # 1 hartree in wavenumbers
EV_HARTREE = 1.0 / 27.211386245988
KCAL = HARTREE_KCAL                    # short alias

# ------------------------------------------------------------------ names used in the CSV files
# Geometry levels (column "geometry_level")
LV_DB = "db_qcisd_mg3"        # published NHTBH38/08 database geometry
LV_OWN = "own_b3lyp"          # our DF-B3LYP-D3(BJ)/def2-TZVP(D) optimisation
LV_SCANPEAK = "own_scanpeak"  # transition state taken from the top of our relaxed scan (fallback)
LV_SCAN = "own_scan"          # a point of our relaxed scan
LV_XTB = "xtb_gfn2"           # GFN2-xTB geometry (scout)
LV_SKALA = "skala_relaxed"    # Skala-relaxed geometry (GPU notebook)
LV_GRID = "rigid_grid"        # v4 rigid HCN grid
LV_NB4 = "v4_nb4"             # v4 NB4 path geometry
# Order in which a stored geometry is picked when the same species exists at several levels (review 2026-11-15, row 4):
# our own B3LYP geometry first (the HEADLINE protocol), the published database geometry last (I11 cross-check only).
LEVEL_PREFERENCE = [LV_OWN, LV_SCANPEAK, LV_DB]
# A barrier or reaction energy needs ALL its species at ONE geometry family; a family is a set of levels that count as "the same protocol".
# own_scanpeak is a transition state taken from the top of our own B3LYP scan: same B3LYP protocol, flagged in the label.
FAMILY_LEVELS = {"own": (LV_OWN, LV_SCANPEAK), "db": (LV_DB,)}
FAMILY_OF = {LV_OWN: "own", LV_SCANPEAK: "own", LV_DB: "db"}
HEADLINE_FAMILY = "own"       # 'own' (default), 'db', or 'any' (first complete family per quantity: own, then db); set from config.json by the check

# Method names (column "method").  Both notebooks must use exactly these strings.
# Skala: "Skala-1.1" is the HEADLINE = the published functional WITH its D3 term (skala package default with_dftd3=True; the D3 energy is
# stored separately in "extra"); "Skala-1.1-noD3" = the same SCF minus that D3 energy (written as a second row).  v4 used no D3 ("v4-skala").
M_SKALA, M_PBE, M_R2SCAN, M_B3LYP, M_WB97MV = "Skala-1.1", "PBE", "r2SCAN", "B3LYP-D3BJ", "wB97M-V"
M_SKALA_NOD3 = "Skala-1.1-noD3"
M_WB97XV = "wB97X-V"                       # fallback anchor (plan section 5)
M_CCSDT_FC, M_CCSDT_AE, M_XTB, M_B2PLYP, M_DFB3LYP = "CCSD(T)-fc", "CCSD(T)-ae", "xTB-GFN2", "B2PLYP", "DF-B3LYP-D3BJ"
M_MP2DF = "DF-MP2-fc"                      # density-fitted MP2, frozen core: only used for the basis correction of the CCSD(T) reference
M_CCSDT_BC = "CCSD(T)+dMP2"                # DERIVED (never stored): CCSD(T)-fc/TZVP(D) + [MP2/QZVPP(D) - MP2/TZVP(D)]
LADDER = [M_SKALA, M_PBE, M_R2SCAN, M_B3LYP, M_WB97MV]          # the five functionals of the paper (Skala = with D3)
LADDER_ALL = [M_SKALA, M_SKALA_NOD3, M_PBE, M_R2SCAN, M_B3LYP, M_WB97MV]
ANCHORS = (M_WB97MV, M_WB97XV)             # anchor functional and its fallback
CORE_METHODS = (M_DFB3LYP, M_CCSDT_FC, M_WB97MV, M_WB97XV)      # methods whose failure can affect a system (checks I1, I6)

COLUMNS = ["system", "point_id", "coordinate", "geometry_level", "method", "basis",
           "energy_hartree", "converged", "t1", "imaginary_modes", "seconds", "hardware", "versions", "extra"]
# "extra" is a small JSON text for things that do not deserve their own column
# (zero-point energy, RHF stability, error messages, frozen-core flag ...).

PHASE_CSV = {  # file names for the phases; GPU names follow the plan
    "phase00": "phase00_timing.json", "phase01": "phase01_stationary.csv", "phase02": "phase02_scans.csv",
    "phase03": "phase03_ccsdt.csv", "phase04": "phase04_scout_ref.csv", "phase05": "phase05_hcn_grid.csv",
    "phase06": "phase06_ccsdt_basis.csv", "phase07": "phase07_optional.csv", "phase08": "phase08_mp2_basis.csv",
    "phase10": "phase10_timing.json", "phase11": "phase11_dft.csv", "phase12": "phase12_basis.csv",
    "phase13": "phase13_scout.csv", "phase14": "phase14_skala_relaxed.csv"}

# ------------------------------------------------------------------ root folder
ROOT = None


def in_colab():
    return "google.colab" in sys.modules


def setup_root(root=None, mount=True):
    """Pick the working folder and create it.  Order: argument, env B1V5_ROOT, Google Drive, local folder."""
    global ROOT
    root = root or os.environ.get("B1V5_ROOT")
    if not root and mount and in_colab():
        from google.colab import drive
        for attempt in range(3):                      # the mount sometimes fails on the first try (popup not answered in time, credentials not yet propagated)
            try:
                drive.mount("/content/drive", force_remount=(attempt > 0))
                if os.path.isdir("/content/drive/MyDrive"):
                    root = "/content/drive/MyDrive/skala-b1-v5"
                    break
            except Exception as ex:
                print(f"Google Drive mount, attempt {attempt + 1} of 3 failed ({repr(ex)[:80]}); " + ("trying again in 20 s: please click Allow in the Drive popup." if attempt < 2 else ""))
            if attempt < 2:
                time.sleep(20)
        if root is None:   # no Drive: carry on locally and say so
            print("*" * 90); print("COULD NOT MOUNT GOOGLE DRIVE -> using a local folder (/content/skala-b1-v5-local). Nothing is saved to Drive.")
            print("The last cell will then NOT release the runtime and will offer a zip to download. Better: stop, Runtime > Restart, Run all again and click Allow."); print("*" * 90)
    root = root or os.path.join(os.getcwd(), "skala-b1-v5-local")
    os.makedirs(os.path.join(root, "geoms"), exist_ok=True)
    ROOT = os.path.abspath(root)
    print("Working folder:", ROOT)
    return ROOT


def path(*parts):
    if ROOT is None:
        setup_root(mount=False)
    return os.path.join(ROOT, *parts)


# ------------------------------------------------------------------ hardware and versions
def hardware_info():
    """One short text line: CPU model, cores, memory, GPU (if any)."""
    cpu = platform.processor() or platform.machine()
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    mem = ""
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                mem = f", {int(line.split()[1]) / 1048576:.0f} GB RAM"; break
    except Exception:
        pass
    gpu = ""
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, timeout=10).stdout.strip()
        gpu = f", GPU {out.splitlines()[0]}" if out else ""
    except Exception:
        pass
    return f"{cpu}, {os.cpu_count()} cores{mem}{gpu}"


def package_versions(names=("pyscf", "geometric", "tblite", "ase", "numpy", "scipy", "pandas", "skala", "skala-cuda12x", "gpu4pyscf",
                            "gpu4pyscf-cuda12x", "cupy-cuda12x", "torch", "huggingface_hub", "pyscf-dispersion")):
    out = []
    for n in names:
        try:
            out.append(f"{n}={importlib.metadata.version(n)}")
        except Exception:
            pass
    return ";".join(out) + f";python={platform.python_version()}"


_HW, _VER = [None], [None]


def hw():
    if _HW[0] is None:
        _HW[0] = hardware_info()
    return _HW[0]


def ver():
    if _VER[0] is None:
        _VER[0] = package_versions()
    return _VER[0]


class Timer:
    """with Timer() as t: ...  then t.seconds."""
    def __enter__(self):
        self.t0 = time.time(); return self

    def __exit__(self, *a):
        self.seconds = time.time() - self.t0


def now():
    return datetime.datetime.now().isoformat(timespec="seconds")


# ------------------------------------------------------------------ results tables (CSV, long format)
def make_row(system, point_id, level, method, basis, energy, converged=True, coordinate="", t1="", imag="",
             seconds=0.0, extra=None):
    return dict(system=system, point_id=point_id, coordinate=coordinate, geometry_level=level, method=method, basis=basis,
                energy_hartree=("" if energy is None else repr(float(energy))), converged=int(bool(converged)), t1=t1,
                imaginary_modes=imag, seconds=f"{seconds:.2f}", hardware=hw(), versions=ver(),
                extra=json.dumps(extra or {}, sort_keys=True))


def row_key(r):
    return (str(r["system"]), str(r["point_id"]), str(r["geometry_level"]), str(r["method"]), str(r["basis"]))


def read_rows(csv_name):
    """All rows of a CSV as a list of dicts ([] when the file does not exist)."""
    p = path(csv_name)
    if not os.path.exists(p):
        return []
    with open(p, newline="") as fh:
        return list(csv.DictReader(fh))


def append_row(csv_name, row):
    """Add one row safely: write a temporary file, then rename it over the old one."""
    p = path(csv_name)
    old = open(p).read() if os.path.exists(p) else ",".join(COLUMNS) + "\n"
    if old and not old.endswith("\n"):
        old += "\n"
    buf = io.StringIO()
    csv.DictWriter(buf, fieldnames=COLUMNS, extrasaction="ignore", lineterminator="\n").writerow(row)
    tmp = p + ".tmp"
    with open(tmp, "w") as fh:
        fh.write(old + buf.getvalue()); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, p)


def have(csv_name, system, point_id, level, method, basis, retry_failed=False):
    """True when this exact calculation is already in the CSV (so we skip it)."""
    key = (system, str(point_id), level, method, basis)
    for r in read_rows(csv_name):
        if row_key(r) == key and not (retry_failed and r["converged"] in ("0", "False")):
            return True
    return False


def run_point(csv_name, system, point_id, level, method, basis, fn, coordinate="", retry_failed=False, quiet=False):
    """Do one calculation unless it is already saved.

    fn() must return a dict with 'energy' and optionally 'converged', 't1', 'imag', 'extra'.
    A crash inside fn is stored as a failed row (converged=0) and the run goes on.
    Returns the new row, or None when skipped.
    """
    if have(csv_name, system, point_id, level, method, basis, retry_failed):
        if not quiet:
            print(f"  skip (already done): {system} {point_id} {method} {basis}")
        return None
    check_time()                      # stops the phase cleanly if a time limit has passed
    t0 = time.time()
    try:
        out = fn()
        row = make_row(system, point_id, level, method, basis, out.get("energy"), out.get("converged", True), coordinate,
                       out.get("t1", ""), out.get("imag", ""), time.time() - t0, out.get("extra"))
    except PhaseTimeout:
        raise
    except Exception as ex:
        row = make_row(system, point_id, level, method, basis, float("nan"), False, coordinate, "", "", time.time() - t0,
                       {"error": repr(ex)[:300]})
    append_row(csv_name, row)
    if not quiet:
        e = row["energy_hartree"]
        print(f"  {system} {point_id} {method} {basis}: E={float(e) if e not in ('', 'nan') else float('nan'):.8f} "
              f"conv={row['converged']} {row['seconds']} s")
    return row


def extra_of(row):
    try:
        return json.loads(row.get("extra") or "{}")
    except Exception:
        return {}


# ------------------------------------------------------------------ DONE markers and a small phase ledger
def is_done(phase):
    return os.path.exists(path(f"DONE_{phase}"))


def mark_done(phase, note=""):
    with open(path(f"DONE_{phase}"), "w") as fh:
        fh.write(f"{now()} {note}\n")


def log_phase(phase, seconds, note=""):
    """Keep a running list of what ran and how long it took (phase_log.json)."""
    p = path("phase_log.json")
    log = json.load(open(p)) if os.path.exists(p) else {}
    rec = log.get(phase, {"seconds": 0.0, "runs": 0})
    rec["seconds"] += seconds; rec["runs"] += 1; rec["last"] = now(); rec["note"] = note
    log[phase] = rec
    json.dump(log, open(p + ".tmp", "w"), indent=1); os.replace(p + ".tmp", p)


def write_json(name, obj):
    p = path(name)
    json.dump(obj, open(p + ".tmp", "w"), indent=1, default=float); os.replace(p + ".tmp", p)


def read_json(name, default=None):
    p = path(name)
    return json.load(open(p)) if os.path.exists(p) else default


# ------------------------------------------------------------------ guards: errors never stop the notebook, clocks stop phases cleanly
import traceback, contextlib

MAX_TOTAL_HOURS = 3.2        # whole-notebook budget (the notebook sets its own value)
DEFAULT_PHASE_MIN = 85.0     # per-phase wall-time limit in minutes (plan: no phase over 85 min)
_CLOCK = {"t0": time.time(), "phase": None, "deadline": None, "total_deadline": None}


class PhaseTimeout(Exception):
    """Raised at the next calculation after a time limit has passed; the phase then stops cleanly."""


def start_clock(max_total_hours=None):
    """Call once at the top of the notebook."""
    global MAX_TOTAL_HOURS
    if max_total_hours is not None:
        MAX_TOTAL_HOURS = max_total_hours
    _CLOCK["t0"] = time.time()
    _CLOCK["total_deadline"] = _CLOCK["t0"] + 3600.0 * MAX_TOTAL_HOURS


def elapsed_hours():
    return (time.time() - _CLOCK["t0"]) / 3600.0


def check_time():
    """Called before every single calculation (run_point does it for you)."""
    now_ = time.time()
    if _CLOCK["deadline"] and now_ > _CLOCK["deadline"]:
        raise PhaseTimeout(f"phase time limit reached in {_CLOCK['phase']}")
    if _CLOCK["total_deadline"] and now_ > _CLOCK["total_deadline"]:
        raise PhaseTimeout(f"total time budget of {MAX_TOTAL_HOURS} h reached")


ERRORS_CSV = "phase_errors.csv"


def log_error(phase, system, exc, note=""):
    """Append one line (phase, system, short traceback) to phase_errors.csv. Never raises."""
    try:
        tb = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        where = traceback.extract_tb(exc.__traceback__)[-1:] if exc.__traceback__ else []
        where = "; ".join(f"{os.path.basename(f.filename)}:{f.lineno} {f.name}" for f in where)
        p = path(ERRORS_CSV)
        new = not os.path.exists(p)
        with open(p, "a", newline="") as fh:
            w = csv.writer(fh)
            if new:
                w.writerow(["time", "phase", "system", "error", "where", "note"])
            w.writerow([now(), phase, system, tb[:300], where[:200], note])
        print(f"  !! {phase} {system}: {tb[:200]}  (logged in {ERRORS_CSV})")
    except Exception:
        print("  !! could not log error", repr(exc)[:200])


@contextlib.contextmanager
def item_guard(phase, system=""):
    """Wrap ONE item (a system, a species, a point) of a loop: a crash is logged and the loop carries on.
    A time-limit stop (PhaseTimeout) is passed up so the whole phase ends cleanly."""
    try:
        yield
    except PhaseTimeout:
        raise
    except Exception as ex:
        log_error(phase, system, ex)


def _set_status(phase, **kw):
    st = read_json("phase_status.json", {})
    st.setdefault(phase, {}).update(kw)
    write_json("phase_status.json", st)


@contextlib.contextmanager
def phase(name, title="", max_min=None, force=False):
    """Wrap a whole phase.  Whatever happens inside, the notebook goes on to the next cell.
        with phase("phase01", "C1 geometries", max_min=40) as ph:
            if ph.skip: ...           # (the body is simply skipped when the phase is already DONE)
    The phase is marked DONE only if it ran to the end without error or time-out."""
    class _Ph:
        skip = False
    ph = _Ph()
    print(f"\n===== {name} {title} =====  (notebook time so far {elapsed_hours():.2f} h)")
    if is_done(name) and not force:
        print(f"  {name} already DONE (marker found) -> skipped. Delete DONE_{name} to run it again.")
        _set_status(name, status="already done", title=title)
        ph.skip = True
        yield ph
        return
    t0 = time.time()
    _CLOCK["phase"] = name
    _CLOCK["deadline"] = t0 + 60.0 * (max_min or DEFAULT_PHASE_MIN)
    status, note = "done", ""
    try:
        check_time()
    except PhaseTimeout as ex:          # the total budget is already used up: do not start this phase
        ph.skip = True; status, note = "stopped (time limit)", str(ex)
        print("  ** not started:", ex)
    try:
        yield ph
    except PhaseTimeout as ex:
        status, note = "stopped (time limit)", str(ex)
        print("  ** phase stopped cleanly:", ex, "- rerun the notebook to continue, finished points are kept.")
    except Exception as ex:
        status, note = "FAILED", repr(ex)[:200]
        log_error(name, "(whole phase)", ex)
    finally:
        _CLOCK["deadline"] = None
        secs = time.time() - t0
        n_err = 0
        try:
            n_err = sum(1 for r in csv.DictReader(open(path(ERRORS_CSV))) if r["phase"] == name) if os.path.exists(path(ERRORS_CSV)) else 0
        except Exception:
            pass
        if status == "done" and n_err:
            status = "done with errors"
        if status == "done":
            mark_done(name, title)
        log_phase(name, secs, status)
        _set_status(name, status=status, title=title, seconds=round(secs, 1), errors=n_err, note=note, finished=now())
        print(f"  {name}: {status}, {secs / 60:.1f} min")


def skip_phase(name, title, reason):
    """Record 'skipped: missing input ...' and carry on."""
    print(f"  {name} skipped: {reason}")
    _set_status(name, status="skipped", title=title, note=reason, finished=now())
    log_error(name, "", RuntimeError("skipped: " + reason), note="skipped")


def missing_inputs(*names):
    return [n for n in names if not os.path.exists(path(n))]


def write_run_summary(extra_lines=()):
    """RUN_SUMMARY.md: what ran, what failed, what was skipped, total hours."""
    st = read_json("phase_status.json", {})
    L = ["# Run summary", "", f"Written: {now()}", f"Hardware: {hw()}", f"Versions: {ver()}",
         f"Notebook time: {elapsed_hours():.2f} h (budget {MAX_TOTAL_HOURS} h)", "", "| phase | title | status | minutes | errors | note |", "|---|---|---|---|---|---|"]
    for k in sorted(st):
        v = st[k]
        L.append(f"| {k} | {v.get('title', '')} | {v.get('status', '')} | {v.get('seconds', 0) / 60:.1f} | {v.get('errors', '')} | {str(v.get('note', ''))[:80]} |")
    ran = [k for k in st if st[k].get("status", "").startswith(("done", "already"))]
    bad = [k for k in st if st[k].get("status", "") in ("FAILED",) or "error" in st[k].get("status", "") or "stopped" in st[k].get("status", "")]
    skp = [k for k in st if st[k].get("status") == "skipped"]
    L += ["", f"Ran: {', '.join(sorted(ran)) or 'nothing'}", f"Failed or stopped early: {', '.join(sorted(bad)) or 'none'}",
          f"Skipped: {', '.join(sorted(skp)) or 'none'}", ""]
    L += list(extra_lines)
    if os.path.exists(path(ERRORS_CSV)):
        L += ["", "## Errors (phase_errors.csv)", ""]
        for r in list(csv.DictReader(open(path(ERRORS_CSV))))[:60]:
            L.append(f"- {r['phase']} / {r['system']}: {r['error'][:140]}")
    open(path("RUN_SUMMARY.md"), "w").write("\n".join(L) + "\n")
    print("\n".join(L[:30]))
    return "\n".join(L)


# ------------------------------------------------------------------ geometries
# An "atoms" list is [(symbol, (x, y, z)), ...] in Angstrom (same format as fpes.py).
def write_xyz(p, atoms, comment=""):
    lines = [str(len(atoms)), comment] + [f"{s:2s} {x: .8f} {y: .8f} {z: .8f}" for s, (x, y, z) in atoms]
    open(p, "w").write("\n".join(lines) + "\n")


def read_xyz(p):
    L = open(p).read().strip().splitlines()
    n = int(L[0].split()[0])
    return [(l.split()[0], tuple(float(v) for v in l.split()[1:4])) for l in L[2:2 + n]]


def save_geom(name, atoms, charge=0, **meta):
    """Write geoms/<name>.json and geoms/<name>.xyz (atomic)."""
    d = dict(name=name, charge=int(charge), atoms=[[s, list(map(float, c))] for s, c in atoms], saved=now(), **meta)
    pj = path("geoms", name + ".json")
    json.dump(d, open(pj + ".tmp", "w"), indent=1, default=float); os.replace(pj + ".tmp", pj)
    write_xyz(path("geoms", name + ".xyz"), atoms, f"{name} charge={charge} {json.dumps(meta, default=float)[:120]}")


def load_geom(name):
    """Return dict(atoms=[(sym,(x,y,z))...], charge=..., ...) or None."""
    pj = path("geoms", name + ".json")
    if not os.path.exists(pj):
        return None
    d = json.load(open(pj))
    d["atoms"] = [(s, tuple(c)) for s, c in d["atoms"]]
    return d


def geom_name(system, species, level):
    return f"{system}_{species}_{level}".replace("-", "m").replace("+", "p")


def best_geom(system, species):
    """The preferred stored geometry of a stationary species (our own B3LYP first, then the scan-peak TS, then the database).  Returns (level, geom) or (None, None)."""
    for lv in LEVEL_PREFERENCE:
        g = load_geom(geom_name(system, species, lv))
        if g:
            return lv, g
    return None, None


def geometry_of(system, species, level):
    return load_geom(geom_name(system, species, level))


# ------------------------------------------------------------------ geometry builders (from fpes.py; same formulas as v4)
M_C, M_N = 12.0, 14.003074
R_CN_DEFAULT = 1.16


def hcn_geometry(theta_deg, r, R_CN=R_CN_DEFAULT):
    """HCN/HNC rigid grid of v4: C and N on the z axis, H at distance r from the C-N centre of mass.
    theta = 0: H on the C side (HCN); theta = 180: H on the N side (HNC).  (Copy of fpes.hcn_geometry.)"""
    zc = -R_CN * M_N / (M_C + M_N)
    zn = R_CN * M_C / (M_C + M_N)
    t = math.radians(theta_deg)
    return [("C", (0.0, 0.0, zc)), ("N", (0.0, 0.0, zn)), ("H", (r * math.sin(t), 0.0, -r * math.cos(t)))]


def hcn_grid_axes(n_theta=25, n_r=17):
    return np.linspace(0.0, 180.0, n_theta), np.linspace(1.05, 2.30, n_r)


def nh3_geometry(umbrella_deg, r_NH=1.012):
    """NH3 with the three H below the plane by `umbrella_deg` (0 = planar TS, about 22 = minimum).  (Copy of fpes.nh3_geometry.)"""
    t = math.radians(umbrella_deg)
    rho, z = r_NH * math.cos(t), -r_NH * math.sin(t)
    atoms = [("N", (0.0, 0.0, 0.0))]
    for k in range(3):
        p = 2 * math.pi * k / 3
        atoms.append(("H", (rho * math.cos(p), rho * math.sin(p), z)))
    return atoms


def ch4_geometry(stretch, r_CH=1.087):
    """CH4 with one C-H bond stretched; the other three at tetrahedral directions.  (Copy of fpes.ch4_geometry.)"""
    v = 1 / math.sqrt(3)
    dirs = [(v, v, v), (-v, -v, v), (-v, v, -v), (v, -v, -v)]
    atoms = [("C", (0.0, 0.0, 0.0))]
    for k, d in enumerate(dirs):
        L = stretch if k == 0 else r_CH
        atoms.append(("H", (d[0] * L, d[1] * L, d[2] * L)))
    return atoms


CH4_STRETCH = np.linspace(0.90, 1.70, 9)   # the 9 stored v4 points


def _unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v)


def _ch3(c, axis, r=1.09, ang=109.5, phase=0.0):
    """Three H atoms of a CH3 group on carbon at c.  `axis` points from C to the substituent;
    the H atoms sit at angle `ang` from that axis (109.5 = normal tetrahedral CH3X)."""
    c, a = np.asarray(c, float), _unit(axis)
    t = np.array([1.0, 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1.0, 0])
    u = _unit(t - a * np.dot(t, a)); w = np.cross(a, u)
    out = []
    for k in range(3):
        p = phase + 2 * math.pi * k / 3
        d = math.cos(math.radians(ang)) * a + math.sin(math.radians(ang)) * (math.cos(p) * u + math.sin(p) * w)
        out.append(("H", tuple(c + r * d)))
    return out


def _pt(s, x, y, z):
    return (s, (float(x), float(y), float(z)))


# ------------------------------------------------------------------ the system database
# tier A: CCSD(T) on whole path; B: CCSD(T) at stationary points only; C: literature + ladder only.
# zpe: how the literature value treats zero-point energy ("excluded", "included", "unknown").
# reactants / products: the separated species that define the reaction energy and "from reactants" barriers.
# rc / pc: reactant / product complex species (SN2 only).  For symmetric SN2 reactions pc is the same species as rc.
SYSTEMS = {
    "hcn": dict(n=1, name="HCN to HNC", cls="H shift", tier="A", sn2=False, db=True, zpe="excluded",
                reactants=["HCN"], products=["HNC"], rc=None, pc=None, ts="TS"),
    "nh3": dict(n=2, name="NH3 inversion", cls="inversion", tier="A", sn2=False, db=False, zpe="excluded",
                reactants=["NH3"], products=["NH3"], rc=None, pc=None, ts="TS"),
    "h2co": dict(n=3, name="H2CO to H2 + CO", cls="elimination", tier="A", sn2=False, db=False, zpe="included",
                 reactants=["H2CO"], products=["H2", "CO"], rc=None, pc=None, ts="TS"),
    "ch3nc": dict(n=4, name="CH3NC to CH3CN", cls="group shift", tier="B", sn2=False, db=False, zpe="unknown",
                  reactants=["CH3NC"], products=["CH3CN"], rc=None, pc=None, ts="TS"),
    "sn2_ff": dict(n=5, name="F- + CH3F", cls="SN2", tier="B", sn2=True, db=True, zpe="excluded",
                   reactants=["F-", "CH3F"], products=["F-", "CH3F"], rc="RC", pc="RC", ts="TS"),
    "sn2_clcl": dict(n=6, name="Cl- + CH3Cl", cls="SN2", tier="C", sn2=True, db=True, zpe="excluded",
                     reactants=["Cl-", "CH3Cl"], products=["Cl-", "CH3Cl"], rc="RC", pc="RC", ts="TS"),
    "sn2_fcl": dict(n=7, name="F- + CH3Cl", cls="SN2", tier="C", sn2=True, db=True, zpe="excluded",
                    reactants=["F-", "CH3Cl"], products=["CH3F", "Cl-"], rc="RC", pc="PC", ts="TS"),
    "sn2_ohf": dict(n=8, name="OH- + CH3F", cls="SN2", tier="C", sn2=True, db=True, zpe="excluded",
                    reactants=["OH-", "CH3F"], products=["CH3OH", "F-"], rc="RC", pc="PC", ts="TS"),
    "ch4": dict(n=9, name="CH4 C-H stretch", cls="bond stretch", tier="A", sn2=False, db=False, zpe="n/a",
                reactants=["CH4"], products=[], rc=None, pc=None, ts=None),
}
REACTIONS = [k for k in SYSTEMS if k != "ch4"]          # the eight reactions
SYSTEM_ORDER = list(SYSTEMS)
TIER_B = [k for k, v in SYSTEMS.items() if v["tier"] == "B"]

# Which species (stationary points and fragments) each system needs, and what kind they are:
# 'min' = minimum, 'ts' = transition state.
SPECIES_KIND = {
    "hcn": {"HCN": "min", "HNC": "min", "TS": "ts"},
    "nh3": {"NH3": "min", "TS": "ts"},
    "h2co": {"H2CO": "min", "TS": "ts", "H2": "min", "CO": "min"},
    "ch3nc": {"CH3NC": "min", "CH3CN": "min", "TS": "ts"},
    "sn2_ff": {"F-": "min", "CH3F": "min", "RC": "min", "TS": "ts"},
    "sn2_clcl": {"Cl-": "min", "CH3Cl": "min", "RC": "min", "TS": "ts"},
    "sn2_fcl": {"F-": "min", "CH3Cl": "min", "RC": "min", "TS": "ts", "PC": "min", "CH3F": "min", "Cl-": "min"},
    "sn2_ohf": {"OH-": "min", "CH3F": "min", "RC": "min", "TS": "ts", "PC": "min", "CH3OH": "min", "F-": "min"},
    "ch4": {"CH4": "min"},
}
CHARGE = {"F-": -1, "Cl-": -1, "OH-": -1}


def species_charge(system, species):
    """Total charge: -1 for the anion, its complexes and the TS of every SN2 system, else 0."""
    if SYSTEMS[system]["sn2"]:
        return 0 if species in ("CH3F", "CH3Cl", "CH3OH") else -1
    return 0


def basis_for(system, size="main"):
    """Basis set name. size: 'small' | 'main' | 'large'. SN2 systems get the diffuse (D) version.
    BASIS_OVERRIDE can replace the sizes (used only for the quick sandbox test)."""
    base = {"small": "def2-SVP", "main": "def2-TZVP", "large": "def2-QZVPP"}[size]
    if BASIS_OVERRIDE.get(size):
        return BASIS_OVERRIDE[size]
    return base + "D" if SYSTEMS[system]["sn2"] else base


BASIS_OVERRIDE = {}


def _build_start_geometries():
    """Start geometries (Angstrom). Minima are standard experimental-like values; transition-state guesses were refined with GFN2-xTB
    so that geomeTRIC starts close.  Nothing here is a result: C1 re-optimises everything it uses."""
    G = {}
    # --- system 1: HCN -> HNC (linear minima, triangular TS)
    G[("hcn", "HCN")] = [_pt("C", 0, 0, 0), _pt("N", 0, 0, 1.156), _pt("H", 0, 0, -1.065)]
    G[("hcn", "HNC")] = [_pt("C", 0, 0, 0), _pt("N", 0, 0, 1.169), _pt("H", 0, 0, 1.169 + 0.995)]
    th = math.radians(72.0)
    G[("hcn", "TS")] = [_pt("C", 0, 0, 0), _pt("N", 0, 0, 1.18), _pt("H", 1.19 * math.sin(th), 0, 1.19 * math.cos(th))]
    # --- system 2: NH3 inversion
    G[("nh3", "NH3")] = nh3_geometry(22.0)
    G[("nh3", "TS")] = nh3_geometry(0.0, 0.995)
    # --- system 3: H2CO -> H2 + CO
    G[("h2co", "H2CO")] = [_pt("C", 0, 0, 0), _pt("O", 0, 0, 1.205), _pt("H", 0.9435, 0, -0.585), _pt("H", -0.9435, 0, -0.585)]
    # TS found by DF-B3LYP/def2-SVP from three different guesses (all converged to this structure): C-H 1.10, C...H 1.68, H...H 1.31
    G[("h2co", "TS")] = [_pt("C", -0.132, 0, -0.014), _pt("O", -0.228, 0, 1.146), _pt("H", 1.466, 0, -0.542), _pt("H", 0.258, 0, -1.048)]
    G[("h2co", "H2")] = [_pt("H", 0, 0, -0.3705), _pt("H", 0, 0, 0.3705)]
    G[("h2co", "CO")] = [_pt("C", 0, 0, 0), _pt("O", 0, 0, 1.128)]
    # --- system 4: CH3NC -> CH3CN
    G[("ch3nc", "CH3NC")] = [_pt("C", 0, 0, 0), _pt("N", 0, 0, 1.424), _pt("C", 0, 0, 2.594)] + _ch3((0, 0, 0), (0, 0, 1), 1.09, 109.5)
    G[("ch3nc", "CH3CN")] = [_pt("C", 0, 0, 0), _pt("C", 0, 0, 1.46), _pt("N", 0, 0, 2.62)] + _ch3((0, 0, 0), (0, 0, 1), 1.09, 109.5)
    mid = np.array([0, 0, 0.595])
    cme = np.array([1.80, 0, 0.595])
    G[("ch3nc", "TS")] = [_pt("C", 0, 0, 0), _pt("N", 0, 0, 1.19), ("C", tuple(cme))] + _ch3(cme, mid - cme, 1.09, 100.0)
    # --- systems 5-8: SN2 (X- + CH3Y).  Backside geometry: X ... C - Y on the z axis.
    def ch3x(sym, d, r=1.09, sgn=+1, c0=0.0):
        """CH3-sym with carbon at z=c0 and sym on +/- z at distance d."""
        c = np.array([0, 0, c0]); ax = np.array([0, 0, sgn])
        return [_pt("C", *c), _pt(sym, 0, 0, c0 + sgn * d)] + _ch3(c, ax, r, 108.5)
    G[("sn2_ff", "F-")] = [_pt("F", 0, 0, 0)]
    G[("sn2_ff", "CH3F")] = ch3x("F", 1.38)
    G[("sn2_ff", "RC")] = ch3x("F", 1.43, sgn=-1)[:2] + [_pt("F", 0, 0, 2.55)] + ch3x("F", 1.43, sgn=-1)[2:]
    G[("sn2_ff", "TS")] = [_pt("C", 0, 0, 0), _pt("F", 0, 0, 1.83), _pt("F", 0, 0, -1.83)] + [_pt("H", 1.07 * math.cos(a), 1.07 * math.sin(a), 0) for a in (0, 2.0944, 4.1888)]
    G[("sn2_clcl", "Cl-")] = [_pt("Cl", 0, 0, 0)]
    G[("sn2_clcl", "CH3Cl")] = ch3x("Cl", 1.785)
    G[("sn2_clcl", "RC")] = ch3x("Cl", 1.82, sgn=-1)[:2] + [_pt("Cl", 0, 0, 3.15)] + ch3x("Cl", 1.82, sgn=-1)[2:]
    G[("sn2_clcl", "TS")] = [_pt("C", 0, 0, 0), _pt("Cl", 0, 0, 2.33), _pt("Cl", 0, 0, -2.33)] + [_pt("H", 1.07 * math.cos(a), 1.07 * math.sin(a), 0) for a in (0, 2.0944, 4.1888)]
    # F- + CH3Cl -> CH3F + Cl-
    G[("sn2_fcl", "F-")] = [_pt("F", 0, 0, 0)]
    G[("sn2_fcl", "CH3Cl")] = ch3x("Cl", 1.785)
    G[("sn2_fcl", "RC")] = ch3x("Cl", 1.85, sgn=-1)[:2] + [_pt("F", 0, 0, 2.6)] + ch3x("Cl", 1.85, sgn=-1)[2:]
    G[("sn2_fcl", "TS")] = [_pt("C", 0, 0, 0), _pt("F", 0, 0, 1.85), _pt("Cl", 0, 0, -2.20)] + [_pt("H", 1.07 * math.cos(a), 1.07 * math.sin(a), 0.03) for a in (0, 2.0944, 4.1888)]
    G[("sn2_fcl", "PC")] = ch3x("F", 1.43, sgn=-1)[:2] + [_pt("Cl", 0, 0, 3.15)] + ch3x("F", 1.43, sgn=-1)[2:]
    G[("sn2_fcl", "CH3F")] = ch3x("F", 1.38)
    G[("sn2_fcl", "Cl-")] = [_pt("Cl", 0, 0, 0)]
    # OH- + CH3F -> CH3OH + F-
    G[("sn2_ohf", "OH-")] = [_pt("O", 0, 0, 0), _pt("H", 0, 0, 0.96)]
    G[("sn2_ohf", "CH3F")] = ch3x("F", 1.38)
    # reactant complex: hydrogen-bonded HO- ... H-CH2F (O near one methyl H)
    cf = ch3x("F", 1.38)
    hb = np.array(cf[2][1]); u = _unit(hb - np.array(cf[0][1]))
    o = hb + 1.9 * u
    G[("sn2_ohf", "RC")] = cf + [_pt("O", *o), _pt("H", o[0] + 0.96 * u[0], o[1] + 0.96 * u[1], o[2] + 0.96 * u[2])]
    # TS: HO...C...F nearly collinear, O-H bent away
    G[("sn2_ohf", "TS")] = [_pt("C", 0, 0, 0), _pt("F", 0, 0, -1.90), _pt("O", 0, 0, 1.95), _pt("H", 0.90, 0, 2.30)] + [_pt("H", 1.07 * math.cos(a), 1.07 * math.sin(a), 0.03) for a in (0, 2.0944, 4.1888)]
    G[("sn2_ohf", "CH3OH")] = [_pt("C", 0, 0, 0), _pt("O", 0, 0, 1.42), _pt("H", 0.89, 0, 1.73)] + _ch3((0, 0, 0), (0, 0, 1), 1.09, 109.0)
    # product complex: CH3OH ... F-, F- bound to the O-H hydrogen
    G[("sn2_ohf", "PC")] = G[("sn2_ohf", "CH3OH")] + [_pt("F", 2.45, 0, 1.95)]
    G[("sn2_ohf", "F-")] = [_pt("F", 0, 0, 0)]
    # --- CH4
    G[("ch4", "CH4")] = ch4_geometry(1.087)
    return G


START_GEOMS = _build_start_geometries()


def system_species(system):
    return list(SPECIES_KIND[system])


def start_atoms(system, species):
    return START_GEOMS[(system, species)]


def n_electrons(atoms, charge=0):
    Z = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "Cl": 17}
    return sum(Z[s] for s, _ in atoms) - charge


def preview_systems():
    for k, v in SYSTEMS.items():
        print(f"{v['n']:>2} {k:9s} {v['name']:18s} class={v['cls']:12s} tier={v['tier']}  species={system_species(k)}")


# mount Drive (the one click) or fall back to a local folder, make the folders
setup_root()

**Cell 2: calculation helpers** (CPU helpers, stored v4 numbers, geometry-pack tools, GPU helpers). Only defines functions and constants.

In [ ]:
# ===== b1v5_cpu.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_cpu.py  --  calculation helpers for the CPU notebook (PySCF, geomeTRIC, xTB).
# Pasted into the second code cell of the CPU notebook; also importable.
# Needs b1v5_common (already loaded in the notebook).  Plain functions, no hidden state.
#
# Levels of theory used:
#   geometry / hessian : density-fitted B3LYP-D3(BJ), def2-TZVP (def2-TZVPD for SN2 species)
#   reference energy   : CCSD(T), frozen core, RHF reference, def2-TZVP(D)
#   scout geometry     : GFN2-xTB via tblite
# ==============================================================================
import os, time, tempfile, logging, math, json
import numpy as np
try:
    b1v5_common_loaded = COLUMNS  # noqa  (names exist when the notebook cell above ran)
except NameError:
    from b1v5_common import *     # noqa

Z_OF = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "Cl": 17}
MEM_MB = 9000   # PySCF memory limit; the plan uses 9 GB on a 12 GB Colab machine


def quiet_geometric():
    """geomeTRIC is very chatty; keep only warnings."""
    for name in list(logging.root.manager.loggerDict) + ["geometric", "geometric.nifty", ""]:
        if name == "" or name.startswith("geometric"):
            lg = logging.getLogger(name)
            if name:
                lg.setLevel(logging.ERROR)
    try:
        import geometric.nifty as nf
        nf.logger.setLevel(logging.ERROR)
        for h in list(nf.logger.handlers):
            h.setLevel(logging.ERROR)
    except Exception:
        pass


def atoms_str(atoms):
    return "; ".join(f"{s} {x:.8f} {y:.8f} {z:.8f}" for s, (x, y, z) in atoms)


def make_mol(atoms, charge, basis):
    from pyscf import gto
    return gto.M(atom=atoms_str(atoms), basis=basis, charge=int(charge), spin=0, unit="Angstrom", verbose=0, max_memory=MEM_MB)


def mol_atoms(mol):
    xyz = mol.atom_coords() * BOHR
    return [(mol.atom_pure_symbol(i), tuple(map(float, xyz[i]))) for i in range(mol.natm)]


# ------------------------------------------------------------------ DFT (geometries, Hessians, zero-point energies)
def make_ks(atoms, charge, basis, xc="b3lyp", disp="d3bj", df=True, grid_level=3):
    from pyscf import dft
    mol = make_mol(atoms, charge, basis)
    mf = dft.RKS(mol, xc=xc)
    if df:
        mf = mf.density_fit()
    if disp:
        mf.disp = disp
    mf.grids.level = grid_level
    mf.conv_tol = 1e-9
    mf.max_cycle = 150
    return mf


def dft_energy(atoms, charge, basis, **kw):
    mf = make_ks(atoms, charge, basis, **kw)
    e = mf.kernel()
    if not mf.converged:                       # one more try with Newton steps
        mf = mf.newton(); e = mf.kernel()
    return dict(energy=float(e), converged=bool(mf.converged))


def frequencies(mf):
    """Harmonic analysis from the analytic Hessian.
    Returns (frequencies in cm-1 with imaginary ones negative, zero-point energy in hartree, displacement vectors of the imaginary modes)."""
    from pyscf.hessian import thermo
    mol = mf.mol
    if mol.natm == 1:
        return [], 0.0, []
    h = mf.Hessian().kernel()
    r = thermo.harmonic_analysis(mol, h)
    # PySCF returns imaginary modes as complex numbers (0 + 1900j); we turn them into negative numbers
    fq = [(-abs(f.imag) if abs(np.imag(f)) > 1e-6 else float(np.real(f))) for f in np.array(r["freq_wavenumber"], dtype=complex)]
    zpe = 0.5 * sum(f for f in fq if f > 0) / HARTREE_CM1
    order = [i for i in np.argsort(fq) if fq[i] < -20.0]
    return fq, zpe, [np.array(r["norm_mode"][i]) for i in order]


def analyse_stationary(atoms, charge, basis, **kw):
    """Energy + harmonic frequencies + zero-point energy at a fixed geometry."""
    mf = make_ks(atoms, charge, basis, **kw)
    e = mf.kernel()
    ok = bool(mf.converged)
    fq, zpe, modes = frequencies(mf)
    n_imag = int(sum(1 for f in fq if f < -20.0))      # -20 cm-1 guards against numerical noise near zero
    return dict(energy=float(e), converged=ok, imag=n_imag, imag_modes=modes,
                extra=dict(zpe_hartree=zpe, lowest_freq_cm1=(min(fq) if fq else None), freqs=[round(float(f), 1) for f in fq][:12]))


def refine_stationary(atoms, charge, basis, kind, **kw):
    """Frequencies at `atoms`; if there are MORE imaginary modes than wanted (minimum 0, transition state 1), push the geometry along
    the extra mode(s) and optimise once more (rule 4.4: one retry from a fresh guess).  Returns (atoms, analysis, retried)."""
    want = 1 if kind == "ts" else 0
    a = analyse_stationary(atoms, charge, basis, **kw)
    if a["imag"] <= want or len(atoms) < 3:
        return atoms, a, False
    extra_modes = a["imag_modes"][want:]
    d = sum(extra_modes)
    d = 0.25 * d / np.abs(d).max()                     # largest atom moves 0.25 A
    pushed = [(s, tuple(np.array(c) + d[i])) for i, (s, c) in enumerate(atoms)]
    r = opt_dft(pushed, charge, basis, transition=(kind == "ts"), hessian=(kind == "ts"), **kw)
    a2 = analyse_stationary(r["atoms"], charge, basis, **kw)
    if abs(a2["imag"] - want) < abs(a["imag"] - want):
        a2["converged"] = bool(a2["converged"] and r["converged"])
        return r["atoms"], a2, True
    return atoms, a, True


def constraint_text(items):
    """items: list of ('distance'|'angle'|'dihedral', atom indices (0-based), value) -> geomeTRIC constraint file text."""
    lines = ["$set"]
    for kind, idx, val in items:
        lines.append(f"{kind} " + " ".join(str(i + 1) for i in idx) + f" {val:.6f}")
    return "\n".join(lines) + "\n"


def opt_dft(atoms, charge, basis, constraints=None, transition=False, hessian=False, maxsteps=120, **kw):
    """Geometry optimisation (minimum or transition state) with geomeTRIC.  `constraints` as for constraint_text.
    hessian=True gives geomeTRIC the analytic Hessian at the first step (needed for TS searches)."""
    from pyscf.geomopt import geometric_solver
    quiet_geometric()
    mf = make_ks(atoms, charge, basis, **kw)
    opts = dict(convergence_set="GAU", maxsteps=maxsteps)
    if transition:
        opts["transition"] = True
    if hessian:
        opts["hessian"] = True
    with tempfile.TemporaryDirectory() as d:
        if constraints:
            cp = os.path.join(d, "cons.txt"); open(cp, "w").write(constraint_text(constraints)); opts["constraints"] = cp
        conv, mol = geometric_solver.kernel(mf, assert_convergence=False, **opts)
    return dict(atoms=mol_atoms(mol), converged=bool(conv))


# ------------------------------------------------------------------ CCSD(T)
def ccsdt(atoms, charge, basis, frozen_core=True, stability=True):
    """CCSD(T) with RHF reference.  Returns energy, convergence, T1 diagnostic and RHF stability in 'extra'."""
    from pyscf import scf, cc
    from pyscf.data.elements import chemcore
    mol = make_mol(atoms, charge, basis)
    mf = scf.RHF(mol); mf.conv_tol = 1e-10; mf.max_cycle = 200
    mf.kernel()
    if not mf.converged:
        mf = mf.newton(); mf.kernel()
    ncore = chemcore(mol) if frozen_core else 0
    extra = dict(frozen_core=bool(frozen_core), n_frozen=int(ncore), e_scf=float(mf.e_tot), nbf=int(mol.nao))
    if stability and mol.nelectron > 2:
        try:
            _, _, st_i, st_e = mf.stability(internal=True, external=True, return_status=True)
            extra["stable"] = int(bool(st_i) and bool(st_e)); extra["stable_internal"] = int(bool(st_i)); extra["stable_external"] = int(bool(st_e))
        except Exception as ex:
            extra["stable"] = ""; extra["stability_error"] = repr(ex)[:100]
    mycc = cc.CCSD(mf); mycc.frozen = ncore; mycc.conv_tol = 1e-8; mycc.max_cycle = 100
    mycc.kernel()
    et = mycc.ccsd_t()
    try:
        t1 = float(mycc.get_t1_diagnostic())
    except Exception:                                  # older PySCF without the helper
        t1 = float(np.sqrt(np.linalg.norm(mycc.t1) ** 2 / (2 * mycc.t1.shape[0])))
    extra.update(e_ccsd_corr=float(mycc.e_corr), e_t=float(et))
    return dict(energy=float(mf.e_tot + mycc.e_corr + et), converged=bool(mf.converged and mycc.converged), t1=round(t1, 5), extra=extra)


# ------------------------------------------------------------------ DF-MP2 for the basis correction of the CCSD(T) reference (review row 5)
def dfmp2(atoms, charge, basis, frozen_core=True):
    """Total DF-MP2 energy (RHF with density fitting + DF-MP2 correlation, frozen core), used ONLY for the difference
    MP2/def2-QZVPP(D) - MP2/def2-TZVP(D) that is added to the CCSD(T) energy of the same geometry.
    The fitting error of the two bases largely cancels in that difference.  RHF: auxiliary basis def2-universal-jkfit; MP2: PySCF default RI basis."""
    from pyscf import scf, mp
    from pyscf.data.elements import chemcore
    mol = make_mol(atoms, charge, basis)
    mf = scf.RHF(mol).density_fit(auxbasis="def2-universal-jkfit"); mf.conv_tol = 1e-9; mf.max_cycle = 200
    mf.kernel()
    if not mf.converged:
        mf = scf.RHF(mol).density_fit(auxbasis="def2-universal-jkfit").newton(); mf.conv_tol = 1e-9; mf.kernel()
    ncore = chemcore(mol) if frozen_core else 0
    pt = mp.dfmp2.DFMP2(mf); pt.frozen = ncore
    pt.kernel()
    return dict(energy=float(mf.e_tot + pt.e_corr), converged=bool(mf.converged),
                extra=dict(frozen_core=bool(frozen_core), n_frozen=int(ncore), e_scf=float(mf.e_tot), e_mp2_corr=float(pt.e_corr), nbf=int(mol.nao),
                           auxbasis_jk="def2-universal-jkfit", auxbasis_mp2="pyscf default RI"))


# ------------------------------------------------------------------ B2PLYP (optional phase): hybrid + scaled DF-MP2
def b2plyp(atoms, charge, basis):
    """B2PLYP = 0.53 HF + 0.47 B88 exchange + 0.73 LYP + 0.27 * MP2 correlation (Grimme 2006); frozen core, density fitting.
    NOT yet checked against a published energy (see README)."""
    from pyscf import dft, mp
    from pyscf.data.elements import chemcore
    mol = make_mol(atoms, charge, basis)
    mf = dft.RKS(mol); mf.xc = "0.53*HF + 0.47*B88, 0.73*LYP"; mf = mf.density_fit()
    mf.grids.level = 3; mf.conv_tol = 1e-9; mf.kernel()
    pt = mp.dfmp2.DFMP2(mf); pt.frozen = chemcore(mol)
    pt.kernel()
    return dict(energy=float(mf.e_tot + 0.27 * pt.e_corr), converged=bool(mf.converged),
                extra=dict(e_hybrid=float(mf.e_tot), e_mp2_corr=float(pt.e_corr)))


# ------------------------------------------------------------------ xTB (tblite) with a geomeTRIC engine
def xtb_eg(atoms, charge, method="GFN2-xTB"):
    """Energy (hartree) and gradient (hartree/bohr, flat) for GFN2-xTB."""
    from tblite.interface import Calculator
    nums = np.array([Z_OF[s] for s, _ in atoms])
    pos = np.array([c for _, c in atoms]) / BOHR
    calc = Calculator(method, nums, pos, charge=float(charge))
    calc.set("verbosity", 0)
    res = calc.singlepoint()
    return float(res.get("energy")), np.array(res.get("gradient")).ravel()


def xtb_energy(atoms, charge):
    try:
        e, _ = xtb_eg(atoms, charge)
        return dict(energy=e, converged=True)
    except Exception:
        return dict(energy=float("nan"), converged=False)


def _xtb_engine(atoms, charge):
    import geometric.engine as ge, geometric.molecule as gm

    class XtbEngine(ge.Engine):
        def __init__(self):
            M = gm.Molecule(); M.elem = [s for s, _ in atoms]; M.xyzs = [np.array([c for _, c in atoms])]
            super().__init__(M)
            self.n = 0

        def calc_new(self, coords, dirname):
            self.n += 1
            at = [(s, tuple(c * BOHR)) for (s, _), c in zip(atoms, coords.reshape(-1, 3))]
            e, g = xtb_eg(at, charge)
            return {"energy": e, "gradient": g}
    return XtbEngine()


def opt_xtb(atoms, charge, constraints=None, transition=False, hessian=None, maxiter=300):
    """Geometry optimisation with GFN2-xTB.  A single atom is returned unchanged."""
    if len(atoms) == 1:
        return dict(atoms=atoms, converged=True, energy=xtb_eg(atoms, charge)[0])
    import geometric.optimize as go
    quiet_geometric()
    eng = _xtb_engine(atoms, charge)
    with tempfile.TemporaryDirectory() as d:
        kw = dict(customengine=eng, input=os.path.join(d, "x"), maxiter=maxiter, convergence_set="GAU", transition=transition)
        if constraints:
            cp = os.path.join(d, "cons.txt"); open(cp, "w").write(constraint_text(constraints)); kw["constraints"] = cp
        if hessian is not None:
            hp = os.path.join(d, "hess.txt"); np.savetxt(hp, hessian); kw["hessian"] = "file:" + hp
        try:
            M = go.run_optimizer(**kw); conv = True
        except Exception:                      # not converged, or xTB SCF failure: report 'not converged', never crash
            conv = False; M = None
    if M is None:
        return dict(atoms=atoms, converged=False, energy=float("nan"))
    new = [(s, tuple(map(float, c))) for s, c in zip(M.elem, M.xyzs[-1])]
    return dict(atoms=new, converged=conv, energy=xtb_eg(new, charge)[0])


# ------------------------------------------------------------------ relaxed scans
def coord_value(kind, idx, atoms):
    """Value of a distance (A), angle (deg) or dihedral (deg) in `atoms`."""
    p = [np.array(atoms[i][1]) for i in idx]
    if kind == "distance":
        return float(np.linalg.norm(p[0] - p[1]))
    if kind == "angle":
        a, b = p[0] - p[1], p[2] - p[1]
        return float(np.degrees(np.arccos(np.clip(np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b), -1, 1))))
    b0, b1, b2 = p[0] - p[1], p[2] - p[1], p[3] - p[2]
    b1 = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1) * b1; w = b2 - np.dot(b2, b1) * b1
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1, v), w), np.dot(v, w))))


# One driving coordinate per reaction (atom numbers 0-based, in the order of the species' start geometry in b1v5_common).
# 'start' = species whose geometry is point zero of the walk; values(g0) = the n driven values in path order (reactant side first).
# The walk starts at the value closest to the start geometry and goes out in both directions, each step starting from the previous one.
SCANS = {
    "hcn": dict(kind="angle", idx=(2, 0, 1), start="HCN", n=21, label="H-C-N angle (deg)",
                values=lambda g0: np.array([180.0] + list(np.linspace(171.0, 6.0, 20)))),
    "nh3": dict(kind="dihedral", idx=(0, 1, 2, 3), start="NH3", n=21, label="N-H1-H2-H3 dihedral (deg)",
                values=lambda g0: np.linspace(coord_value("dihedral", (0, 1, 2, 3), g0), -coord_value("dihedral", (0, 1, 2, 3), g0), 21)),
    "h2co": dict(kind="distance", idx=(2, 3), start="TS", n=21, label="H...H distance (A)",
                 values=lambda g0: np.linspace(1.85, 0.80, 21)),
    "ch3nc": dict(kind="angle", idx=(0, 1, 2), start="CH3NC", n=11, label="C-N-C angle (deg)",
                  values=lambda g0: np.array([180.0] + list(np.linspace(171.0, 9.0, 10)))),
    "sn2_ff": dict(kind="distance", idx=(0, 2), start="RC", n=11, label="C...F(attacking) distance (A)",
                   values=lambda g0: np.linspace(coord_value("distance", (0, 2), g0), 1.43, 11)),
    # the three systems below are scanned with xTB only (C4); DFT paths for them are an optional GPU extra
    "sn2_clcl": dict(kind="distance", idx=(0, 2), start="RC", n=11, label="C...Cl(attacking) distance (A)", dft=False,
                     values=lambda g0: np.linspace(coord_value("distance", (0, 2), g0), 1.80, 11)),
    "sn2_fcl": dict(kind="distance", idx=(0, 2), start="RC", n=11, label="C...F(attacking) distance (A)", dft=False,
                    values=lambda g0: np.linspace(coord_value("distance", (0, 2), g0), 1.40, 11)),
    "sn2_ohf": dict(kind="distance", idx=(0, 5), start="RC", n=11, label="C...O(attacking) distance (A)", dft=False,
                    values=lambda g0: np.linspace(coord_value("distance", (0, 5), g0), 1.42, 11)),
}
SCAN_STRIDE = 1      # the sandbox test sets 3: every third point only
# CH4 has no scan: the 9 stored v4 stretch points are reused (b1v5_common.CH4_STRETCH).


def _walk(system, g0, charge, relax, energy, store, save):
    """Generic relaxed walk.  relax(atoms, constraint) -> dict(atoms, converged); energy(atoms) -> dict(energy, converged)."""
    sp = SCANS[system]
    vals = np.asarray(sp["values"](g0), float)
    keep = sorted(set(range(0, len(vals), SCAN_STRIDE)) | {len(vals) - 1})
    vals = vals[keep]
    n = len(vals)
    c0 = coord_value(sp["kind"], sp["idx"], g0)
    i0 = int(np.argmin(np.abs(vals - c0)))
    pts = {int(p["index"]): p for p in store if int(p["index"]) < n}
    def one(k, prev_atoms, free=False):
        t0 = time.time()
        if free:
            r = dict(atoms=prev_atoms, converged=True)
        else:
            r = relax(prev_atoms, (sp["kind"], sp["idx"], float(vals[k])))
        e = energy(r["atoms"])
        pts[k] = dict(index=k, coordinate=float(vals[k]) if not free else c0, atoms=r["atoms"], energy=e["energy"],
                      converged=bool(r["converged"] and e["converged"]), seconds=time.time() - t0)
        save([pts[j] for j in sorted(pts)])
        print(f"   scan {system} point {k + 1}/{n}  coordinate {pts[k]['coordinate']:.3f}  E={e['energy']:.6f}  converged={pts[k]['converged']}  {time.time() - t0:.0f} s")
    if i0 not in pts:
        one(i0, g0, free=True)
    for k in range(i0 + 1, n):                # towards the end of the list
        if k not in pts:
            check_time(); one(k, pts[k - 1]["atoms"])
    for k in range(i0 - 1, -1, -1):           # towards the beginning
        if k not in pts:
            check_time(); one(k, pts[k + 1]["atoms"])
    return [pts[j] for j in sorted(pts)]


def scan_points_dft(system, g0, charge, basis, store, save, maxsteps=100):
    """Relaxed scan with DF-B3LYP-D3(BJ); `store` = finished points, `save(points)` is called after every point."""
    return _walk(system, g0, charge,
                 lambda at, c: opt_dft(at, charge, basis, constraints=[c], maxsteps=maxsteps),
                 lambda at: dft_energy(at, charge, basis), store, save)


def scan_points_xtb(system, g0, charge):
    """The same walk with GFN2-xTB (fast; no resume needed)."""
    return _walk(system, g0, charge, lambda at, c: opt_xtb(at, charge, constraints=[c]),
                 lambda at: xtb_energy(at, charge), [], lambda pts: None)


def peak_of(points):
    """Index of the highest-energy scan point, or None when the highest point is an end point (no peak found)."""
    E = np.array([p["energy"] for p in points])
    k = int(np.nanargmax(E))
    return None if k in (0, len(E) - 1) else k


def parabola_peak(points, k):
    """Three-point parabola through the highest point and its two neighbours; returns (coordinate, energy) of the vertex."""
    x = np.array([points[j]["coordinate"] for j in (k - 1, k, k + 1)]); y = np.array([points[j]["energy"] for j in (k - 1, k, k + 1)])
    a, b, c = np.polyfit(x, y, 2)
    if a >= 0:
        return float(x[1]), float(y[1])
    xv = -b / (2 * a)
    if not (x.min() <= xv <= x.max()):
        return float(x[1]), float(y[1])
    return float(xv), float(np.polyval([a, b, c], xv))


# ===== b1v5_v4_stored.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
"""Stored v4 numbers for check I9 (auto-generated by make_v4_stored.py; do not edit)."""
import numpy as np

HCN_THETA = [0.0, 7.5, 15.0, 22.5, 30.0, 37.5, 45.0, 52.5, 60.0, 67.5, 75.0, 82.5, 90.0, 97.5, 105.0, 112.5, 120.0, 127.5, 135.0, 142.5, 150.0, 157.5, 165.0, 172.5, 180.0]
HCN_R = [1.05, 1.128125, 1.20625, 1.284375, 1.3625, 1.440625, 1.51875, 1.596875, 1.675, 1.753125, 1.83125, 1.909375, 1.9875, 2.065625, 2.14375, 2.221875, 2.3]
HCN_GRID_SKALA = [[-91.778622114, -92.386440443, -92.760506165, -92.99296046, -93.136740797, -93.223664944, -93.27345773, -93.298920946, -93.308372884, -93.307284482, -93.299319406, -93.287159079, -93.272676101, -93.257159379, -93.241398465, -93.225973741, -93.211166683], [-91.898477948, -92.452455448, -92.797754229, -93.013975615, -93.148245689, -93.229418474, -93.275700608, -93.299004768, -93.307162841, -93.305322369, -93.296956591, -93.284622249, -93.270121096, -93.254658365, -93.239023674, -93.223749968, -93.209091858], [-92.18568181, -92.615274601, -92.89118761, -93.066967193, -93.177017978, -93.243417414, -93.280675709, -93.298463451, -93.303252899, -93.299399264, -93.289972217, -93.277198038, -93.262653485, -93.247397598, -93.232122798, -93.21725873, -93.20301715], [-92.506053656, -92.804524872, -93.002116388, -93.130256274, -93.210792115, -93.258756982, -93.284554605, -93.295259768, -93.295711418, -93.289337965, -93.278624408, -93.265380338, -93.250879836, -93.235992118, -93.221272627, -93.207017695, -93.193420746], [-92.772601423, -92.967403736, -93.099009057, -93.184986122, -93.238419606, -93.269061157, -93.283829725, -93.287594203, -93.283859732, -93.275158945, -93.2633821, -93.249914707, -93.235714376, -93.2214247, -93.207459107, -93.194052893, -93.1814412], [-92.961482217, -93.084781499, -93.168352661, -93.22225172, -93.254528948, -93.271289863, -93.277023859, -93.275077854, -93.267974168, -93.257576486, -93.245219893, -93.23188695, -93.218273486, -93.204807191, -93.191805643, -93.179514588, -93.168069453], [-93.082374697, -93.159040526, -93.210056204, -93.241588109, -93.258677255, -93.265255216, -93.264393591, -93.258451482, -93.249193156, -93.237895577, -93.225488143, -93.212626021, -93.199806014, -93.187356701, -93.175542694, -93.164476178, -93.154138376], [-93.153441046, -93.200260548, -93.229936831, -93.246483974, -93.253254051, -93.252977719, -93.247823007, -93.239457592, -93.229082446, -93.217582969, -93.205556026, -93.193462176, -93.18164181, -93.170366597, -93.159769151, -93.149853702, -93.140559], [-93.192303424, -93.219793498, -93.235422004, -93.242032954, -93.242059329, -93.237491001, -93.229860446, -93.220302943, -93.209621698, -93.198345308, -93.18690233, -93.175711604, -93.164951554, -93.154730923, -93.14505627, -93.136026656, -93.127560403], [-93.212419346, -93.227182883, -93.23354707, -93.233721597, -93.229550779, -93.222479818, -93.213577892, -93.203505485, -93.192706468, -93.181647842, -93.170716193, -93.160140374, -93.150098202, -93.140614769, -93.131741815, -93.123475535, -93.11585912], [-93.222804563, -93.229528973, -93.230040044, -93.226217079, -93.219518238, -93.210999594, -93.201227749, -93.190572134, -93.179536542, -93.168485617, -93.157700272, -93.147429027, -93.137800838, -93.128884561, -93.120608861, -93.113006282, -93.106043774], [-93.229065022, -93.23111419, -93.228190528, -93.221979896, -93.213788976, -93.20424434, -93.193698947, -93.182515696, -93.171172184, -93.159934912, -93.149042853, -93.138724575, -93.12910155, -93.120283831, -93.112273448, -93.105029996, -93.098493819], [-93.234042377, -93.233939751, -93.229283651, -93.221861697, -93.212783956, -93.20240085, -93.191123383, -93.179337207, -93.167472829, -93.155870004, -93.144681209, -93.13410744, -93.124301945, -93.115322297, -93.107217225, -93.099954743, -93.093495316], [-93.23838202, -93.238387874, -93.23349302, -93.225607316, -93.2159419, -93.204910947, -93.192931944, -93.180517933, -93.168082599, -93.155936751, -93.144277175, -93.133283979, -93.12309153, -93.113764163, -93.105314871, -93.097777778, -93.091086649], [-93.240935699, -93.243474393, -93.239869728, -93.232330289, -93.222535493, -93.211234152, -93.198773877, -93.185701424, -93.172539591, -93.159704946, -93.147419501, -93.135827646, -93.125059808, -93.115193129, -93.106254842, -93.098283696, -93.091251296], [-93.239220791, -93.247143019, -93.246852158, -93.240969716, -93.231616128, -93.220320449, -93.207734746, -93.194262109, -93.180457168, -93.166849014, -93.15379152, -93.141471637, -93.130009456, -93.119474284, -93.109914898, -93.101352832, -93.093788261], [-93.229455136, -93.246710526, -93.252422591, -93.250101644, -93.242454754, -93.23159184, -93.218981956, -93.205310479, -93.191032029, -93.176683001, -93.162776118, -93.149605495, -93.137320963, -93.12603446, -93.115773589, -93.106573334, -93.09844401], [-93.206098397, -93.238282921, -93.253780181, -93.257602299, -93.253484212, -93.24427757, -93.232047409, -93.218273393, -93.203670436, -93.18872283, -93.173985226, -93.159910535, -93.146753578, -93.134616511, -93.123574367, -93.113637656, -93.104828118], [-93.16170826, -93.216899909, -93.247443013, -93.260858207, -93.262645088, -93.256736543, -93.245950763, -93.232411318, -93.217587519, -93.202180578, -93.186707484, -93.171682162, -93.157545912, -93.144495566, -93.132588968, -93.121878738, -93.112341913], [-93.088100873, -93.177374105, -93.22990958, -93.257258293, -93.267700341, -93.266970811, -93.259035398, -93.246643752, -93.231862784, -93.216083739, -93.20004852, -93.184247087, -93.169191909, -93.155197244, -93.142408591, -93.130853428, -93.120517884], [-92.979362159, -93.116555968, -93.199192008, -93.245279088, -93.267293824, -93.273621872, -93.269772346, -93.259459318, -93.245339613, -93.229384526, -93.212852994, -93.196372829, -93.180452815, -93.165556445, -93.15185602, -93.139456829, -93.128361478], [-92.838202843, -93.036955919, -93.157008058, -93.22580248, -93.261556985, -93.276194482, -93.277270867, -93.26967842, -93.256754328, -93.240960082, -93.224051634, -93.20700908, -93.190392038, -93.17466047, -93.160172984, -93.147034555, -93.135256845], [-92.684454384, -92.9512498, -93.110953864, -93.203192646, -93.252800991, -93.275642773, -93.281603825, -93.276791955, -93.265237906, -93.249840954, -93.232747529, -93.21534309, -93.198275097, -93.182000722, -93.166914848, -93.153194482, -93.140897887], [-92.559048552, -92.882599568, -93.074081757, -93.184651975, -93.244889272, -93.273923774, -93.283537675, -93.280846129, -93.270382513, -93.255377635, -93.238237833, -93.22059866, -93.203275774, -93.186713987, -93.171300171, -93.157245609, -93.144615903], [-92.509809992, -92.855969625, -93.059820144, -93.177419827, -93.241687981, -93.273061993, -93.284039366, -93.282150952, -93.272114612, -93.257268473, -93.240124047, -93.222403514, -93.204995895, -93.188334728, -93.172818324, -93.158644772, -93.145919046]]
HCN_GRID_XTB = [[-3.815595521, -4.52382572, -4.947806469, -5.200550764, -5.348586871, -5.432333997, -5.476685662, -5.497028139, -5.502858809, -5.50002374, -5.492112238, -5.481324519, -5.469023195, -5.45608733, -5.44294013, -5.429928866, -5.417317754], [-3.956674201, -4.599904589, -4.989267009, -5.222834112, -5.360026479, -5.43759144, -5.478421744, -5.496777655, -5.501511552, -5.498096874, -5.489906667, -5.479015634, -5.466716632, -5.453841549, -5.440780354, -5.427882943, -5.41539858], [-4.293682587, -4.78622853, -5.092111727, -5.278254242, -5.388179339, -5.450024551, -5.481837108, -5.495088412, -5.497007739, -5.49211969, -5.483241524, -5.47212384, -5.459881683, -5.447202764, -5.434418275, -5.421869837, -5.409767441], [-4.665009304, -4.998623193, -5.211116937, -5.342093455, -5.419453122, -5.462190583, -5.482924899, -5.489884205, -5.488358865, -5.481716592, -5.472086177, -5.460816446, -5.448796577, -5.436484017, -5.424211394, -5.412261447, -5.400797573], [-4.965644026, -5.17479266, -5.309919838, -5.393380799, -5.441930684, -5.467469685, -5.478150691, -5.47942011, -5.47485332, -5.466769461, -5.456663794, -5.445525211, -5.433950966, -5.422254558, -5.410760516, -5.399664658, -5.389088234], [-5.168275135, -5.293363982, -5.374023566, -5.423023908, -5.450259426, -5.46294205, -5.466121333, -5.463238347, -5.4565957, -5.447717053, -5.437623763, -5.426980565, -5.416108933, -5.405339574, -5.394888596, -5.384888399, -5.375423766], [-5.287124095, -5.359584706, -5.405297683, -5.4318062, -5.445005714, -5.449242745, -5.447602478, -5.442220598, -5.434552194, -5.425595145, -5.416021429, -5.406159795, -5.396335511, -5.386768131, -5.377590257, -5.368887323, -5.360712779], [-5.347596246, -5.388258197, -5.412635755, -5.425337692, -5.429948441, -5.429119947, -5.424752352, -5.418180141, -5.410343963, -5.401885341, -5.393126925, -5.38438587, -5.375871261, -5.367705198, -5.359963079, -5.352691323, -5.345916455], [-5.372915549, -5.394531833, -5.406091731, -5.41049818, -5.409993336, -5.406242631, -5.40045417, -5.393507522, -5.386001748, -5.378254491, -5.370571773, -5.363139691, -5.35605874, -5.349382513, -5.343137576, -5.337335506, -5.331979073], [-5.380590037, -5.390725266, -5.39450724, -5.393829403, -5.39015451, -5.384589946, -5.377979828, -5.370904936, -5.363717797, -5.356736656, -5.350132932, -5.343976479, -5.338280948, -5.33303381, -5.328213646, -5.32379843, -5.319768588], [-5.382163479, -5.385441649, -5.384492911, -5.380622486, -5.37483189, -5.367935343, -5.360526636, -5.353052559, -5.345931213, -5.339411741, -5.333586837, -5.328433293, -5.323872222, -5.319816487, -5.316192086, -5.312942162, -5.310024762], [-5.383640153, -5.383140742, -5.37951797, -5.373777162, -5.366639404, -5.358680743, -5.350351517, -5.34218697, -5.334639808, -5.328028647, -5.322455288, -5.317825166, -5.313960397, -5.310690226, -5.307883321, -5.305447151, -5.303318295], [-5.386875099, -5.384805006, -5.379987667, -5.373348002, -5.365478045, -5.356822735, -5.347797785, -5.338901707, -5.330636471, -5.323465395, -5.317604379, -5.312949672, -5.309248326, -5.306256692, -5.303792407, -5.301731259, -5.29999075], [-5.391686762, -5.389849083, -5.384941328, -5.378033245, -5.369839555, -5.360847324, -5.351432398, -5.342061045, -5.3332216, -5.325421928, -5.318981761, -5.313865472, -5.309823085, -5.306587918, -5.303953473, -5.301776797, -5.299962073], [-5.39704653, -5.397318801, -5.393395088, -5.386830191, -5.378648652, -5.369544982, -5.359956151, -5.350338491, -5.341160944, -5.332877464, -5.325798399, -5.319960867, -5.315195445, -5.31128009, -5.308024531, -5.305287738, -5.302970189], [-5.400848719, -5.405714582, -5.404323292, -5.398996265, -5.391309995, -5.382308218, -5.372692213, -5.362927365, -5.35346963, -5.344714888, -5.336931999, -5.33020553, -5.324463869, -5.319566592, -5.315371044, -5.311758344, -5.308636172], [-5.398988695, -5.412309556, -5.415875166, -5.413241591, -5.40685124, -5.398341062, -5.388782768, -5.378879074, -5.369066509, -5.359725477, -5.351117079, -5.343368464, -5.336491455, -5.330427562, -5.325092185, -5.320400392, -5.316276617], [-5.384584507, -5.412515761, -5.424849425, -5.427208754, -5.423431868, -5.416084622, -5.406843776, -5.396776777, -5.386544419, -5.376516859, -5.366992578, -5.358149956, -5.350070083, -5.342764562, -5.336203791, -5.330339119, -5.325115759], [-5.347725065, -5.39985452, -5.42679089, -5.437620404, -5.438479574, -5.433409226, -5.425028947, -5.414997739, -5.404336029, -5.393630283, -5.383193822, -5.373275894, -5.364020039, -5.355497274, -5.347727105, -5.340695665, -5.334369832], [-5.276520676, -5.366919296, -5.416803622, -5.440974914, -5.449282568, -5.448091884, -5.44145207, -5.431908528, -5.42102027, -5.409715915, -5.398481155, -5.387596571, -5.37727681, -5.367645056, -5.358764496, -5.350653847, -5.343301452], [-5.160985989, -5.308013682, -5.39130408, -5.434731611, -5.453826122, -5.458411907, -5.454602412, -5.446171712, -5.435423342, -5.423721559, -5.411850836, -5.400179186, -5.388969629, -5.378398882, -5.368568427, -5.359526832, -5.351283083], [-5.001557351, -5.224124281, -5.350875639, -5.418951016, -5.451757917, -5.463747187, -5.463716025, -5.45698485, -5.44675934, -5.434938749, -5.422615287, -5.4103657, -5.398474111, -5.38716979, -5.376588063, -5.366803212, -5.357842924], [-4.821175372, -5.128955533, -5.303276511, -5.397789729, -5.44512999, -5.46495036, -5.468980188, -5.46418731, -5.454708981, -5.44298458, -5.430413549, -5.417773757, -5.405415351, -5.393594332, -5.382476593, -5.372156397, -5.362676507], [-4.670900974, -5.050344292, -5.263548247, -5.379266906, -5.438124834, -5.464118071, -5.471363624, -5.468145518, -5.459330341, -5.447770664, -5.43510272, -5.422241841, -5.409616945, -5.397492074, -5.386055461, -5.375414275, -5.365621088], [-4.611374162, -5.019446469, -5.247912114, -5.371858283, -5.435151644, -5.463512581, -5.471992281, -5.469384277, -5.460834637, -5.449352335, -5.436663024, -5.423731946, -5.411020841, -5.398796116, -5.38725394, -5.376506014, -5.366608313]]
HCN_GRID_PBE = [[-91.699079644, -92.308764927, -92.68400033, -92.917173695, -93.061658709, -93.149398484, -93.200132288, -93.226508022, -93.236855132, -93.236742928, -93.229926746, -93.218948469, -93.205529668, -93.190835657, -93.17565414, -93.16051533, -93.145772242], [-91.819187181, -92.374866117, -92.72119265, -92.938110061, -93.073117987, -93.155187952, -93.202485378, -93.226775994, -93.235872178, -93.235030689, -93.227813753, -93.216641839, -93.203159817, -93.188484602, -93.173372481, -93.158334098, -93.143709758], [-92.107209532, -92.538003911, -92.814655095, -92.991225239, -93.102189504, -93.169630946, -93.207943649, -93.226732767, -93.232512764, -93.22973957, -93.221471605, -93.209808335, -93.196189179, -93.181599058, -93.166710438, -93.151978846, -93.137709426], [-92.428465088, -92.72766205, -92.926036624, -93.055108353, -93.136733791, -93.185836529, -93.212667652, -93.224338854, -93.225809957, -93.22054476, -93.210958885, -93.198732586, -93.185031649, -93.170663595, -93.156184858, -93.141976204, -93.128292769], [-92.695881256, -92.891300378, -93.023790288, -93.1108081, -93.165425403, -93.197169462, -93.212936377, -93.217690767, -93.214988933, -93.207363817, -93.196609158, -93.183990416, -93.170397222, -93.156453904, -93.142595799, -93.129123006, -93.116236427], [-92.885859835, -93.009680778, -93.094135368, -93.149105443, -93.182457108, -93.200241534, -93.207014993, -93.206153985, -93.20012655, -93.190717423, -93.179208159, -93.166513975, -93.153287942, -93.139993306, -93.126956492, -93.114402668, -93.102481661], [-93.007961831, -93.085070046, -93.136880027, -93.169364537, -93.187475435, -93.195144775, -93.195404416, -93.19053939, -93.1822421, -93.171750418, -93.159962994, -93.147528755, -93.134915681, -93.122458858, -93.110394248, -93.098884006, -93.088032338], [-93.080649344, -93.127767583, -93.158081309, -93.175501995, -93.183307944, -93.184137527, -93.180046612, -93.172597694, -93.162957156, -93.151985016, -93.140310722, -93.128391657, -93.116559095, -93.105049884, -93.094029104, -93.083606693, -93.073848985], [-93.120940683, -93.148809699, -93.165160808, -93.172734754, -93.173809813, -93.170213871, -93.163374957, -93.15438469, -93.144062672, -93.13301696, -93.121693431, -93.110416398, -93.099418821, -93.088864198, -93.078863146, -93.069484154, -93.060763552], [-93.141797874, -93.157304353, -93.164646804, -93.165861129, -93.162607792, -93.156203781, -93.147674032, -93.137804283, -93.127188339, -93.116268599, -93.105369604, -93.094725151, -93.084499289, -93.074802898, -93.065705084, -93.057243361, -93.049431194], [-93.15219517, -93.159886647, -93.161490603, -93.158630527, -93.152594759, -93.144388147, -93.134785602, -93.124380075, -93.113621168, -93.102845426, -93.092299577, -93.082158765, -93.072541158, -93.063520425, -93.05513643, -93.047403503, -93.04031784], [-93.157816049, -93.160893209, -93.159020379, -93.153621499, -93.145794308, -93.136377446, -93.126008582, -93.115173238, -93.104240277, -93.093486111, -93.083112598, -93.073258987, -93.064014411, -93.055428277, -93.047519772, -93.040286519, -93.033710472], [-93.161725051, -93.162685599, -93.159065557, -93.152263021, -93.143332271, -93.133064655, -93.122055433, -93.110755628, -93.099507716, -93.08856777, -93.078119644, -93.06828639, -93.059141305, -93.050718704, -93.043023695, -93.036040285, -93.02973776], [-93.164901064, -93.165978396, -93.162137709, -93.15491727, -93.14546309, -93.134627892, -93.123050113, -93.111213088, -93.099482677, -93.088129711, -93.077346151, -93.067256419, -93.057930692, -93.049396475, -93.041649095, -93.034661677, -93.028392223], [-93.166610604, -93.170150293, -93.16768048, -93.161072322, -93.15171423, -93.140635838, -93.128606607, -93.116205835, -93.103868305, -93.091913349, -93.080564911, -93.0699678, -93.060202478, -93.051298528, -93.043247447, -93.036013854, -93.02954612], [-93.164553463, -93.173435754, -93.17428454, -93.169559494, -93.161088945, -93.15022626, -93.137975349, -93.125081858, -93.112097077, -93.099421989, -93.087338367, -93.076031444, -93.065607331, -93.05610895, -93.047530464, -93.039833074, -93.032958112], [-93.15475641, -93.172924856, -93.179747044, -93.178646754, -93.172176929, -93.162218255, -93.150145939, -93.136962316, -93.123395011, -93.109968434, -93.097051576, -93.084892621, -93.073641407, -93.063367502, -93.054078129, -93.045736709, -93.038281046], [-93.131370262, -93.164496212, -93.181061044, -93.186055719, -93.183182753, -93.175140662, -93.163871653, -93.150762876, -93.136802678, -93.122692667, -93.108927226, -93.095844156, -93.083658341, -93.072482478, -93.062347875, -93.053228214, -93.045062484], [-93.086715627, -93.142975345, -93.174587399, -93.189102063, -93.192023942, -93.187297123, -93.177711966, -93.165222585, -93.151192672, -93.136574902, -93.122036137, -93.108036876, -93.094880571, -93.082741486, -93.071688506, -93.061714247, -93.052763412], [-93.012280536, -93.102997518, -93.156720707, -93.185168961, -93.196656707, -93.196991957, -93.190192228, -93.179015997, -93.165351844, -93.150496079, -93.135345843, -93.12052063, -93.106436646, -93.09334634, -93.081369763, -93.07052724, -93.060775107], [-92.901928416, -93.041220046, -93.125371076, -93.172692464, -93.195729732, -93.20296081, -93.200091915, -93.190956979, -93.178131761, -93.163355496, -93.147812919, -93.132317786, -93.117419207, -93.103462333, -93.090627604, -93.07897027, -93.068462065], [-92.758460769, -92.960251681, -93.082303631, -93.152578587, -93.189451657, -93.204916428, -93.206801117, -93.200241344, -93.188625819, -93.174206051, -93.158494529, -93.142518185, -93.126969285, -93.112290295, -93.098725056, -93.086366607, -93.07520506], [-92.602324, -92.873022979, -93.035318543, -93.129300213, -93.180178734, -93.203885046, -93.21052898, -93.206593166, -93.196283019, -93.182355278, -93.166639046, -93.150363053, -93.134352019, -93.119136088, -93.105016927, -93.092121255, -93.080456823], [-92.475178809, -92.803184271, -92.997778843, -93.110312726, -93.171943534, -93.201895929, -93.212129404, -93.210183688, -93.200889053, -93.187381505, -93.171725028, -93.155295256, -93.139011679, -93.123467062, -93.109002734, -93.095770247, -93.083789517], [-92.425318032, -92.776135758, -92.983307857, -93.102952216, -93.168656336, -93.200955622, -93.212531672, -93.211331026, -93.202420067, -93.189077052, -93.17345258, -93.156976898, -93.140603903, -93.12494862, -93.1103674, -93.097020266, -93.084931471]]
CH3NC_THETA = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0, 140.0, 150.0, 160.0, 170.0, 180.0]
CH3NC_GRID_SKALA = [[-131.181552874, -131.936909686, -132.310113178, -132.487307426, -132.563370837, -132.58606786, -132.580593037, -132.562368093, -132.539210171, -132.515063352, -132.491960512, -132.471081063, -132.45389761], [-131.295058196, -131.986216526, -132.330112942, -132.493594358, -132.563272753, -132.582776025, -132.576174386, -132.557714443, -132.534735143, -132.510985585, -132.488387738, -132.468075674, -132.451585961], [-131.5667565, -132.105355544, -132.377324869, -132.506625382, -132.559974327, -132.57207549, -132.563030368, -132.54439567, -132.522252116, -132.499771722, -132.478641449, -132.459733392, -132.444877417], [-131.860523207, -132.234073857, -132.425440824, -132.515541687, -132.550051717, -132.554130103, -132.542717128, -132.524367957, -132.503668642, -132.48312455, -132.464057926, -132.447194478, -132.435149672], [-132.089339172, -132.332409023, -132.457170304, -132.514204877, -132.533058351, -132.53097954, -132.518218538, -132.500778102, -132.481998748, -132.463756903, -132.446990148, -132.433313448, -132.423392258], [-132.24228401, -132.39402167, -132.471208438, -132.504270392, -132.512029761, -132.505907853, -132.492798919, -132.476731626, -132.460006227, -132.443965009, -132.429675071, -132.419814518, -132.411576099], [-132.341236361, -132.430523911, -132.474715922, -132.491245809, -132.491484343, -132.482749869, -132.469577628, -132.454764377, -132.43989608, -132.426078265, -132.415574104, -132.407338458, -132.400393273], [-132.401071654, -132.452646733, -132.475307772, -132.480349279, -132.475119143, -132.46434862, -132.451043815, -132.437000335, -132.423356982, -132.411451608, -132.403472616, -132.396728666, -132.391015684], [-132.432571012, -132.465201611, -132.476392985, -132.474623875, -132.465593719, -132.452965435, -132.438960365, -132.424932462, -132.411982189, -132.402097947, -132.394923769, -132.388933632, -132.383932575], [-132.444452224, -132.471248257, -132.478386819, -132.473801545, -132.462761432, -132.448761043, -132.4338698, -132.419354905, -132.406307431, -132.39706658, -132.390036371, -132.38394011, -132.378834182], [-132.440440026, -132.472073188, -132.481093415, -132.476855086, -132.465397256, -132.450620932, -132.434839224, -132.419458464, -132.405605869, -132.395421872, -132.387900341, -132.38161444, -132.376319236], [-132.419416607, -132.467066453, -132.484016941, -132.483160052, -132.472578456, -132.457407867, -132.440698672, -132.424195122, -132.409078875, -132.396816097, -132.388317913, -132.381427382, -132.375767326], [-132.378129089, -132.45499522, -132.486394821, -132.492031158, -132.483726253, -132.468672205, -132.451048167, -132.433166408, -132.416453069, -132.402014071, -132.391984745, -132.383991251, -132.377413465], [-132.313070303, -132.434124828, -132.487130544, -132.502673871, -132.498127891, -132.483622263, -132.465032457, -132.445595758, -132.427091578, -132.410631819, -132.39768175, -132.388662163, -132.381244649], [-132.222219442, -132.402628764, -132.484342091, -132.51318669, -132.514008375, -132.500759395, -132.481340617, -132.4601884, -132.439763872, -132.421341329, -132.405832713, -132.394894872, -132.386579819], [-132.099907013, -132.356874067, -132.475213661, -132.520927255, -132.528826547, -132.517506134, -132.497494777, -132.474746863, -132.452438224, -132.432202598, -132.414879991, -132.401498617, -132.392081911], [-131.955710081, -132.300469303, -132.459897, -132.524156452, -132.539993255, -132.531271235, -132.511261263, -132.487334336, -132.463377758, -132.441452438, -132.422571082, -132.407362907, -132.396998297], [-131.830607365, -132.250965717, -132.444354357, -132.52385274, -132.546376209, -132.540054407, -132.520336999, -132.495770664, -132.470824924, -132.447877367, -132.428053823, -132.411913185, -132.400377112], [-131.77982797, -132.230973033, -132.437834298, -132.523170076, -132.548290889, -132.542972583, -132.523442646, -132.49866805, -132.473440333, -132.450170914, -132.430007278, -132.413485596, -132.401423751]]
CH3NC_GRID_XTB = [[-7.265161166, -8.053902166, -8.433192461, -8.604660684, -8.671640872, -8.686799561, -8.676786649, -8.655093303, -8.628725456, -8.601599028, -8.575737175, -8.552270364, -8.531798751], [-7.386327212, -8.105849621, -8.453347212, -8.610251605, -8.670714696, -8.683118449, -8.672103184, -8.650227948, -8.624079297, -8.597379469, -8.572011955, -8.549081793, -8.529135067], [-7.672298886, -8.228409543, -8.499431407, -8.620965867, -8.665579247, -8.67136172, -8.658062749, -8.635959009, -8.61061125, -8.585222326, -8.561348372, -8.540001431, -8.521583849], [-7.974327733, -8.355551619, -8.542500883, -8.624688782, -8.651750469, -8.65047209, -8.63512695, -8.613420917, -8.58976166, -8.566595117, -8.545213742, -8.526396949, -8.510370292], [-8.197512777, -8.442831285, -8.563025732, -8.613864131, -8.627269529, -8.621028426, -8.604999293, -8.58482399, -8.563868803, -8.543831477, -8.525813045, -8.510262279, -8.497238838], [-8.335614645, -8.486618835, -8.560738369, -8.59025228, -8.594845059, -8.586105694, -8.570887082, -8.553400601, -8.535916186, -8.51979279, -8.505701105, -8.493816313, -8.484062773], [-8.429365551, -8.51043089, -8.549566778, -8.562823071, -8.560860227, -8.550730459, -8.537122921, -8.522795745, -8.509275846, -8.497371478, -8.487312439, -8.479059629, -8.472445264], [-8.488815465, -8.527737055, -8.541697807, -8.541265396, -8.532849615, -8.520857289, -8.508332633, -8.496858868, -8.487093558, -8.479051564, -8.472573199, -8.467455087, -8.463492656], [-8.520954583, -8.539925814, -8.540555952, -8.531313417, -8.517312943, -8.502218171, -8.488945937, -8.479049838, -8.471922718, -8.466705869, -8.462867303, -8.460081484, -8.458118997], [-8.533794087, -8.547241588, -8.543872296, -8.53164899, -8.515209951, -8.497492427, -8.481947325, -8.471892311, -8.465712802, -8.461754419, -8.459173039, -8.457504965, -8.456451099], [-8.531102911, -8.5497999, -8.54899702, -8.537782707, -8.521382874, -8.503199706, -8.486401523, -8.474689162, -8.46742552, -8.462784855, -8.459772484, -8.457844668, -8.456648363], [-8.512569117, -8.5476819, -8.555784917, -8.548914241, -8.534356638, -8.516863771, -8.499958081, -8.486429031, -8.476651678, -8.469671229, -8.464678592, -8.461156938, -8.458752073], [-8.475804057, -8.54135222, -8.565199174, -8.565873016, -8.554489399, -8.537859999, -8.520339821, -8.504621331, -8.491765571, -8.481674216, -8.47391678, -8.468050675, -8.463690315], [-8.420850931, -8.531591495, -8.576808784, -8.587143293, -8.579462042, -8.563348666, -8.544501617, -8.526150111, -8.509952684, -8.49643797, -8.485556692, -8.477018916, -8.470452823], [-8.345935764, -8.514446847, -8.586233904, -8.608170213, -8.604785064, -8.589233726, -8.5688978, -8.547958498, -8.528581204, -8.511791359, -8.497846718, -8.486621302, -8.477801692], [-8.234644846, -8.480375986, -8.586911914, -8.623740641, -8.626002202, -8.611678847, -8.590336635, -8.567270433, -8.545256886, -8.525694736, -8.50910786, -8.495514976, -8.484672052], [-8.092077463, -8.430186284, -8.577405493, -8.631525996, -8.640547079, -8.628183343, -8.606528009, -8.582039612, -8.558150401, -8.536544112, -8.517972118, -8.502570509, -8.490158146], [-7.963063343, -8.382178355, -8.564476488, -8.633418575, -8.648332988, -8.637900044, -8.616368434, -8.591150547, -8.566175138, -8.543346123, -8.523562772, -8.50704285, -8.493649269], [-7.909904571, -8.362029314, -8.558477804, -8.633374038, -8.650670774, -8.641054355, -8.619637255, -8.594208419, -8.568882751, -8.545651243, -8.525463341, -8.508567019, -8.494841136]]
NB4_HCN_GEOMS = [[0.0, 1.6749999999999998], [15.0, 1.5968749999999998], [30.0, 1.5187499999999998], [45.0, 1.440625], [60.0, 1.284375], [75.0, 1.20625], [90.0, 1.128125], [105.0, 1.20625], [120.0, 1.3625], [127.5, 1.5187499999999998], [150.0, 1.5187499999999998], [180.0, 1.5187499999999998]]   # (theta_deg, r_A) of the 12 NB4 path points, for fpes.hcn_geometry
NB4_HCN_E = {"ccsd_t": [-93.31333514680848, -93.30478580033724, -93.29058825866117, -93.27077707956026, -93.24661052922545, -93.23453955337713, -93.23765081655874, -93.24305451803149, -93.2468394984869, -93.23743811708243, -93.27512572559105, -93.28804518990142], "pbe": [-93.34668701546074, -93.33773378408505, -93.3239298741205, -93.30565030348075, -93.28362613997106, -93.27227791084712, -93.27352510622592, -93.27835248645113, -93.28295992891209, -93.27480527860124, -93.31048596454919, -93.32190724460166], "b3lyp": [-93.46359904501735, -93.45493944022938, -93.44022144289309, -93.42054222982475, -93.39819001345172, -93.38747942128933, -93.39051919139376, -93.39658299740643, -93.40130524526258, -93.39220529452754, -93.42887320181973, -93.44129544993841], "skala": [-93.42741501539396, -93.41902239866492, -93.40447554196335, -93.3852106549393, -93.36223488281567, -93.34959479005283, -93.35352257950996, -93.35929379381243, -93.36206424578519, -93.35178973580315, -93.38989520285153, -93.40363388597093]}   # hartree; CCSD(T) is all-electron def2-TZVP
NB4_HCN_MAE = {'skala': 0.564, 'pbe': 1.242, 'b3lyp': 1.374}   # kcal/mol, NUMBERS.md
SCOUT_STORED = {'hcn': 48.481, 'ch3nc': 68.822}   # kcal/mol, NUMBERS.md, two-call scout barrier

def bottleneck(E, theta):
    """Minimum-bottleneck barrier between the two basins on a 4-connected grid (same algorithm as fpes.bottleneck_barrier).
    Returns (barrier_hartree, saddle_cell, hcn_cell)."""
    E = np.array([[np.inf if v is None else v for v in row] for row in E], float); nt, nr = E.shape
    hcn = np.where(np.asarray(theta) < 45.0)[0]; hnc = np.where(np.asarray(theta) > 135.0)[0]
    a = np.unravel_index(np.argmin(E[hcn]), E[hcn].shape); a = (hcn[a[0]], a[1])
    b = np.unravel_index(np.argmin(E[hnc]), E[hnc].shape); b = (hnc[b[0]], b[1])
    parent = list(range(nt * nr))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    active = np.zeros(nt * nr, bool)
    for flat in np.argsort(E, axis=None):
        if not np.isfinite(E.flat[flat]):
            break
        active[flat] = True; i, j = divmod(int(flat), nr)
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < nt and 0 <= nj < nr and active[ni * nr + nj]:
                ra, rb = find(flat), find(ni * nr + nj)
                if ra != rb:
                    parent[ra] = rb
        if find(a[0] * nr + a[1]) == find(b[0] * nr + b[1]):
            return E[i, j] - E[a], (i, j), a
    return float("nan"), None, a


def scout_barrier_kcal(system):
    """Two-call scout of v4: Skala energy at the xTB saddle minus Skala energy at the xTB first minimum (kcal/mol), from the stored grids."""
    th, sk, xt = (HCN_THETA, HCN_GRID_SKALA, HCN_GRID_XTB) if system == "hcn" else (CH3NC_THETA, CH3NC_GRID_SKALA, CH3NC_GRID_XTB)
    _, saddle, a = bottleneck(xt, th)
    S = np.array([[np.nan if v is None else v for v in row] for row in sk], float)
    return float((S[saddle] - S[a]) * 627.5094740631)



# ===== b1v5_geompack.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_geompack.py  --  the GEOMETRY PACK of the 1-hour design (PLAN_1hour.md, decision D1 and section 7.1).
#
# What it is, in plain words:
#   The geometry work (optimisations, transition states with ONE imaginary mode, Hessians, zero-point energies, relaxed scans, GFN2-xTB
#   minima and peaks) is deterministic CPU work.  It was done ONCE, in the sandbox, with the same helper functions the full notebooks use
#   (b1v5_cpu.py: opt_dft, refine_stationary, scan_points_dft, opt_xtb, ...), and stored in ONE JSON file with a SHA-256.  Both 1-hour
#   notebooks carry that file inside them (gzip + base64 in a cell), check its SHA-256, unpack it into geoms/ and phase01 / phase02 /
#   phase04 rows, and the CPU notebook re-verifies the geometries with a gradient test (phase C0).
#
# This file holds:   (1) the builder (also used by REBUILD_GEOMS = True on Colab and by build_geompack.py in the sandbox),
#                    (2) pack loading, hashing, verification and installation into the working folder.
# Pasted unchanged into both 1-hour notebooks (after b1v5_common.py and b1v5_cpu.py, which share the namespace); also importable.
# ==============================================================================
import os, sys, io, json, gzip, base64, hashlib, math, time, platform, datetime
import numpy as np
try:
    b1v5_common_loaded = COLUMNS      # noqa  (names exist when the cells above ran)
except NameError:
    from b1v5_common import *         # noqa
try:
    b1v5_cpu_loaded = ccsdt           # noqa
except NameError:
    from b1v5_cpu import *            # noqa

PACK_FORMAT = "b1v5-geompack-1"
CORE_SYSTEMS = ["hcn", "nh3", "h2co", "ch3nc", "sn2_ff", "sn2_clcl", "ch4"]      # six reactions + the CH4 stretch (PLAN_1hour D3)
EXT78_SYSTEMS = ["sn2_fcl", "sn2_ohf"]                                           # F- + CH3Cl and OH- + CH3F: first extension items
DFT_PATH_SYSTEMS = ["hcn", "nh3", "h2co"]                                        # relaxed DFT scans of the core (CH4: 9 stored rigid points)
PATH_N = {"hcn": 11, "nh3": 9, "h2co": 11}                                       # plan section 2: C3b 'HCN 11 points, NH3 9, H2CO 11, CH4 9'
XTB_N = 21                                                                       # xTB scans are cheap: 21 points for a finer peak
GRAD_RMS_TOL = 3.0e-4                                                            # pass criterion of the gradient test (hartree/bohr)
GRAD_RMS_BUILD_TARGET = 1.5e-4                                                   # the builder re-optimises tighter above this
LEVELS_TEXT = ("geometry: DF-B3LYP-D3(BJ) (PySCF density fitting, grid level 3, conv_tol 1e-9), def2-TZVP (def2-TZVPD for every SN2 species), "
               "geomeTRIC convergence set GAU; transition states by geomeTRIC eigenvector following with the analytic Hessian at step 0; "
               "frequencies and zero-point energy from the analytic DF-B3LYP-D3(BJ) Hessian; xTB: GFN2-xTB (tblite) with a geomeTRIC engine")


# ------------------------------------------------------------------ hashing
def canon_atoms(atoms):
    return [[s, [round(float(v), 8) for v in c]] for s, c in atoms]


def struct_sha(system, species, level, charge, atoms):
    blob = json.dumps([system, species, level, int(charge), canon_atoms(atoms)], separators=(",", ":")).encode()
    return hashlib.sha256(blob).hexdigest()


def path_sha(system, pts):
    blob = json.dumps([system, [[int(p["index"]), round(float(p["coordinate"]), 8), canon_atoms([(s, tuple(c)) for s, c in p["atoms"]])] for p in pts]], separators=(",", ":")).encode()
    return hashlib.sha256(blob).hexdigest()


def pack_bytes(pack):
    """Canonical bytes of a pack (sorted keys, no spaces): what the SHA-256 of the pack is computed over."""
    return json.dumps(pack, sort_keys=True, separators=(",", ":")).encode()


def pack_sha256(pack):
    return hashlib.sha256(pack_bytes(pack)).hexdigest()


def pack_encode(pack):
    """gzip + base64 text of a pack (what the notebooks embed)."""
    return base64.b64encode(gzip.compress(pack_bytes(pack), 9)).decode()


def pack_decode(text):
    return json.loads(gzip.decompress(base64.b64decode(text)).decode())


def key_of(system, species, level):
    return f"{system}/{species}/{level}"


# ------------------------------------------------------------------ the relaxed-scan specifications of the 1-hour design
def _scan_specs(ts_angle_hcn=72.0, n_xtb=None):
    """One driving coordinate per reaction, as in b1v5_cpu.SCANS, but with 11 / 9 / 11 points for the DFT paths (HCN, NH3, H2CO) and, if n_xtb is
    given, n_xtb points for the xTB scans (all six reactions).  The HCN grid contains the angle of our own transition state, so the path
    has a point at the saddle (constrained optimisation there = the transition state)."""
    n = n_xtb
    X = {}
    X["hcn"] = dict(kind="angle", idx=(2, 0, 1), start="HCN", label="H-C-N angle (deg)",
                    values=(lambda g0: np.array([180.0] + list(np.linspace(171.0, 6.0, n - 1)))) if n else
                    (lambda g0: np.array(sorted({180.0, round(float(ts_angle_hcn), 3)} | set(np.round(np.linspace(171.0, 6.0, 9), 3)), reverse=True))))
    X["nh3"] = dict(kind="dihedral", idx=(0, 1, 2, 3), start="NH3", label="N-H1-H2-H3 dihedral (deg)",
                    values=lambda g0, k=(n or PATH_N["nh3"]): np.linspace(coord_value("dihedral", (0, 1, 2, 3), g0), -coord_value("dihedral", (0, 1, 2, 3), g0), k))
    X["h2co"] = dict(kind="distance", idx=(2, 3), start="TS", label="H...H distance (A)",
                     values=lambda g0, k=(n or PATH_N["h2co"]): np.linspace(1.85, 0.80, k))
    X["ch3nc"] = dict(kind="angle", idx=(0, 1, 2), start="CH3NC", label="C-N-C angle (deg)",
                      values=lambda g0, k=(n or 11): np.array([180.0] + list(np.linspace(171.0, 9.0, k - 1))))
    X["sn2_ff"] = dict(kind="distance", idx=(0, 2), start="RC", label="C...F(attacking) distance (A)",
                       values=lambda g0, k=(n or 11): np.linspace(coord_value("distance", (0, 2), g0), 1.43, k))
    X["sn2_clcl"] = dict(kind="distance", idx=(0, 2), start="RC", label="C...Cl(attacking) distance (A)",
                         values=lambda g0, k=(n or 11): np.linspace(coord_value("distance", (0, 2), g0), 1.80, k))
    X["sn2_fcl"] = dict(kind="distance", idx=(0, 2), start="RC", label="C...F(attacking) distance (A)",
                        values=lambda g0, k=(n or 11): np.linspace(coord_value("distance", (0, 2), g0), 1.40, k))
    X["sn2_ohf"] = dict(kind="distance", idx=(0, 5), start="RC", label="C...O(attacking) distance (A)",
                        values=lambda g0, k=(n or 11): np.linspace(coord_value("distance", (0, 5), g0), 1.42, k))
    return X


def _walk_spec(sp, g0, relax, energy, store, save, label=""):
    """The relaxed walk of b1v5_cpu._walk with an explicit scan specification (so that DFT and xTB scans can differ)."""
    vals = np.asarray(sp["values"](g0), float)
    n = len(vals)
    c0 = coord_value(sp["kind"], sp["idx"], g0)
    i0 = int(np.argmin(np.abs(vals - c0)))
    pts = {int(p["index"]): p for p in store if int(p["index"]) < n}

    def one(k, prev_atoms, free=False):
        t0 = time.time()
        r = dict(atoms=prev_atoms, converged=True) if free else relax(prev_atoms, (sp["kind"], sp["idx"], float(vals[k])))
        e = energy(r["atoms"])
        pts[k] = dict(index=k, coordinate=float(vals[k]) if not free else c0, atoms=r["atoms"], energy=e["energy"],
                      converged=bool(r["converged"] and e["converged"]), seconds=time.time() - t0)
        save([pts[j] for j in sorted(pts)])
        print(f"   {label} point {k + 1}/{n}  coordinate {pts[k]['coordinate']:.3f}  E={e['energy']:.6f}  converged={pts[k]['converged']}  {time.time() - t0:.0f} s", flush=True)
    if i0 not in pts:
        one(i0, g0, free=True)
    for k in range(i0 + 1, n):
        if k not in pts:
            one(k, pts[k - 1]["atoms"])
    for k in range(i0 - 1, -1, -1):
        if k not in pts:
            one(k, pts[k + 1]["atoms"])
    return [pts[j] for j in sorted(pts)]


# ------------------------------------------------------------------ the builder
class PackBuilder:
    """Builds the pack step by step in a work folder (resumable: every finished step is written to pack_work.json).
        PackBuilder(work).build(systems)   then   .assemble(path_out)
    Every step records its seconds in work['timings']."""

    def __init__(self, work, hardware=None, log=print):
        self.work = os.path.abspath(work)
        os.makedirs(self.work, exist_ok=True)
        self.fn = os.path.join(self.work, "pack_work.json")
        self.W = json.load(open(self.fn)) if os.path.exists(self.fn) else dict(structures={}, paths={}, xtb={}, timings=[], notes=[])
        self.log = log
        self.hardware = hardware

    def save(self):
        json.dump(self.W, open(self.fn + ".tmp", "w"), indent=1, default=float)
        os.replace(self.fn + ".tmp", self.fn)

    def tick(self, step, system, species, seconds, note=""):
        self.W["timings"].append(dict(step=step, system=system, species=species, seconds=round(float(seconds), 1), note=note))
        self.log(f"  [{system}:{species}] {step}: {seconds:.1f} s {note}")
        self.save()

    # ---- one stationary structure: optimise, frequencies, gradient check
    def stationary(self, system, sp, kind):
        key = key_of(system, sp, LV_OWN)
        if key in self.W["structures"]:
            return
        ch, basis = species_charge(system, sp), basis_for(system, "main")
        at = start_atoms(system, sp)
        t0 = time.time()
        conv = True
        if len(at) > 1:
            r = opt_dft(at, ch, basis, transition=(kind == "ts"), hessian=(kind == "ts"))
            at, conv = r["atoms"], r["converged"]
        self.tick("optimisation", system, sp, time.time() - t0, f"kind={kind} converged={conv}")
        t0 = time.time()
        at, a, retried = refine_stationary(at, ch, basis, kind)
        self.tick("frequencies + zero-point energy" + (" (+ retry along the extra mode)" if retried else ""), system, sp, time.time() - t0, f"imag={a['imag']}")
        a = self.regrid_analysis(at, ch, basis, kind, a, system, sp)
        g = self.gradient(at, ch, basis, system, sp)
        if g["rms"] > GRAD_RMS_BUILD_TARGET and len(at) > 1:            # re-optimise tighter once so that the gradient test on Colab has margin
            t0 = time.time()
            at2, a2, g2 = self.tighten(at, ch, basis, kind)
            self.tick("tighter re-optimisation (GAU_TIGHT)", system, sp, time.time() - t0, f"rms {g['rms']:.1e} -> {g2['rms']:.1e}, imag {a['imag']} -> {a2['imag']}")
            if g2["rms"] < g["rms"] and abs(a2["imag"] - (1 if kind == "ts" else 0)) <= abs(a["imag"] - (1 if kind == "ts" else 0)):
                at, a, g = at2, a2, g2
        self.W["structures"][key] = dict(system=system, species=sp, level=LV_OWN, kind=kind, charge=int(ch), basis=basis, atoms=canon_atoms(at),
                                         energy_hartree=a["energy"], scf_converged=bool(a["converged"]), opt_converged=bool(conv), imag=int(a["imag"]),
                                         zpe_hartree=a["extra"].get("zpe_hartree"), lowest_freq_cm1=a["extra"].get("lowest_freq_cm1"), freqs=a["extra"].get("freqs"),
                                         retried=bool(retried), rms_grad=g["rms"], max_grad=g["max"], grad_seconds=g["seconds"], hessian_grid=int(a["extra"].get("hessian_grid", 3)),
                                         sha256=struct_sha(system, sp, LV_OWN, ch, at))
        self.save()

    def regrid_analysis(self, at, ch, basis, kind, a, system, sp):
        """Found after the first sandbox build (Cl- + CH3Cl, def2-TZVPD): at integration grid level 3 the very flat modes of an anion with diffuse functions can come out
        imaginary (TS: -134 cm-1, at level 5 the same geometry gives +206 cm-1 and exactly one imaginary mode).  If the number of imaginary modes is not the wanted one,
        the Hessian is repeated ONCE at grid level 5 at the stored geometry and taken if it is closer to the wanted number.  The geometry, the energy and the gradient test
        stay at level 3 (the level the notebooks use); only frequencies and zero-point energy come from level 5 (recorded as hessian_grid = 5)."""
        want = 1 if kind == "ts" else 0
        if len(at) < 2 or a["imag"] == want:
            return a
        t0 = time.time()
        a5 = analyse_stationary(at, ch, basis, grid_level=5)
        self.tick("Hessian repeated at grid level 5", system, sp, time.time() - t0, f"imag {a['imag']} -> {a5['imag']}")
        if abs(a5["imag"] - want) < abs(a["imag"] - want):
            a5["energy"], a5["converged"] = a["energy"], a["converged"]
            a5["extra"]["hessian_grid"] = 5
            return a5
        return a

    def regrid(self, system, sp):
        """Apply regrid_analysis to a structure that is already in the work folder (used for the structures built before this step existed)."""
        key = key_of(system, sp, LV_OWN)
        v = self.W["structures"][key]
        a = dict(energy=v["energy_hartree"], converged=v["scf_converged"], imag=v["imag"], extra=dict(zpe_hartree=v["zpe_hartree"], lowest_freq_cm1=v["lowest_freq_cm1"], freqs=v["freqs"]))
        a2 = self.regrid_analysis([(s, tuple(c)) for s, c in v["atoms"]], v["charge"], v["basis"], v["kind"], a, system, sp)
        if a2 is not a:
            v.update(imag=int(a2["imag"]), zpe_hartree=a2["extra"]["zpe_hartree"], lowest_freq_cm1=a2["extra"]["lowest_freq_cm1"], freqs=a2["extra"]["freqs"], hessian_grid=5)
            fb = key_of(system, sp, LV_SCANPEAK)
            if v["kind"] == "ts" and v["imag"] == 1 and fb in self.W["structures"]:
                del self.W["structures"][fb]
                self.W["notes"].append(f"{system}: the scan-peak fallback TS was dropped: the own TS has one imaginary mode at grid level 5")
            self.W["notes"].append(f"{system}:{sp}: frequencies and zero-point energy from a grid level 5 Hessian (level 3 gave {a['imag']} imaginary modes)")
            self.save()

    def gradient(self, atoms, ch, basis, system="", sp=""):
        t0 = time.time()
        mf = make_ks(atoms, ch, basis)
        e = mf.kernel()
        g = np.asarray(mf.nuc_grad_method().kernel(), float) if len(atoms) > 0 else np.zeros((1, 3))
        sec = time.time() - t0
        rec = dict(rms=float(np.sqrt(np.mean(g ** 2))), max=float(np.abs(g).max()), seconds=round(sec, 1), energy=float(e))
        if system:
            self.tick("gradient at the stored geometry", system, sp, sec, f"rms={rec['rms']:.2e} max={rec['max']:.2e}")
        return rec

    def tighten(self, atoms, ch, basis, kind):
        from pyscf.geomopt import geometric_solver
        quiet_geometric()
        mf = make_ks(atoms, ch, basis)
        opts = dict(convergence_set="GAU_TIGHT", maxsteps=120)
        if kind == "ts":
            opts["transition"] = True; opts["hessian"] = True
        conv, mol = geometric_solver.kernel(mf, assert_convergence=False, **opts)
        new = mol_atoms(mol)
        a2 = analyse_stationary(new, ch, basis)
        return new, a2, self.gradient(new, ch, basis)

    # ---- relaxed DFT scan of one system (HCN, NH3, H2CO), 11 / 9 / 11 points
    def dft_path(self, system):
        if system in self.W["paths"] and self.W["paths"][system].get("complete"):
            return
        ts = self.W["structures"].get(key_of(system, "TS", LV_OWN))
        ts_angle = 72.0
        if system == "hcn" and ts:
            ts_angle = coord_value("angle", (2, 0, 1), [(s, tuple(c)) for s, c in ts["atoms"]])
        spec = _scan_specs(ts_angle)[system]
        start = self.W["structures"][key_of(system, spec["start"], LV_OWN)]
        g0 = [(s, tuple(c)) for s, c in start["atoms"]]
        ch, basis = species_charge(system, spec["start"]), basis_for(system, "main")
        old = self.W["paths"].get(system, {}).get("points", [])
        store = [dict(p, atoms=[(s, tuple(c)) for s, c in p["atoms"]]) for p in old]

        def save(pts):
            self.W["paths"][system] = dict(system=system, label=spec["label"], basis=basis, level=LV_SCAN, complete=False,
                                           points=[dict(index=int(q["index"]), coordinate=float(q["coordinate"]), atoms=canon_atoms(q["atoms"]), energy=float(q["energy"]),
                                                        converged=bool(q["converged"]), seconds=round(float(q.get("seconds", 0.0)), 1)) for q in pts])
            self.save()
        t0 = time.time()
        pts = _walk_spec(spec, g0, lambda at, c: opt_dft(at, ch, basis, constraints=[c], maxsteps=100), lambda at: dft_energy(at, ch, basis), store, save, label=f"{system} DFT scan")
        P = self.W["paths"][system]
        P["complete"] = True; P["sha256"] = path_sha(system, P["points"])
        k = peak_of(pts)
        P["peak_index"] = k
        self.tick("relaxed DFT scan", system, "path", time.time() - t0, f"{len(pts)} points, peak index {k}")

    # ---- the scan-peak fallback for a TS search that failed
    def scanpeak_fallback(self, system):
        spec = _scan_specs()[system]
        start = self.W["structures"].get(key_of(system, spec["start"], LV_OWN))
        if not start:
            return
        g0 = [(s, tuple(c)) for s, c in start["atoms"]]
        ch, basis = species_charge(system, spec["start"]), basis_for(system, "main")
        t0 = time.time()
        pts = _walk_spec(spec, g0, lambda at, c: opt_dft(at, ch, basis, constraints=[c], maxsteps=100), lambda at: dft_energy(at, ch, basis), [], lambda p: None, label=f"{system} fallback scan")
        k = peak_of(pts)
        if k is None:
            self.W["notes"].append(f"{system}: TS search failed and the fallback scan has no peak")
            return
        a = analyse_stationary(pts[k]["atoms"], species_charge(system, "TS"), basis)
        self.W["structures"][key_of(system, "TS", LV_SCANPEAK)] = dict(system=system, species="TS", level=LV_SCANPEAK, kind="ts", charge=int(species_charge(system, "TS")), basis=basis,
                                                                    atoms=canon_atoms(pts[k]["atoms"]), energy_hartree=a["energy"], scf_converged=bool(a["converged"]), opt_converged=True,
                                                                    imag=int(a["imag"]), zpe_hartree=a["extra"].get("zpe_hartree"), lowest_freq_cm1=a["extra"].get("lowest_freq_cm1"),
                                                                    freqs=a["extra"].get("freqs"), retried=False, rms_grad=None, max_grad=None, grad_seconds=None,
                                                                    sha256=struct_sha(system, "TS", LV_SCANPEAK, species_charge(system, "TS"), pts[k]["atoms"]), from_scan_point=int(k))
        self.tick("scan-peak fallback for the TS", system, "TS", time.time() - t0, f"point {k}")

    # ---- the CH4 stretch: the nine stored rigid points (geometry only, energies from C3b / G1b)
    def ch4_path(self):
        if "ch4" in self.W["paths"]:
            return
        pts = [dict(index=i, coordinate=float(s), atoms=canon_atoms(ch4_geometry(float(s))), energy=None, converged=True, seconds=0.0) for i, s in enumerate(CH4_STRETCH)]
        self.W["paths"]["ch4"] = dict(system="ch4", label="C-H stretch (A)", basis=basis_for("ch4", "main"), level=LV_SCAN, complete=True, points=pts, sha256=path_sha("ch4", pts))
        self.save()

    # ---- GFN2-xTB scout geometries: minimum (reactant or complex) and the peak of a relaxed xTB scan
    def xtb(self, system):
        if system in self.W["xtb"]:
            return
        S = SYSTEMS[system]
        rmin = S["rc"] or S["reactants"][0]
        ch = species_charge(system, rmin)
        own = self.W["structures"].get(key_of(system, rmin, LV_OWN))
        a0 = [(s, tuple(c)) for s, c in own["atoms"]] if own else start_atoms(system, rmin)
        t0 = time.time()
        r = opt_xtb(a0, ch)
        e_min = xtb_energy(r["atoms"], ch)["energy"]
        self.tick("xTB minimum", system, rmin, time.time() - t0, f"converged={r['converged']}")
        spec = _scan_specs(n_xtb=XTB_N)[system]
        if spec["start"] == rmin:
            g0 = r["atoms"]
        else:
            st = self.W["structures"].get(key_of(system, spec["start"], LV_OWN))
            g0 = [(s, tuple(c)) for s, c in st["atoms"]] if st else start_atoms(system, spec["start"])
        t0 = time.time()
        pts = _walk_spec(spec, g0, lambda at, c: opt_xtb(at, species_charge(system, spec["start"]), constraints=[c]),
                         lambda at: xtb_energy(at, species_charge(system, spec["start"])), [], lambda p: None, label=f"{system} xTB scan")
        k = peak_of(pts)
        ec = species_charge(system, "TS")
        rec = dict(system=system, reactant_species=rmin, charge=int(ch), method="GFN2-xTB (tblite)", scan=dict(label=spec["label"], n_points=len(pts)),
                   minimum=dict(atoms=canon_atoms(r["atoms"]), energy_hartree=float(e_min), converged=bool(r["converged"]), sha256=struct_sha(system, rmin, LV_XTB, ch, r["atoms"])))
        if k is None:
            mid = pts[len(pts) // 2]
            rec["peak"] = dict(atoms=canon_atoms(mid["atoms"]), energy_hartree=None, converged=False, no_peak=True, scan_max_index=int(np.nanargmax([q["energy"] for q in pts])),
                               sha256=struct_sha(system, "TS", LV_XTB, ec, mid["atoms"]))
        else:
            rec["peak"] = dict(atoms=canon_atoms(pts[k]["atoms"]), energy_hartree=float(pts[k]["energy"]), converged=bool(pts[k]["converged"]), no_peak=False, scan_point=int(k),
                               coordinate=float(pts[k]["coordinate"]), sha256=struct_sha(system, "TS", LV_XTB, ec, pts[k]["atoms"]))
        self.W["xtb"][system] = rec
        self.tick("xTB relaxed scan + peak", system, "TS", time.time() - t0, f"{len(pts)} points, peak {k}")

    # ---- everything for a list of systems
    def build(self, systems):
        for system in systems:
            for sp, kind in SPECIES_KIND[system].items():
                with item_guard("geompack", f"{system}:{sp}"):
                    self.stationary(system, sp, kind)
            ts = self.W["structures"].get(key_of(system, "TS", LV_OWN))
            if SYSTEMS[system]["ts"] and (ts is None or ts["imag"] != 1):
                with item_guard("geompack", f"{system}:fallback"):
                    self.scanpeak_fallback(system)
            if system in DFT_PATH_SYSTEMS:
                with item_guard("geompack", f"{system}:path"):
                    self.dft_path(system)
            if system == "ch4":
                self.ch4_path()
        for system in systems:
            if system in REACTIONS:
                with item_guard("geompack", f"{system}:xtb"):
                    self.xtb(system)

    # ---- the pack file
    def assemble(self, systems, design, extra_note=""):
        W = self.W
        pack = dict(format=PACK_FORMAT, design=design, systems=list(systems), created=now(), levels=LEVELS_TEXT, note=extra_note,
                    builder=dict(script="build_geompack.py + b1v5_geompack.py + b1v5_cpu.py", versions=ver(), hardware=self.hardware or hw(), python=platform.python_version()),
                    structures={k: v for k, v in sorted(W["structures"].items()) if v["system"] in systems},
                    paths={k: v for k, v in sorted(W["paths"].items()) if k in systems},
                    xtb={k: v for k, v in sorted(W["xtb"].items()) if k in systems},
                    timings=[t for t in W["timings"] if t["system"] in systems], notes=W["notes"])
        return pack


# ------------------------------------------------------------------ loading and verification
def load_pack_text(text, expected_sha256=None):
    """Decode an embedded pack; returns (pack, sha256 of its canonical bytes, matches).  Never raises on a hash mismatch (the caller decides)."""
    pack = pack_decode(text)
    sha = pack_sha256(pack)
    return pack, sha, (expected_sha256 is None or sha == expected_sha256)


def verify_hashes(pack):
    """Re-compute the SHA-256 of every structure and path.  Returns (n_ok, list of problems)."""
    ok, bad = 0, []
    for k, v in pack["structures"].items():
        h = struct_sha(v["system"], v["species"], v["level"], v["charge"], [(s, tuple(c)) for s, c in v["atoms"]])
        if h == v["sha256"]:
            ok += 1
        else:
            bad.append("structure " + k)
    for k, v in pack["paths"].items():
        if path_sha(k, v["points"]) == v.get("sha256"):
            ok += 1
        else:
            bad.append("path " + k)
    for k, v in pack["xtb"].items():
        for part, sp in (("minimum", v["reactant_species"]), ("peak", "TS")):
            p = v[part]
            if struct_sha(k, sp, LV_XTB, p.get("charge", v["charge"] if part == "minimum" else species_charge(k, "TS")), [(s, tuple(c)) for s, c in p["atoms"]]) == p["sha256"]:
                ok += 1
            else:
                bad.append(f"xtb {k} {part}")
    return ok, bad


def pack_geoms(pack, systems=None):
    """Yield (system, species, level, kind, charge, atoms) for every structure that is a geometry (own, scan-peak TS, xTB)."""
    for k, v in pack["structures"].items():
        if systems is None or v["system"] in systems:
            yield v["system"], v["species"], v["level"], v["kind"], v["charge"], [(s, tuple(c)) for s, c in v["atoms"]]
    for sysname, v in pack["xtb"].items():
        if systems is not None and sysname not in systems:
            continue
        yield sysname, v["reactant_species"], LV_XTB, "min", v["charge"], [(s, tuple(c)) for s, c in v["minimum"]["atoms"]]
        p = v["peak"]
        yield sysname, "TS", LV_XTB, "ts", species_charge(sysname, "TS"), [(s, tuple(c)) for s, c in p["atoms"]]


def install_pack(pack, systems=None, sha=None, hardware_label="geometry pack (sandbox build)"):
    """Write the pack into the working folder: geoms/*.json + *.xyz, geoms/<system>_path.json, and the rows of phase01_stationary.csv (DF-B3LYP-D3(BJ)
    energy, imaginary modes, zero-point energy), phase02_scans.csv (path energies) and phase04_scout_ref.csv (xTB energies).
    Idempotent: a geometry file with the same atoms is left alone; rows that exist are not duplicated.  Returns a dict of counts."""
    n = dict(geoms=0, geoms_kept=0, paths=0, rows=0)
    for system, sp, lv, kind, ch, at in pack_geoms(pack, systems):
        name = geom_name(system, sp, lv)
        g = load_geom(name)
        if g is not None and canon_atoms(g["atoms"]) == canon_atoms(at):
            n["geoms_kept"] += 1
            continue
        save_geom(name, at, ch, species=sp, kind=kind, level=lv, opt_converged=True, basis=basis_for(system, "main"), from_pack=True, pack_sha256=sha or "")
        n["geoms"] += 1
    for system, v in pack["paths"].items():
        if systems is not None and system not in systems:
            continue
        p = path("geoms", f"{system}_path.json")
        d = dict(system=system, level=LV_SCAN, basis=v["basis"], label=v["label"], saved=now(), from_pack=True, pack_sha256=sha or "",
                 points=[dict(index=int(q["index"]), coordinate=float(q["coordinate"]), atoms=[[s, list(map(float, c))] for s, c in q["atoms"]],
                              energy=(None if q.get("energy") is None else float(q["energy"])), converged=bool(q.get("converged", True))) for q in v["points"]])
        json.dump(d, open(p + ".tmp", "w"), indent=1); os.replace(p + ".tmp", p)
        n["paths"] += 1
    def put(csv_name, system, pid, level, method, basis, e, conv, imag="", coordinate="", extra=None):
        if have(csv_name, system, pid, level, method, basis):
            return
        r = make_row(system, pid, level, method, basis, e, conv, coordinate, "", imag, 0.0, dict(extra or {}, from_pack=True))
        r["hardware"] = hardware_label
        append_row(csv_name, r)
        n["rows"] += 1
    for k, v in pack["structures"].items():
        if systems is not None and v["system"] not in systems:
            continue
        put(PHASE_CSV["phase01"], v["system"], v["species"], v["level"], M_DFB3LYP, v["basis"], v["energy_hartree"], v["scf_converged"] and v["opt_converged"], imag=v["imag"],
            extra=dict(zpe_hartree=v["zpe_hartree"], lowest_freq_cm1=v["lowest_freq_cm1"], freqs=v["freqs"], retried=v["retried"], rms_grad_build=v["rms_grad"]))
    for system, v in pack["paths"].items():
        if systems is not None and system not in systems or system == "ch4":
            continue
        for q in v["points"]:
            put(PHASE_CSV["phase02"], system, f"p{int(q['index']):02d}", LV_SCAN, M_DFB3LYP, v["basis"], q["energy"], q["converged"], coordinate=f"{q['coordinate']:.4f}")
    for system, v in pack["xtb"].items():
        if systems is not None and system not in systems:
            continue
        for part, sp in (("minimum", v["reactant_species"]), ("peak", "TS")):
            p = v[part]
            if p.get("no_peak"):
                put(PHASE_CSV["phase04"], system, "TS", LV_XTB, M_XTB, "GFN2", float("nan"), False, extra={"no_peak": True, "scan_max_index": p.get("scan_max_index")})
            else:
                put(PHASE_CSV["phase04"], system, sp, LV_XTB, M_XTB, "GFN2", p["energy_hartree"], p["converged"])
    return n


def gradient_test(pack, systems=None, tol=GRAD_RMS_TOL, which=None, quiet=False, csv_name="phase00_verify.csv", admit=None, run=None):
    """The gradient-norm test of phase C0: DF-B3LYP-D3(BJ) energy + gradient at every stored structure in `which` (a list of (system, species) pairs;
    default every unimolecular structure and the SN2 transition states).  PASS: RMS gradient below `tol` hartree/bohr.  One row per structure in
    `csv_name` (energy = the energy recomputed here; extra holds RMS, max, the stored pack energy and the difference).
    `run(csv, system, pid, level, method, basis, fn, kind)` lets the caller route every calculation through the deadline queue."""
    out = []
    todo = which if which is not None else [(v["system"], v["species"]) for k, v in pack["structures"].items()
                                           if v["level"] in (LV_OWN, LV_SCANPEAK) and (not SYSTEMS[v["system"]]["sn2"] or v["species"] == "TS") and v["system"] != "ch4"
                                           and len(v["atoms"]) > 2]          # polyatomic structures only (no H2, CO, atoms)
    for system, sp in todo:
        if systems is not None and system not in systems:
            continue
        v = pack["structures"].get(key_of(system, sp, LV_OWN)) or pack["structures"].get(key_of(system, sp, LV_SCANPEAK))
        if v is None:
            continue
        at = [(s, tuple(c)) for s, c in v["atoms"]]
        def fn(at=at, v=v):
            mf = make_ks(at, v["charge"], v["basis"])
            e = float(mf.kernel())
            g = np.asarray(mf.nuc_grad_method().kernel(), float)
            rms, mx = float(np.sqrt(np.mean(g ** 2))), float(np.abs(g).max())
            return dict(energy=e, converged=bool(mf.converged), imag=v["imag"],
                        extra=dict(rms_grad=rms, max_grad=mx, tolerance=tol, passed=bool(rms < tol), pack_energy=v["energy_hartree"], dE_vs_pack=e - v["energy_hartree"],
                                   rms_grad_at_build=v["rms_grad"]))
        pid = sp
        if run is not None:
            row = run(csv_name, system, pid, v["level"], "DF-B3LYP-D3BJ-grad", v["basis"], fn, f"grad:{system}")
        else:
            row = run_point(csv_name, system, pid, v["level"], "DF-B3LYP-D3BJ-grad", v["basis"], fn, quiet=quiet)
        out.append((system, sp, row))
    return out


def grad_summary(csv_name="phase00_verify.csv", tol=GRAD_RMS_TOL):
    rows = [r for r in read_rows(csv_name) if r["method"] == "DF-B3LYP-D3BJ-grad"]
    n_pass = n_fail = 0
    worst = (0.0, "")
    for r in rows:
        x = extra_of(r)
        if x.get("rms_grad") is None or r["converged"] not in ("1", "True"):
            n_fail += 1; continue
        if x["rms_grad"] < tol:
            n_pass += 1
        else:
            n_fail += 1
        if x["rms_grad"] > worst[0]:
            worst = (x["rms_grad"], f"{r['system']}:{r['point_id']}")
    return dict(n=len(rows), passed=n_pass, failed=n_fail, worst_rms=worst[0], worst_where=worst[1], tolerance=tol)


# ------------------------------------------------------------------ REBUILD_GEOMS: the same builder, on Colab, into the working folder
def rebuild_geoms(systems, work_subdir="geompack_rebuild", pack_sha=None):
    """Not part of the 1-hour budget (extension EXT-geom, about 1.5 to 2 hours, resumable).  Builds every structure with the full-size machinery,
    writes the new geometries over the pack's (the pack versions are copied to geoms_pack/ first) and replaces the phase01 / phase02 / phase04 rows."""
    import shutil
    work = path(work_subdir)
    b = PackBuilder(work)
    b.build(systems)
    pack = b.assemble(systems, "rebuilt on this machine")
    d_old = path("geoms_pack")
    os.makedirs(d_old, exist_ok=True)
    for f in os.listdir(path("geoms")):
        if f.endswith((".json", ".xyz")) and not os.path.exists(os.path.join(d_old, f)):
            shutil.copy2(path("geoms", f), os.path.join(d_old, f))
    for name in (PHASE_CSV["phase01"], PHASE_CSV["phase02"], PHASE_CSV["phase04"]):
        if os.path.exists(path(name)):
            os.replace(path(name), path(name.replace(".csv", "_from_pack.csv")))
    for f in list(os.listdir(path("geoms"))):
        if f.endswith((".json", ".xyz")):
            os.remove(path("geoms", f))
    n = install_pack(pack, systems, sha=pack_sha_of(pack), hardware_label=hw())
    json.dump(pack, open(path("geompack_rebuilt.json"), "w"), indent=1, default=float)
    return n


def pack_sha_of(pack):
    return pack_sha256(pack)


# ===== b1v5_gpu.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_gpu.py  --  calculation helpers and phase logic for the GPU notebook (G-1 to G4 and the optional phase).
# Pasted into the second code cell of the GPU notebook (after b1v5_common.py, b1v5_cpu.py and b1v5_v4_stored.py,
# which share the same namespace); it is not meant to be imported on its own.
#
# What is in here, in plain words:
#   1. detect the GPU and decide how much work to do
#   2. one function per functional (PBE, r2SCAN, B3LYP-D3(BJ), wB97M-V) and for Skala-1.1, on the GPU or the CPU,
#      with a ladder of retries when the SCF does not converge
#   3. a geometry optimiser (geomeTRIC) that takes energies and gradients from any of them
#   4. the phases: G-1 (rebuild missing CPU inputs), G0 (checks), G1 (all functionals), G2 (basis check),
#      G3 (xTB scout check), G4 (Skala-relaxed geometries), optional (paths of three more SN2 systems)
# Stand-in mode (STANDIN = True) swaps Skala and the large functionals for cheap PySCF functionals in tiny bases
# so that the whole notebook can be tested on a plain CPU.  Its numbers are NOT results.
#
# Skala and its D3 term (review 2026-11-15, rows 1 and 2) -- what was READ in the skala 2026.9 wheel (pip download, not run):
#   * SkalaKS / SkalaRKS (pyscf and gpu4pyscf) and the ASE calculator all default to with_dftd3=True: the PACKAGE DEFAULT IS D3 ON.
#   * The D3 variant is not hard-coded in the package: the dispersion model is built as DFTD3Dispersion(mol, xc.get_d3_settings()), where the
#     string "b3lyp5" is stored inside the weights file (extra file 'expected_d3_settings', identical in skala-1.1.fun and skala-1.1-rev1.fun).
#     pyscf-dispersion then uses its default version 'd3bj' (rational / Becke-Johnson damping) and atm=False (two-body only, no Axilrod-Teller-Muto
#     term).  In pyscf-dispersion 1.7.0 the parameter set "b3lyp5" gives the same energy as "b3lyp".  The D3 energy is added in energy_nuc().
#   * In skala 2026.9 the name "skala-1.1" resolves to the file skala-1.1-rev1.fun (the same trained model, retraced); "skala-1.1-rev0" is the
#     original skala-1.1.fun.  Both weights files are verified against a SHA-256 table inside the package.
#   * skala 2026.9 requires pyscf >= 2.11 and < 2.15 (the GPU notebook pins pyscf 2.12.1; the CPU notebook uses 2.15.0).
# WHAT THIS CODE ASSUMES: Skala-1.1 (headline) = D3 on, the package default; Skala-1.1-noD3 = the same SCF minus the stored D3 energy; the v4 gate
# (G0, I9) = D3 off, as in v4.  The D3 energy, the D3 settings string and the dispersion flag are written into the "extra" of every Skala row.
# Nothing here falls back silently: if with_dftd3 cannot be set, or no D3 energy comes back, the point fails loudly.
# ==============================================================================
import os, re, io, sys, json, csv, glob, math, time, shutil, tarfile, tempfile, subprocess, contextlib, importlib.util
import numpy as np
try:
    b1v5_common_loaded = COLUMNS  # noqa  (names exist when the notebook cells above ran)
except NameError:
    from b1v5_common import *     # noqa
try:
    b1v5_cpu_loaded = ccsdt       # noqa
except NameError:
    from b1v5_cpu import *        # noqa

# ------------------------------------------------------------------ settings (the notebook's settings cell overwrites these)
STANDIN = False               # True: cheap stand-in functionals and tiny bases on CPU (plumbing test, NOT results)
RETRY_FAILED = False
FORCE_RERUN = False
SYSTEMS_TO_RUN = list(SYSTEM_ORDER)
GPU_NAME, GPU_MEM_GB, GPU_CLASS = "", 0.0, "NONE"
NO_GPU = False
CALC_MAX_SEC = 1200           # one single SCF may not run longer than this (seconds); the point is then logged as failed
CONV_TOL = 1e-9               # SCF convergence (hartree), as in v4
GRID_LEVEL = 3                # DFT integration grid level, as in v4
GPU_QZ_MAX_NBF = 700          # G2: skip a large-basis calculation with more basis functions than this
CPU_FALLBACK_MAX_NBF = 220    # a failed GPU calculation is retried on the CPU only below this size
STOP_ON_G0_FAIL = True        # G0 reproduction of stored v4 energies off by more than 1e-4 hartree: stop the later phases
G4_SYSTEMS = ["hcn", "ch3nc", "sn2_ff"]
G4_OPTIONAL = ["nh3", "h2co"]
G4_DELTA = {"angle": 4.0, "distance": 0.04, "dihedral": 4.0}   # spacing of the three constrained points around the TS coordinate
G4_MAX_SHIFT = 3              # how often the three-point window may move if the top is not in the middle
G4_CONV = "GAU_LOOSE"         # geomeTRIC convergence set for the Skala optimisations
G4_MAXITER = 60
OPT_PATH_SYSTEMS = ["sn2_clcl", "sn2_fcl", "sn2_ohf"]
REGEN_PRIORITY = list(SYSTEM_ORDER)

CSV_REPRO = "phase10_repro.csv"
CSV_OPT = "phase16_optional.csv"
LV_SCANGPU = "own_scan_gpu"   # level name for the optional GPU relaxed scans (B3LYP without dispersion)
M_B3LYP_NODISP = "B3LYP-nodisp"   # used only to relax the optional GPU scans

# Method names and the order in which they are run when time is short (the last ones are cut first: PBE, then r2SCAN)
PRIORITY = [M_SKALA, "ANCHOR", M_B3LYP, M_R2SCAN, M_PBE]

# what G0 learns, saved in phase10_timing.json and reloaded by every later phase
GSTATE = dict(routes={}, anchor=M_WB97MV, gate_ok=None, skala_ok=None, skala_route="gpu", skala_mol_mode="cpu",
              grad_ok=None, grad_variant="plain", loaded=False)
SKALA_NAME = "skala-1.1"      # in skala 2026.9 this is the rev1 file; use "skala-1.1-rev0" to load the original weights file (see header)
DM_CACHE = {}


# ------------------------------------------------------------------ 1. GPU detection and scaling
def detect_gpu():
    """(name, memory in GB, class).  class is A100 (also H100), L4, T4, OTHER (any other GPU) or NONE."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=15).stdout.strip()
    except Exception:
        out = ""
    if not out:
        return "", 0.0, "NONE"
    name, _, mem = out.splitlines()[0].partition(",")
    name = name.strip()
    try:
        mem_gb = float(mem.strip()) / 1024.0
    except Exception:
        mem_gb = 0.0
    up = name.upper()
    if any(k in up for k in ("A100", "H100", "H200", "A800", "GH200")):
        cls = "A100"
    elif re.search(r"\bL4\b", up):
        cls = "L4"
    elif "T4" in up:
        cls = "T4"
    else:
        cls = "OTHER"
    return name, mem_gb, cls


# What each GPU class gets.  T4 (and unknown cards) trim the optional work; point counts of the plan are never reduced.
GPU_PLAN = {
    "A100": dict(optional=True, g4_extra=True, qz_max_nbf=900, calc_max_sec=1500),
    "L4": dict(optional=True, g4_extra=True, qz_max_nbf=700, calc_max_sec=1500),
    "T4": dict(optional=False, g4_extra=False, qz_max_nbf=550, calc_max_sec=1200),
    "OTHER": dict(optional=False, g4_extra=False, qz_max_nbf=550, calc_max_sec=1200),
    "NONE": dict(optional=False, g4_extra=False, qz_max_nbf=300, calc_max_sec=1200),
}


def gpu_available():
    """True when a GPU is present and gpu4pyscf can be imported."""
    return GPU_CLASS != "NONE" and importlib.util.find_spec("gpu4pyscf") is not None


def gpu_cleanup():
    """Free GPU memory between calculations (never raises)."""
    try:
        import gc
        gc.collect()
        try:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
        except Exception:
            pass
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    except Exception:
        pass


class CalcTimeout(Exception):
    """One single calculation took longer than CALC_MAX_SEC."""


@contextlib.contextmanager
def calc_time_limit(sec):
    """Stops one calculation after `sec` seconds (only where the operating system allows it; otherwise no limit)."""
    import signal, threading
    if not sec or threading.current_thread() is not threading.main_thread() or not hasattr(signal, "setitimer"):
        yield
        return

    def _h(signum, frame):
        raise CalcTimeout(f"single calculation longer than {sec} s")
    old = signal.signal(signal.SIGALRM, _h)
    signal.setitimer(signal.ITIMER_REAL, float(sec))
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)


# ------------------------------------------------------------------ 2. functionals
# method name -> PySCF functional string (and dispersion).  The same strings are used on the GPU and the CPU.
XC_SPEC = {
    M_PBE: dict(xc="pbe"),
    M_R2SCAN: dict(xc="r2scan"),
    M_B3LYP: dict(xc="b3lyp", disp="d3bj"),
    M_WB97MV: dict(xc="wb97m_v"),
    M_WB97XV: dict(xc="wb97x_v"),
    M_B3LYP_NODISP: dict(xc="b3lyp"),
    "v4-pbe": dict(xc="pbe"),
    "v4-b3lyp": dict(xc="b3lyp"),                 # v4 used plain B3LYP (no dispersion)
}
# cheap stand-ins (STANDIN = True only): Skala-1.1 -> BLYP, wB97M-V -> wB97X (no non-local term)
STANDIN_SPEC = {
    M_SKALA: dict(xc="blyp", disp="d3bj", d3xc="b3lyp5"),        # the stand-in carries a D3 term too, so the with / without D3 plumbing is tested
    M_PBE: dict(xc="pbe"), M_R2SCAN: dict(xc="r2scan"), M_B3LYP: dict(xc="b3lyp", disp="d3bj"),
    M_WB97MV: dict(xc="wb97x"), M_WB97XV: dict(xc="wb97x"), M_B3LYP_NODISP: dict(xc="b3lyp"),
    "v4-pbe": dict(xc="pbe"), "v4-b3lyp": dict(xc="b3lyp"), "v4-skala": dict(xc="blyp"),
}
V4_BASE = {"v4-pbe": M_PBE, "v4-b3lyp": M_B3LYP, "v4-skala": M_SKALA}   # which method's route a v4 method follows
V4_KEY = {"v4-ccsd_t": "ccsd_t", "v4-pbe": "pbe", "v4-b3lyp": "b3lyp", "v4-skala": "skala"}


def is_skala(method):
    return method in (M_SKALA, "v4-skala") and not STANDIN


def spec_of(method):
    return (STANDIN_SPEC if STANDIN else XC_SPEC)[method]


def route_of(method):
    """'gpu', 'cpu', 'standin' or 'none' for a method (from G0; the GPU when G0 has not said anything)."""
    if STANDIN:
        return "standin"
    m = V4_BASE.get(method, method)
    if m == M_SKALA:
        return GSTATE.get("skala_route", "gpu") if GSTATE.get("skala_ok") is not False else "none"
    r = GSTATE["routes"].get(m)
    if r:
        return r
    return "gpu" if gpu_available() else "cpu"


def d3_energy(atoms, charge, basis, xc, version="d3bj"):
    """DFT-D3(BJ) dispersion energy (hartree), computed with the PySCF dispersion package on the CPU (two-body, atm=False: the same call that
    Skala uses; xc is only the NAME of the damping-parameter set, 'b3lyp5' for Skala and 'b3lyp' for B3LYP-D3(BJ) give the same energy)."""
    from pyscf.dispersion.dftd3 import DFTD3Dispersion
    mol = make_mol(atoms, charge, basis)
    return float(DFTD3Dispersion(mol, xc, version=version, atm=False).get_dispersion()["energy"])


def _to_np(x):
    return x.get() if hasattr(x, "get") else np.asarray(x)


def _build_skala(mol, route, df=False, variant="plain", d3=True):
    """Skala-1.1 through its PySCF bindings (GPU: skala.gpu4pyscf, CPU: skala.pyscf).
    d3=True: with its D3 term (the package default and the published functional; headline).  d3=False: dispersion off, ONLY for the v4 gate.
    The keyword with_dftd3 is always passed explicitly, so the setting never depends on a package default; an older signature without it
    raises an error (review row 2: no silent fallback that runs whatever the default is)."""
    os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")      # PyTorch >= 2.6 loading fix used in the v7 optimiser notebook
    if route == "gpu":
        from skala.gpu4pyscf import SkalaKS
    else:
        from skala.pyscf import SkalaKS
    m = mol.to_gpu() if (route == "gpu" and (variant == "v7" or GSTATE.get("skala_mol_mode") == "gpu")) else mol
    try:
        mf = SkalaKS(m, xc=SKALA_NAME, with_dftd3=bool(d3))
    except TypeError as ex:
        raise RuntimeError("this skala version does not accept with_dftd3: refusing to guess the dispersion setting "
                           f"(install the pinned skala 2026.9; original error: {ex!r})") from ex
    if d3 and getattr(mf, "with_dftd3", None) is None:
        raise RuntimeError("Skala was asked for its D3 term but built none (no 'expected_d3_settings' in the weights?): refusing to label the row 'with D3'")
    if variant == "v7":      # the settings of the tested v7 optimiser notebook (density fitting, full grid, no small-density cut)
        from pyscf import dft
        mf.grids.level = GRID_LEVEL
        mf.grids.prune = dft.gen_grid.nwchem_prune
        mf.small_rho_cutoff = 0.0
        mf.max_memory = 2000
        mf = mf.density_fit(auxbasis="def2-universal-jkfit")
        mf.init_guess = "minao"
    elif df:
        mf = mf.density_fit()
    return mf


def skala_d3_info(mf):
    """(D3 energy in hartree or None, name of the D3 parameter set or None) of a finished Skala calculation."""
    d3 = None
    try:
        d3 = mf.scf_summary.get("dispersion")
    except Exception:
        pass
    w = getattr(mf, "with_dftd3", None)
    if d3 is None and w:
        try:
            d3 = w.get_energy()
        except Exception:
            d3 = None
    return (None if d3 is None else float(d3)), (getattr(w, "_xc", None) if w else None)


def _build_mf(method, mol, route, df=False, variant="plain"):
    from pyscf import dft
    if is_skala(method):
        return _build_skala(mol, route, df, variant, d3=(method != "v4-skala"))
    mf = dft.RKS(mol, xc=spec_of(method)["xc"])
    mf.grids.level = GRID_LEVEL
    if route == "gpu":
        mf = mf.to_gpu()
    if df:
        mf = mf.density_fit()
    return mf


def _scf_once(method, atoms, charge, basis, route, dm0=None, df=False, level_shift=0.0, max_cycle=100, variant="plain"):
    """One SCF.  Returns dict(energy, converged, dm)."""
    mol = make_mol(atoms, charge, basis)
    mf = _build_mf(method, mol, route, df, variant)
    mf.conv_tol = CONV_TOL
    mf.max_cycle = max_cycle
    if level_shift:
        mf.level_shift = level_shift
    with calc_time_limit(CALC_MAX_SEC):
        e = float(mf.kernel(dm0=dm0)) if dm0 is not None else float(mf.kernel())
    conv = bool(getattr(mf, "converged", True))
    dm = None
    try:
        dm = _to_np(mf.make_rdm1())
    except Exception:
        pass
    d3, d3xc = skala_d3_info(mf) if is_skala(method) else (None, None)
    del mf
    return dict(energy=e, converged=conv, dm=dm, nbf=int(mol.nao), d3=d3, d3xc=d3xc)


def energy_of(method, atoms, charge, basis, dm_key=None, variant="plain", force_df=False):
    """Energy of one geometry with one method, with the retry ladder of plan section 5:
    warm start from the neighbouring density, fresh start, level shift, density fitting, then the CPU (not for Skala).
    force_df=True runs ONLY the density-fitted calculation (used to recompute the partner species of a barrier whose other species needed the
    density-fitting retry, review row 11).  Skala: the energy includes its D3 term; the D3 energy is returned in extra['d3_hartree'].
    Returns dict(energy, converged, extra).  Raises only if every attempt crashed."""
    route = route_of(method)
    if route == "none":
        raise RuntimeError(f"{method} is not available (see G0)")
    dm0 = DM_CACHE.get(dm_key) if dm_key else None
    nao = None
    if dm0 is not None:
        try:
            nao = make_mol(atoms, charge, basis).nao
            if dm0.shape != (nao, nao):
                dm0 = None
        except Exception:
            dm0 = None
    attempts = []
    if force_df:
        attempts.append(("density fitting", dict(df=True)))
        attempts.append(("density fitting + level shift", dict(df=True, level_shift=0.3, max_cycle=300)))
    else:
        if dm0 is not None:
            attempts.append(("warm start", dict(dm0=dm0)))
        attempts.append(("default", {}))
        attempts.append(("level shift", dict(level_shift=0.3, max_cycle=300)))
        if route == "gpu":
            attempts.append(("density fitting", dict(df=True)))
            if not is_skala(method):
                attempts.append(("cpu", dict(route="cpu")))
        if is_skala(method):
            # rescue for a Skala SCF that does not converge (seen on the Cl- + CH3Cl TS in the first Colab run): start from the converged PBE density
            attempts.append(("PBE start + level shift", dict(rescue=True, level_shift=0.3, max_cycle=300)))
            attempts.append(("PBE start + level shift 0.5 + density fitting", dict(rescue=True, df=True, level_shift=0.5, max_cycle=400)))
    best, last_err, tried = None, None, []
    for name, kw in attempts:
        kw = dict(kw)
        r_route = kw.pop("route", route)
        if kw.pop("rescue", False):
            try:
                pr = _scf_once(M_PBE, atoms, charge, basis, r_route)
                if not pr["converged"] or pr["dm"] is None:
                    tried.append(name + " skipped: PBE start did not converge")
                    continue
                kw["dm0"] = pr["dm"]
            except PhaseTimeout:
                raise
            except Exception as ex:
                tried.append(name + " skipped: PBE start crashed " + repr(ex)[:60])
                gpu_cleanup()
                continue
        if r_route == "cpu" and route == "gpu":
            if nao is None:
                nao = make_mol(atoms, charge, basis).nao
            if nao > CPU_FALLBACK_MAX_NBF:
                continue
        check_time()
        try:
            res = _scf_once(method, atoms, charge, basis, r_route, variant=variant, **kw)
            tried.append(name)
            if res["converged"]:
                best = dict(res, attempt=name, route=r_route, df=bool(kw.get("df")))
                break
            if best is None:
                best = dict(res, attempt=name + " (not converged)", route=r_route, df=bool(kw.get("df")))
        except PhaseTimeout:
            raise
        except Exception as ex:
            last_err = ex
            tried.append(name + " crashed: " + repr(ex)[:80])
            gpu_cleanup()
    gpu_cleanup()
    if best is None:
        raise last_err if last_err else RuntimeError("no SCF attempt possible")
    e = best["energy"]
    sp = {} if is_skala(method) else spec_of(method)
    d3_hartree, d3_set = None, None
    if is_skala(method):
        if method != "v4-skala":                                  # D3 on: the energy already contains it; it must have come back
            if best.get("d3") is None:
                raise RuntimeError("Skala returned no D3 energy although with_dftd3=True: refusing to store the row as 'with D3'")
            d3_hartree, d3_set = best["d3"], best.get("d3xc")
    elif sp.get("disp") and np.isfinite(e):
        d3_hartree = d3_energy(atoms, charge, basis, sp.get("d3xc", sp["xc"]), sp["disp"])
        d3_set = sp.get("d3xc", sp["xc"])
        e += d3_hartree
    if dm_key and best["converged"] and best["dm"] is not None:
        DM_CACHE[dm_key] = best["dm"]
    extra = dict(route=best["route"], attempt=best["attempt"], nbf=best["nbf"], df=best["df"])
    if d3_hartree is not None:
        extra.update(d3_hartree=float(d3_hartree), d3_set=d3_set, d3_version="d3bj, two-body (atm=False)")
    if method in (M_SKALA, "v4-skala"):
        extra.update(with_dftd3=bool(method != "v4-skala"), skala_name=SKALA_NAME, weights=GSTATE.get("weights_short", ""))
    if len(tried) > 1:
        extra["tried"] = tried
    if STANDIN:
        extra["standin"] = True
    return dict(energy=float(e), converged=bool(best["converged"]), extra=extra)


def reset_dm(prefix=None):
    """Forget stored densities (call at the start of every new path)."""
    for k in list(DM_CACHE):
        if prefix is None or k[0] == prefix:
            DM_CACHE.pop(k, None)


# ------------------------------------------------------------------ energies and gradients for the optimiser
class EG:
    """Energy and gradient of one method for one molecule (charge and basis fixed); used by geomeTRIC.
    eg(atoms) -> (energy in hartree, gradient in hartree/bohr as a flat array); energy(atoms) -> dict(energy, converged)."""

    def __init__(self, method, charge, basis, variant="plain"):
        self.method, self.charge, self.basis, self.variant = method, charge, basis, variant
        self.dm0, self.n_grad, self.n_energy = None, 0, 0

    def _run(self, atoms, want_grad):
        route = route_of(self.method)
        if route == "none":
            raise RuntimeError(f"{self.method} is not available")
        mol = make_mol(atoms, self.charge, self.basis)
        last = None
        for kw in ({"dm0": self.dm0}, {}, {"level_shift": 0.3, "max_cycle": 300}):
            if kw.get("dm0", 0) is None:
                continue
            mf = _build_mf(self.method, mol, route, False, self.variant)
            mf.conv_tol = CONV_TOL * (0.1 if want_grad else 1.0)
            mf.max_cycle = kw.get("max_cycle", 100)
            if kw.get("level_shift"):
                mf.level_shift = kw["level_shift"]
            with calc_time_limit(CALC_MAX_SEC):
                e = float(mf.kernel(dm0=kw["dm0"]) if "dm0" in kw else mf.kernel())
            if getattr(mf, "converged", True):
                try:
                    self.dm0 = _to_np(mf.make_rdm1())
                except Exception:
                    self.dm0 = None
                g = None
                if want_grad:
                    with calc_time_limit(CALC_MAX_SEC):
                        g = np.asarray(_to_np(mf.nuc_grad_method().kernel()), float).ravel()
                gpu_cleanup()
                return e, g
            last = e
            gpu_cleanup()
        raise RuntimeError(f"SCF did not converge in the gradient engine (last energy {last})")

    def energy(self, atoms):
        check_time()
        self.n_energy += 1
        e, _ = self._run(atoms, False)
        return dict(energy=e, converged=True)

    def eg(self, atoms):
        check_time()
        self.n_grad += 1
        e, g = self._run(atoms, True)
        return e, g


def opt_custom(atoms, charge, eg, constraints=None, convergence_set="GAU_LOOSE", maxiter=60):
    """Geometry optimisation (minimum) with geomeTRIC, energies and gradients from `eg(atoms)`.
    Returns dict(atoms, converged, n_steps, error).  When the optimiser gives up, the last geometry it tried is returned."""
    if len(atoms) == 1:
        return dict(atoms=atoms, converged=True, n_steps=0, error="")
    import geometric.optimize as go
    import geometric.engine as ge, geometric.molecule as gm
    quiet_geometric()

    class _Engine(ge.Engine):
        def __init__(self):
            M = gm.Molecule()
            M.elem = [s for s, _ in atoms]
            M.xyzs = [np.array([c for _, c in atoms])]
            super().__init__(M)
            self.n, self.last = 0, None

        def calc_new(self, coords, dirname):
            check_time()                           # lets the phase watchdog stop an optimisation between two gradients
            self.n += 1
            at = [(s, tuple(c * BOHR)) for (s, _), c in zip(atoms, coords.reshape(-1, 3))]
            self.last = at
            e, g = eg(at)
            return {"energy": e, "gradient": np.asarray(g, float).ravel()}
    eng = _Engine()
    err = ""
    with tempfile.TemporaryDirectory() as d:
        kw = dict(customengine=eng, input=os.path.join(d, "x"), maxiter=maxiter, convergence_set=convergence_set, transition=False)
        if constraints:
            cp = os.path.join(d, "cons.txt")
            open(cp, "w").write(constraint_text(constraints))
            kw["constraints"] = cp
        try:
            M = go.run_optimizer(**kw)
            conv = True
        except PhaseTimeout:
            raise
        except Exception as ex:
            conv, M, err = False, None, repr(ex)[:160]
    if M is not None:
        new = [(s, tuple(map(float, c))) for s, c in zip(M.elem, M.xyzs[-1])]
    else:
        new = eng.last or atoms
    return dict(atoms=new, converged=conv, n_steps=eng.n, error=err)


def fd_gradient_check(method, atoms, charge, basis, variant="plain", h=0.0025, n_comp=2):
    """Analytic gradient against a central finite difference of the energy (same engine), on the two largest components.
    Returns dict(ratio list, net_gradient, ok, status).  ratio = analytic / finite difference."""
    eng = EG(method, charge, basis, variant)
    e0, g = eng.eg(atoms)
    g = g.reshape(-1, 3)
    order = np.argsort(-np.abs(g).ravel())
    picks, seen_atoms = [], set()
    for k in order:                                    # largest components, on different atoms when possible
        i, c = divmod(int(k), 3)
        if i in seen_atoms and len(seen_atoms) < len(atoms):
            continue
        picks.append((i, c)); seen_atoms.add(i)
        if len(picks) == n_comp:
            break
    ratios, rows = [], []
    for i, c in picks:
        ep, em = [], []
        for sgn, store in ((+1, ep), (-1, em)):
            at = [(s, tuple(np.array(x) + (sgn * h if (j == i) else 0.0) * np.eye(3)[c])) for j, (s, x) in enumerate(atoms)]
            store.append(EG(method, charge, basis, variant).energy(at)["energy"])
        fd = (ep[0] - em[0]) / (2 * h / BOHR)          # hartree per bohr
        an = float(g[i, c])
        r = an / fd if abs(fd) > 1e-7 else float("nan")
        ratios.append(r); rows.append(dict(atom=int(i), xyz="xyz"[c], analytic=an, finite_difference=float(fd), ratio=r))
    net = float(np.linalg.norm(g.sum(axis=0)))
    worst = max([abs(r - 1.0) for r in ratios if np.isfinite(r)] or [9.9])
    status = "PASS" if (worst <= 0.05 and net <= 1e-3) else ("WARN" if (worst <= 0.10 and net <= 3e-3) else "FAIL")
    return dict(components=rows, net_gradient=net, worst_ratio_deviation=float(worst), status=status, ok=status in ("PASS", "WARN"),
                energy=float(e0), variant=variant)


# ------------------------------------------------------------------ Skala weights on Drive
def hf_hub_dir():
    home = os.environ.get("HF_HOME") or os.path.join(os.path.expanduser("~"), ".cache", "huggingface")
    return os.environ.get("HF_HUB_CACHE") or os.path.join(home, "hub")


def restore_weights_cache():
    """Unpack the Skala weights from Drive (saved by an earlier session) when this session has none yet."""
    tgz, hub = path("skala_weights_cache.tgz"), hf_hub_dir()
    if os.path.exists(tgz) and not glob.glob(os.path.join(hub, "models--*kala*")):
        try:
            os.makedirs(hub, exist_ok=True)
            with tarfile.open(tgz) as t:
                t.extractall(hub)
            return "restored from Drive"
        except Exception as ex:
            return "could not restore: " + repr(ex)[:80]
    return "nothing to restore" if not os.path.exists(tgz) else "already present locally"


def save_weights_cache():
    """Copy the Skala weights (Hugging Face cache) to Drive once, so later sessions need no download."""
    tgz, hub = path("skala_weights_cache.tgz"), hf_hub_dir()
    dirs = glob.glob(os.path.join(hub, "models--*kala*"))
    if not dirs:
        return "no Hugging Face cache for Skala found (weights may ship inside the package)"
    if os.path.exists(tgz):
        return "already on Drive"
    try:
        with tarfile.open(tgz + ".tmp", "w:gz") as t:
            for d in dirs:
                t.add(d, arcname=os.path.basename(d))
        os.replace(tgz + ".tmp", tgz)
        return f"saved {os.path.getsize(tgz) / 1e6:.0f} MB to Drive"
    except Exception as ex:
        return "could not save: " + repr(ex)[:80]


def skala_weights_info():
    """Review row 10: which weights file is used.  Returns SHA-256 of the file, its name, the Hugging Face revision (the snapshot folder name
    of the download), the SHA-256 the skala package expects, and the D3 parameter-set name stored in the file ('expected_d3_settings')."""
    import hashlib, zipfile
    info = dict(skala_name=SKALA_NAME, standin=STANDIN)
    if STANDIN:
        info["note"] = "stand-in: no Skala weights are used"
        return info
    p, expected = None, None
    try:
        from skala.functional import resolve_functional_artifact
        art = resolve_functional_artifact(SKALA_NAME)
        p, expected = str(art.path), getattr(art, "expected_hash", None)
    except Exception as ex:
        info["resolve_error"] = repr(ex)[:160]
        cand = sorted(glob.glob(os.path.join(hf_hub_dir(), "models--microsoft--skala-*", "snapshots", "*", "*.fun")), key=os.path.getmtime)
        p = cand[-1] if cand else None
    if not p or not os.path.exists(p):
        info["note"] = "weights file not found"
        return info
    info["file"] = os.path.basename(os.path.realpath(p))
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    info["sha256"] = h.hexdigest()
    info["package_expected_sha256"] = expected
    info["hash_matches_package_table"] = (None if expected is None else expected == info["sha256"])
    m = re.search(r"snapshots[/\\]([0-9a-f]{40})", p)
    info["huggingface_revision"] = m.group(1) if m else "unknown (file not in a Hugging Face snapshot folder)"
    info["huggingface_repo"] = "microsoft/" + SKALA_NAME.partition("-rev")[0]
    try:
        with zipfile.ZipFile(p) as z:
            nm = [n for n in z.namelist() if n.endswith("extra/expected_d3_settings")]
            info["expected_d3_settings"] = z.read(nm[0]).decode("utf-8").strip() if nm else "not found in file"
    except Exception as ex:
        info["expected_d3_settings"] = "could not read: " + repr(ex)[:80]
    GSTATE["weights_short"] = info["sha256"][:12]
    return info


# ------------------------------------------------------------------ saved state of G0
def save_gstate(**extra):
    d = dict(routes=GSTATE["routes"], anchor=GSTATE["anchor"], gate_ok=GSTATE["gate_ok"], skala_ok=GSTATE["skala_ok"],
             skala_route=GSTATE["skala_route"], skala_mol_mode=GSTATE["skala_mol_mode"], grad_ok=GSTATE["grad_ok"],
             grad_variant=GSTATE["grad_variant"], weights_short=GSTATE.get("weights_short", ""))
    old = read_json("phase10_timing.json", {}) or {}
    old.update(d)
    old.update(extra)
    write_json("phase10_timing.json", old)


def run_gphase(name, title, fn, max_min, force=None, blocked=None):
    """Run one phase of the notebook: skip it with a clear message when it must not start, otherwise guard it (errors are logged, never raised)."""
    why = blocked if blocked is not None else phase_blocked(name)
    if why:
        skip_phase(name, title, why)
        return
    with phase(name, title, max_min=max_min, force=(FORCE_RERUN if force is None else force)) as ph:
        if not ph.skip:
            fn()


def load_gstate():
    """Read what G0 learned (routes per functional, anchor, gates).  Silent when G0 has not run."""
    st = read_json("phase10_timing.json", None)
    if not st:
        return False
    for k in ("routes", "anchor", "gate_ok", "skala_ok", "skala_route", "skala_mol_mode", "grad_ok", "grad_variant", "weights_short"):
        if k in st and st[k] is not None:
            GSTATE[k] = st[k]
    GSTATE["loaded"] = True
    return True


def anchor_method():
    return GSTATE.get("anchor") or M_WB97MV


def ladder_methods(stage="stationary"):
    """The functionals to run, in priority order.  stage 'path' leaves out CPU-routed ones (CPU is for stationary points only)."""
    out = []
    for m in PRIORITY:
        m = anchor_method() if m == "ANCHOR" else m
        r = route_of(m)
        if r == "none":
            continue
        if stage == "path" and r == "cpu":
            continue
        out.append(m)
    return out


def phase_blocked(name):
    """Reason (text) why a GPU phase must not start, or None.  Used before `with phase(...)`."""
    if name == "phase10":                                    # G0 always runs: with no GPU it is the diagnostics-only phase
        return None
    if NO_GPU and not STANDIN:
        return "no GPU detected: only the G0 diagnostics run (Runtime > Change runtime type > GPU, then Run all again)"
    load_gstate()
    if name in ("phase11", "phase11p", "phase12", "phase13", "phase14", "phase16") and GSTATE.get("gate_ok") is False and STOP_ON_G0_FAIL:
        return "G0 reproduction of the stored v4 energies failed (difference over 1e-4 hartree): old and new numbers must not be mixed. Fix the install, delete DONE_phase10 and run again (or set STOP_ON_G0_FAIL = False)"
    if name == "phase14":
        if GSTATE.get("skala_ok") is False:
            return "Skala is not usable (see G0): G4 skipped"
        if GSTATE.get("grad_ok") is not True:
            return "the Skala gradient check of G0 did not pass (or did not run): G4 skipped and the limit is stated in the summary"
    return None


# ------------------------------------------------------------------ geometry helpers shared by the phases
def g_save_path(system, pts, label="", suffix="path", level=LV_SCAN):
    p = path("geoms", f"{system}_{suffix}.json")
    d = dict(system=system, level=level, basis=basis_for(system, "main"), label=label, saved=now(),
             points=[dict(index=int(q["index"]), coordinate=float(q["coordinate"]), atoms=[[s, list(map(float, c))] for s, c in q["atoms"]],
                          energy=(None if q.get("energy") is None else float(q["energy"])), converged=bool(q.get("converged", True))) for q in pts])
    json.dump(d, open(p + ".tmp", "w"), indent=1)
    os.replace(p + ".tmp", p)


def g_load_path(system, suffix="path"):
    p = path("geoms", f"{system}_{suffix}.json")
    if not os.path.exists(p):
        return []
    try:
        return [dict(q, atoms=[(s, tuple(c)) for s, c in q["atoms"]]) for q in json.load(open(p))["points"]]
    except Exception:
        return []


def g_expected_points(system):
    if system == "ch4":
        return len(CH4_STRETCH)
    n = SCANS[system]["n"]
    return len(sorted(set(range(0, n, SCAN_STRIDE)) | {n - 1}))


def g_scout_species(system):
    S = SYSTEMS[system]
    pmin = S["pc"] or (S["products"][0] if len(S["products"]) == 1 else None)
    return [S["rc"] or S["reactants"][0], "TS", pmin]


def g_levels(system, sp):
    """Geometry levels stored for a species, preferred first."""
    return [lv for lv in LEVEL_PREFERENCE if geometry_of(system, sp, lv)]


# ------------------------------------------------------------------ G-1: rebuild what the CPU notebook did not leave (same helpers, budget-limited)
def g_missing_inputs(systems):
    miss = dict(stationary=[], paths=[], xtb=[], tiers=not os.path.exists(path("tiers.json")))
    for s in systems:
        for sp, kind in SPECIES_KIND[s].items():
            if geometry_of(s, sp, LV_OWN) is None and geometry_of(s, sp, LV_SCANPEAK) is None:      # our own B3LYP geometry is the headline protocol
                miss["stationary"].append((s, sp, kind))
        if s == "ch4" or (s in SCANS and SCANS[s].get("dft", True)):
            if len(g_load_path(s)) < g_expected_points(s):
                miss["paths"].append(s)
        if s in REACTIONS:
            names = [geom_name(s, x, LV_XTB) for x in g_scout_species(s) if x]
            if any(load_geom(n) is None for n in names):
                miss["xtb"].append(s)
    return miss


def g_regen_stationary(system, sp, kind):
    """Same as C1 of the CPU notebook: optimise one species (minimum or transition state), frequencies, CSV row."""
    ch, basis = species_charge(system, sp), basis_for(system, "main")
    name = geom_name(system, sp, LV_OWN)
    g = load_geom(name)
    if g is None:
        t0 = time.time()
        at = start_atoms(system, sp)
        if len(at) > 1:
            r = opt_dft(at, ch, basis, transition=(kind == "ts"), hessian=(kind == "ts"))
            at, conv = r["atoms"], r["converged"]
        else:
            conv = True
        save_geom(name, at, ch, species=sp, kind=kind, level=LV_OWN, opt_converged=bool(conv), basis=basis, seconds=round(time.time() - t0, 1), regenerated_by="GPU notebook G-1")
        g = load_geom(name)

    def calc():
        at, a, retried = refine_stationary(g["atoms"], ch, basis, kind)
        if retried and at is not g["atoms"]:
            save_geom(name, at, ch, species=sp, kind=kind, level=LV_OWN, opt_converged=True, basis=basis, retried=True)
        a["converged"] = bool(a["converged"] and g.get("opt_converged", True))
        a.pop("imag_modes", None)
        a["extra"]["retried"] = bool(retried)
        return a
    run_point(PHASE_CSV["phase01"], system, sp, LV_OWN, M_DFB3LYP, basis, calc, retry_failed=RETRY_FAILED)


def g_regen_scan(system):
    """Same as C2 of the CPU notebook: one relaxed scan (geometry file plus one CSV row per point)."""
    if system == "ch4":
        g_save_path("ch4", [dict(index=i, coordinate=float(s), atoms=ch4_geometry(float(s)), energy=None, converged=True) for i, s in enumerate(CH4_STRETCH)], "C-H stretch (A)")
        return
    sp = SCANS[system]
    g0 = geometry_of(system, sp["start"], LV_OWN) or best_geom(system, sp["start"])[1]
    if g0 is None:
        raise RuntimeError(f"missing start geometry {sp['start']} for the scan of {system}")
    ch, basis = species_charge(system, sp["start"]), basis_for(system, "main")
    store = g_load_path(system)
    try:
        pts = scan_points_dft(system, g0["atoms"], ch, basis, store, lambda pts, system=system, sp=sp: g_save_path(system, pts, sp["label"]))
    finally:
        for q in g_load_path(system):
            run_point(PHASE_CSV["phase02"], system, f"p{q['index']:02d}", LV_SCAN, M_DFB3LYP, basis_for(system, "main"),
                      lambda q=q: dict(energy=q["energy"], converged=q["converged"]), coordinate=f"{q['coordinate']:.4f}", quiet=True)
    if best_geom(system, "TS")[1] is None:           # TS search failed: the top of the scan becomes the TS (plan section 5)
        k = peak_of(pts)
        if k is None:
            raise RuntimeError("no peak along the scan and no TS")
        save_geom(geom_name(system, "TS", LV_SCANPEAK), pts[k]["atoms"], ch, species="TS", kind="ts", level=LV_SCANPEAK, from_scan_point=k)


def g_regen_xtb(system):
    """Same as the geometry part of C4: GFN2-xTB minima, scan peak and product (no CCSD(T) here)."""
    csv_ = PHASE_CSV["phase04"]
    rmin, _, pmin = g_scout_species(system)
    ch = species_charge(system, rmin)
    names = {sp: geom_name(system, sp, LV_XTB) for sp in (rmin, "TS", pmin) if sp}
    lv, g = best_geom(system, rmin)
    a0 = g["atoms"] if g else start_atoms(system, rmin)
    r = opt_xtb(a0, ch)
    save_geom(names[rmin], r["atoms"], ch, species=rmin, level=LV_XTB, converged=r["converged"])
    if pmin:
        lv, g = best_geom(system, pmin)
        r2 = opt_xtb(g["atoms"] if g else start_atoms(system, pmin), species_charge(system, pmin))
        save_geom(names[pmin], r2["atoms"], species_charge(system, pmin), species=pmin, level=LV_XTB, converged=r2["converged"])
    sp_ = SCANS[system]
    if sp_["start"] == rmin:
        g0 = r["atoms"]
    else:
        lv0, gg = best_geom(system, sp_["start"])
        g0 = gg["atoms"] if gg else start_atoms(system, sp_["start"])
    pts = scan_points_xtb(system, g0, species_charge(system, sp_["start"]))
    k = peak_of(pts)
    if k is None:
        save_geom(names["TS"], pts[len(pts) // 2]["atoms"], ch, species="TS", level=LV_XTB, no_peak=True)
        append_row(csv_, make_row(system, "TS", LV_XTB, M_XTB, "GFN2", float("nan"), False, extra={"no_peak": True, "scan_max_index": int(np.nanargmax([q["energy"] for q in pts]))}))
    else:
        save_geom(names["TS"], pts[k]["atoms"], ch, species="TS", level=LV_XTB, scan_point=k)
    for sp, nm in names.items():
        gg = load_geom(nm)
        if gg and not gg.get("no_peak"):
            run_point(csv_, system, sp, LV_XTB, M_XTB, "GFN2", lambda gg=gg, sp=sp: xtb_energy(gg["atoms"], species_charge(system, sp)), retry_failed=RETRY_FAILED)


def g_regen():
    """Phase G-1: find what is missing, rebuild it with the CPU helpers, and say so."""
    systems = [s for s in REGEN_PRIORITY if s in SYSTEMS_TO_RUN]
    miss = g_missing_inputs(systems)
    n = len(miss["stationary"]) + len(miss["paths"]) + len(miss["xtb"])
    print(f"  missing from the CPU notebook: {len(miss['stationary'])} stationary geometries, {len(miss['paths'])} paths, {len(miss['xtb'])} xTB scout sets, tiers.json {'missing' if miss['tiers'] else 'present'}")
    lines = []
    if miss["tiers"]:
        write_json("tiers.json", {k: v["tier"] for k, v in SYSTEMS.items()})
        lines.append("tiers.json was missing: plan defaults written (the CPU notebook C0 timing rule was not applied)")
    if n == 0:
        lines.append("nothing missing: CPU inputs found")
    for s, sp, kind in miss["stationary"]:
        with item_guard("phase09", f"{s}:{sp}"):
            print(f"  regenerating geometry {s} {sp} ({kind})")
            g_regen_stationary(s, sp, kind)
            lines.append(f"regenerated stationary geometry {s}:{sp}")
    for s in miss["paths"]:
        with item_guard("phase09", f"{s}:path"):
            print(f"  regenerating relaxed scan {s}")
            g_regen_scan(s)
            lines.append(f"regenerated path {s}")
    for s in miss["xtb"]:
        with item_guard("phase09", f"{s}:xtb"):
            print(f"  regenerating xTB scout geometries {s}")
            g_regen_xtb(s)
            lines.append(f"regenerated xTB scout geometries {s}")
    left = g_missing_inputs(systems)
    left_n = len(left["stationary"]) + len(left["paths"]) + len(left["xtb"])
    lines.append(f"still missing after G-1: {left_n} item(s)" + ("" if not left_n else " (" + ", ".join([f"{a}:{b}" for a, b, _ in left["stationary"]] + [x + ":path" for x in left["paths"]] + [x + ":xtb" for x in left["xtb"]])[:300] + ")"))
    write_json("phase09_regen.json", dict(missing_before=dict(stationary=[f"{a}:{b}" for a, b, _ in miss["stationary"]], paths=miss["paths"], xtb=miss["xtb"], tiers=miss["tiers"]),
                                           log=lines, finished=now()))
    for l in lines:
        print("  G-1:", l)
    return lines


# ------------------------------------------------------------------ G0
def g_v4_series(method, k_list):
    """The NB4 HCN path points (v4 geometry list) with one v4 method; rows go to phase10_repro.csv."""
    basis = "STO-3G" if STANDIN else "def2-TZVP"
    reset_dm("nb4")
    for k in k_list:
        th, r_ = NB4_HCN_GEOMS[k]
        at = hcn_geometry(th, r_)
        if method == "v4-ccsd_t":
            fn = lambda at=at: ccsdt(at, 0, basis, frozen_core=False, stability=False)
        else:
            fn = lambda at=at: energy_of(method, at, 0, basis, dm_key=("nb4", method, basis))
        run_point(CSV_REPRO, "hcn_nb4", f"nb4_{k:02d}", LV_NB4, method, basis, fn, coordinate=f"{th},{r_}", retry_failed=RETRY_FAILED, quiet=(len(k_list) > 1))


def g_gate():
    """Compare every rerun v4 energy with the stored value.  Returns dict(max_diff, per_method, status)."""
    per, worst_d = {}, 0.0
    for r in read_rows(CSV_REPRO):
        key = V4_KEY.get(r["method"])
        if not key or r["energy_hartree"] in ("", "nan"):
            continue
        k = int(r["point_id"].split("_")[-1])
        d = abs(float(r["energy_hartree"]) - NB4_HCN_E[key][k])
        per[r["method"]] = max(per.get(r["method"], 0.0), d)
        worst_d = max(worst_d, d)
    status = "no data" if not per else ("PASS" if worst_d <= 1e-5 else ("WARN" if worst_d <= 1e-4 else "FAIL"))
    return dict(max_diff=worst_d, per_method=per, status=status)


def g_probe_one(method, atoms, basis):
    """Does this functional run on the GPU, and does it agree with the CPU?  Returns a record; route = 'gpu' | 'cpu' | 'none'."""
    from pyscf import dft
    rec = dict(method=method, basis=basis, gpu_ok=False, cpu_ok=False, gpu_seconds=None, cpu_seconds=None, gpu_cpu_diff=None, error_gpu="", error_cpu="")
    sp = spec_of(method)
    e_cpu = None
    t0 = time.time()
    try:                                                   # CPU reference, native PySCF (with the dispersion switch for B3LYP)
        mf = dft.RKS(make_mol(atoms, 0, basis), xc=sp["xc"])
        mf.grids.level = GRID_LEVEL
        mf.conv_tol = 1e-9
        if sp.get("disp"):
            mf.disp = sp["disp"]
        with calc_time_limit(CALC_MAX_SEC):
            e_cpu = float(mf.kernel())
        rec["cpu_ok"] = bool(mf.converged) and np.isfinite(e_cpu)
        rec["cpu_seconds"] = round(time.time() - t0, 1)
        rec["e_cpu"] = e_cpu
    except Exception as ex:
        rec["error_cpu"] = repr(ex)[:150]
    if gpu_available() and not STANDIN:
        t0 = time.time()
        try:
            res = _scf_once(method, atoms, 0, basis, "gpu")
            e = res["energy"] + (d3_energy(atoms, 0, basis, sp["xc"], sp["disp"]) if sp.get("disp") else 0.0)
            rec["gpu_seconds"] = round(time.time() - t0, 1)
            rec["e_gpu"] = e
            if e_cpu is not None:
                rec["gpu_cpu_diff"] = abs(e - e_cpu)
                rec["gpu_ok"] = bool(res["converged"]) and abs(e - e_cpu) < 5e-5
            else:
                rec["gpu_ok"] = bool(res["converged"])
        except Exception as ex:
            rec["error_gpu"] = repr(ex)[:150]
        gpu_cleanup()
    else:
        rec["error_gpu"] = "no GPU / gpu4pyscf in this session" if not STANDIN else "stand-in: GPU not used"
    rec["route"] = "gpu" if rec["gpu_ok"] else ("cpu" if rec["cpu_ok"] else "none")
    return rec


def g_probe_functionals():
    """Plan section 5: which functionals work on the GPU.  Sets GSTATE['routes'] and GSTATE['anchor']."""
    atoms, basis = start_atoms("hcn", "HCN"), ("STO-3G" if STANDIN else "def2-SVP")
    recs = {}
    for m in (M_PBE, M_R2SCAN, M_B3LYP, M_WB97MV):
        with item_guard("phase10", f"probe {m}"):
            recs[m] = g_probe_one(m, atoms, basis)
            r = recs[m]
            print(f"  probe {m:11s}: GPU {'ok' if r['gpu_ok'] else 'NO'} ({r['gpu_seconds']} s, difference to CPU {r['gpu_cpu_diff']}), CPU {'ok' if r['cpu_ok'] else 'NO'} ({r['cpu_seconds']} s) -> route {r['route']}"
                  + (f"  [{r['error_gpu']}]" if r["error_gpu"] and not r["gpu_ok"] else ""))
    routes = {m: r["route"] for m, r in recs.items()}
    anchor = M_WB97MV
    if routes.get(M_WB97MV, "none") == "none":                # fallback of plan section 5: wB97X-V
        with item_guard("phase10", "probe wB97X-V"):
            recs[M_WB97XV] = g_probe_one(M_WB97XV, atoms, basis)
            routes[M_WB97XV] = recs[M_WB97XV]["route"]
            anchor = M_WB97XV if routes[M_WB97XV] != "none" else M_WB97MV
            print(f"  wB97M-V does not run: the anchor becomes {anchor} (route {routes[M_WB97XV]})")
    elif routes[M_WB97MV] == "cpu" and not STANDIN:
        print("  wB97M-V runs on the CPU only: it will be computed at stationary points only (plan section 5)")
    for m in (M_PBE, M_R2SCAN, M_B3LYP):
        if routes.get(m) == "cpu" and not STANDIN:
            print(f"  {m} runs on the CPU only: stationary points only (plan section 5)")
    if STANDIN:                                               # the stand-in runs everything on the CPU, paths included
        routes = {m: "standin" for m in list(routes) + [M_WB97XV]}
    GSTATE["routes"], GSTATE["anchor"] = routes, anchor
    return recs


def g_skala_check():
    """Does Skala run, on the GPU (else the CPU)?  Sets skala_route / skala_ok (preliminary) and returns a record."""
    atoms, basis = start_atoms("hcn", "HCN"), ("STO-3G" if STANDIN else "def2-SVP")
    rec = dict(basis=basis, tried=[])
    if STANDIN:
        GSTATE["skala_route"], GSTATE["skala_ok"] = "standin", True
        t0 = time.time()
        r = energy_of(M_SKALA, atoms, 0, basis)
        rec.update(route="standin", energy=r["energy"], seconds=round(time.time() - t0, 1), note="stand-in functional, not Skala")
        return rec
    for route, mode in (("gpu", "cpu"), ("gpu", "gpu"), ("cpu", "cpu")):
        if route == "gpu" and not gpu_available():
            continue
        GSTATE["skala_route"], GSTATE["skala_mol_mode"], GSTATE["skala_ok"] = route, mode, True
        t0 = time.time()
        try:
            res = _scf_once(M_SKALA, atoms, 0, basis, route)
            gpu_cleanup()
            if res["converged"] and np.isfinite(res["energy"]):
                rec.update(route=route, mol_mode=mode, energy=res["energy"], seconds=round(time.time() - t0, 1))
                print(f"  Skala runs: route {route} (molecule object: {mode}), E = {res['energy']:.8f}, {rec['seconds']} s")
                return rec
            rec["tried"].append(f"{route}/{mode}: not converged")
        except PhaseTimeout:
            raise
        except Exception as ex:
            rec["tried"].append(f"{route}/{mode}: {repr(ex)[:140]}")
            gpu_cleanup()
    GSTATE["skala_ok"] = False
    print("  Skala does NOT run in this session:", rec["tried"])
    return rec


def g_skala_gradient_check():
    """Plan G0: one Skala gradient against finite differences.  If the plain engine fails, the settings of the tested v7 optimiser notebook are tried."""
    basis = "STO-3G" if STANDIN else basis_for("hcn", "main")
    atoms = hcn_geometry(60.0, 1.40)            # a bent, off-equilibrium HCN with a large gradient
    out = dict(basis=basis, tests=[])
    for variant in (["plain"] if STANDIN else ["plain", "v7"]):
        with item_guard("phase10", f"gradient check {variant}"):
            t0 = time.time()
            r = fd_gradient_check(M_SKALA, atoms, 0, basis, variant=variant)
            r["seconds"] = round(time.time() - t0, 1)
            out["tests"].append(r)
            print(f"  gradient check ({variant}): {r['status']}, ratios {[round(c['ratio'], 4) for c in r['components']]}, net gradient {r['net_gradient']:.1e}")
            if r["ok"]:
                GSTATE["grad_ok"], GSTATE["grad_variant"] = True, variant
                out["chosen"] = variant
                return out
    GSTATE["grad_ok"] = False
    return out


def g0_run():
    """Phase G0 (everything in one place, every step guarded)."""
    info = dict(started=now(), hardware=hw(), versions=ver(), gpu_name=GPU_NAME, gpu_class=GPU_CLASS, gpu_mem_gb=round(GPU_MEM_GB, 1), standin=STANDIN)
    with item_guard("phase10", "weights cache"):
        info["weights_restore"] = restore_weights_cache()
        print("  Skala weights:", info["weights_restore"])
    if NO_GPU and not STANDIN:
        info["note"] = "no GPU: diagnostics only"
        save_gstate(**info)
        print("  ** NO GPU: G0 stops here after the diagnostics. **")
        return info
    with item_guard("phase10", "probe functionals"):
        info["probe"] = g_probe_functionals()
        save_gstate(**info)
    with item_guard("phase10", "Skala check"):
        info["skala"] = g_skala_check()
        save_gstate(**info)
    with item_guard("phase10", "weights cache save"):
        if GSTATE.get("skala_ok") and not STANDIN:
            info["weights_save"] = save_weights_cache()
            print("  Skala weights on Drive:", info["weights_save"])
    with item_guard("phase10", "weights hash"):                       # review row 10: SHA-256 of the weights and the Hugging Face revision
        info["skala_weights"] = skala_weights_info()
        w = info["skala_weights"]
        print("  Skala weights:", {k: w.get(k) for k in ("file", "sha256", "huggingface_revision", "hash_matches_package_table", "expected_d3_settings")})
        save_gstate(**info)
    # reproduce the first stored NB4 HCN point with all four v4 methods, then the other 11 points (DFT methods) for check I9
    for m in ("v4-ccsd_t", "v4-pbe", "v4-b3lyp", "v4-skala"):
        if m == "v4-skala" and not GSTATE.get("skala_ok"):
            continue
        with item_guard("phase10", f"repro {m}"):
            g_v4_series(m, [0])
    first = g_gate()
    print(f"  point 0 of NB4: largest |difference| to the stored v4 energy {first['max_diff']:.1e} hartree ({first['status']}); per method {first['per_method']}")
    for m in ("v4-pbe", "v4-b3lyp", "v4-skala"):
        if m == "v4-skala" and not GSTATE.get("skala_ok"):
            continue
        with item_guard("phase10", f"repro all points {m}"):
            g_v4_series(m, list(range(12)))
    gate = g_gate()
    info["gate"] = gate
    if STANDIN:
        GSTATE["gate_ok"] = True
        print("  stand-in: the gate is not applied (stand-in energies differ from the stored v4 energies by design)")
    elif gate["status"] == "no data":
        GSTATE["gate_ok"] = None
        print("  gate: no reproduced energies (all attempts failed); later phases run, but nothing was verified")
    else:
        GSTATE["gate_ok"] = gate["status"] != "FAIL"
        if gate["status"] == "WARN":
            print("  gate WARN: differences between 1e-5 and 1e-4 hartree: version drift, state it in the paper")
        if gate["status"] == "FAIL":
            print("  ** GATE FAILED: differences over 1e-4 hartree. The later phases will not start (STOP_ON_G0_FAIL). **")
    save_gstate(**info)
    if GSTATE.get("skala_ok"):
        with item_guard("phase10", "Skala gradient"):
            info["gradient"] = g_skala_gradient_check()
    else:
        GSTATE["grad_ok"] = False
    save_gstate(**info)
    return info


# ------------------------------------------------------------------ G1 / G2 / G3 building blocks
def ensure_nod3(csv_name, system, pid, level, basis, coordinate=""):
    """'Skala without D3' (review row 1): the SAME SCF as the headline Skala row minus the D3 energy stored in its extra.
    Writes the second row (method Skala-1.1-noD3) unless it exists; also repairs a run that was cut between the two rows.
    A failed headline row gives a failed noD3 row (so that the two variants always have the same points)."""
    if have(csv_name, system, pid, level, M_SKALA_NOD3, basis, RETRY_FAILED):
        return None
    src = [r for r in read_rows(csv_name) if row_key(r) == (system, str(pid), level, M_SKALA, basis)]
    if not src:
        return None
    src.sort(key=lambda r: r["converged"] in ("1", "True"))            # a repeated headline row (RETRY_FAILED) that converged is used, else the last failed one
    r = src[-1]
    x = extra_of(r)
    ok = r["converged"] in ("1", "True") and r["energy_hartree"] not in ("", "nan") and x.get("d3_hartree") is not None
    e = float(r["energy_hartree"]) - float(x["d3_hartree"]) if ok else float("nan")
    row = make_row(system, pid, level, M_SKALA_NOD3, basis, e, ok, coordinate or r["coordinate"], seconds=0.0,
                   extra=dict(x, derived_from=M_SKALA, same_scf=True, with_dftd3=False))
    append_row(csv_name, row)
    return row


def g_energy_row(csv_name, system, pid, level, method, basis, atoms, charge, coordinate="", dm_key=None):
    """One energy, saved at once (skipped when already in the CSV).  For Skala the second row 'Skala-1.1-noD3' is written too."""
    row = run_point(csv_name, system, pid, level, method, basis,
                    lambda: energy_of(method, atoms, charge, basis, dm_key=dm_key), coordinate=coordinate, retry_failed=RETRY_FAILED, quiet=True)
    if method == M_SKALA:
        ensure_nod3(csv_name, system, pid, level, basis, coordinate)
    return row


def g_align_retry_modes(csv_name, standin_levels=None):
    """Review row 11: when one species of a barrier was computed with the density-fitting retry (extra.df = true) and its partner without,
    recompute the partner(s) with density fitting too.  The new row gets the point id 'SP@df' (same CSV), so nothing is overwritten and the
    check (item I14) uses the aligned energy.  Returns the number of rows added."""
    rows = [r for r in read_rows(csv_name) if "@" not in r["point_id"] and not re.match(r"^p\d+$", r["point_id"]) and r["converged"] in ("1", "True")]
    groups = {}
    for r in rows:
        groups.setdefault((r["system"], r["geometry_level"], r["method"], r["basis"]), []).append(r)
    n = 0
    for (system, level, method, basis), rs in groups.items():
        if method in (M_SKALA_NOD3,) or method.startswith("v4-"):
            continue
        fitted = [r for r in rs if extra_of(r).get("df")]
        if not fitted or len(fitted) == len(rs):
            continue
        need = [r for r in rs if not extra_of(r).get("df")]
        for r in need:
            g = geometry_of(system, r["point_id"], level)
            if g is None:
                continue
            with item_guard("phase11", f"{system}:{r['point_id']}:{method}:align"):
                new = run_point(csv_name, system, r["point_id"] + "@df", level, method, basis,
                                lambda g=g, system=system, r=r, method=method, basis=basis: energy_of(method, g["atoms"], species_charge(system, r["point_id"]), basis, force_df=True),
                                retry_failed=RETRY_FAILED, quiet=True)
                if new is not None:
                    n += 1
                    if method == M_SKALA:
                        ensure_nod3(csv_name, system, r["point_id"] + "@df", level, basis)
    if n:
        print(f"  retry-mode alignment: {n} partner species recomputed with density fitting in {csv_name}")
    return n


def g1_system(system):
    """G1, stationary points of one system, all functionals, at EVERY stored geometry level (review row 4): our own B3LYP geometries first
    (the headline protocol), then the downloaded database geometries wherever the CPU notebook found them (I11 cross-check).
    The headline level is complete for the whole system before the cross-check level is started."""
    basis = basis_for(system, "main")
    present = {sp: g_levels(system, sp) for sp in SPECIES_KIND[system]}
    for sp, lvs in present.items():
        if not lvs:
            log_error("phase11", f"{system}:{sp}", RuntimeError(f"skipped: missing input geometry for {sp} (CPU notebook C1 / G-1)"), note="skipped")
    levels = [lv for lv in LEVEL_PREFERENCE if any(lv in lvs for lvs in present.values())]
    for lv in levels:                                      # own_b3lyp (and own_scanpeak) first, database last
        for sp in SPECIES_KIND[system]:
            if lv not in present[sp]:
                continue
            g = geometry_of(system, sp, lv)
            for m in ladder_methods("stationary"):
                with item_guard("phase11", f"{system}:{sp}:{m}"):
                    g_energy_row(PHASE_CSV["phase11"], system, sp, lv, m, basis, g["atoms"], species_charge(system, sp))


def g1_path(system, method):
    """G1, one functional along the relaxed path of one system (densities of the previous point are used as the start)."""
    pts = g_load_path(system)
    if not pts:
        log_error("phase11p", f"{system}:path", RuntimeError(f"skipped: missing input geoms/{system}_path.json (CPU notebook C2 / G-1)"), note="skipped")
        return
    basis = basis_for(system, "main")
    reset_dm(system)
    for q in pts:
        with item_guard("phase11p", f"{system}:p{q['index']}:{method}"):
            g_energy_row(PHASE_CSV["phase11"], system, f"p{q['index']:02d}", LV_SCAN, method, basis, q["atoms"], species_charge(system, "TS"),
                         coordinate=f"{q['coordinate']:.4f}", dm_key=(system, method, basis))


def g1_dft_stationary():
    """G1, stage 1: stationary points and fragments at every stored geometry level, then the retry-mode alignment."""
    for system in [s for s in SYSTEMS_TO_RUN]:
        with item_guard("phase11", system):
            print(f"  {system}: stationary points")
            g1_system(system)
    with item_guard("phase11", "retry-mode alignment"):
        g_align_retry_modes(PHASE_CSV["phase11"])


def g1_dft_paths():
    """G1, stage 2 (its own phase 'phase11p' so that no phase is longer than 85 minutes): path points, one functional at a time."""
    systems = [s for s in SYSTEMS_TO_RUN]
    skipped = [m for m in ladder_methods("stationary") if m not in ladder_methods("path")]
    if skipped:
        print(f"  note: {', '.join(skipped)} run on the CPU, so they are computed at stationary points only (not along paths)")
    for method in ladder_methods("path"):                    # one functional at a time (cheapest to cut last)
        for system in systems:
            with item_guard("phase11p", f"{system}:path:{method}"):
                print(f"  {system}: path, {method}")
                g1_path(system, method)


def g1_dft():
    g1_dft_stationary()
    g1_dft_paths()


def g2_basis():
    """G2: the same stationary points in the small and the large basis, method by method (PBE and r2SCAN are cut first if time runs out)."""
    csv2 = PHASE_CSV["phase12"]
    species = [(s, sp) for s in SYSTEMS_TO_RUN if s != "ch4" for sp in SPECIES_KIND[s]]
    for m in ladder_methods("stationary"):
        for size in ("large", "small"):
            print(f"  {m}, {size} basis")
            for system, sp in species:
                with item_guard("phase12", f"{system}:{sp}:{m}:{size}"):
                    lv, g = best_geom(system, sp)
                    if g is None:
                        log_error("phase12", f"{system}:{sp}", RuntimeError("skipped: missing input geometry"), note="skipped")
                        continue
                    b, ch = basis_for(system, size), species_charge(system, sp)
                    if size == "large":
                        nbf = make_mol(g["atoms"], ch, b).nao
                        if nbf > GPU_QZ_MAX_NBF:
                            if not have(csv2, system, sp, lv, m, b):
                                log_error("phase12", f"{system}:{sp}", RuntimeError(f"skipped: {b} has {nbf} basis functions, above the limit {GPU_QZ_MAX_NBF} for this GPU"), note="skipped")
                            continue
                    g_energy_row(csv2, system, sp, lv, m, b, g["atoms"], ch)
    with item_guard("phase12", "retry-mode alignment"):
        g_align_retry_modes(csv2)


def g3_scout():
    """G3 (review row 9): ALL five functionals (Skala with and without D3, PBE, r2SCAN, B3LYP-D3(BJ), wB97M-V) at the xTB minimum, peak and product of
    every reaction, so that the geometry penalty can be shown not to be Skala-specific.  xTB's own barrier (from the C4 energies) is recorded too."""
    csv3 = PHASE_CSV["phase13"]
    reactions = [s for s in SYSTEMS_TO_RUN if s in REACTIONS]
    order = sorted(reactions, key=lambda s: (SYSTEMS[s]["tier"] != "C", SYSTEM_ORDER.index(s)))   # Tier C first (it has no CCSD(T))
    for m in ladder_methods("stationary"):
        for system in order:
            basis = basis_for(system, "main")
            for sp in g_scout_species(system):
                if not sp:
                    continue
                with item_guard("phase13", f"{system}:{sp}:{m}"):
                    g = load_geom(geom_name(system, sp, LV_XTB))
                    if g is None:
                        log_error("phase13", f"{system}:{sp}", RuntimeError("skipped: missing input xTB geometry (CPU notebook C4 / G-1)"), note="skipped")
                        continue
                    if g.get("no_peak"):
                        continue                                # recorded by the CPU notebook as a result: nothing to compute
                    g_energy_row(csv3, system, sp, LV_XTB, m, basis, g["atoms"], species_charge(system, sp))
    with item_guard("phase13", "retry-mode alignment"):
        g_align_retry_modes(csv3)
    with item_guard("phase13", "xTB own barriers"):
        out = {}
        e = {(r["system"], r["point_id"]): float(r["energy_hartree"]) for r in read_rows(PHASE_CSV["phase04"])
             if r["method"] == M_XTB and r["geometry_level"] == LV_XTB and r["converged"] in ("1", "True") and r["energy_hartree"] not in ("", "nan")}
        for system in reactions:
            rmin = g_scout_species(system)[0]
            out[system] = None if (system, "TS") not in e or (system, rmin) not in e else round((e[(system, "TS")] - e[(system, rmin)]) * HARTREE_KCAL, 3)
        write_json("phase13_xtb_barriers.json", dict(unit="kcal/mol", barrier_TS_minus_reactant_or_complex=out,
                                                      note="GFN2-xTB energies at GFN2-xTB geometries (from CPU notebook C4); null = no xTB peak or missing"))
        print("  xTB's own barriers (kcal/mol):", out)


# ------------------------------------------------------------------ G4: Skala-relaxed minima and transition states
def g4_minima(system):
    S = SYSTEMS[system]
    out = []
    for sp in S["reactants"] + S["products"] + [x for x in (S["rc"], S["pc"]) if x]:
        if sp not in out and SPECIES_KIND[system].get(sp) == "min":
            out.append(sp)
    return out


def g4_system(system, variant):
    S, spc = SYSTEMS[system], SCANS.get(system)
    basis, csv4 = basis_for(system, "main"), PHASE_CSV["phase14"]

    def relax(atoms, ch, constraints=None):
        eng = EG(M_SKALA, ch, basis, variant)
        return opt_custom(atoms, ch, eng.eg, constraints=constraints, convergence_set=G4_CONV, maxiter=G4_MAXITER)

    def e_std(atoms, ch):                                     # the energy with the same engine as G1 (not the gradient engine)
        return energy_of(M_SKALA, atoms, ch, basis)
    # ---- minima
    for sp in g4_minima(system):
        with item_guard("phase14", f"{system}:{sp}"):
            lv, g = best_geom(system, sp)
            if g is None:
                log_error("phase14", f"{system}:{sp}", RuntimeError("skipped: missing input geometry"), note="skipped")
                continue
            ch, nm = species_charge(system, sp), geom_name(system, sp, LV_SKALA)
            gs = load_geom(nm)
            if gs is None:
                t0 = time.time()
                r = relax(g["atoms"], ch)
                save_geom(nm, r["atoms"], ch, species=sp, kind="min", level=LV_SKALA, opt_converged=bool(r["converged"]), n_gradients=r["n_steps"], seconds=round(time.time() - t0, 1), error=r["error"])
                gs = load_geom(nm)
            run_point(csv4, system, sp, LV_SKALA, M_SKALA, basis,
                      lambda gs=gs, ch=ch: (lambda e: dict(energy=e["energy"], converged=bool(e["converged"] and gs.get("opt_converged", True)),
                                                           extra=dict(e["extra"], n_gradients=gs.get("n_gradients"), opt_error=gs.get("error", ""), start_level=lv)))(e_std(gs["atoms"], ch)),
                      retry_failed=RETRY_FAILED)
            ensure_nod3(csv4, system, sp, LV_SKALA, basis)
    # ---- transition state: constrained optimisations around the TS coordinate, then a parabola through the top three points
    if not S["ts"] or spc is None:
        return
    with item_guard("phase14", f"{system}:TS"):
        if have(csv4, system, "TS", LV_SKALA, M_SKALA, basis) and not RETRY_FAILED:
            return
        lv, gTS = best_geom(system, "TS")
        if gTS is None:
            log_error("phase14", f"{system}:TS", RuntimeError("skipped: missing input TS geometry"), note="skipped")
            return
        ch = species_charge(system, "TS")
        kind, idx = spc["kind"], spc["idx"]
        c_ts, delta = coord_value(kind, idx, gTS["atoms"]), G4_DELTA[kind]
        sp_file = path("geoms", f"{system}_skala_ts_scan.json")
        store = json.load(open(sp_file)) if os.path.exists(sp_file) else {}

        def point(c, start):
            key = f"{c:.5f}"
            if key not in store:
                t0 = time.time()
                r = relax(start, ch, constraints=[(kind, idx, float(c))])
                e = e_std(r["atoms"], ch)
                store[key] = dict(coordinate=float(c), atoms=[[s, list(x)] for s, x in r["atoms"]], energy=e["energy"], converged=bool(r["converged"] and e["converged"]),
                                  d3=e["extra"].get("d3_hartree"), n_gradients=r["n_steps"], seconds=round(time.time() - t0, 1), error=r["error"])
                json.dump(store, open(sp_file + ".tmp", "w"), indent=1)
                os.replace(sp_file + ".tmp", sp_file)
                print(f"   {system} TS window: coordinate {c:.4f}  E = {e['energy']:.8f}  ({r['n_steps']} gradients)")
            return store[key]
        for c in (c_ts - delta, c_ts, c_ts + delta):
            point(c, gTS["atoms"])
        for shift in range(G4_MAX_SHIFT + 1):           # move the window if the top is not in the middle
            keys = sorted(store, key=float)
            k = int(np.nanargmax([store[k_]["energy"] for k_ in keys]))
            if (0 < k < len(keys) - 1) or shift == G4_MAX_SHIFT:
                break
            new_c = float(keys[0]) - delta if k == 0 else float(keys[-1]) + delta
            point(new_c, [(s, tuple(x)) for s, x in store[keys[k]]["atoms"]])
        keys = sorted(store, key=float)
        pts = [dict(coordinate=store[k_]["coordinate"], energy=store[k_]["energy"]) for k_ in keys]
        k = int(np.nanargmax([p["energy"] for p in pts]))
        interior = 0 < k < len(pts) - 1
        if interior:
            xv, ev = parabola_peak(pts, k)
        else:
            xv, ev = pts[k]["coordinate"], pts[k]["energy"]
        top = store[keys[k]]
        save_geom(geom_name(system, "TS", LV_SKALA), [(s, tuple(x)) for s, x in top["atoms"]], ch, species="TS", kind="ts", level=LV_SKALA,
                  vertex_coordinate=xv, interior=bool(interior), method="constrained optimisation + parabola vertex (not a saddle-point search)")
        conv_ts = bool(interior and all(store[x]["converged"] for x in keys[max(0, k - 1):k + 2]))
        d3s = [store[k_].get("d3") for k_ in keys]
        nod3 = None
        if all(v is not None for v in d3s):            # the same window without the D3 term (same SCFs, same geometries)
            pts_n = [dict(coordinate=store[k_]["coordinate"], energy=store[k_]["energy"] - store[k_]["d3"]) for k_ in keys]
            ev_n = parabola_peak(pts_n, k)[1] if interior else pts_n[k]["energy"]
            nod3 = (ev_n, [round(p["energy"], 8) for p in pts_n])
        common = dict(method="constrained optimisation + three-point parabola VERTEX (not a saddle-point search)", interior=bool(interior), window=[round(float(x), 4) for x in keys],
                      vertex_coordinate=xv, reference_ts_coordinate=c_ts, standin=STANDIN)
        if nod3 is not None and not have(csv4, system, "TS", LV_SKALA, M_SKALA_NOD3, basis):      # written first: a cut between the two rows is repaired on rerun
            append_row(csv4, make_row(system, "TS", LV_SKALA, M_SKALA_NOD3, basis, nod3[0], conv_ts, coordinate=f"{xv:.4f}",
                                      extra=dict(common, window_energies=nod3[1], derived_from=M_SKALA, same_scf=True, with_dftd3=False)))
        row = make_row(system, "TS", LV_SKALA, M_SKALA, basis, ev, conv_ts, coordinate=f"{xv:.4f}",
                       extra=dict(common, window_energies=[round(p["energy"], 8) for p in pts], window_d3=[None if v is None else round(v, 8) for v in d3s], with_dftd3=True))
        append_row(csv4, row)
        print(f"   {system} TS: parabola vertex at {xv:.4f} (reference TS {c_ts:.4f}), E = {ev:.8f}, interior maximum: {interior}")


def g4_report(system):
    """Print Skala//Skala against Skala//reference barrier for one system (from the CSV files)."""
    S, basis = SYSTEMS[system], basis_for(system, "main")

    def E(csv_, sp, lv=None):
        for r in read_rows(csv_):
            if r["system"] == system and r["point_id"] == sp and r["method"] == M_SKALA and r["basis"] == basis and r["converged"] in ("1", "True") and r["energy_hartree"] not in ("", "nan") and (lv is None or r["geometry_level"] == lv):
                if lv is not None or r["geometry_level"] in LEVEL_PREFERENCE:
                    return float(r["energy_hartree"])
        return None
    out = {}
    for tag, csv_, lv in (("Skala//Skala", PHASE_CSV["phase14"], LV_SKALA), ("Skala//reference", PHASE_CSV["phase11"], None)):
        ts = E(csv_, "TS", lv)
        parts = [E(csv_, sp, lv) for sp in S["reactants"]]
        out[tag] = None if ts is None or any(p is None for p in parts) else (ts - sum(parts)) * HARTREE_KCAL
    if out["Skala//Skala"] is not None and out["Skala//reference"] is not None:
        print(f"   {system}: barrier Skala//Skala {out['Skala//Skala']:.2f}, Skala//reference {out['Skala//reference']:.2f}, difference {out['Skala//Skala'] - out['Skala//reference']:+.2f} kcal/mol")
    return out


def g4_relaxed(systems=None):
    variant = GSTATE.get("grad_variant", "plain")
    todo = [s for s in (systems or G4_SYSTEMS) if s in SYSTEMS_TO_RUN]
    summary = {}
    for system in todo:
        with item_guard("phase14", system):
            print(f"  {system}: Skala-relaxed minima and TS (gradient settings: {variant})")
            g4_system(system, variant)
            summary[system] = g4_report(system)
    old = read_json("phase14_summary.json", {}) or {}
    old.update(summary)
    write_json("phase14_summary.json", old)


# ------------------------------------------------------------------ optional phase: GPU relaxed paths for further systems, then all functionals along them
def gopt_paths():
    csv_o = CSV_OPT
    for system in [s for s in OPT_PATH_SYSTEMS if s in SCANS and s in SYSTEMS_TO_RUN]:
        with item_guard("phase16", f"{system}:scan"):
            sp = SCANS[system]
            lv, g0 = best_geom(system, sp["start"])
            if g0 is None:
                log_error("phase16", system, RuntimeError(f"skipped: missing start geometry {sp['start']}"), note="skipped")
                continue
            ch, basis = species_charge(system, sp["start"]), basis_for(system, "main")
            eng = EG(M_B3LYP_NODISP, ch, basis)
            print(f"  {system}: relaxed scan with {M_B3LYP_NODISP} on the GPU ({sp['label']})")
            _walk(system, g0["atoms"], ch,
                  lambda at, c, ch=ch: opt_custom(at, ch, eng.eg, constraints=[c], convergence_set=G4_CONV, maxiter=G4_MAXITER),
                  lambda at, eng=eng: eng.energy(at), g_load_path(system, "pathgpu"),
                  lambda pts, system=system, sp=sp: g_save_path(system, pts, sp["label"] + " (GPU B3LYP, no dispersion)", suffix="pathgpu", level=LV_SCANGPU))
    for m in ladder_methods("path"):
        for system in [s for s in OPT_PATH_SYSTEMS if s in SCANS and s in SYSTEMS_TO_RUN]:
            with item_guard("phase16", f"{system}:path:{m}"):
                pts = g_load_path(system, "pathgpu")
                if not pts:
                    continue
                basis = basis_for(system, "main")
                reset_dm(system)
                ch = species_charge(system, SCANS[system]["start"])
                print(f"  {system}: functional {m} along the GPU path")
                for q in pts:
                    with item_guard("phase16", f"{system}:p{q['index']}:{m}"):
                        g_energy_row(csv_o, system, f"p{q['index']:02d}", LV_SCANGPU, m, basis, q["atoms"], ch, coordinate=f"{q['coordinate']:.4f}", dm_key=(system, m, basis))


# ------------------------------------------------------------------ summary helpers
def csv_stats(csv_name):
    rows = read_rows(csv_name)
    if not rows:
        return None
    bad = [r for r in rows if r["converged"] in ("0", "False") or r["energy_hartree"] in ("", "nan")]
    return dict(rows=len(rows), failed=len(bad))


def g_summary_lines():
    L = ["", "## GPU notebook details", "", f"GPU: {GPU_NAME or 'none'} ({GPU_CLASS}, {GPU_MEM_GB:.0f} GB)"]
    if STANDIN:
        L += ["", "**STAND-IN DRY RUN: every number in this folder comes from cheap CPU functionals in tiny bases. None of it is Skala or a result.**"]
    st = read_json("phase10_timing.json", {}) or {}
    L.append(f"Routes per functional: {st.get('routes')}; anchor functional: {st.get('anchor')}")
    L.append(f"G0 gate (stored v4 energies reproduced, Skala WITHOUT D3 as in v4): {st.get('gate_ok')}; Skala usable: {st.get('skala_ok')}; Skala gradient check passed: {st.get('grad_ok')} (settings: {st.get('grad_variant')})")
    w = st.get("skala_weights") or {}
    L.append(f"Skala weights: {w.get('skala_name')} -> file {w.get('file')}, SHA-256 {w.get('sha256')}, Hugging Face revision {w.get('huggingface_revision')}, "
             f"matches the package table: {w.get('hash_matches_package_table')}, D3 parameter set stored in the weights: {w.get('expected_d3_settings')}")
    L.append("Skala-1.1 = with its D3 term (package default, headline); Skala-1.1-noD3 = same SCF minus the stored D3 energy; D3 off only in the v4 gate.")
    if st.get("anchor") and st.get("anchor") != M_WB97MV:
        L.append(f"NOTE: the anchor functional is {st['anchor']}, not wB97M-V: check item R2 of the CPU notebook looks for wB97M-V rows and will say NOT RUN.")
    L += ["", "| table | rows | failed or unconverged |", "|---|---|---|"]
    for name in (CSV_REPRO, PHASE_CSV["phase11"], PHASE_CSV["phase12"], PHASE_CSV["phase13"], PHASE_CSV["phase14"], CSV_OPT):
        s = csv_stats(name)
        L.append(f"| {name} | {s['rows'] if s else 'not written'} | {s['failed'] if s else ''} |")
    cfg = read_json("config.json", {}) or {}
    if cfg.get("smoke") and not STANDIN:
        L.append("WARNING: config.json says the CPU notebook ran as a smoke test (small bases). Its numbers are not the paper numbers.")
    if os.path.exists(path("phase09_regen.json")):
        r = read_json("phase09_regen.json", {})
        L += ["", "G-1 (rebuilt CPU inputs): " + "; ".join(r.get("log", []))[:400]]
    return L

**Cell 3: check module, literature table and figure code** (pasted unchanged). Only defines functions.

In [ ]:
# ===== b1v5_lit_table.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_lit_table.py  --  the literature table LIT for Paper B1 v5 (plan section 4.1).
# Pasted into the check cell (C7) of the notebooks; also importable.
#
# Every value is in kcal/mol.  Columns:
#   id         L1 ... L19 as in the plan
#   system     key of the system database (b1v5_common.SYSTEMS)
#   quantity   what is compared (see QUANTITIES below)
#   value, unc literature value and its stated uncertainty (None = none stated)
#   source     where it comes from
#   zp         zero-point convention of the literature number: "excluded", "included" or "unknown"
#   verified   "yes", "partial" (value yes, convention not) or "no".  ONLY verified == "yes" rows enter error statistics
#              (review 2026-11-15, row 6: L6 CH3NC is "no" until the full paper is read; L19 is a placeholder without a value).
#              L3 (= L1 minus L2) is kept as a row but is not used in pooled statistics (STATS_EXCLUDED_IDS).
#
# QUANTITIES (how the check computes the computed counterpart):
#   barrier_fwd        E(TS) - E(separated reactants)           (single reactant for systems 1 to 4)
#   barrier_rev        E(TS) - E(separated products)
#   barrier_fwd_cplx   E(TS) - E(reactant complex)              (SN2 only)
#   barrier_rev_cplx   E(TS) - E(product complex)               (SN2 only)
#   reaction_energy    E(separated products) - E(separated reactants)
# ==============================================================================
_NHT = "NHTBH38/08 database page, comp.chem.umn.edu/db/dbs/nhtbh38.html (QCISD/MG3 geometries)"

LIT = [
    dict(id="L1", system="hcn", quantity="barrier_fwd", value=48.07, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L2", system="hcn", quantity="barrier_rev", value=32.82, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L3", system="hcn", quantity="reaction_energy", value=15.25, unc=None, source=_NHT + "; difference of L1 and L2", zp="excluded", verified="yes"),
    dict(id="L4", system="nh3", quantity="barrier_fwd", value=5.12, unc=None,
         source="Lin et al., J. Chem. Phys. 117, 11265 (2002): 1790 cm-1, CCSD(T)/CBS with corrections", zp="excluded", verified="yes"),
    dict(id="L5", system="h2co", quantity="barrier_fwd", value=81.9, unc=0.3,
         source="Feller, Dupuis, Garrett, J. Chem. Phys. 113, 218 (2000), theory (abstract read)", zp="included", verified="yes"),
    dict(id="L6", system="ch3nc", quantity="barrier_fwd", value=38.25, unc=0.25,
         source="Nguyen and Stanton, Sanibel 2018 abstract, HEAT-345Q (experiment 38.2 to 38.4); the abstract calls it an 'activation energy': "
                "may be an Arrhenius-type or 0 K quantity, not an electronic barrier. Full paper (Nguyen, Thorpe, Bross, Ruscic, Stanton, "
                "J. Phys. Chem. Lett. 2018) not yet read", zp="unknown", verified="no"),
    dict(id="L7", system="sn2_ff", quantity="barrier_fwd", value=-0.34, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L8", system="sn2_ff", quantity="barrier_fwd_cplx", value=13.38, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L9", system="sn2_clcl", quantity="barrier_fwd", value=3.10, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L10", system="sn2_clcl", quantity="barrier_fwd_cplx", value=13.41, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L11", system="sn2_fcl", quantity="barrier_fwd", value=-12.54, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L12", system="sn2_fcl", quantity="barrier_rev", value=20.11, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L13", system="sn2_fcl", quantity="barrier_fwd_cplx", value=3.44, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L14", system="sn2_fcl", quantity="barrier_rev_cplx", value=29.42, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L15", system="sn2_ohf", quantity="barrier_fwd", value=-2.44, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L16", system="sn2_ohf", quantity="barrier_rev", value=17.66, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L17", system="sn2_ohf", quantity="barrier_fwd_cplx", value=10.96, unc=None, source=_NHT, zp="excluded", verified="yes"),
    dict(id="L18", system="sn2_ohf", quantity="barrier_rev_cplx", value=47.20, unc=None, source=_NHT, zp="excluded", verified="yes"),
    # PLACEHOLDER: the CH3NC -> CH3CN reaction enthalpy is reported in the same paper (abstract only seen); VALUE NOT YET READ.
    dict(id="L19", system="ch3nc", quantity="reaction_energy", value=None, unc=None,
         source="Nguyen, Thorpe, Bross, Ruscic, Stanton, J. Phys. Chem. Lett. 2018 (OSTI 1461519): reaction enthalpy, value not yet read; "
                "convention (0 K or 298 K, enthalpy or energy) unknown", zp="unknown", verified="no"),
]

# Information only: never scored, never joined to a PASS/WARN/FAIL.
LIT_INFO = [
    dict(id="I-a", system="hcn", quantity="HNC minus HCN (0 K)", value=14.89, unc=None,
         source="ATcT, from the v4 literature check (not re-read)", zp="included", verified="no"),
    dict(id="I-b", system="h2co", quantity="barrier_fwd (experiment)", value=79.2, unc=0.8,
         source="Feller, Dupuis, Garrett 2000, experiment", zp="included", verified="yes"),
]

LIT_BY_ID = {r["id"]: r for r in LIT}
STATS_EXCLUDED_IDS = {"L3"}            # L3 = L1 - L2: not independent, never in pooled statistics (only in the supplement table)


def lit_scorable(l):
    """True when the row has a number to compare with (L19 is a placeholder)."""
    return l.get("value") is not None


def lit_in_stats(l):
    """True when the row may enter ANY error statistic: has a value, is verified, and is not the dependent row L3."""
    return lit_scorable(l) and l.get("verified") == "yes" and l["id"] not in STATS_EXCLUDED_IDS


# ===== b1v5_check.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_check.py  --  the automated check (cell C7) of Paper B1 v5, plan section 4.
# Pasted into the C7 cell of the notebooks (after b1v5_common, b1v5_lit_table and b1v5_v4_stored); also importable.
#
# What it does, in plain words:
#   1. reads every phase*.csv in the working folder,
#   2. compares our numbers with the literature table LIT (checks R1 to R4),
#   3. runs internal sanity checks I1 to I13,
#   4. prints one line per item: PASS / WARN / FAIL (or NOT RUN when the data is missing),
#   5. writes check_report.csv and check_report.md,
#   6. decides, system by system, where it goes in the paper (rule 4.4) and whether the paper has enough systems.
# It never crashes on missing files: a missing input just gives "NOT RUN".
# All energy differences are in kcal/mol unless written otherwise.
#
# Changes after the final paper-strategy review (2026-11-15), in short:
#   row 3  I6 is scored on wB97M-V (or its fallback) and CCSD(T) only; other functionals give INFO ("no barrier" is a finding).
#          I1 removes a METHOD's entry, not the system, unless a core method (DF-B3LYP, CCSD(T), anchor) fails.
#   row 4  one geometry level per quantity: all species of a barrier must come from the same geometry family, else NOT RUN.
#          Headline family = our own B3LYP geometries; the database geometries are the I11 cross-check.
#   row 5  R1 and the reference statistics use CCSD(T) + [MP2/QZVPP(D) - MP2/TZVP(D)]; the raw CCSD(T) is kept as a column.
#   row 6  unverified literature rows (L6) and placeholders (L19) are never scored PASS or FAIL and never enter statistics.
#   row 7  PASS tolerance for references below 10 kcal/mol is max(0.5, 10% of the reference).
#   row 11 a barrier must not mix density-fitted and unfitted energies (I14).
# ==============================================================================
import os, re, csv, json, glob, math
import numpy as np

try:
    COLUMNS  # noqa  (defined when b1v5_common was pasted above)
except NameError:
    from b1v5_common import *  # noqa
try:
    LIT  # noqa
except NameError:
    from b1v5_lit_table import *  # noqa
try:
    scout_barrier_kcal  # noqa
    HAVE_V4 = True
except NameError:
    try:
        from b1v5_v4_stored import *  # noqa
        HAVE_V4 = True
    except ImportError:
        HAVE_V4 = False

# ------------------------------------------------------------------ tolerances (kcal/mol unless stated); the author may tighten them
TOL = dict(R1=(1.5, 3.0), R2=(2.0, 4.0), R4_flag=5.0, I1_path=(0.98, 0.95), I2=(0.02, 0.03), I5_floor=0.1, I5_peak=(0.5, 1.5),
           I8=(0.5, 1.0), I9_energy=(1e-5, 1e-4), I9_mae=0.02, I9_scout=0.01, I10=0.3, I11_geom=(0.3, 1.0), I11_skala=(0.5, 1.0), I12=(1.0, 3.0))
V4_MAE = {"v4-pbe": 1.242, "v4-b3lyp": 1.374, "v4-skala": 0.564}
V4_METHODS = {"v4-ccsd_t": "ccsd_t", "v4-pbe": "pbe", "v4-b3lyp": "b3lyp", "v4-skala": "skala"}   # method name in rows -> key in b1v5_v4_stored.NB4_HCN_E
SMALL_REF = 10.0           # kcal/mol: for |reference| below this the PASS tolerance is max(REL_FLOOR, REL_FRAC * |reference|) (review row 7)
REL_FLOOR, REL_FRAC = 0.5, 0.10
I6_SCORED = (M_CCSDT_FC,) + tuple(ANCHORS)       # only these methods can FAIL item I6 (review row 3)
NOT_A_FUNCTIONAL = (M_CCSDT_BC, M_MP2DF, M_SKALA_NOD3, M_DFB3LYP, M_CCSDT_AE, M_XTB)   # skipped where "every functional" is looped
PASS, WARN, FAIL, NOTRUN, NA, INFO = "PASS", "WARN", "FAIL", "NOT RUN", "N/A", "INFO"
_RANK = {PASS: 0, NA: 0, INFO: 0, NOTRUN: 1, WARN: 2, FAIL: 3}


def worst(statuses):
    s = [x for x in statuses if x]
    return max(s, key=lambda x: _RANK[x]) if s else NOTRUN


def cap_warn(status):
    return WARN if status == PASS else status


# ------------------------------------------------------------------ loading
_PATH_ID = re.compile(r"^(p\d+|g_\d+_\d+|nb4_\d+)$")


def is_path_id(pid):
    return bool(_PATH_ID.match(str(pid)))


def _f(x):
    try:
        v = float(x)
        return v if math.isfinite(v) else None
    except Exception:
        return None


def load_rows():
    """All rows of all phase*.csv, parsed.  Returns (rows, files_found)."""
    rows, files = [], []
    for fn in sorted(glob.glob(path("phase[0-9]*.csv"))):
        files.append(os.path.basename(fn))
        try:
            for r in csv.DictReader(open(fn, newline="")):
                x = extra_of(r)
                rows.append(dict(file=os.path.basename(fn), system=r["system"], pid=r["point_id"], level=r["geometry_level"],
                                 method=r["method"], basis=r["basis"], E=_f(r["energy_hartree"]), conv=r["converged"] in ("1", "True", "true"),
                                 t1=_f(r.get("t1")), imag=(int(float(r["imaginary_modes"])) if _f(r.get("imaginary_modes")) is not None else None),
                                 coord=_f(r.get("coordinate")), x=x))
        except Exception as ex:
            print("  could not read", fn, repr(ex)[:100])
    return rows, files


def level_label(rows):
    """Text for the geometry level of the rows that make up one quantity."""
    lv = sorted({r["level"] for r in rows})
    if lv == [LV_OWN]:
        return LV_OWN
    if set(lv) <= {LV_OWN, LV_SCANPEAK}:
        return LV_OWN + "+scanpeak_TS"
    return "+".join(lv)


class Data:
    """Index of the CSV rows with the lookups the checks need."""
    def __init__(self):
        self.rows, self.files = load_rows()
        cfg = read_json("config.json", {}) or {}
        for k, v in (cfg.get("basis_override") or {}).items():
            BASIS_OVERRIDE[k] = v
        # headline geometry family ('own' by default; 'db' or 'any' only if the CPU notebook was run with OWN_GEOMETRIES_FOR_ALL = False)
        self.headline = cfg.get("headline_family") or "own"
        self.idx = {}
        for r in self.rows:
            self.idx.setdefault((r["system"], r["pid"], r["method"], r["basis"]), []).append(r)
        self.tiers = read_json("tiers.json", {}) or {k: v["tier"] for k, v in SYSTEMS.items()}
        self.excluded = set()          # (system, point, level, basis, method) excluded by I2 / I3
        for r in self.rows:
            if r["method"].startswith("CCSD(T)") and ((r["t1"] or 0) > TOL["I2"][1] or r["x"].get("stable") in (0, "0", False)):
                self.excluded.add((r["system"], r["pid"], r["level"], r["basis"], r["method"]))
        self.mixed = {}                # (system, method, basis) -> dict about density-fitted / unfitted mixing (item I14)
        self.n_derived = 0
        self._build_derived()

    def _build_derived(self):
        """CCSD(T)/TZVP(D) + [DF-MP2/QZVPP(D) - DF-MP2/TZVP(D)] at every geometry where all three energies exist (review row 5).
        The derived rows live only in the index (never in self.rows), so the other checks do not see them."""
        mp2 = {}
        for r in self.rows:
            if r["method"] == M_MP2DF and r["conv"] and r["E"] is not None:
                mp2.setdefault((r["system"], r["pid"], r["level"]), {})[r["basis"]] = r["E"]
        for r in self.rows:
            if r["method"] != M_CCSDT_FC or not r["conv"] or r["E"] is None or r["system"] not in SYSTEMS:
                continue
            mb, lb = basis_for(r["system"], "main"), basis_for(r["system"], "large")
            if r["basis"] != mb or (r["system"], r["pid"], r["level"], r["basis"], r["method"]) in self.excluded:
                continue
            m = mp2.get((r["system"], r["pid"], r["level"]), {})
            if mb in m and lb in m:
                d = m[lb] - m[mb]
                row = dict(r, method=M_CCSDT_BC, E=r["E"] + d, file="derived", x=dict(r["x"], raw_E=r["E"], dmp2_hartree=d, derived=True))
                self.idx.setdefault((r["system"], r["pid"], M_CCSDT_BC, r["basis"]), []).append(row)
                self.n_derived += 1

    def rows_for(self, system, point, method, basis):
        return self.idx.get((system, point, method, basis), [])

    def _levels(self, level):
        """Tuple of geometry levels for a request: None = the headline family, 'own' / 'db' = a family, anything else = that exact level."""
        if level is None:
            return [FAMILY_LEVELS[f] for f in (("own", "db") if self.headline == "any" else (self.headline,))]
        return [FAMILY_LEVELS[level]] if level in FAMILY_LEVELS else [(level,)]

    def _ok_rows(self, system, sp, method, basis, levels, allow_excluded=False):
        rs = [r for r in self.rows_for(system, sp, method, basis) if r["conv"] and r["E"] is not None and r["x"].get("interior") is not False
              and r["level"] in levels]
        if not allow_excluded:
            rs = [r for r in rs if (r["system"], r["pid"], r["level"], r["basis"], r["method"]) not in self.excluded]
        return sorted(rs, key=lambda r: levels.index(r["level"]))

    def energies(self, system, species, method, basis, level=None, allow_excluded=False):
        """All `species` of one quantity at ONE geometry family (review row 4).  Returns (dict species -> row, level label) or None
        when no single family has every species (the quantity is then NOT RUN, never silently mixed)."""
        for levels in self._levels(level):
            got = {}
            for sp in dict.fromkeys(species):
                rs = self._ok_rows(system, sp, method, basis, levels, allow_excluded)
                if not rs:
                    break
                got[sp] = rs[0]
            else:
                return got, level_label(list(got.values()))
        return None

    def aligned(self, system, sp, method, basis, level, df=True):
        """Partner energy recomputed with the same retry mode (pid 'SP@df'), or None."""
        rs = self._ok_rows(system, sp + ("@df" if df else ""), method, basis, (level,))
        return rs[0] if rs else None

    def E(self, system, sp, method, basis, level=None, allow_excluded=False):
        """Energy (hartree) of ONE stationary species (own geometry first, then the database one); None if absent.
        Anything that combines species must use energies() or quantity(), which keep the geometry level fixed."""
        lv = self._levels(level) if level else [tuple(LEVEL_PREFERENCE)]
        for levels in lv:
            rs = self._ok_rows(system, sp, method, basis, levels, allow_excluded)
            if rs:
                return rs[0]["E"]
        return None

    def zpe(self, system, sp):
        for lv in (LV_OWN, LV_SCANPEAK):
            for r in self.rows_for(system, sp, M_DFB3LYP, basis_for(system, "main")):
                if r["level"] == lv and r["x"].get("zpe_hartree") is not None:
                    return float(r["x"]["zpe_hartree"])
        return None

    def methods_with(self, system, basis):
        return sorted({k[2] for k in self.idx if k[0] == system and k[3] == basis and not is_path_id(k[1]) and "@" not in k[1]})


def _quantity_species(system, q):
    """(plus species, minus species) of a quantity; None when the quantity does not exist for this system."""
    S = SYSTEMS[system]
    if q == "barrier_fwd":
        return [S["ts"]] if S["ts"] else None, S["reactants"]
    if q == "barrier_rev":
        return [S["ts"]] if S["ts"] else None, S["products"]
    if q == "barrier_fwd_cplx":
        return ([S["ts"]] if S["ts"] else None), ([S["rc"]] if S["rc"] else [])
    if q == "barrier_rev_cplx":
        return ([S["ts"]] if S["ts"] else None), ([S["pc"]] if S["pc"] else [])
    if q == "reaction_energy":
        return S["products"], S["reactants"]
    return None, None


def quantity_info(D, system, q, method, basis, zpe=False, level=None):
    """Computed counterpart of a LIT quantity.  Returns dict(value in kcal/mol, level label, aligned) or None when anything is missing,
    a species comes from another geometry family, or fitted and unfitted energies would be mixed without a recomputed partner."""
    plus, minus = _quantity_species(system, q)
    if not plus or not minus:
        return None
    if zpe:
        z = [D.zpe(system, sp) for sp in plus + minus]
        if any(v is None for v in z):
            return None
        return dict(value=(sum(z[:len(plus)]) - sum(z[len(plus):])) * HARTREE_KCAL, level="own_b3lyp", aligned=False)
    got = D.energies(system, plus + minus, method, basis, level)
    if got is None:
        return None
    rows, lab = got
    aligned = False
    dfs = {sp: bool(r["x"].get("df")) for sp, r in rows.items()}
    if len(set(dfs.values())) > 1:                       # review row 11: fitted and unfitted energies in one barrier
        fit_levels = {sp: rows[sp]["level"] for sp in rows}
        new = {}
        for sp, f in dfs.items():
            if not f:
                a = D.aligned(system, sp, method, basis, fit_levels[sp], df=True)
                if a is None:
                    D.mixed[(system, method, basis)] = dict(species=sorted(sp for sp, f2 in dfs.items() if f2), partners_missing=sorted(sp for sp, f2 in dfs.items() if not f2), aligned=False)
                    return None
                new[sp] = a
        rows = dict(rows, **new); aligned = True
        D.mixed[(system, method, basis)] = dict(species=sorted(sp for sp, f2 in dfs.items() if f2), partners_missing=[], aligned=True)
    e = sum(rows[sp]["E"] for sp in plus) - sum(rows[sp]["E"] for sp in minus)
    return dict(value=e * HARTREE_KCAL, level=lab, aligned=aligned)


def quantity(D, system, q, method, basis, zpe=False, level=None, **kw):
    """Value (kcal/mol) of a quantity or None.  See quantity_info."""
    i = quantity_info(D, system, q, method, basis, zpe=zpe, level=level)
    return None if i is None else i["value"]


def reference_quantity(D, system, q, basis):
    """The CCSD(T) reference of a quantity: basis-corrected when the MP2 rows exist (review row 5), else raw.
    Returns dict(value, raw, corrected: bool, level) or None."""
    c = quantity_info(D, system, q, M_CCSDT_BC, basis)
    r = quantity_info(D, system, q, M_CCSDT_FC, basis)
    if c is not None:
        return dict(value=c["value"], raw=None if r is None else r["value"], corrected=True, level=c["level"])
    if r is not None:
        return dict(value=r["value"], raw=r["value"], corrected=False, level=r["level"])
    return None


def compare_to_lit(D, lit, method, basis, level=None):
    """Computed value after the zero-point rule of plan 4.1.  Returns dict(value, diff, note, forced_warn, level) or None."""
    if lit.get("value") is None:
        return None
    qi = quantity_info(D, lit["system"], lit["quantity"], method, basis, level=level)
    if qi is None:
        return None
    el, lab = qi["value"], qi["level"]
    zp = lit["zp"]
    dz = quantity(D, lit["system"], lit["quantity"], method, basis, zpe=True)
    base = dict(level=lab, aligned=qi["aligned"])
    if zp == "excluded":
        return dict(base, value=el, diff=el - lit["value"], note="electronic energy vs zero-point-excluded reference", forced_warn=False)
    if zp == "included":
        if dz is None:
            return dict(base, value=None, diff=None, note="needs the B3LYP zero-point energies from C1", forced_warn=False, missing_zpe=True)
        v = el + dz
        return dict(base, value=v, diff=v - lit["value"], note=f"zero-point difference {dz:+.2f} added", forced_warn=False)
    # unknown convention (review row 6): the electronic value is compared, nothing is picked to look better; never better than WARN
    return dict(base, value=el, diff=el - lit["value"], note="zero-point convention unknown: electronic value shown, not scored", forced_warn=True)


def pass_tolerance(tol, ref):
    """PASS limit; for small references (|ref| below 10 kcal/mol) at most max(0.5, 10% of |ref|)  (review row 7)."""
    if ref is not None and abs(ref) < SMALL_REF:
        return min(tol[0], max(REL_FLOOR, REL_FRAC * abs(ref)))
    return tol[0]


def grade(diff, tol, ref=None):
    d = abs(diff)
    return PASS if d <= pass_tolerance(tol, ref) else (WARN if d <= tol[1] else FAIL)


# ------------------------------------------------------------------ report container
class Report:
    def __init__(self):
        self.items = []

    def add(self, item, scope, status, value="", ref="", diff="", tol="", msg="", raw_value="", raw_difference=""):
        self.items.append(dict(item=item, scope=scope, status=status, value=value, reference=ref, difference=diff, tolerance=tol, message=msg,
                               raw_value=raw_value, raw_difference=raw_difference))
        fmt = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)
        print(f"{status:8s} {item:4s} {scope:22s} value={fmt(value):>9s} ref={fmt(ref):>9s} diff={fmt(diff):>8s} tol={tol!s:>9s}  {msg}")

    def status(self, item, scope):
        return [x["status"] for x in self.items if x["item"] == item and sys_of(x["scope"]) == scope]

    def counts(self):
        c = {k: 0 for k in (PASS, WARN, FAIL, NOTRUN, NA, INFO)}
        for x in self.items:
            c[x["status"]] += 1
        return c


def sys_of(scope):
    """The system key inside an item scope such as 'L1 hcn:barrier_fwd' or 'nh3 CCSD(T)'."""
    for tok in re.split(r"[ :]", scope):
        if tok in SYSTEMS:
            return tok
    return ""


def _fnum(x, n=2):
    return "" if x is None else round(float(x), n)


# ------------------------------------------------------------------ literature checks
R2_CAUSE = {"sn2": "missing diffuse functions, or wrong fragment reference (complex vs separated)", "h2co": "zero-point handling",
            "ch3nc": "zero-point handling (convention unknown)", "other": "geometry level or functional string"}


def literature_checks(D, R):
    for lit in LIT:
        s = lit["system"]; tier = D.tiers.get(s, SYSTEMS[s]["tier"]); mb, lb = basis_for(s, "main"), basis_for(s, "large")
        tag = f"{lit['id']} {s}:{lit['quantity']}"
        if not lit_scorable(lit):                     # placeholder row (L19): nothing to compare with yet
            R.add("R3", tag, WARN, "", "", "", "never PASS", f"PLACEHOLDER, value not yet read, excluded from all statistics. Source: {lit['source'][:90]}")
            continue
        unverified = lit["verified"] != "yes"
        # --- R1 protocol: our CCSD(T) (basis-corrected) vs literature (tier A and B only)
        if tier not in ("A", "B"):
            R.add("R1", tag, NA, msg="tier C: no CCSD(T) planned for this system")
        else:
            c = compare_to_lit(D, lit, M_CCSDT_BC, mb); corrected = True
            raw = compare_to_lit(D, lit, M_CCSDT_FC, mb)
            if c is None or c.get("value") is None:
                c, corrected = raw, False
            if c is None or c.get("value") is None:
                R.add("R1", tag, NOTRUN, msg="no usable CCSD(T) rows at one geometry level" + (" (zero-point energies missing)" if c is not None else " (excluded by I2/I3, missing, or species at different levels)"))
            else:
                st = grade(c["diff"], TOL["R1"], lit["value"])
                if unverified or c["forced_warn"]:
                    st = WARN                                    # never PASS, and never FAIL, on an unverified reference
                elif not corrected and st == PASS:
                    st = WARN                                    # no PASS without the MP2 basis correction
                msg = c["note"] + f"; geometry level {c['level']}"
                msg += ("; reference = CCSD(T) + [MP2/" + basis_for(s, "large") + " - MP2/" + mb + "]") if corrected else "; MP2 basis correction NOT available: raw CCSD(T) used (no PASS possible)"
                if unverified:
                    msg += "; reference unverified: not scored"
                if st == WARN and not unverified:
                    msg += "; likely cause: remaining basis incompleteness, zero-point convention, or geometry level"
                if st == FAIL:
                    msg += "; RISK: a referee will doubt every reference number. FALLBACK: literature becomes the only reference for this system, CCSD(T) path errors go to the supplement"
                R.add("R1", tag, st, _fnum(c["value"]), lit["value"], _fnum(c["diff"]), f"{TOL['R1'][0]}/{TOL['R1'][1]}" + (f" (PASS {pass_tolerance(TOL['R1'], lit['value']):.2f})" if pass_tolerance(TOL['R1'], lit['value']) != TOL['R1'][0] else ""), msg,
                      raw_value=_fnum(raw["value"]) if raw and raw.get("value") is not None else "", raw_difference=_fnum(raw["diff"]) if raw and raw.get("value") is not None else "")
        # --- R2 anchor: wB97M-V (fallback wB97X-V) in the large basis vs literature (all systems)
        c, used, anchor = None, lb, M_WB97MV
        for anchor in ANCHORS:
            for b in (lb, mb):
                c = compare_to_lit(D, lit, anchor, b)
                if c is not None and c.get("value") is not None:
                    used = b; break
            if c is not None and c.get("value") is not None:
                break
        if c is None or c.get("value") is None:
            R.add("R2", tag, NOTRUN, msg="no wB97M-V (or fallback wB97X-V) rows at one geometry level (GPU notebook G1/G2 not run yet)")
        else:
            st = grade(c["diff"], TOL["R2"], lit["value"])
            if unverified or c["forced_warn"]:
                st = WARN if st in (PASS, FAIL) else st
            msg = c["note"] + f"; geometry level {c['level']}" + ("" if used == lb else f"; large basis {lb} not available, scored at {used}")
            if anchor != M_WB97MV:
                msg += f"; FALLBACK ANCHOR {anchor} used (wB97M-V does not run)"
            if unverified:
                msg += "; reference unverified: not scored"
            if st == WARN and not unverified:
                msg += "; likely cause: " + R2_CAUSE["sn2" if SYSTEMS[s]["sn2"] else (s if s in R2_CAUSE else "other")]
            if st == FAIL:
                msg += "; geometry or setup error suspected. FALLBACK: system leaves the main text (rule 4.4)"
            R.add("R2", tag, st, _fnum(c["value"]), lit["value"], _fnum(c["diff"]), f"{TOL['R2'][0]}/{TOL['R2'][1]}" + (f" (PASS {pass_tolerance(TOL['R2'], lit['value']):.2f})" if pass_tolerance(TOL['R2'], lit['value']) != TOL['R2'][0] else ""), msg)
        # --- R3 unverified reference
        if unverified:
            R.add("R3", tag, WARN, lit["value"], lit["value"], "", "never PASS",
                  f"verified={lit['verified']} ({lit['zp']} zero-point): EXCLUDED from all error statistics; confirm source before submission. Source: {lit['source'][:90]}")
        # --- R4 Skala (with and without D3) and the other functionals vs literature (reported, not scored)
        errs = {}
        for m in LADDER_ALL:
            c = compare_to_lit(D, lit, m, mb)
            if c is not None and c.get("value") is not None:
                errs[m] = c["diff"]
        if not errs:
            R.add("R4", tag, NOTRUN, msg="no functional rows (GPU notebook G1 not run yet)")
        else:
            sk = errs.get(M_SKALA)
            msg = "signed errors (method minus literature): " + ", ".join(f"{m} {e:+.2f}" for m, e in errs.items()) + ("; reference unverified, not in statistics" if unverified else "")
            if sk is not None and abs(sk) > TOL["R4_flag"]:
                R.add("R4", tag, WARN, _fnum(sk), lit["value"], _fnum(sk), f">{TOL['R4_flag']}", "Skala error is large: re-run that point from a fresh guess. " + msg)
            else:
                R.add("R4", tag, INFO, _fnum(sk), lit["value"], _fnum(sk), "not scored", msg)


# ------------------------------------------------------------------ internal checks
def _path(D, system, method, basis):
    rs = [r for k, v in D.idx.items() if k[0] == system and k[2] == method and k[3] == basis and is_path_id(k[1]) for r in v]
    rs = sorted([r for r in rs if r["conv"] and r["E"] is not None], key=lambda r: int(re.sub(r"\D", "", r["pid"]) or 0))
    return rs


def i1_scf(D, R):
    """SCF convergence.  A failed point removes THAT METHOD's entry (it is already left out of every quantity); it fails the SYSTEM only when
    a core method (DF-B3LYP geometries, CCSD(T), the anchor functional) is affected (review row 3)."""
    for s in SYSTEM_ORDER:
        rs = [r for r in D.rows if r["system"] == s and r["method"] != M_XTB and "@" not in r["pid"]]
        if not rs:
            R.add("I1", s, NOTRUN, msg="no rows"); continue
        good = {(r["pid"], r["method"], r["basis"], r["level"]) for r in rs if r["conv"]}          # a failed row that was later repeated and converged does not count
        rs = [r for r in rs if r["conv"] or (r["pid"], r["method"], r["basis"], r["level"]) not in good]
        stat = [r for r in rs if not is_path_id(r["pid"])]; pth = [r for r in rs if is_path_id(r["pid"])]
        bad_core = [f"{r['pid']}/{r['method']}" for r in stat if not r["conv"] and r["method"] in CORE_METHODS]
        bad_other = [f"{r['pid']}/{r['method']}" for r in stat if not r["conv"] and r["method"] not in CORE_METHODS]
        frac_core, frac_other = {}, {}
        for m in sorted({r["method"] for r in pth}):
            xs = [r["conv"] for r in pth if r["method"] == m]
            (frac_core if m in CORE_METHODS else frac_other)[m] = sum(xs) / len(xs)
        worst_core = min(frac_core.values()) if frac_core else None
        weak_other = {m: f for m, f in frac_other.items() if f < TOL["I1_path"][0]}
        frac = worst_core
        if bad_core:
            R.add("I1", s, FAIL, len(bad_core), "", "", "0 unconverged (core methods)", "unconverged stationary points of a core method: " + ", ".join(bad_core[:6]))
        elif worst_core is not None and worst_core < TOL["I1_path"][1]:
            R.add("I1", s, FAIL, _fnum(worst_core, 3), "", "", ">=0.95", f"only {worst_core:.1%} of path points converged for a core method")
        elif bad_other or weak_other or (worst_core is not None and worst_core < TOL["I1_path"][0]):
            msg = []
            if bad_other:
                msg.append("method entries removed (unconverged stationary points): " + ", ".join(bad_other[:8]))
            if weak_other:
                msg.append("path convergence below 98% for " + ", ".join(f"{m} {f:.0%}" for m, f in weak_other.items()))
            if worst_core is not None and worst_core < TOL["I1_path"][0]:
                msg.append(f"core method path convergence {worst_core:.1%}")
            R.add("I1", s, WARN, _fnum(frac, 3) if frac is not None else len(bad_other), "", "", ">=0.98 path, all stationary",
                  "; ".join(msg) + ". Only these method entries are dropped, the system stays (rule 4.4)")
        else:
            R.add("I1", s, PASS, _fnum(frac, 3) if frac is not None else "", "", "", ">=0.98 path, all stationary",
                  f"{len(stat)} stationary rows" + (f", {len(pth)} path rows" if pth else ""))


def i2_i3_cc(D, R):
    for s in SYSTEM_ORDER:
        rs = [r for r in D.rows if r["system"] == s and r["method"].startswith("CCSD(T)") and r["file"] != "phase06_ccsdt_basis.csv" and r["level"] != LV_XTB]
        if not rs:
            R.add("I2", s, NA if D.tiers.get(s) == "C" else NOTRUN, msg="no CCSD(T) rows" + (" (tier C)" if D.tiers.get(s) == "C" else ""))
            R.add("I3", s, NA if D.tiers.get(s) == "C" else NOTRUN, msg="no CCSD(T) rows")
            continue
        t1 = [(r["t1"], r["pid"]) for r in rs if r["t1"] is not None]
        if not t1:
            R.add("I2", s, NOTRUN, msg="no T1 values")
        else:
            mx = max(t1)
            flagged = [f"{p}({v:.3f})" for v, p in t1 if v >= TOL["I2"][0]]
            st = PASS if mx[0] < TOL["I2"][0] else (WARN if mx[0] <= TOL["I2"][1] else FAIL)
            msg = f"max T1 {mx[0]:.4f} at {mx[1]}" + (f"; flagged: {', '.join(flagged[:6])}" if flagged else "")
            if st == FAIL:
                msg += "; point(s) excluded from reference statistics"
            R.add("I2", s, st, _fnum(mx[0], 4), "", "", f"<{TOL['I2'][0]} / {TOL['I2'][1]}", msg)
        stab = [(r["x"].get("stable"), r["pid"]) for r in rs if r["x"].get("stable") not in (None, "")]
        if not stab:
            R.add("I3", s, NOTRUN, msg="no stability results")
        else:
            bad = [p for v, p in stab if str(v) in ("0", "False")]
            R.add("I3", s, FAIL if bad else PASS, len(bad), "", "", "all stable", ("unstable RHF, points excluded: " + ", ".join(bad[:6])) if bad else f"{len(stab)} points stable")


def i4_imag(D, R):
    for s in SYSTEM_ORDER:
        sts, msgs, n_own, n_db = [], [], 0, 0
        for sp, kind in SPECIES_KIND[s].items():
            own = [r for r in D.rows if r["system"] == s and r["pid"] == sp and r["method"] == M_DFB3LYP and r["level"] in (LV_OWN, LV_SCANPEAK) and r["imag"] is not None]
            if own:
                n_own += 1
                want = 1 if kind == "ts" else 0
                if own[0]["imag"] != want:
                    sts.append(FAIL); msgs.append(f"{sp}: {own[0]['imag']} imaginary modes, expected {want}")
            elif any(r["system"] == s and r["pid"] == sp and r["level"] == LV_DB for r in D.rows):
                n_db += 1
        if n_own == 0 and n_db == 0:
            R.add("I4", s, NOTRUN if s != "ch4" else NA, msg="no frequency data" if s != "ch4" else "single minimum; not needed"); continue
        if not sts:
            if n_db and not n_own:
                sts.append(WARN); msgs.append("database geometry, not tested")
            else:
                sts.append(PASS); msgs.append(f"{n_own} own stationary points tested" + (f"; {n_db} database geometries not tested" if n_db else ""))
        R.add("I4", s, worst(sts), "", "", "", "TS 1, minima 0", "; ".join(msgs))


def _e_pair(D, s, sps, m, basis, level=None):
    """Energies of the species `sps` at one geometry family: dict species -> hartree, or None."""
    g = D.energies(s, sps, m, basis, level)
    return None if g is None else {k: r["E"] for k, r in g[0].items()}


def i5_i6(D, R):
    for s in SYSTEM_ORDER:
        S = SYSTEMS[s]
        mb = basis_for(s, "main")
        meths = [m for m in D.methods_with(s, mb) if m not in (M_CCSDT_BC, M_MP2DF, M_SKALA_NOD3, M_XTB)]
        # ---- I6: transition state above both minima.  SCORED on CCSD(T) and the anchor functional only (review row 3);
        #      for every other functional "no barrier" is a finding (INFO), never a reason to drop the system.
        if S["ts"]:
            lows = [x for x in dict.fromkeys(([S["rc"], S["pc"]] if S["sn2"] else S["reactants"][:1] + (S["products"] if len(S["products"]) == 1 and S["products"] != S["reactants"] else []))) if x]
            sum_prod = (not S["sn2"]) and len(S["products"]) > 1
            bad, ok, info = [], 0, []
            for m in meths:
                comps = [([S["ts"], sp], sp, lambda e, sp=sp: e[sp]) for sp in lows]
                if sum_prod:
                    comps.append(([S["ts"]] + S["products"], "sum of products", lambda e: sum(e[q] for q in S["products"])))
                for need, label, f in comps:
                    e = _e_pair(D, s, need, m, mb)
                    if e is None:
                        continue
                    below = e[S["ts"]] <= f(e)
                    if m in I6_SCORED:
                        ok += 1
                        if below:
                            bad.append(f"{m} (TS {(e[S['ts']] - f(e)) * HARTREE_KCAL:+.1f} vs {label})")
                    elif below:
                        info.append((m, label, (e[S["ts"]] - f(e)) * HARTREE_KCAL))
            R.add("I6", s, FAIL if bad else (PASS if ok else NOTRUN), len(bad), "", "", "TS above minima (CCSD(T), anchor)",
                  ("TS not above minimum: " + "; ".join(bad[:4])) if bad else (f"{ok} CCSD(T)/anchor comparisons all fine" if ok else "no CCSD(T) or anchor energies yet at one geometry level"))
            for m, label, d in info:
                R.add("I6", f"{s} {m}", INFO, _fnum(d), "", "", "not scored", f"no barrier with {m}: TS {d:+.1f} kcal/mol vs {label}; a finding for the paper, the system stays")
        else:
            R.add("I6", s, NA, msg="no transition state (bond stretch)")
        # ---- I5: path single-peaked, peak close to the TS energy
        if not S["ts"]:
            R.add("I5", s, NA, msg="no reaction path peak expected (bond stretch)"); continue
        done = False
        for m in [M_CCSDT_FC, M_WB97MV, M_B3LYP, M_DFB3LYP, M_PBE]:
            pts = _path(D, s, m, mb)
            if len(pts) < 5:
                continue
            r0 = S["reactants"][0] if not S["sn2"] else S["rc"]
            ee = _e_pair(D, s, [r0, S["ts"]], m, mb)
            e0 = ee[r0] if ee else (_e_pair(D, s, [r0], m, mb) or {}).get(r0)
            e0 = e0 if e0 is not None else pts[0]["E"]
            E = (np.array([r["E"] for r in pts]) - e0) * HARTREE_KCAL      # relative to the reactant minimum
            d = np.diff(E); d[np.abs(d) < TOL["I5_floor"]] = 0
            sg = np.sign(d[d != 0]); n_max = int(np.sum((sg[:-1] > 0) & (sg[1:] < 0))) if len(sg) > 1 else 0
            ets = ee[S["ts"]] if ee else None
            diff = None if ets is None else float(E.max() - (ets - e0) * HARTREE_KCAL)
            if n_max != 1:
                st, msg = FAIL, f"{n_max} maxima along {len(pts)} {m} path points (need exactly one)"
            elif diff is None:
                st, msg = WARN, "one maximum, but no TS/minimum energy to compare with"
            else:
                a = abs(diff); st = PASS if a <= TOL["I5_peak"][0] else (WARN if a <= TOL["I5_peak"][1] else FAIL)
                msg = f"one maximum ({len(pts)} {m} points); path peak minus TS energy {diff:+.2f}"
            R.add("I5", s, st, _fnum(diff), "", _fnum(diff), f"{TOL['I5_peak'][0]}/{TOL['I5_peak'][1]}", msg)
            done = True
            break
        if not done:
            R.add("I5", s, NOTRUN, msg="fewer than 5 path points in any method")


def i7_rank(D, R):
    mb = basis_for("hcn", "main")
    b = {m: quantity(D, "hcn", "barrier_fwd", m, mb) for m in (M_PBE, M_B3LYP, M_CCSDT_FC)}
    if any(v is None for v in b.values()):
        R.add("I7", "hcn", NOTRUN, msg="needs PBE, B3LYP and CCSD(T) at the HCN stationary points")
    else:
        ok = b[M_PBE] < b[M_B3LYP] < b[M_CCSDT_FC]
        R.add("I7", "hcn", PASS if ok else WARN, "", "", "", "PBE < B3LYP < CCSD(T)", f"barriers PBE {b[M_PBE]:.2f}, B3LYP {b[M_B3LYP]:.2f}, CCSD(T) {b[M_CCSDT_FC]:.2f}" + ("" if ok else "; order differs: check functional strings and dispersion flags"))
    for s in [k for k in SYSTEMS if SYSTEMS[k]["sn2"]]:
        mb = basis_for(s, "main")
        v = {m: quantity(D, s, "barrier_fwd_cplx", m, mb) for m in LADDER}
        if any(x is None for x in v.values()):
            R.add("I7", s, NOTRUN, msg="needs all five functionals"); continue
        ok = min(v, key=v.get) == M_PBE
        R.add("I7", s, PASS if ok else WARN, "", "", "", "PBE lowest", "central barriers: " + ", ".join(f"{m} {x:.2f}" for m, x in v.items()) + ("" if ok else "; PBE is not lowest: check functional strings and dispersion flags"))


def i8_basis(D, R):
    for s in REACTIONS:
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"
        mb, lb = basis_for(s, "main"), basis_for(s, "large")
        sh = {}
        for m in sorted({k[2] for k in D.idx if k[0] == s} - {M_CCSDT_BC, M_MP2DF, M_XTB}):
            a, b = quantity(D, s, q, m, mb), quantity(D, s, q, m, lb)
            if a is not None and b is not None:
                sh[m] = b - a
        if not sh:
            R.add("I8", s, NOTRUN, msg=f"no method has the barrier in both {mb} and {lb}"); continue
        w = max(sh.values(), key=abs)
        st = grade(w, TOL["I8"])
        msg = f"{mb} to {lb} shifts: " + ", ".join(f"{m} {x:+.2f}" for m, x in sh.items())
        if st == WARN:
            msg += "; quote the larger basis as the headline"
        if st == FAIL:
            msg += "; this system is reported at the large basis only"
        R.add("I8", s, st, _fnum(w), "", "", f"{TOL['I8'][0]}/{TOL['I8'][1]}", msg)


def i9_v4(D, R):
    if not HAVE_V4:
        R.add("I9", "v4", NOTRUN, msg="stored v4 numbers (b1v5_v4_stored) not available"); return
    # (a) energies of NB4 point(s) vs stored
    worst_d, n, per = 0.0, 0, []
    for m, key in V4_METHODS.items():
        for r in D.rows:
            if r["system"] == "hcn_nb4" and r["method"] == m and r["E"] is not None:
                k = int(r["pid"].split("_")[-1])
                d = abs(r["E"] - NB4_HCN_E[key][k]); n += 1; worst_d = max(worst_d, d); per.append(f"{m}[{k}] {d:.1e}")
    if n == 0:
        R.add("I9", "v4 energies", NOTRUN, msg="no rerun of stored NB4 points yet (C0 does CCSD(T); GPU G0 does the DFT methods)")
    else:
        st = PASS if worst_d <= TOL["I9_energy"][0] else (WARN if worst_d <= TOL["I9_energy"][1] else FAIL)
        msg = f"{n} stored energies rerun, largest |difference| {worst_d:.1e} hartree"
        if st == WARN:
            msg += "; version drift, state it"
        if st == FAIL:
            msg += "; STOP: do not mix old and new numbers"
        R.add("I9", "v4 energies", st, f"{worst_d:.1e}", "", "", f"{TOL['I9_energy'][0]:g}/{TOL['I9_energy'][1]:g} hartree", msg)
    # (b) path mean errors recomputed from all 12 points
    ref = np.array(NB4_HCN_E["ccsd_t"]); rel = lambda E: (np.array(E) - E[0]) * HARTREE_KCAL
    res = []
    for m, mae in V4_MAE.items():
        pts = {int(r["pid"].split("_")[-1]): r["E"] for r in D.rows if r["system"] == "hcn_nb4" and r["method"] == m and r["E"] is not None}
        if len(pts) == 12:
            E = [pts[k] for k in range(12)]
            res.append((m, float(np.mean(np.abs(rel(E) - rel(ref)))), mae))
    if not res:
        R.add("I9", "v4 path errors", NOTRUN, msg="needs all 12 NB4 HCN path points for PBE, B3LYP and Skala (GPU G0 extended)")
    else:
        st = worst([PASS if abs(a - b) <= TOL["I9_mae"] else FAIL for _, a, b in res])
        R.add("I9", "v4 path errors", st, "", "", "", f"{TOL['I9_mae']}", ", ".join(f"{m} {a:.3f} vs {b:.3f}" for m, a, b in res))
    # (c) scout values from the stored arrays
    out = []
    for s, stored in SCOUT_STORED.items():
        v = scout_barrier_kcal(s); out.append((s, v, stored))
    st = worst([PASS if abs(v - t) <= TOL["I9_scout"] else FAIL for _, v, t in out])
    R.add("I9", "v4 scout values", st, "", "", "", f"{TOL['I9_scout']}", ", ".join(f"{s} {v:.3f} vs {t:.3f}" for s, v, t in out))


def i10_fc(D, R):
    out = []
    for s in REACTIONS:
        mb = basis_for(s, "main")
        a, b = quantity(D, s, "barrier_fwd", M_CCSDT_AE, mb), quantity(D, s, "barrier_fwd", M_CCSDT_FC, mb)
        if a is not None and b is not None:
            out.append((s, a - b))
    if not out:
        R.add("I10", "all", NOTRUN, msg="no system has both all-electron and frozen-core CCSD(T) (phase C6)")
    for s, d in out:
        st = PASS if abs(d) <= TOL["I10"] else WARN
        R.add("I10", s, st, _fnum(d), "", "", f"{TOL['I10']}", f"all-electron minus frozen-core barrier {d:+.2f}" + ("" if st == PASS else "; larger: report both"))


def i11_geom(D, R):
    """Own B3LYP geometries (headline) against the published database geometries (cross-check), for every reaction that has both
    (review row 4), then Skala//Skala against Skala//reference."""
    for s in REACTIONS:
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"
        mb = basis_for(s, "main")
        if not SYSTEMS[s]["db"]:
            continue
        res = None
        for m in (M_CCSDT_FC,) + ANCHORS:
            basis = mb
            a, b = quantity(D, s, q, m, basis, level="own"), quantity(D, s, q, m, basis, level="db")
            if a is not None and b is not None:
                res = (m, a, b); break
        if res is None:
            R.add("I11", f"{s} own vs database", NOTRUN, msg="needs the barrier at both our own and the downloaded database geometries (database not downloaded, or G1 not run at both levels?)")
        else:
            m, a, b = res
            st = grade(a - b, (TOL["I11_geom"][0], TOL["I11_geom"][1]))
            R.add("I11", f"{s} own vs database", st, _fnum(a), _fnum(b), _fnum(a - b), f"{TOL['I11_geom'][0]}/{TOL['I11_geom'][1]}",
                  f"own minus database {m} barrier" + ("; report both barriers" if st == FAIL else ""))
    any_ = False
    for s in REACTIONS:
        mb = basis_for(s, "main")
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"
        sk = quantity(D, s, q, M_SKALA, mb, level=LV_SKALA)
        rf = quantity(D, s, q, M_SKALA, mb)
        if sk is not None and rf is not None:
            any_ = True
            st = grade(sk - rf, TOL["I11_skala"])
            R.add("I11", f"{s} Skala//Skala", st, _fnum(sk), _fnum(rf), _fnum(sk - rf), f"{TOL['I11_skala'][0]}/{TOL['I11_skala'][1]}", "Skala//Skala (TS = parabola vertex) minus Skala//reference barrier" + ("; report both barriers" if st == FAIL else ""))
    if not any_:
        R.add("I11", "Skala//Skala", NOTRUN, msg="GPU phase G4 not run yet")


def i12_scout(D, R):
    for s in REACTIONS:
        S = SYSTEMS[s]; mb = basis_for(s, "main"); rmin = S["rc"] or S["reactants"][0]
        xrows = [r for r in D.rows if r["system"] == s and r["level"] == LV_XTB and r["pid"] == S["ts"]]
        if any((not r["conv"]) or r["x"].get("no_peak") for r in xrows if r["method"] == M_XTB):
            R.add("I12", s, WARN, msg="xTB found no peak along the scan: recorded as a result (outside the recipe's validity)"); continue
        for m, tag in ((M_CCSDT_FC, "CCSD(T)"), (M_WB97MV, "wB97M-V"), (M_WB97XV, "wB97X-V")):
            x = _e_pair(D, s, [S["ts"], rmin], m, mb, level=LV_XTB)
            r = _e_pair(D, s, [S["ts"], rmin], m, mb)
            if x is None or r is None:
                continue
            xb, rb = (x[S["ts"]] - x[rmin]) * HARTREE_KCAL, (r[S["ts"]] - r[rmin]) * HARTREE_KCAL
            pen = xb - rb
            st = grade(pen, TOL["I12"])
            R.add("I12", f"{s} {tag}", st, _fnum(xb), _fnum(rb), _fnum(pen), f"{TOL['I12'][0]}/{TOL['I12'][1]}",
                  "scout (xTB geometries) minus reference-geometry barrier" + ("; system outside the recipe's validity (a result)" if st == FAIL else ""))
            break
        else:
            R.add("I12", s, NOTRUN, msg="no CCSD(T) or wB97M-V energies at xTB geometries (C4 / G3)")


def i13_spread(D, R):
    for s in REACTIONS:
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"; mb = basis_for(s, "main")
        v = {m: quantity(D, s, q, m, mb) for m in LADDER}
        std = [v[m] for m in (M_PBE, M_R2SCAN, M_B3LYP, M_WB97MV) if v[m] is not None]
        if len(std) < 2:
            R.add("I13", s, NOTRUN, msg="needs at least two standard functionals"); continue
        lo, hi = min(std), max(std); sk = v[M_SKALA]
        where = "no Skala value" if sk is None else ("inside" if lo <= sk <= hi else "outside")
        R.add("I13", s, INFO, _fnum(hi - lo), "", "", "information only", f"standard functionals {lo:.2f} to {hi:.2f} (range {hi - lo:.2f}); Skala {'' if sk is None else f'{sk:.2f} '}{where}")


def i14_retry_mode(D, R):
    """Review row 11: a barrier must not mix density-fitted (retry) and unfitted energies.  The GPU notebook recomputes the partner species
    with density fitting (pid 'SP@df'); the quantity then uses that partner.  Without it the barrier is dropped from every table."""
    n_rows = sum(1 for r in D.rows if r["file"] != "derived" and r["x"].get("route") is not None)
    if not n_rows:
        R.add("I14", "all", NOTRUN, msg="no GPU rows yet"); return
    n_df = sum(1 for r in D.rows if r["x"].get("df") and "@" not in r["pid"])
    for s in REACTIONS:
        mb = basis_for(s, "main")
        for m in LADDER_ALL + [M_WB97XV]:
            for q in {"barrier_fwd", "barrier_fwd_cplx"}:
                quantity_info(D, s, q, m, mb)                 # fills D.mixed
    shown = 0
    for (s, m, b), v in sorted(D.mixed.items()):
        shown += 1
        if v["aligned"]:
            R.add("I14", f"{s} {m}", INFO, "", "", "", "same retry mode", f"density-fitted retry for {', '.join(v['species'])}: partner(s) recomputed with density fitting, barrier uses them")
        else:
            R.add("I14", f"{s} {m}", WARN, "", "", "", "same retry mode",
                  f"barrier would mix density-fitted ({', '.join(v['species'])}) and unfitted ({', '.join(v['partners_missing'])}) energies: dropped from all tables until the partner is recomputed (rerun G1 of the GPU notebook)")
    if not shown:
        R.add("I14", "all", PASS, n_df, "", "", "no mixed retry modes", f"{n_df} density-fitted retry point(s) among {n_rows} GPU rows; no barrier mixes fitted and unfitted energies")
    n_path_df = sum(1 for r in D.rows if r["x"].get("df") and is_path_id(r["pid"]))
    if n_path_df:
        R.add("I14", "paths", INFO, n_path_df, "", "", "information", f"{n_path_df} path point(s) were computed with the density-fitting retry (error of order 0.1 kcal/mol in a path profile)")


def i15_basis_correction(D, R):
    """Size of the MP2 basis correction of the CCSD(T) barrier (review row 5)."""
    any_ = False
    for s in REACTIONS:
        if D.tiers.get(s, SYSTEMS[s]["tier"]) not in ("A", "B"):
            continue
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"; mb = basis_for(s, "main")
        ref = reference_quantity(D, s, q, mb)
        if ref is None:
            continue
        if not ref["corrected"]:
            R.add("I15", s, NOTRUN, msg="no MP2 basis correction rows (C3b) for the barrier species: raw CCSD(T) used"); continue
        any_ = True
        d = ref["value"] - ref["raw"]
        R.add("I15", s, WARN if abs(d) > 3.0 else INFO, _fnum(d), "", "", "information (WARN above 3)",
              f"MP2/{basis_for(s, 'large')} minus MP2/{mb} shifts the CCSD(T) {q} by {d:+.2f} (raw {ref['raw']:.2f} -> corrected {ref['value']:.2f}), level {ref['level']}")
    if not any_ and not [x for x in R.items if x["item"] == "I15"]:
        R.add("I15", "all", NOTRUN, msg="C3b (MP2 basis correction) has not run yet")


# ------------------------------------------------------------------ decision (rule 4.4) and paper-level rule
def decide(D, R):
    out = {}
    for s in SYSTEM_ORDER:
        S = SYSTEMS[s]
        if s == "ch4":
            out[s] = ("negative result (F6)", "bond stretch, not a reaction: T1 and stability decide whether the reference is used"); continue
        sts = lambda item: [x["status"] for x in R.items if x["item"] == item and sys_of(x["scope"]) == s]
        i1, i4, i6 = worst(sts("I1")), worst(sts("I4")), worst(sts("I6"))
        r2 = sts("R2"); r1 = [x for x in sts("R1") if x != NA]
        i2i3 = sts("I2") + sts("I3")
        if FAIL in (i1, i4, i6):
            why = [n for n, v in (("I1", i1), ("I4", i4), ("I6", i6)) if v == FAIL]
            out[s] = ("SUPPLEMENT", f"{'/'.join(why)} failed: retry once from a fresh initial guess; if it fails again, supplement with the failure stated in one sentence in the main text"); continue
        if FAIL in r2:
            out[s] = ("SUPPLEMENT", "R2 failed (wB97M-V far from literature): geometry or setup error suspected; failure stated in the main text"); continue
        missing = [n for n, v in (("I1", i1), ("I4", i4), ("I6", i6)) if v == NOTRUN]
        if r2 == [] or all(x == NOTRUN for x in r2):
            missing.append("R2")
        if D.tiers.get(s, S["tier"]) in ("A", "B") and (not r1 or all(x == NOTRUN for x in r1)) and FAIL not in i2i3:
            missing.append("R1")
        if missing:
            out[s] = ("INCOMPLETE", "not run yet: " + ", ".join(missing)); continue
        if FAIL in r1 or FAIL in i2i3:
            out[s] = ("MAIN, CCSD(T) role withdrawn", "R1 failed or I2/I3 exclude the transition state: kept on the literature reference"); continue
        out[s] = ("MAIN", "I1 (core methods), I4, I6 (CCSD(T) and anchor) not failed; R2 PASS or WARN" + (" (R1 too)" if r1 else ""))
    return out


def paper_level(dec):
    main = [s for s in REACTIONS if dec[s][0].startswith("MAIN")]
    inc = [s for s in REACTIONS if dec[s][0] == "INCOMPLETE"]
    classes = {SYSTEMS[s]["cls"] for s in main}
    n = len(main)
    if n >= 6 and len(classes) >= 3:
        v = "FULL PAPER as defined"
    elif n >= 4:
        v = "add verified extras, or submit as a short communication titled as a case study"
    else:
        v = "DO NOT SUBMIT: fix the protocol"
    if inc:
        v = f"PROVISIONAL ({len(inc)} system(s) still incomplete: {', '.join(inc)}); so far: " + v
    return f"{n} of 8 reactions in the main text, {len(classes)} classes: {v}"


# ------------------------------------------------------------------ main entry
def run_check(root=None, write=True):
    """Run everything.  Returns dict(items, counts, decisions, paper)."""
    if root:
        setup_root(root, mount=False)
    print("=" * 100); print("Paper B1 v5 automated check (C7)"); print("=" * 100)
    D = Data()
    print(f"CSV files found: {', '.join(D.files) or 'none'}  ({len(D.rows)} rows)\n")
    R = Report()
    for name, fn in (("literature checks R1-R4", literature_checks), ("I1", i1_scf), ("I2/I3", i2_i3_cc), ("I4", i4_imag), ("I5/I6", i5_i6),
                     ("I7", i7_rank), ("I8", i8_basis), ("I9", i9_v4), ("I10", i10_fc), ("I11", i11_geom), ("I12", i12_scout), ("I13", i13_spread),
                     ("I14", i14_retry_mode), ("I15", i15_basis_correction)):
        try:
            fn(D, R)
        except Exception as ex:                       # a bug in one check must never stop the others
            R.add(name.split()[0], "check code", WARN, msg=f"check crashed: {repr(ex)[:150]}")
    dec = decide(D, R)
    paper = paper_level(dec)
    c = R.counts()
    print("\n" + "-" * 100); print("Decision per system (rule 4.4)")
    for s in SYSTEM_ORDER:
        print(f"  {SYSTEMS[s]['n']} {SYSTEMS[s]['name']:18s} -> {dec[s][0]:30s} {dec[s][1]}")
    print("\nPaper-level rule:", paper)
    print("\nCounts: " + "  ".join(f"{k} {v}" for k, v in c.items()))
    if write:
        with open(path("check_report.csv"), "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=["item", "scope", "status", "value", "reference", "difference", "tolerance", "message", "raw_value", "raw_difference"]); w.writeheader(); w.writerows(R.items)
        md = ["# Check report, Paper B1 v5", "", f"Written {now()}.  Tolerances in kcal/mol unless stated.", "",
              "Counts: " + ", ".join(f"{k} {v}" for k, v in c.items()), "", "## Decision per system", "", "| # | System | Tier | Decision | Why |", "|---|---|---|---|---|"]
        for s in SYSTEM_ORDER:
            md.append(f"| {SYSTEMS[s]['n']} | {SYSTEMS[s]['name']} | {D.tiers.get(s, SYSTEMS[s]['tier'])} | {dec[s][0]} | {dec[s][1]} |")
        md += ["", f"**Paper-level rule:** {paper}", "", "## All items", "", "| status | item | scope | value | reference | difference | tolerance | message | raw CCSD(T) value | raw difference |", "|---|---|---|---|---|---|---|---|---|---|"]
        for x in R.items:
            md.append(f"| {x['status']} | {x['item']} | {x['scope']} | {x['value']} | {x['reference']} | {x['difference']} | {x['tolerance']} | {str(x['message']).replace('|', '/')} | {x.get('raw_value', '')} | {x.get('raw_difference', '')} |")
        open(path("check_report.md"), "w").write("\n".join(md) + "\n")
        print("\nWrote check_report.csv and check_report.md in", path(""))
    return dict(items=R.items, counts=c, decisions=dec, paper=paper)


# ===== b1v5_gpu_figs.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_gpu_figs.py  --  phase G5: figures and tables for Paper B1 v5, regenerated from the CSV files.
# Pasted into the G5 cell of the GPU notebook, after the check module (b1v5_check.py), whose classes
# (Data, quantity, compare_to_lit, ...) it reuses so that errors are computed exactly as the automated check does.
#
# Writes into the Drive folder:
#   figures/fig1_design.png ... fig6_hcn_map_ch4.png      the six main figures of plan section 1
#   numbers_v5.csv                                        every number used in a figure or table, one per line
#   table1_systems.csv, table2_errors.csv, table2_detail.csv, table3_scout_cost.csv   and  summary_tables.xlsx (if openpyxl is present)
#   and the analysis outputs of review section 4: findings_outcome.csv (F1 to F6 against the FROZEN criteria), path_statistics.csv,
#   basis_table.csv, scout_table.csv, geometry_table.csv, skala_d3_table.csv, reference_quality.csv, excluded_and_failed.csv
# A figure whose data is missing is skipped with a message; the other figures are still made.
# ==============================================================================
import os, csv, json, math
import numpy as np
try:
    COLUMNS  # noqa
except NameError:
    from b1v5_common import *  # noqa

NUMS = []                       # rows of numbers_v5.csv
FIG_STATUS = {}                 # figure/table name -> "made" or the reason it was skipped
# one colour per method, fixed (colour-blind-safe set, in the order of the reference palette); markers differ as well
METHOD_STYLE = {
    M_SKALA: ("#2a78d6", "o"), M_SKALA_NOD3: ("#7fb3ee", "h"), M_PBE: ("#eb6834", "v"), M_R2SCAN: ("#1baf7a", "^"), M_B3LYP: ("#e87ba4", "s"),
    M_WB97MV: ("#6250d6", "D"), "wB97X-V": ("#6250d6", "D"), M_CCSDT_FC: ("#0b0b0b", "P"), M_CCSDT_BC: ("#0b0b0b", "P"),
}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"


def _plt():
    import matplotlib
    try:
        matplotlib.use("Agg", force=False)
    except Exception:
        pass
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "font.size": 9,
                         "axes.edgecolor": "#898781", "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
                         "legend.frameon": False, "figure.dpi": 100, "savefig.dpi": 200, "axes.axisbelow": True})
    return plt


def _methods_present(D, basis_of=None):
    """Ladder methods that have any row (anchor shown under its real name)."""
    have_ = {r["method"] for r in D.rows}
    out = [m for m in (M_SKALA, M_SKALA_NOD3, M_WB97MV, M_WB97XV, M_B3LYP, M_R2SCAN, M_PBE) if m in have_]
    return out


def _save(fig, name, standin=False):
    os.makedirs(path("figures"), exist_ok=True)
    if standin:
        fig.text(0.5, 0.5, "STAND-IN DRY RUN\nnot Skala, not results", ha="center", va="center", fontsize=26, color="#c3c2b7", alpha=0.45, rotation=20)
    p = path("figures", name)
    fig.savefig(p, bbox_inches="tight")
    try:
        from IPython.display import display, Image
        display(Image(filename=p, width=760))
    except Exception:
        pass
    _plt().close(fig)
    print("  wrote", p)
    return p


def _nofig(plt, msg):
    fig, ax = plt.subplots(figsize=(5, 2))
    ax.text(0.5, 0.5, msg, ha="center", va="center", color=INK2); ax.set_axis_off()
    return fig


def num(table, system, quantity, method, basis, level, value, ref=None, note=""):
    NUMS.append(dict(table=table, system=system, quantity=quantity, method=method, basis=basis, geometry_level=level,
                     value="" if value is None else round(float(value), 4), reference="" if ref is None else round(float(ref), 4),
                     error="" if (value is None or ref is None) else round(float(value) - float(ref), 4), unit="kcal/mol", note=note))


def primary_q(system):
    return "barrier_fwd_cplx" if SYSTEMS[system]["sn2"] else "barrier_fwd"


def lit_primary(system, verified_only=True):
    """The primary literature barrier of a reaction (SN2: from the complex).  verified_only=True: None for unverified rows (L6)."""
    for l in LIT:
        if l["system"] == system and l["quantity"] == primary_q(system) and lit_scorable(l):
            if verified_only and not lit_in_stats(l):
                return None
            return l
    return None


def boot_ci(x, n=2000, seed=12345):
    """95% percentile bootstrap interval of the mean absolute error, resampling single values (kept for simple lists)."""
    x = np.abs(np.asarray(x, float))
    if len(x) < 3:
        return (float("nan"), float("nan"))
    rng = np.random.RandomState(seed)
    m = x[rng.randint(0, len(x), (n, len(x)))].mean(axis=1)
    return tuple(float(v) for v in np.percentile(m, [2.5, 97.5]))


def boot_by_reaction(groups, n=2000, seed=12345):
    """95% bootstrap interval of the mean absolute error, resampling REACTIONS (not rows): `groups` = {reaction: [signed errors]}.
    The rows of one reaction are dependent (L3 = L1 - L2, forward and reverse barriers, SN2 from reactants and from the complex), so
    they stay together.  Fewer than 3 reactions: no interval."""
    keys = [k for k, v in groups.items() if len(v)]
    if len(keys) < 3:
        return (float("nan"), float("nan"))
    rng = np.random.RandomState(seed)
    arr = [np.abs(np.asarray(groups[k], float)) for k in keys]
    means = []
    for _ in range(n):
        pick = rng.randint(0, len(keys), len(keys))
        means.append(float(np.concatenate([arr[i] for i in pick]).mean()))
    return tuple(float(v) for v in np.percentile(means, [2.5, 97.5]))


TIE = 0.5                       # kcal/mol: two mean absolute errors (or per-reaction errors) within this are a TIE (review row 8, F5)
CLASS_ORDER = ["H shift", "inversion", "elimination", "group shift", "SN2"]


def stat_methods(D):
    """Methods that get a line in Table 2: the five functionals (Skala with and without D3), the CCSD(T) reference (corrected and raw)."""
    have_ = {k[2] for k in D.idx}
    return [m for m in (M_SKALA, M_SKALA_NOD3, M_WB97MV, M_WB97XV, M_B3LYP, M_R2SCAN, M_PBE, M_CCSDT_BC, M_CCSDT_FC) if m in have_]


def headline_rows(D, method, kind="main"):
    """ONE primary barrier per reaction with a verified reference (SN2 from the complex).  Returns a list of dicts."""
    out = []
    for s in REACTIONS:
        l = lit_primary(s)
        if l is None:
            continue
        c = compare_to_lit(D, l, method, basis_for(s, kind))
        if c is not None and c.get("value") is not None:
            out.append(dict(system=s, cls=SYSTEMS[s]["cls"], lit_id=l["id"], computed=c["value"], literature=l["value"], error=c["diff"], level=c["level"]))
    return out


def pooled_rows(D, method, kind="main"):
    """Every verified literature row except the dependent L3 (supplement statistics)."""
    out = []
    for l in LIT:
        if not lit_in_stats(l):
            continue
        c = compare_to_lit(D, l, method, basis_for(l["system"], kind))
        if c is not None and c.get("value") is not None:
            out.append(dict(system=l["system"], cls=SYSTEMS[l["system"]]["cls"], lit_id=l["id"], computed=c["value"], literature=l["value"], error=c["diff"], level=c["level"]))
    return out


def _stat_line(method, kind, label, rows, by="system"):
    if not rows:
        return None
    e = np.array([r["error"] for r in rows], float)
    groups = {}
    for r in rows:
        groups.setdefault(r[by], []).append(r["error"])
    lo, hi = boot_by_reaction(groups)
    return dict(method=method, basis_kind=kind, set=label, n_rows=len(e), n_reactions=len(groups), MAE=round(float(np.mean(np.abs(e))), 3), MSE=round(float(np.mean(e)), 3),
                MaxAE=round(float(np.max(np.abs(e))), 3), MAE_ci95_low=round(lo, 3), MAE_ci95_high=round(hi, 3),
                geometry_level="; ".join(sorted({r["level"] for r in rows})), reactions=", ".join(sorted(groups)))


# ------------------------------------------------------------------ tables
def table1_systems(D):
    rows = []
    for s in SYSTEM_ORDER:
        S = SYSTEMS[s]
        lits = [l for l in LIT if l["system"] == s]
        fmt = lambda l: f"{l['id']} {l['quantity']} " + ("(value not yet read)" if l["value"] is None else f"{l['value']:+.2f}") + (f" +/- {l['unc']}" if l["unc"] else "") + ("" if l["verified"] == "yes" else " [UNVERIFIED, not in statistics]")
        rows.append(dict(number=S["n"], system=S["name"], key=s, cls=S["cls"], tier=D.tiers.get(s, S["tier"]),
                         species=", ".join(SPECIES_KIND[s]), literature=("; ".join(fmt(l) for l in lits) or "none (negative result)"),
                         source=("; ".join(sorted({l["source"][:70] for l in lits})) or ""), zero_point=S["zpe"], verified=", ".join(sorted({l["verified"] for l in lits})) or ""))
    return rows


def table2_errors(D):
    """Table 2 (review row 8).  Summary lines per method and basis: (a) HEADLINE = one primary barrier per verified reaction (SN2 from the complex),
    (b) per class, (c) pooled verified rows without the dependent L3 (supplement).  n is printed with every mean; the bootstrap resamples reactions.
    The detail table has every literature row with signed error, reference, convention, geometry level and basis (supplement)."""
    out, detail = [], []
    for m in stat_methods(D):
        for kind in ("main", "large"):
            if m in (M_CCSDT_BC, M_CCSDT_FC) and kind == "large":
                continue
            for l in LIT:
                if not lit_scorable(l):
                    continue
                c = compare_to_lit(D, l, m, basis_for(l["system"], kind))
                if c is None or c.get("value") is None:
                    continue
                prim = lit_primary(l["system"]) is l
                detail.append(dict(method=m, basis_kind=kind, basis=basis_for(l["system"], kind), lit_id=l["id"], system=l["system"], quantity=l["quantity"],
                                   computed=round(c["value"], 3), literature=l["value"], error=round(c["diff"], 3), zero_point_convention=l["zp"], geometry_level=c["level"],
                                   verified=l["verified"], in_headline=bool(prim and lit_in_stats(l)), in_pooled=lit_in_stats(l), note=c["note"][:80]))
                num("table2", l["system"], l["quantity"] + " " + l["id"], m, basis_for(l["system"], kind), c["level"], c["value"], l["value"], c["note"][:60])
            h = headline_rows(D, m, kind)
            line = _stat_line(m, kind, "HEADLINE: one primary barrier per reaction", h)
            if line:
                out.append(line)
            for cls in CLASS_ORDER:
                cl = _stat_line(m, kind, f"class {cls}", [r for r in h if r["cls"] == cls])
                if cl:
                    out.append(cl)
            pl = _stat_line(m, kind, "pooled verified rows, L3 excluded (supplement)", pooled_rows(D, m, kind))
            if pl:
                out.append(pl)
    return out, detail


def skala_d3_table(D):
    """Skala with and without D3, per reaction (review section 4, item 7)."""
    rows = []
    for s in REACTIONS:
        q, mb = primary_q(s), basis_for(s, "main")
        a, b = quantity_info(D, s, q, M_SKALA, mb), quantity_info(D, s, q, M_SKALA_NOD3, mb)
        if a is None and b is None:
            continue
        l = lit_primary(s, verified_only=False)
        lit_ok = l is not None and lit_in_stats(l)
        ref = reference_quantity(D, s, q, mb)
        rows.append(dict(system=SYSTEMS[s]["name"], key=s, quantity=q, geometry_level=(a or b)["level"],
                         skala_with_D3=None if a is None else round(a["value"], 3), skala_without_D3=None if b is None else round(b["value"], 3),
                         D3_contribution=None if a is None or b is None else round(a["value"] - b["value"], 3),
                         literature=None if l is None or l["value"] is None else l["value"], literature_verified=bool(lit_ok),
                         error_with_D3=None if (a is None or not lit_ok) else round(a["value"] - l["value"], 3),
                         error_without_D3=None if (b is None or not lit_ok) else round(b["value"] - l["value"], 3),
                         CCSDT_reference=None if ref is None else round(ref["value"], 3), reference_is_basis_corrected=None if ref is None else ref["corrected"]))
        for tag, v in (("with D3", a), ("without D3", b)):
            if v is not None:
                num("skala_d3", s, q, M_SKALA if tag == "with D3" else M_SKALA_NOD3, mb, v["level"], v["value"], l["value"] if lit_ok else None)
    return rows


def table3_scout_cost(D, record=True):
    """Scout check per reaction (review row 9): ALL functionals at the reference (own) geometry, the xTB geometries and, for Skala, the Skala-relaxed
    geometry; xTB's own barrier; the split of the error into geometry penalty and functional error; measured cost per point."""
    scout = []
    for s in REACTIONS:
        q, mb = primary_q(s), basis_for(s, "main")
        lit = lit_primary(s)
        ref = reference_quantity(D, s, q, mb)
        xtb = quantity(D, s, q, M_XTB, "GFN2", level=LV_XTB)
        target = ref["value"] if ref else (lit["value"] if lit else None)
        tname = ("CCSD(T)+dMP2" if ref and ref["corrected"] else ("CCSD(T) raw" if ref else ("literature" if lit else "none")))
        for m in [x for x in (M_SKALA, M_SKALA_NOD3, M_WB97MV, M_WB97XV, M_B3LYP, M_R2SCAN, M_PBE) if any(k[2] == x for k in D.idx if k[0] == s)]:
            a = quantity(D, s, q, m, mb)
            b = quantity(D, s, q, m, mb, level=LV_XTB)
            c = quantity(D, s, q, m, mb, level=LV_SKALA) if m in (M_SKALA, M_SKALA_NOD3) else None
            if all(v is None for v in (a, b, c)):
                continue
            scout.append(dict(system=SYSTEMS[s]["name"], key=s, method=m, quantity=q, literature=(lit["value"] if lit else None), reference_used=tname, reference=target,
                              CCSDT_ref_geometry=(ref["value"] if ref else None), ref_geometry=a, xtb_geometry=b, relaxed_geometry=c, xtb_own_barrier=xtb,
                              geometry_penalty_xtb=(None if a is None or b is None else b - a),
                              functional_error_at_ref_geometry=(None if a is None or target is None else a - target),
                              total_error_at_xtb_geometry=(None if b is None or target is None else b - target),
                              skala_relaxed_minus_reference_geometry=(None if a is None or c is None else c - a)))
            for tag, v in (("reference geometry", a), ("xTB geometry", b), ("own-method relaxed geometry (TS = parabola vertex)", c)):
                if v is not None and record:
                    num("table3", s, q, m, mb, tag, v, lit["value"] if lit else None)
    cost = {}
    for fn in ("phase11_dft.csv", "phase12_basis.csv", "phase13_scout.csv", "phase14_skala_relaxed.csv", "phase04_scout_ref.csv"):
        for r in read_rows(fn):
            try:
                sec = float(r["seconds"])
            except Exception:
                continue
            if r["converged"] in ("1", "True") and float(r["seconds"] or 0) > 0:
                cost.setdefault((r["method"], r["hardware"].split(", GPU")[-1] if "GPU" in r["hardware"] else "CPU"), []).append(sec)
    cost_rows = [dict(method=m, hardware=h, points=len(v), mean_seconds=round(float(np.mean(v)), 2), median_seconds=round(float(np.median(v)), 2)) for (m, h), v in sorted(cost.items())]
    for c in (cost_rows if record else []):
        num("table3", "", "seconds per point", c["method"], "", c["hardware"], c["mean_seconds"], None, f"{c['points']} points, unit seconds")
    return scout, cost_rows


# ------------------------------------------------------------------ figures
def fig1_design(D, plt, standin):
    cols = [("stationary\nenergies", lambda r: r["file"] == "phase11_dft.csv" and not is_path_id(r["pid"])),
            ("path\nenergies", lambda r: r["file"] in ("phase11_dft.csv", "phase16_optional.csv") and is_path_id(r["pid"])),
            ("CCSD(T)\nrows", lambda r: r["method"].startswith("CCSD(T)")),
            ("MP2 basis\ncorrection", lambda r: r["file"] == "phase08_mp2_basis.csv"),
            ("basis\ncheck", lambda r: r["file"] == "phase12_basis.csv"),
            ("xTB scout\n(G3)", lambda r: r["file"] == "phase13_scout.csv"),
            ("Skala-\nrelaxed (G4)", lambda r: r["file"] == "phase14_skala_relaxed.csv")]
    M = np.array([[sum(1 for r in D.rows if r["system"] == s and f(r) and r["E"] is not None) for _, f in cols] for s in SYSTEM_ORDER])
    fig, ax = plt.subplots(figsize=(8.2, 4.2))
    ax.imshow((M > 0).astype(float), cmap="Blues", vmin=0, vmax=2.2, aspect="auto")
    ax.grid(False)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c for c, _ in cols])
    ax.set_yticks(range(len(SYSTEM_ORDER)))
    ax.set_yticklabels([f"{SYSTEMS[s]['n']} {SYSTEMS[s]['name']}  ({SYSTEMS[s]['cls']}, tier {D.tiers.get(s, SYSTEMS[s]['tier'])})" for s in SYSTEM_ORDER])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, str(M[i, j]) if M[i, j] else "-", ha="center", va="center", color=INK, fontsize=8)
    ax.xaxis.tick_top(); ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title("Figure 1. Study design: number of energies saved per system and calculation type", loc="left", fontsize=9, pad=28)
    for i, s in enumerate(SYSTEM_ORDER):
        for j, (c, _) in enumerate(cols):
            num("fig1", s, "rows saved: " + c.replace("\n", " "), "", "", "", M[i, j])
    return _save(fig, "fig1_design.png", standin)


def fig2_barrier_errors(D, plt, standin):
    methods = [m for m in _methods_present(D) if m not in (M_CCSDT_FC, M_CCSDT_BC)]
    systems = [s for s in REACTIONS if lit_primary(s)]                      # verified references only: CH3NC (L6) is left out
    data = {m: [] for m in methods + [M_CCSDT_BC]}
    for s in systems:
        l = lit_primary(s)
        for m in data:
            c = compare_to_lit(D, l, m, basis_for(s, "main"))
            if c is None and m == M_CCSDT_BC:                               # no MP2 basis correction for this system: raw CCSD(T)
                c = compare_to_lit(D, l, M_CCSDT_FC, basis_for(s, "main"))
            v = None if c is None or c.get("value") is None else c["diff"]
            data[m].append(v)
            if v is not None:
                num("fig2", s, l["quantity"] + " error vs " + l["id"], m, basis_for(s, "main"), "preferred", c["value"], l["value"])
    if not any(v is not None for m in methods for v in data[m]):
        FIG_STATUS["fig2"] = "skipped: no functional energies at stationary points yet"
        return _save(_nofig(plt, "Figure 2: no functional energies yet"), "fig2_barrier_errors.png", standin)
    fig, ax = plt.subplots(figsize=(8.6, 4.0))
    w = 0.8 / len(methods)
    x = np.arange(len(systems))
    for k, m in enumerate(methods):
        v = np.array([np.nan if y is None else y for y in data[m]], float)
        ax.bar(x + (k - (len(methods) - 1) / 2) * w, v, w * 0.9, color=METHOD_STYLE[m][0], label=m, edgecolor="white", linewidth=0.5)
    v = np.array([np.nan if y is None else y for y in data[M_CCSDT_BC]], float)
    if np.isfinite(v).any():
        ax.plot(x, v, linestyle="none", marker="P", color=INK, markersize=7, label="CCSD(T) + MP2 basis correction (tiers A, B)")
    ax.axhline(0, color=INK, lw=0.8)
    ax.axhspan(-1.0, 1.0, color="#f0efec", zorder=0)
    ax.set_xticks(x); ax.set_xticklabels([f"{SYSTEMS[s]['n']} {SYSTEMS[s]['name']}" for s in systems], rotation=20, ha="right")
    ax.set_ylabel("signed error vs literature (kcal/mol)")
    ax.set_title(f"Figure 2. Barrier errors by functional, own B3LYP geometries (def2-TZVP, def2-TZVPD for SN2; shaded: +/- 1 kcal/mol; n = {len(systems)} reactions with a verified reference; "
                 "Skala-1.1 = with D3)", loc="left", fontsize=8)
    ax.legend(ncol=3, loc="best", fontsize=8)
    return _save(fig, "fig2_barrier_errors.png", standin)


def _path_series(D, system, method, basis):
    return {r["pid"]: (r["E"], r["coord"]) for r in _path(D, system, method, basis)}


def fig3_paths(D, plt, standin):
    systems = [s for s in ("hcn", "nh3", "h2co", "ch4") if s in SYSTEMS]
    fig, axes = plt.subplots(2, 2, figsize=(9, 6.2))
    made = 0
    for ax, s in zip(axes.ravel(), systems):
        mb = basis_for(s, "main")
        ref = _path_series(D, s, M_CCSDT_BC, mb)                              # basis-corrected reference (review row 5)
        raw_note = ""
        if len(ref) < 3:
            ref, raw_note = _path_series(D, s, M_CCSDT_FC, mb), " (raw CCSD(T): no MP2 correction)"
        ax.set_title(f"{SYSTEMS[s]['n']} {SYSTEMS[s]['name']}{raw_note}", loc="left", fontsize=8)
        label = SCANS[s]["label"] if s in SCANS else "C-H distance (A)"
        ax.set_xlabel(label); ax.set_ylabel("error vs CCSD(T)+dMP2 (kcal/mol)")
        if len(ref) < 3:
            ax.text(0.5, 0.5, "no CCSD(T) path rows", transform=ax.transAxes, ha="center", color=INK2); continue
        for m in _methods_present(D):
            if m in (M_CCSDT_FC, M_CCSDT_BC):
                continue
            ser = _path_series(D, s, m, mb)
            common = [p for p in sorted(ref) if p in ser]
            if len(common) < 3:
                continue
            e = np.array([(ser[p][0] - ser[common[0]][0]) - (ref[p][0] - ref[common[0]][0]) for p in common]) * HARTREE_KCAL
            xs = [ref[p][1] if ref[p][1] is not None else i for i, p in enumerate(common)]
            col, mk = METHOD_STYLE[m]
            ax.plot(xs, e, marker=mk, color=col, lw=1.2, ms=4, label=m)
            made += 1
            for p, x_, v in zip(common, xs, e):
                num("fig3", s, "path error " + p, m, mb, LV_SCAN, v, 0.0, f"coordinate {x_:.3f}; relative to first common point")
        ax.axhline(0, color=INK, lw=0.8)
    if made == 0:
        plt.close(fig)
        FIG_STATUS["fig3"] = "skipped: needs CCSD(T) path rows (CPU notebook C3) and path energies (G1)"
        return _save(_nofig(plt, "Figure 3: needs CCSD(T) paths and G1 paths"), "fig3_path_errors.png", standin)
    axes.ravel()[0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 3. Error along the relaxed paths against the basis-corrected CCSD(T) (energy relative to the first path point)", x=0.01, ha="left", fontsize=9)
    fig.tight_layout()
    return _save(fig, "fig3_path_errors.png", standin)


def fig4_basis(D, plt, standin):
    methods = [m for m in _methods_present(D) if m != M_CCSDT_FC]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    made = 0
    for ax, (a_kind, b_kind, title) in zip(axes, (("main", "large", "main to large basis (TZVP to QZVPP)"), ("small", "main", "small to main basis (SVP to TZVP)"))):
        systems = [s for s in REACTIONS]
        w = 0.8 / max(1, len(methods))
        x = np.arange(len(systems))
        for k, m in enumerate(methods):
            v = []
            for s in systems:
                q = primary_q(s)
                a, b = quantity(D, s, q, m, basis_for(s, a_kind)), quantity(D, s, q, m, basis_for(s, b_kind))
                v.append(np.nan if a is None or b is None else b - a)
                if a is not None and b is not None:
                    num("fig4", s, f"basis shift {a_kind}->{b_kind} of {q}", m, f"{basis_for(s, a_kind)}->{basis_for(s, b_kind)}", "preferred", b - a)
            v = np.array(v, float)
            if np.isfinite(v).any():
                made += 1
            ax.bar(x + (k - (len(methods) - 1) / 2) * w, v, w * 0.9, color=METHOD_STYLE[m][0], label=m, edgecolor="white", linewidth=0.5)
        for yv in (-0.5, 0.5):
            ax.axhline(yv, color=INK2, lw=0.8, ls="--")
        ax.axhline(0, color=INK, lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels([f"{SYSTEMS[s]['n']}" for s in systems])
        ax.set_xlabel("reaction number"); ax.set_title(title, loc="left", fontsize=9)
    axes[0].set_ylabel("barrier shift (kcal/mol)")
    axes[0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 4. Basis-set shifts of the barrier (dashed: +/- 0.5 kcal/mol)", x=0.01, ha="left", fontsize=9)
    if made == 0:
        FIG_STATUS["fig4"] = "skipped: no basis-check rows yet (G2)"
        plt.close(fig)
        return _save(_nofig(plt, "Figure 4: no basis-check rows yet (G2)"), "fig4_basis_shifts.png", standin)
    fig.tight_layout()
    return _save(fig, "fig4_basis_shifts.png", standin)


def fig5_scout(D, plt, standin):
    systems = [s for s in REACTIONS]
    x = np.arange(len(systems))
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), gridspec_kw=dict(width_ratios=[1.5, 1]))
    ax, ax2 = axes
    made = 0
    anc = anchor_method() if "anchor_method" in globals() else M_WB97MV
    for key, (lvl, mk, lab) in {"ref": (None, "o", "Skala (D3), own B3LYP geometry"), "xtb": (LV_XTB, "s", "Skala, xTB geometries (scout)"), "skala": (LV_SKALA, "^", "Skala, Skala-relaxed (TS = parabola vertex)")}.items():
        v = []
        for s in systems:
            val = quantity(D, s, primary_q(s), M_SKALA, basis_for(s, "main"), **({"level": lvl} if lvl else {}))
            v.append(np.nan if val is None else val)
            if val is not None:
                num("fig5", s, primary_q(s), M_SKALA, basis_for(s, "main"), lvl or "preferred", val, (lit_primary(s) or {}).get("value"))
        v = np.array(v, float)
        if np.isfinite(v).any():
            made += 1
        ax.plot(x, v, linestyle="none", marker=mk, color=METHOD_STYLE[M_SKALA][0], ms=7, mfc=(METHOD_STYLE[M_SKALA][0] if key == "ref" else "white"), label=lab)
    lv = [(lit_primary(s, verified_only=False) or {}).get("value", np.nan) for s in systems]
    lv = [np.nan if v is None else v for v in lv]
    ax.plot(x, lv, linestyle="none", marker="_", color=INK, ms=16, mew=2, label="literature")
    cc = [(reference_quantity(D, s, primary_q(s), basis_for(s, "main")) or {}).get("value") for s in systems]
    ax.plot(x, [np.nan if c is None else c for c in cc], linestyle="none", marker="P", color=INK, mfc="white", ms=7, label="CCSD(T) + dMP2")
    ax.set_xticks(x); ax.set_xticklabels([str(SYSTEMS[s]["n"]) for s in systems]); ax.set_xlabel("reaction number"); ax.set_ylabel("barrier (kcal/mol)")
    ax.legend(fontsize=7, loc="best")
    ax.set_title("three routes to the barrier", loc="left", fontsize=9)
    w = 0.38
    meths_pen = [m for m in (M_SKALA, anc, M_B3LYP, M_R2SCAN, M_PBE) if any(r["method"] == m and r["level"] == LV_XTB for r in D.rows)] or [M_SKALA, anc]
    w = 0.8 / max(1, len(meths_pen))
    for k, m in enumerate(meths_pen):
        pen = []
        for s in systems:
            a = quantity(D, s, primary_q(s), m, basis_for(s, "main"))
            b = quantity(D, s, primary_q(s), m, basis_for(s, "main"), level=LV_XTB)
            pen.append(np.nan if a is None or b is None else b - a)
            if a is not None and b is not None:
                num("fig5", s, "geometry penalty (xTB minus reference geometry)", m, basis_for(s, "main"), LV_XTB, b - a)
        ax2.bar(x + (k - (len(meths_pen) - 1) / 2) * w, pen, w * 0.9, color=METHOD_STYLE.get(m, ("#898781", "o"))[0], label=m, edgecolor="white")
    ax2.axhspan(-1, 1, color="#f0efec", zorder=0); ax2.axhline(0, color=INK, lw=0.8)
    ax2.set_xticks(x); ax2.set_xticklabels([str(SYSTEMS[s]["n"]) for s in systems]); ax2.set_xlabel("reaction number")
    ax2.set_ylabel("xTB minus reference geometry (kcal/mol)"); ax2.set_title("geometry penalty (shaded: +/- 1)", loc="left", fontsize=9)
    ax2.legend(fontsize=7)
    fig.suptitle("Figure 5. Scout check: Skala energies at xTB geometries against Skala's own barrier", x=0.01, ha="left", fontsize=9)
    if made == 0:
        FIG_STATUS["fig5"] = "skipped: no Skala barriers yet (G1/G3/G4)"
    fig.tight_layout()
    return _save(fig, "fig5_scout_check.png", standin)


def _v4_grid(a):
    return np.array([[np.nan if v is None else v for v in row] for row in a], float)


def fig6_hcn_map_ch4(D, plt, standin):
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
    cc_npz = path("phase05_hcn_grid.npz")
    have_grid = os.path.exists(cc_npz)
    if have_grid:
        z = np.load(cc_npz, allow_pickle=True)
        Ecc, theta, rr = z["E"], np.asarray(z["theta"], float), np.asarray(z["r"], float)

        def rel(E):
            hc = np.where(theta < 45)[0]
            i = np.unravel_index(np.nanargmin(np.where(np.isfinite(E[hc]), E[hc], np.nan)), E[hc].shape)
            return (E - E[hc][i]) * HARTREE_KCAL
        rc = rel(Ecc)
        for ax, (nm, G) in zip(axes[:2], (("Skala (stored v4 grid)", HCN_GRID_SKALA), ("PBE (stored v4 grid)", HCN_GRID_PBE))):
            try:
                Eg = _v4_grid(G)
                err = rel(Eg) - rc
                err = np.where(rc <= 120.0, err, np.nan)
                lim = np.nanmax(np.abs(err)) if np.isfinite(err).any() else 1.0
                lim = max(1.0, min(lim, 8.0))
                im = ax.imshow(err.T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim,
                               extent=[theta[0], theta[-1], rr[0], rr[-1]])
                ax.grid(False)
                fig.colorbar(im, ax=ax, label="error vs CCSD(T)/def2-SVP (kcal/mol)")
                ax.set_xlabel("theta (deg; 0 = HCN, 180 = HNC)"); ax.set_ylabel("r (A)"); ax.set_title(nm, loc="left", fontsize=9)
                n_ok = int(np.isfinite(err).sum())
                for (i, j), v in np.ndenumerate(err):
                    if np.isfinite(v):
                        num("fig6", "hcn_grid", "map error", nm.split()[0], "def2-SVP", LV_GRID, v, 0.0, f"theta {theta[i]:.1f}, r {rr[j]:.3f}")
                ax.text(0.02, 0.97, f"{n_ok} cells", transform=ax.transAxes, va="top", fontsize=7, color=INK2)
            except Exception as ex:
                ax.text(0.5, 0.5, "map not available:\n" + repr(ex)[:60], transform=ax.transAxes, ha="center", fontsize=7)
    else:
        for ax in axes[:2]:
            ax.text(0.5, 0.5, "no phase05_hcn_grid.npz\n(CPU notebook C5)", transform=ax.transAxes, ha="center", color=INK2); ax.set_axis_off()
        FIG_STATUS["fig6a"] = "skipped: no CCSD(T) HCN grid (C5)"
    ax = axes[2]
    mb = basis_for("ch4", "main")
    ref = _path_series(D, "ch4", M_CCSDT_FC, mb)
    made = 0
    if len(ref) >= 3:
        for m in _methods_present(D):
            if m == M_CCSDT_FC:
                continue
            ser = _path_series(D, "ch4", m, mb)
            common = [p for p in sorted(ref) if p in ser]
            if len(common) < 3:
                continue
            e = np.array([(ser[p][0] - ser[common[0]][0]) - (ref[p][0] - ref[common[0]][0]) for p in common]) * HARTREE_KCAL
            xs = [ref[p][1] if ref[p][1] is not None else i for i, p in enumerate(common)]
            col, mk = METHOD_STYLE[m]
            ax.plot(xs, e, marker=mk, color=col, lw=1.2, ms=4, label=m)
            made += 1
            for p, x_, v in zip(common, xs, e):
                num("fig6", "ch4", "stretch error " + p, m, mb, LV_SCAN, v, 0.0, f"C-H {x_:.3f} A")
        ax.axhline(0, color=INK, lw=0.8); ax.legend(fontsize=7, ncol=2)
    if made == 0:
        ax.text(0.5, 0.5, "no CH4 CCSD(T) or functional rows yet", transform=ax.transAxes, ha="center", color=INK2)
        FIG_STATUS["fig6b"] = "skipped: no CH4 stretch rows"
    ax.set_xlabel("C-H distance (A)"); ax.set_ylabel("error vs CCSD(T) (kcal/mol)"); ax.set_title("CH4 C-H stretch", loc="left", fontsize=9)
    fig.suptitle("Figure 6. HCN rigid-map error (left, middle) and CH4 stretch (right)", x=0.01, ha="left", fontsize=9)
    fig.tight_layout()
    return _save(fig, "fig6_hcn_map_ch4.png", standin)


# ------------------------------------------------------------------ analysis outputs of review section 4
# THE FROZEN OUTCOME CRITERIA.  Dated 2026-11-15, written before any real number exists; the same wording is in PLAN_experiments.md section 7.
# "supports" / "negative" / "inconclusive" are decided by exactly these rules.  Do not change them after a real number has been seen.
FROZEN_DATE = "2026-11-15"
FROZEN = {
    "F1": dict(finding="Barriers at relaxed stationary points, five functionals, headline set (one primary barrier per verified reaction, SN2 from the complex)",
               supports="Skala-1.1 (with D3) mean absolute error is at most 2.0 kcal/mol AND at most the wB97M-V (anchor) mean absolute error + 0.5",
               negative="Skala mean absolute error exceeds the better of B3LYP-D3(BJ) and r2SCAN by more than 0.5 kcal/mol",
               inconclusive="everything else (a tie within 0.5 with B3LYP or r2SCAN, or Skala above 2.0 without losing)"),
    "F2": dict(finding="Error along the relaxed Tier A paths (HCN, NH3, H2CO, CH4) against the basis-corrected CCSD(T)",
               supports="range (max minus min) of the signed Skala error is at most 1.0 kcal/mol on at least 3 of the 4 paths",
               negative="maximum absolute Skala error is at least 2.0 kcal/mol on at least 2 of the 4 paths",
               inconclusive="everything else; always inconclusive when fewer than 3 paths have a usable reference (R1 failed or rows missing)"),
    "F3": dict(finding="Basis-set stability of the primary barrier, def2-TZVP(D) to def2-QZVPP(D)",
               supports="Skala shift at most 0.5 kcal/mol AND Skala's rank among the five functionals unchanged, in at least 75% of the reactions with data (at least 4 reactions)",
               negative="Skala shift above 1.0 kcal/mol in at least 2 reactions",
               inconclusive="everything else, or fewer than 4 reactions with both bases"),
    "F4": dict(finding="Scout check: barriers at xTB geometries against the barrier at our own B3LYP geometries, every functional of G3",
               supports="geometry penalty below 1.0 kcal/mol in at least 75% of the reactions (6 of 8) for EVERY functional computed",
               negative="penalty above 3.0 kcal/mol in at least 3 reactions, or in every reaction of one class (SN2: all four)",
               inconclusive="everything else, or fewer than 4 reactions with xTB-geometry energies"),
    "F5": dict(finding="Robustness of Skala's rank among the five functionals (by absolute error; errors within 0.5 kcal/mol of each other tie)",
               supports="the most frequent rank (ties share a rank) occurs in at least 75% of the reactions (6 of 8)",
               negative="the most frequent rank occurs in fewer than 50% of the reactions",
               inconclusive="everything else, or fewer than 4 reactions"),
    "F6": dict(finding="Negative results kept: CH4 C-H stretch and the rigid HCN map against CCSD(T)/def2-SVP",
               supports="T1 below 0.02 (and RHF stable at the CH4 points) at at least 90% of the CH4 points and HCN grid cells in the 120 kcal/mol window",
               negative="that fraction is below 50%: the reference is truncated at the distance where it fails, and this is said in the paper",
               inconclusive="everything else"),
}


def _rank_with_ties(errs):
    """Rank (1 = smallest absolute error) of each method; errors within TIE of the better neighbour share the rank."""
    items = sorted(errs.items(), key=lambda kv: abs(kv[1]))
    ranks, cur, lead = {}, 0, None
    for m, e in items:
        if lead is None or abs(e) - lead > TIE:
            cur, lead = cur + 1, abs(e)
        ranks[m] = cur
    return ranks


def _functionals(D):
    """The five functionals of the paper: Skala (with D3), the anchor actually present, B3LYP, r2SCAN, PBE."""
    have_ = {k[2] for k in D.idx}
    anc = M_WB97MV if M_WB97MV in have_ else (M_WB97XV if M_WB97XV in have_ else M_WB97MV)
    return [M_SKALA, anc, M_B3LYP, M_R2SCAN, M_PBE]


def _path_errors(D, system, method, ref_method):
    mb = basis_for(system, "main")
    ref, ser = _path_series(D, system, ref_method, mb), _path_series(D, system, method, mb)
    common = [p for p in sorted(ref) if p in ser]
    if len(common) < 3:
        return None
    e = np.array([(ser[p][0] - ser[common[0]][0]) - (ref[p][0] - ref[common[0]][0]) for p in common]) * HARTREE_KCAL
    xs = [ref[p][1] if ref[p][1] is not None else float(i) for i, p in enumerate(common)]
    peak = int(np.argmax([ref[p][0] for p in common]))
    return dict(points=common, err=e, coord=xs, ref_peak_index=peak)


def path_statistics(D):
    """Review section 4, item 3: per Tier A system and method, error statistics along the path against the basis-corrected and the raw CCSD(T)."""
    rows = []
    for s in ("hcn", "nh3", "h2co", "ch4"):
        for ref_m, tag in ((M_CCSDT_BC, "CCSD(T)+dMP2"), (M_CCSDT_FC, "CCSD(T) raw")):
            for m in [x for x in (M_SKALA, M_SKALA_NOD3, M_WB97MV, M_WB97XV, M_B3LYP, M_R2SCAN, M_PBE) if any(k[2] == x and k[0] == s for k in D.idx)]:
                r = _path_errors(D, s, m, ref_m)
                if r is None:
                    continue
                e, k = r["err"], int(np.argmax(np.abs(r["err"])))
                where = "no reaction peak" if s == "ch4" else ("at the TS (+/- 1 point)" if abs(k - r["ref_peak_index"]) <= 1 else ("before the TS" if k < r["ref_peak_index"] else "after the TS"))
                rows.append(dict(system=SYSTEMS[s]["name"], key=s, method=m, reference=tag, n_points=len(e), mean_abs_error=round(float(np.mean(np.abs(e))), 3),
                                 max_abs_error=round(float(np.max(np.abs(e))), 3), max_at_coordinate=round(float(r["coord"][k]), 4), max_location=where,
                                 error_min=round(float(e.min()), 3), error_max=round(float(e.max()), 3), error_range=round(float(e.max() - e.min()), 3)))
    return rows


def basis_table(D):
    """Review section 4, item 4: barrier shifts SVP, TZVP, QZVPP per method; rank change yes or no (Skala's rank among the functionals, main against large basis)."""
    rows, fn = [], _functionals(D)
    for s in REACTIONS:
        q = primary_q(s)
        b = {k: basis_for(s, k) for k in ("small", "main", "large")}
        l = lit_primary(s)
        err = {kind: {} for kind in ("main", "large")}
        for m in fn:
            for kind in ("main", "large"):
                c = compare_to_lit(D, l, m, b[kind]) if l else None
                if c is not None and c.get("value") is not None:
                    err[kind][m] = c["diff"]
        rk = {kind: (_rank_with_ties(err[kind]) if len(err[kind]) == len(fn) else None) for kind in err}
        for m in [x for x in (M_SKALA, M_SKALA_NOD3) + tuple(fn[1:]) if any(k[2] == x and k[0] == s for k in D.idx)]:
            v = {k: quantity(D, s, q, m, b[k]) for k in b}
            if all(x is None for x in v.values()):
                continue
            sh = lambda a_, b_: None if v[a_] is None or v[b_] is None else round(v[b_] - v[a_], 3)
            changed = None
            if rk["main"] and rk["large"] and m in rk["main"] and m in rk["large"]:
                changed = "yes" if rk["main"][m] != rk["large"][m] else "no"
            rows.append(dict(system=SYSTEMS[s]["name"], key=s, method=m, quantity=q, small_basis=b["small"], barrier_small=None if v["small"] is None else round(v["small"], 3),
                             main_basis=b["main"], barrier_main=None if v["main"] is None else round(v["main"], 3), large_basis=b["large"],
                             barrier_large=None if v["large"] is None else round(v["large"], 3), shift_small_to_main=sh("small", "main"), shift_main_to_large=sh("main", "large"),
                             rank_main=None if not rk["main"] or m not in rk["main"] else rk["main"][m], rank_large=None if not rk["large"] or m not in rk["large"] else rk["large"][m],
                             rank_changed=changed))
    return rows


def scout_table(D):
    """Review section 4, item 5 (same content as Table 3, one line per reaction and functional)."""
    sc, _ = table3_scout_cost(D, record=False)
    return sc


def geometry_table(D):
    """Review section 4, item 6: the driving coordinate of the transition state (the key distance or angle) at every stored geometry level."""
    rows = []
    for s in REACTIONS:
        sp = SCANS.get(s)
        if not sp:
            continue
        r = dict(system=SYSTEMS[s]["name"], key=s, coordinate=sp["label"])
        for lv in (LV_OWN, LV_SCANPEAK, LV_DB, LV_XTB, LV_SKALA):
            g = geometry_of(s, "TS", lv)
            r[lv] = None if g is None else round(coord_value(sp["kind"], sp["idx"], g["atoms"]), 4)
        if r.get(LV_SKALA) is not None and r.get(LV_OWN) is not None:
            r["skala_minus_b3lyp"] = round(r[LV_SKALA] - r[LV_OWN], 4)
        if r.get(LV_XTB) is not None and r.get(LV_OWN) is not None:
            r["xtb_minus_b3lyp"] = round(r[LV_XTB] - r[LV_OWN], 4)
        if r.get(LV_DB) is not None and r.get(LV_OWN) is not None:
            r["database_minus_b3lyp"] = round(r[LV_DB] - r[LV_OWN], 4)
        if any(r.get(lv) is not None for lv in (LV_OWN, LV_SCANPEAK, LV_DB, LV_XTB, LV_SKALA)):
            rows.append(r)
    return rows


def reference_quality(D):
    """Review section 4, item 8: T1, RHF stability, frozen core, CCSD(T) against the literature (raw and basis-corrected)."""
    rows = []
    for s in REACTIONS:
        if D.tiers.get(s, SYSTEMS[s]["tier"]) not in ("A", "B"):
            continue
        mb = basis_for(s, "main"); q = primary_q(s)
        cc = [r for r in D.rows if r["system"] == s and r["method"] == M_CCSDT_FC and r["basis"] == mb and not is_path_id(r["pid"])]
        t1 = [r["t1"] for r in cc if r["t1"] is not None]
        stab = [r["x"].get("stable") for r in cc if r["x"].get("stable") not in (None, "")]
        l = lit_primary(s, verified_only=False)
        ref = reference_quantity(D, s, q, mb)
        ae, fc = quantity(D, s, "barrier_fwd", M_CCSDT_AE, mb), quantity(D, s, "barrier_fwd", M_CCSDT_FC, mb)
        rows.append(dict(system=SYSTEMS[s]["name"], key=s, tier=D.tiers.get(s, SYSTEMS[s]["tier"]), quantity=q, max_T1=None if not t1 else round(max(t1), 4),
                         rhf_stable_all=None if not stab else all(str(v) not in ("0", "False") for v in stab), frozen_core=True,
                         all_electron_minus_frozen_core=None if ae is None or fc is None else round(ae - fc, 3),
                         CCSDT_raw=None if ref is None else ref["raw"], CCSDT_plus_dMP2=None if ref is None or not ref["corrected"] else round(ref["value"], 3),
                         literature=None if l is None else l["value"], literature_verified=None if l is None else l["verified"],
                         raw_minus_literature=None if ref is None or ref["raw"] is None or l is None or l["value"] is None else round(ref["raw"] - l["value"], 3),
                         corrected_minus_literature=None if ref is None or not ref["corrected"] or l is None or l["value"] is None else round(ref["value"] - l["value"], 3),
                         geometry_level=None if ref is None else ref["level"]))
    return rows


def excluded_and_failed(D):
    """Review section 4, item 9: every point that is left out or failed, with the reason."""
    rows = []
    for r in D.rows:
        why = None
        if not r["conv"] or r["E"] is None:
            why = "SCF / calculation not converged" + (f": {r['x'].get('error')}" if r["x"].get("error") else "")
        elif (r["system"], r["pid"], r["level"], r["basis"], r["method"]) in D.excluded:
            why = f"excluded from reference statistics: T1 {r['t1']} or unstable RHF"
        elif r["x"].get("interior") is False:
            why = "Skala TS: no interior maximum in the window (parabola vertex not usable)"
        if why:
            rows.append(dict(system=r["system"], point=r["pid"], level=r["level"], method=r["method"], basis=r["basis"], file=r["file"], reason=why))
    for key, v in D.mixed.items():
        if not v["aligned"]:
            rows.append(dict(system=key[0], point=",".join(v["partners_missing"]), level="", method=key[1], basis=key[2], file="",
                             reason="barrier would mix density-fitted and unfitted energies; partner not recomputed"))
    p = path("phase_errors.csv")
    if os.path.exists(p):
        for r in csv.DictReader(open(p)):
            rows.append(dict(system=r["system"], point="", level="", method="", basis="", file="phase_errors.csv", reason=f"{r['phase']}: {r['error'][:140]}"))
    return rows


def _t1_fraction(D):
    """Fraction of CH4 stretch points with T1 < 0.02 and RHF stable, and of HCN grid cells (<= 120 kcal/mol) with T1 < 0.02."""
    out = {}
    ch4 = [r for r in D.rows if r["system"] == "ch4" and r["method"] == M_CCSDT_FC and is_path_id(r["pid"]) and r["conv"]]
    if ch4:
        good = [r for r in ch4 if (r["t1"] is not None and r["t1"] < TOL["I2"][0]) and r["x"].get("stable") not in (0, "0", False)]
        out["ch4"] = (len(good), len(ch4))
    z = path("phase05_hcn_grid.npz")
    if os.path.exists(z):
        try:
            d = np.load(z, allow_pickle=True)
            E, t1, theta = d["E"], d["t1"], np.asarray(d["theta"], float)
            ok = np.isfinite(E)
            hc = np.where(theta < 45)[0]
            emin = np.nanmin(E[hc])
            win = ok & ((E - emin) * HARTREE_KCAL <= 120.0)
            out["hcn_grid"] = (int(np.sum(win & np.isfinite(t1) & (t1 < TOL["I2"][0]))), int(np.sum(win)))
        except Exception:
            pass
    return out


def findings_outcome(D):
    """Review section 4, item 1: F1 to F6 against the frozen criteria: measured value, n, verdict, one sentence for the paper.
    A verdict is only as good as the data behind it: with missing data it says 'inconclusive (data missing)'."""
    rows = []
    fn = _functionals(D)
    anc = fn[1]

    def add(fid, measured, n, verdict, sentence):
        f = FROZEN[fid]
        rows.append(dict(finding=fid, description=f["finding"], criterion_supports=f["supports"], criterion_negative=f["negative"], criterion_inconclusive=f["inconclusive"],
                         frozen_on=FROZEN_DATE, measured=measured, n=n, verdict=verdict, sentence_for_the_paper=sentence))

    # ---- F1
    mae = {}
    for m in fn + [M_SKALA_NOD3]:
        h = headline_rows(D, m)
        if h:
            mae[m] = (float(np.mean([abs(r["error"]) for r in h])), len(h))
    if M_SKALA in mae and anc in mae:
        S, W = mae[M_SKALA][0], mae[anc][0]
        B = min([mae[m][0] for m in (M_B3LYP, M_R2SCAN) if m in mae] or [float("nan")])
        if S <= 2.0 and S <= W + 0.5:
            v = "supports"
        elif B == B and S > B + 0.5:
            v = "negative"
        else:
            v = "inconclusive"
        add("F1", "MAE (kcal/mol, n): " + ", ".join(f"{m} {a:.2f} ({n})" for m, (a, n) in mae.items()), mae[M_SKALA][1], v,
            f"On {mae[M_SKALA][1]} reactions with a verified reference Skala-1.1 (with D3) has a mean absolute barrier error of {S:.2f} kcal/mol against {W:.2f} for {anc}"
            + (f" and {B:.2f} for the better of B3LYP-D3(BJ) and r2SCAN" if B == B else "") + ".")
    else:
        add("F1", "Skala or anchor headline rows missing", 0, "inconclusive (data missing)", "")
    # ---- F2
    sk_paths, n_valid = [], 0
    for s in ("hcn", "nh3", "h2co", "ch4"):
        r = _path_errors(D, s, M_SKALA, M_CCSDT_BC)
        if r is not None:
            n_valid += 1
            sk_paths.append((s, float(r["err"].max() - r["err"].min()), float(np.max(np.abs(r["err"])))))
    if n_valid:
        flat = sum(1 for _, rg, _m in sk_paths if rg <= 1.0)
        big = sum(1 for _, _r, mx in sk_paths if mx >= 2.0)
        v = "inconclusive" if n_valid < 3 else ("supports" if flat >= 3 else ("negative" if big >= 2 else "inconclusive"))
        add("F2", "; ".join(f"{s}: range {rg:.2f}, max |error| {mx:.2f}" for s, rg, mx in sk_paths), n_valid, v,
            f"Along {n_valid} relaxed paths the Skala error against basis-corrected CCSD(T) has a range of {min(r for _, r, _ in sk_paths):.2f} to {max(r for _, r, _ in sk_paths):.2f} kcal/mol.")
    else:
        add("F2", "no Skala and CCSD(T) path rows", 0, "inconclusive (data missing)", "")
    # ---- F3
    shifts, rank_same, n_b = [], 0, 0
    for r in basis_table(D):
        if r["method"] == M_SKALA and r["shift_main_to_large"] is not None:
            n_b += 1
            shifts.append((r["key"], r["shift_main_to_large"]))
            if abs(r["shift_main_to_large"]) <= 0.5 and r["rank_changed"] in ("no", None):
                rank_same += 1
    if n_b:
        big = sum(1 for _, x in shifts if abs(x) > 1.0)
        v = "inconclusive" if n_b < 4 else ("supports" if rank_same >= math.ceil(0.75 * n_b) else ("negative" if big >= 2 else "inconclusive"))
        add("F3", "Skala shifts main to large basis: " + ", ".join(f"{k} {x:+.2f}" for k, x in shifts), n_b, v,
            f"The Skala barrier moves by at most {max(abs(x) for _, x in shifts):.2f} kcal/mol from the triple-zeta to the quadruple-zeta basis in {n_b} reactions.")
    else:
        add("F3", "no basis-check rows", 0, "inconclusive (data missing)", "")
    # ---- F4
    pen, n_react = {}, set()
    for r in table3_scout_cost(D, record=False)[0]:
        if r["geometry_penalty_xtb"] is not None and r["method"] in fn:
            pen.setdefault(r["method"], {})[r["key"]] = r["geometry_penalty_xtb"]
            n_react.add(r["key"])
    if pen:
        n_min = min(len(v) for v in pen.values())
        thr = math.ceil(0.75 * len(n_react))
        small_all = all(sum(1 for x in v.values() if abs(x) < 1.0) >= thr for v in pen.values())
        allp = {k: x for v in pen.values() for k, x in v.items()}
        n_big = len({k for v in pen.values() for k, x in v.items() if abs(x) > 3.0})
        sn2_all = all(any(abs(pen[m].get(k, 0)) > 3.0 for m in pen) for k in ("sn2_ff", "sn2_clcl", "sn2_fcl", "sn2_ohf"))
        v = "inconclusive" if len(n_react) < 4 else ("supports" if small_all else ("negative" if (n_big >= 3 or sn2_all) else "inconclusive"))
        add("F4", "geometry penalty (kcal/mol): " + "; ".join(f"{m}: " + ", ".join(f"{k} {x:+.2f}" for k, x in sorted(vv.items())) for m, vv in pen.items()), len(n_react), v,
            f"Barriers at GFN2-xTB geometries differ from those at B3LYP geometries by at most {max(abs(x) for x in allp.values()):.2f} kcal/mol over {len(n_react)} reactions and {len(pen)} functionals.")
    else:
        add("F4", "no xTB-geometry energies", 0, "inconclusive (data missing)", "")
    # ---- F5
    ranks = []
    for s in REACTIONS:
        l = lit_primary(s)
        if l is None:
            continue
        err = {}
        for m in fn:
            c = compare_to_lit(D, l, m, basis_for(s, "main"))
            if c is not None and c.get("value") is not None:
                err[m] = c["diff"]
        if len(err) == len(fn):
            ranks.append(_rank_with_ties(err)[M_SKALA])
    if ranks:
        from collections import Counter
        modal, cnt = Counter(ranks).most_common(1)[0]
        v = "inconclusive" if len(ranks) < 4 else ("supports" if cnt >= math.ceil(0.75 * len(ranks)) else ("negative" if cnt < 0.5 * len(ranks) else "inconclusive"))
        add("F5", f"Skala rank per reaction (1 = best, ties within {TIE}): {ranks}; most frequent rank {modal} in {cnt} of {len(ranks)}", len(ranks), v,
            f"Skala's rank among the five functionals is {modal} in {cnt} of {len(ranks)} reactions.")
    else:
        add("F5", "needs all five functionals at the stationary points", 0, "inconclusive (data missing)", "")
    # ---- F6
    fr = _t1_fraction(D)
    if fr:
        good, tot = sum(a for a, _ in fr.values()), sum(b for _, b in fr.values())
        frac = good / tot if tot else float("nan")
        v = "supports" if frac >= 0.9 else ("negative" if frac < 0.5 else "inconclusive")
        sk = _path_errors(D, "ch4", M_SKALA, M_CCSDT_BC) or _path_errors(D, "ch4", M_SKALA, M_CCSDT_FC)
        loc = "" if sk is None else f" The largest Skala error along the CH4 stretch is {np.max(np.abs(sk['err'])):.2f} kcal/mol at C-H = {sk['coord'][int(np.argmax(np.abs(sk['err'])))]:.2f} A."
        add("F6", "reference reliable (T1 < 0.02) at: " + ", ".join(f"{k} {a}/{b}" for k, (a, b) in fr.items()), tot, v,
            f"CCSD(T) is reliable (T1 below 0.02) at {good} of {tot} points of the CH4 stretch and the HCN map." + loc)
    else:
        add("F6", "no CH4 / HCN grid CCSD(T) rows", 0, "inconclusive (data missing)", "")
    return rows


# ------------------------------------------------------------------ driver
def g5_make_all(standin=False):
    """Make every figure and table.  Each one is guarded: a failure is logged and the next one still runs."""
    plt = _plt()
    D = Data()
    print(f"  G5 reads {len(D.rows)} rows from {len(D.files)} CSV files")
    NUMS.clear(); FIG_STATUS.clear()
    tables = {}
    with item_guard("phase15", "table1"):
        tables["table1_systems"] = table1_systems(D); FIG_STATUS["table1"] = "made"
    with item_guard("phase15", "table2"):
        t2, det = table2_errors(D)
        tables["table2_errors"], tables["table2_detail"] = t2, det; FIG_STATUS["table2"] = "made" if t2 else "empty (no functional rows yet)"
    with item_guard("phase15", "table3"):
        sc, cost = table3_scout_cost(D)
        tables["table3_scout"], tables["table3_cost"] = sc, cost; FIG_STATUS["table3"] = "made" if sc or cost else "empty"
    for key, fn in (("skala_d3_table", skala_d3_table), ("path_statistics", path_statistics), ("basis_table", basis_table), ("scout_table", scout_table),
                    ("geometry_table", geometry_table), ("reference_quality", reference_quality), ("excluded_and_failed", excluded_and_failed),
                    ("findings_outcome", findings_outcome)):
        with item_guard("phase15", key):
            tables[key] = fn(D); FIG_STATUS[key] = "made" if tables[key] else "empty (data missing)"
    for name, fn in (("fig1", fig1_design), ("fig2", fig2_barrier_errors), ("fig3", fig3_paths), ("fig4", fig4_basis), ("fig5", fig5_scout), ("fig6", fig6_hcn_map_ch4)):
        with item_guard("phase15", name):
            fn(D, plt, standin)
            FIG_STATUS.setdefault(name, "made")
    # CSV files
    def _write(name, rows):
        if not rows:
            return
        keys = []
        for r in rows:
            for k in r:
                if k not in keys:
                    keys.append(k)
        with open(path(name + ".csv"), "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
    for k, rows in tables.items():
        _write(k, rows)
    keys = ["table", "system", "quantity", "method", "basis", "geometry_level", "value", "reference", "error", "unit", "note"]
    with open(path("numbers_v5.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(NUMS)
    print(f"  numbers_v5.csv: {len(NUMS)} numbers")
    try:                                                      # one workbook with every table (needs openpyxl; the CSV files are the fallback)
        import pandas as pd
        with pd.ExcelWriter(path("summary_tables.xlsx"), engine="openpyxl") as xw:
            for k, rows in tables.items():
                if rows:
                    pd.DataFrame(rows).to_excel(xw, sheet_name=k[:31], index=False)
            pd.DataFrame(NUMS).to_excel(xw, sheet_name="numbers", index=False)
        print("  wrote summary_tables.xlsx")
    except Exception as ex:
        print("  summary_tables.xlsx not written (", repr(ex)[:80], ") - the table CSV files hold the same numbers")
    write_json("phase15_figures_status.json", FIG_STATUS)
    for k, v in FIG_STATUS.items():
        print(f"   {k}: {v}")
    return FIG_STATUS

**Cell 4: deadline queue, six-reaction design and GPU phases.** Only defines functions and constants.

In [ ]:
# ===== b1v5_1h.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_1h.py  --  the DEADLINE QUEUE and the six-reaction DESIGN of the 1-hour notebooks (PLAN_1hour.md, decisions D3, D4, D5).
# Pasted into BOTH 1-hour notebooks after b1v5_common.py / b1v5_cpu.py (and b1v5_gpu.py in the GPU notebook), in the same namespace.
# It REDEFINES a few names of b1v5_common (check_time) and adds the queue; nothing in the full-size files was changed.
#
# In plain words:
#   * ONE ordered list of calculations per notebook.  Before EVERY calculation `qrun` asks `q_admit`:
#         start only if   elapsed + 1.5 x (running mean time of this kind of calculation) < DEADLINE_MIN        (the deadline test)
#         and             phase elapsed + mean time <= the phase cap                                           (the cap test)
#     The mean is the running mean of the calculations of that kind done in THIS session; before the first one, a prior (measured in the
#     sandbox, scaled by a speed factor measured at the start) is used.  A calculation that fails the test is NOT started and is listed as
#     "not run (time)" in RUN_SUMMARY.md and queue_not_run.csv.  A phase with refused calculations is not marked DONE, so a second Run all
#     continues exactly there.
#   * After the core list comes a TAIL list (same admission test).  At DEADLINE_MIN nothing new is started; figures, check and summary
#     (FINAL_MIN) always run, whatever the clock says.
#   * Nothing here can start a calculation after the deadline; the worst overrun is the single calculation that was admitted (about 4 min).
# ==============================================================================
import os, sys, json, csv, time, contextlib, signal, threading
import numpy as np
try:
    b1v5_common_loaded = COLUMNS      # noqa
except NameError:
    from b1v5_common import *         # noqa

DEADLINE_MIN = 50.0          # no calculation is started after this many minutes (counted from the first line of the first cell)
FINAL_MIN = 5.0              # reserved for figures, check, summary
ADMIT_FACTOR = 1.5           # a calculation is admitted only if elapsed + ADMIT_FACTOR * mean < deadline
CALC_MAX_SEC_BASE = 300      # one SCF attempt may not run longer than this (the plan: no core calculation over about 4 min)
CALC_TOTAL_SEC = 420         # all retry attempts of one calculation together (GPU notebook): then the point is stored as failed and the queue goes on

# ------------------------------------------------------------------ the six-reaction design
DESIGN_REACTIONS = ["hcn", "nh3", "h2co", "ch3nc", "sn2_ff", "sn2_clcl"]
EXT_REACTIONS = ["sn2_fcl", "sn2_ohf"]
LIT_ALL = list(LIT) if "LIT" in globals() else None
if "ALL_REACTIONS" not in globals():
    ALL_REACTIONS = list(REACTIONS)          # the eight of the full design (b1v5_common)
    ALL_SYSTEM_ORDER = list(SYSTEM_ORDER)
DESIGN = dict(reactions=list(DESIGN_REACTIONS))


def set_design(reactions):
    """Restrict the notebook to these reactions (plus the CH4 stretch).  Rebinds REACTIONS, SYSTEM_ORDER, TIER_B and LIT in the notebook namespace,
    so that the check, the figures and the tables of the full-size modules see only this design."""
    g = globals()
    reactions = [r for r in ALL_REACTIONS if r in set(reactions)]
    DESIGN["reactions"] = list(reactions)
    g["REACTIONS"] = list(reactions)
    g["SYSTEM_ORDER"] = list(reactions) + ["ch4"]
    g["TIER_B"] = [k for k in reactions if SYSTEMS[k]["tier"] == "B"]
    if "LIT_ALL" in g and g["LIT_ALL"] is None:
        g["LIT_ALL"] = list(g["LIT"])
    if g.get("LIT_ALL"):
        g["LIT"][:] = [l for l in g["LIT_ALL"] if l["system"] in reactions]
    return list(reactions)


def lit_out_of_design():
    return [l for l in (LIT_ALL or []) if l["system"] not in DESIGN["reactions"]]


# ------------------------------------------------------------------ the queue
Q = dict(t0=None, deadline_min=DEADLINE_MIN, final_min=FINAL_MIN, phase=None, cap_t=None, final=False, means={}, counts={}, prior={}, scale=1.0,
         refused=[], caps={}, started=[], tail_names=[], log=[])
PRIOR_DEFAULT_S = 20.0


class CalcBudget(Exception):
    """One calculation (all its retry attempts together) used more than CALC_TOTAL_SEC: the point is stored as failed and the queue goes on."""


class QTimeout(PhaseTimeout):
    """The hard stop (deadline + final minutes) arrived inside one calculation."""


def q_start(deadline_min=None, t0=None, final_min=None):
    """Call once at the top of the notebook; t0 = time.time() taken in the very first line of the first cell (install time counts)."""
    if deadline_min is not None:
        Q["deadline_min"] = float(deadline_min)
    if final_min is not None:
        Q["final_min"] = float(final_min)
    Q["t0"] = t0 if t0 else time.time()
    _CLOCK["t0"] = Q["t0"]
    _CLOCK["total_deadline"] = None            # the queue has its own hard stop (check_time below)
    globals()["MAX_TOTAL_HOURS"] = round((Q["deadline_min"] + Q["final_min"]) / 60.0, 2)      # only the number printed in RUN_SUMMARY.md
    return Q["t0"]


def q_now_min():
    return (time.time() - Q["t0"]) / 60.0 if Q["t0"] else 0.0


def q_remaining_min():
    return Q["deadline_min"] - q_now_min()


def q_hard_stop_t():
    return Q["t0"] + 60.0 * (Q["deadline_min"] + Q["final_min"])


def check_time():
    """REPLACES b1v5_common.check_time (same name, so run_point and the optimisers pick it up): stops the phase at its limit and everything at the
    hard stop (deadline + final minutes), except in the final phases (figures, check, summary), which must always run."""
    now_ = time.time()
    if _CLOCK.get("deadline") and now_ > _CLOCK["deadline"]:
        raise PhaseTimeout(f"phase time limit reached in {_CLOCK['phase']}")
    if Q["t0"] and not Q["final"] and now_ > q_hard_stop_t():
        raise PhaseTimeout(f"hard stop ({Q['deadline_min'] + Q['final_min']:.0f} min) reached")
    if Q.get("calc_end") and now_ > Q["calc_end"]:
        raise CalcBudget(f"one calculation used more than {CALC_TOTAL_SEC} s with all its retries")


@contextlib.contextmanager
def phase(name, title="", max_min=None, force=False):
    """REPLACES b1v5_common.phase.  Same behaviour, plus one fix found by the 1-hour tests: a hard stop of the cell (KeyboardInterrupt, e.g. the Colab
    'interrupt execution' button, or SystemExit) is not an Exception, so the original marked the interrupted phase DONE and a rerun skipped its remaining
    points.  Here it is recorded as 'interrupted', gets no DONE marker, and the interrupt is passed on."""
    class _Ph:
        skip = False
    ph = _Ph()
    print(f"\n===== {name} {title} =====  (notebook time so far {elapsed_hours():.2f} h)")
    if is_done(name) and not force:
        print(f"  {name} already DONE (marker found) -> skipped. Delete DONE_{name} to run it again.")
        _set_status(name, status="already done", title=title)
        ph.skip = True
        yield ph
        return
    t0 = time.time()
    _CLOCK["phase"] = name
    _CLOCK["deadline"] = t0 + 60.0 * (max_min or DEFAULT_PHASE_MIN)
    status, note, hard = "done", "", None
    try:
        check_time()
    except PhaseTimeout as ex:
        ph.skip = True; status, note = "stopped (time limit)", str(ex)
        print("  ** not started:", ex)
    try:
        yield ph
    except PhaseTimeout as ex:
        status, note = "stopped (time limit)", str(ex)
        print("  ** phase stopped cleanly:", ex, "- rerun the notebook to continue, finished points are kept.")
    except Exception as ex:
        status, note = "FAILED", repr(ex)[:200]
        log_error(name, "(whole phase)", ex)
    except BaseException as ex:                      # KeyboardInterrupt, SystemExit: not DONE, and the interrupt goes on
        status, note, hard = "interrupted", repr(ex)[:200], ex
    finally:
        _CLOCK["deadline"] = None
        secs = time.time() - t0
        n_err = 0
        try:
            n_err = sum(1 for r in csv.DictReader(open(path(ERRORS_CSV))) if r["phase"] == name) if os.path.exists(path(ERRORS_CSV)) else 0
        except Exception:
            pass
        if status == "done" and n_err:
            status = "done with errors"
        if status == "done":
            mark_done(name, title)
        log_phase(name, secs, status)
        _set_status(name, status=status, title=title, seconds=round(secs, 1), errors=n_err, note=note, finished=now())
        print(f"  {name}: {status}, {secs / 60:.1f} min")
    if hard is not None:
        raise hard


def q_est(kind):
    """Expected seconds of one calculation of this kind: running mean of this session, else the prior (scaled by the measured speed factor)."""
    n = Q["counts"].get(kind, 0)
    if n:
        return Q["means"][kind]
    k = kind
    while True:
        if k in Q["prior"]:
            return Q["prior"][k] * Q["scale"]
        if ":" not in k:
            break
        k = k.rsplit(":", 1)[0]
    return PRIOR_DEFAULT_S * Q["scale"]


def q_record(kind, seconds):
    n = Q["counts"].get(kind, 0)
    Q["means"][kind] = (Q["means"].get(kind, 0.0) * n + float(seconds)) / (n + 1)
    Q["counts"][kind] = n + 1


def q_admit(kind):
    """(True, '') or (False, reason).  The deadline test and the cap test of the module header."""
    est = q_est(kind)
    now_ = time.time()
    if Q["t0"] and not Q["final"] and now_ + ADMIT_FACTOR * est > Q["t0"] + 60.0 * Q["deadline_min"]:
        return False, f"deadline: {q_now_min():.1f} min used, {ADMIT_FACTOR} x {est:.0f} s would pass minute {Q['deadline_min']:.0f}"
    if Q["cap_t"] and now_ + est > Q["cap_t"]:
        return False, f"phase cap of {Q['caps'].get(Q['phase'], '?')} min ({Q['phase']}): {est:.0f} s would pass it"
    return True, ""


@contextlib.contextmanager
def q_time_limit(sec):
    """Interrupt ONE calculation at the hard stop (where the operating system allows it: main thread, SIGALRM)."""
    if not sec or threading.current_thread() is not threading.main_thread() or not hasattr(signal, "setitimer"):
        yield
        return

    def _h(signum, frame):
        raise QTimeout("hard stop reached inside a calculation")
    old = signal.signal(signal.SIGALRM, _h)
    signal.setitimer(signal.ITIMER_REAL, float(sec))
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)


def _limited(fn):
    def g():
        remain = q_hard_stop_t() - time.time() - 20.0
        if "CALC_MAX_SEC" in globals():                     # GPU notebook: the per-attempt limit of b1v5_gpu follows the hard stop
            globals()["CALC_MAX_SEC"] = int(max(20, min(CALC_MAX_SEC_BASE, remain)))
        Q["calc_end"] = time.time() + CALC_TOTAL_SEC if "CALC_MAX_SEC" in globals() else None
        try:
            with q_time_limit(max(20.0, remain)):
                return fn()
        finally:
            Q["calc_end"] = None
    return g


def q_refuse(label, kind, reason):
    Q["refused"].append(dict(phase=Q["phase"], label=label, kind=kind, reason=reason, est_s=round(q_est(kind), 1), minute=round(q_now_min(), 1)))


def qrun(csv_name, system, pid, level, method, basis, fn, kind, coordinate="", retry_failed=False, label=None, quiet=True):
    """run_point with the admission test in front.  Returns the new row, or None (already done, or refused: see Q['refused'])."""
    if have(csv_name, system, pid, level, method, basis, retry_failed):
        return None
    ok, why = q_admit(kind)
    if not ok:
        q_refuse(label or f"{system}:{pid}:{method}:{basis}", kind, why)
        return None
    t0 = time.time()
    row = run_point(csv_name, system, pid, level, method, basis, _limited(fn), coordinate=coordinate, retry_failed=retry_failed, quiet=quiet)
    if row is not None and str(row["converged"]) in ("1", "True"):                    # run_point returns converged as an int, the CSV has text
        q_record(kind, time.time() - t0)
    return row


@contextlib.contextmanager
def qphase(name, title, cap_min, final=False, force=None):
    """A phase of the queue: `cap_min` is its cap.  A phase with refused calculations is left 'stopped (time limit)', never DONE, so that the next
    Run all continues.  final=True: figures / check / summary phases, exempt from the hard stop."""
    if force is None:
        force = bool(globals().get("FORCE_RERUN", False))               # the FORCE_RERUN switch of the settings cell
    Q["phase"], Q["cap_t"], Q["final"] = name, time.time() + 60.0 * cap_min, bool(final)
    Q["caps"][name] = cap_min
    Q["started"].append((name, round(q_now_min(), 1)))
    n0 = len(Q["refused"])
    try:
        with phase(name, title, max_min=cap_min + 2.0, force=force) as ph:
            yield ph
            k = len(Q["refused"]) - n0
            if k and not ph.skip:
                raise PhaseTimeout(f"{k} calculation(s) not run (time): a second Run all continues here")
    finally:
        Q["cap_t"], Q["final"] = None, False
        if len(Q["refused"]) > n0:
            print(f"  ** {len(Q['refused']) - n0} calculation(s) NOT RUN (time) in {name}; first: {Q['refused'][n0]['label']} ({Q['refused'][n0]['reason'][:80]})")


def q_save_log():
    """queue_log.json (means, counts, refused of this session) and queue_not_run.csv (appended)."""
    try:
        old = read_json("queue_log.json", {}) or {}
        sessions = old.get("sessions", [])
        sessions.append(dict(at=now(), elapsed_min=round(q_now_min(), 1), deadline_min=Q["deadline_min"], means={k: round(v, 2) for k, v in Q["means"].items()},
                             counts=dict(Q["counts"]), n_refused=len(Q["refused"]), phases=Q["started"], hardware=hw()))
        write_json("queue_log.json", dict(sessions=sessions))
        p = path("queue_not_run.csv")
        new = not os.path.exists(p)
        with open(p, "a", newline="") as fh:
            w = csv.writer(fh)
            if new:
                w.writerow(["time", "phase", "label", "kind", "estimate_s", "minute", "reason"])
            for r in Q["refused"]:
                w.writerow([now(), r["phase"], r["label"], r["kind"], r["est_s"], r["minute"], r["reason"]])
    except Exception as ex:
        print("could not write the queue log:", repr(ex)[:100])


def queue_lines():
    """Lines for RUN_SUMMARY.md: minutes against caps, what was not run (time), measured mean seconds per kind."""
    st = read_json("phase_status.json", {}) or {}
    L = ["", "## Deadline queue", "",
         f"Deadline: no calculation is started after minute {Q['deadline_min']:.0f}; {Q['final_min']:.0f} more minutes are reserved for figures, check and summary. "
         f"Minutes used at this point: {q_now_min():.1f}. Admission test: elapsed + {ADMIT_FACTOR} x running mean < deadline; a phase also stops at its cap.", "",
         "| phase | minutes used (this and earlier sessions) | cap (min) | status |", "|---|---|---|---|"]
    for k in sorted(st):
        if k in Q["caps"]:
            L.append(f"| {k} | {st[k].get('seconds', 0) / 60:.1f} | {Q['caps'][k]} | {st[k].get('status', '')} |")
    n = len(Q["refused"])
    L += ["", f"**Not run (time) in this session: {n} calculation(s).** Details in queue_not_run.csv; a second Run all continues with them."]
    if n:
        by = {}
        for r in Q["refused"]:
            by.setdefault(r["phase"], []).append(r)
        for ph, rs in by.items():
            L.append(f"- {ph}: {len(rs)} not run, first {rs[0]['label']} ({rs[0]['reason'][:90]}); last {rs[-1]['label']}")
    else:
        L.append("- none: every admitted calculation of the lists ran.")
    if Q["means"]:
        L += ["", "Measured mean seconds per calculation (this session): " + ", ".join(f"{k} {v:.1f} s (n={Q['counts'][k]})" for k, v in sorted(Q["means"].items()))]
    L += ["", f"Speed factor applied to the sandbox priors: {Q['scale']:.2f}"]
    return L


# ------------------------------------------------------------------ the geometry pack in a notebook
def pack_setup(b64_text, expected_sha256, systems=None, allow_mismatch=False, build_note=""):
    """Decode the embedded geometry pack, verify its SHA-256 and every structure's SHA-256, install it into geoms/ and the phase CSV files.
    Returns dict(ok, sha256, n_verified, problems, installed) and writes geompack_installed.json."""
    out = dict(ok=False, expected_sha256=expected_sha256, problems=[])
    try:
        pack, sha, match = load_pack_text(b64_text, expected_sha256)
        out["sha256"] = sha
        n_ok, bad = verify_hashes(pack)
        out.update(n_verified=n_ok, problems=bad, format=pack.get("format"), design=pack.get("design"), builder=pack.get("builder"))
        if not match:
            out["problems"].insert(0, f"pack SHA-256 {sha} differs from the expected {expected_sha256}")
        out["ok"] = bool(match and not bad)
        if out["ok"] or allow_mismatch:
            out["installed"] = install_pack(pack, systems, sha=sha)
        out["systems_in_pack"] = pack.get("systems")
        out["has"] = dict(ext78=all(s in pack.get("systems", []) for s in EXT_REACTIONS))
        out["_pack"] = pack
    except Exception as ex:
        out["problems"].append("could not read the pack: " + repr(ex)[:150])
    meta = {k: v for k, v in out.items() if k != "_pack"}
    write_json("geompack_installed.json", dict(meta, at=now()))
    return out


# ------------------------------------------------------------------ pip with a time cap (cell 1 of the notebooks)
def pip_install_capped(specs, timeout_min):
    """pip install with a wall-clock cap; returns (ok, seconds).  Never raises."""
    import subprocess
    t0 = time.time()
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(specs), capture_output=True, text=True, timeout=60 * timeout_min)
        print(("ok   " if r.returncode == 0 else "FAIL ") + " ".join(specs) + f"  ({time.time() - t0:.0f} s)")
        if r.returncode:
            print(r.stderr[-400:])
        return r.returncode == 0, time.time() - t0
    except Exception as ex:
        print("pip step stopped:", repr(ex)[:120], f"({time.time() - t0:.0f} s)")
        return False, time.time() - t0


# ===== b1v5_1h_check.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_1h_check.py  --  the automated check and the frozen criteria for the SIX-REACTION design of PLAN_1hour.md.
# Pasted after b1v5_check.py (and, in the GPU notebook, after b1v5_gpu_figs.py), in the same namespace.  It does not edit the full-size files:
# it REDEFINES a few functions (i11_geom, decide, paper_level, run_check, findings_outcome) and the FROZEN table.
#
# What changes for six reactions (plan section 5: "F1 and F5 thresholds become 5 of 6 where they were 6 of 8; fix that before the run"):
#   * the check and the figures see only the design (set_design in b1v5_1h.py rebinds REACTIONS, SYSTEM_ORDER and LIT);
#   * every item that cannot exist in the reduced design is printed as NOT RUN (never FAIL) and says why:
#       - literature rows L11 to L18 (F- + CH3Cl, OH- + CH3F): "outside the 1-hour core design (extension EXT-GPU-1)";
#       - I11 own versus database geometry: "dropped by design (decision D2: one geometry protocol, no database download)";
#       - I11 Skala//Skala: "Skala-relaxed geometries are not part of the core (extension EXT-GPU-2)";
#       - I8 / F3 where no QZVPP energies exist (CH3NC and the SN2 systems in the core), I9 path errors (only the first stored v4 point is rerun),
#         I10 (all-electron CCSD(T) for HCN only), CCSD(T) references that the queue did not reach (tier falls back to C): all NOT RUN;
#   * decisions: the two extension reactions get "NOT RUN (extension)", never INCOMPLETE or SUPPLEMENT;
#   * paper-level rule: n of the 6 design reactions (rule: at least 6 reactions in at least 3 classes: the core has exactly six, so one failure turns
#     the full paper into a short communication, as the plan says in section 6);
#   * FROZEN: thresholds are the same 75% / 50% rules, written for the design (F1, F5: 5 of 6 reactions when all six have a verified reference,
#     4 of 5 while CH3NC L6 is unverified; F4: 5 of 6; F3: at least 3 reactions instead of 4, because the core has the basis check for three).
# ==============================================================================
import math
import numpy as np

if "_decide_orig" not in globals():
    _decide_orig = decide                    # noqa  (the full-size rule 4.4, kept)
    _run_check_orig = run_check              # noqa

if "_load_rows_orig" not in globals():
    _load_rows_orig = load_rows              # noqa


def load_rows():
    """As b1v5_check.load_rows, without the gradient-test rows of phase00_verify.csv (method '...-grad'): they are a test, not an energy."""
    rows, files = _load_rows_orig()
    return [r for r in rows if not str(r["method"]).endswith("-grad")], files


NOT_IN_DESIGN = "outside the 1-hour core design (extension session EXT-GPU-1 adds F- + CH3Cl and OH- + CH3F)"


def pack_items(R):
    """P1: SHA-256 of the geometry pack; P2: the gradient test of C0 (RMS gradient below 3e-4 hartree/bohr at the stored geometries)."""
    info = read_json("geompack_installed.json", None)
    if not info:
        R.add("P1", "geometry pack", NOTRUN, msg="geompack_installed.json not found (the pack was not installed, or the geometries were rebuilt: REBUILD_GEOMS)")
    else:
        ok = bool(info.get("ok"))
        R.add("P1", "geometry pack", PASS if ok else FAIL, str(info.get("sha256", ""))[:16], str(info.get("expected_sha256", ""))[:16], "", "SHA-256 equal",
              ("pack and all %s structure/path hashes verified" % info.get("n_verified")) if ok else "pack did not verify: " + "; ".join(info.get("problems", []))[:200])
    try:
        g = grad_summary()
    except Exception:
        g = dict(n=0)
    if not g["n"]:
        R.add("P2", "gradient test", NOTRUN, msg="phase00_verify.csv has no gradient rows (C0 not run, or geometries rebuilt)")
    else:
        bad = g["failed"]
        R.add("P2", "gradient test", PASS if not bad else FAIL, "%.1e" % g["worst_rms"], "", "", "RMS < %g" % g["tolerance"],
              f"{g['passed']} of {g['n']} stored structures pass; worst {g['worst_rms']:.1e} at {g['worst_where']}" + ("" if not bad else "; the stored geometries are not stationary on this machine: treat every barrier with caution (EXT-geom rebuilds them)"))


def design_notes(D, R):
    """NOT RUN lines for everything the six-reaction design cannot do (review: NOT RUN, never FAIL)."""
    pack_items(R)
    for lit in lit_out_of_design():
        tag = f"{lit['id']} {lit['system']}:{lit['quantity']}"
        R.add("R1", tag, NOTRUN, msg=NOT_IN_DESIGN)
        R.add("R2", tag, NOTRUN, msg=NOT_IN_DESIGN)
    for s in ALL_REACTIONS:
        if s not in DESIGN["reactions"]:
            for item in ("I1", "I4", "I5", "I6", "I8", "I12"):
                R.add(item, s, NOTRUN, msg=NOT_IN_DESIGN)


def i11_geom_1h(D, R):
    """I11 in the 1-hour design: the database comparison is dropped by decision D2; Skala//Skala only if the extension G4 ran."""
    R.add("I11", "own vs database geometry", NOTRUN, msg="dropped by design (decision D2 of PLAN_1hour: one geometry protocol, own B3LYP geometries; the database download is untested and blocked in the sandbox)")
    any_ = False
    for s in REACTIONS:
        mb = basis_for(s, "main")
        q = "barrier_fwd_cplx" if SYSTEMS[s]["sn2"] else "barrier_fwd"
        sk = quantity(D, s, q, M_SKALA, mb, level=LV_SKALA)
        rf = quantity(D, s, q, M_SKALA, mb)
        if sk is not None and rf is not None:
            any_ = True
            st = grade(sk - rf, TOL["I11_skala"])
            R.add("I11", f"{s} Skala//Skala", st, _fnum(sk), _fnum(rf), _fnum(sk - rf), f"{TOL['I11_skala'][0]}/{TOL['I11_skala'][1]}",
                  "Skala//Skala (TS = parabola vertex) minus Skala//reference barrier" + ("; report both barriers" if st == FAIL else ""))
    if not any_:
        R.add("I11", "Skala//Skala", NOTRUN, msg="Skala-relaxed geometries are not part of the 1-hour core (extension EXT-GPU-2, tail item 5)")


def decide_1h(D, R):
    out = _decide_orig(D, R)
    for s in ALL_REACTIONS:
        if s not in out:
            out[s] = ("NOT RUN (extension)", NOT_IN_DESIGN)
    return out


def paper_level_1h(dec):
    main = [s for s in DESIGN["reactions"] if dec[s][0].startswith("MAIN")]
    inc = [s for s in DESIGN["reactions"] if dec[s][0] == "INCOMPLETE"]
    classes = {SYSTEMS[s]["cls"] for s in main}
    n = len(main)
    if n >= 6 and len(classes) >= 3:
        v = "FULL PAPER as defined (the core has exactly six reactions: no margin)"
    elif n >= 4:
        v = "add verified extras (extension EXT-GPU-1: F- + CH3Cl, OH- + CH3F), or submit as a short communication titled as a case study"
    else:
        v = "DO NOT SUBMIT: fix the protocol"
    if inc:
        v = f"PROVISIONAL ({len(inc)} system(s) still incomplete: {', '.join(inc)}); so far: " + v
    return f"{n} of {len(DESIGN['reactions'])} design reactions in the main text, {len(classes)} classes: {v}"


def run_check_1h(root=None, write=True):
    """As run_check of b1v5_check.py, for the design: design notes first, I11 replaced, decisions and the paper-level rule for six reactions."""
    if root:
        setup_root(root, mount=False)
    print("=" * 100); print("Paper B1 v5 automated check (1-hour design, %d reactions)" % len(DESIGN["reactions"])); print("=" * 100)
    D = Data()
    print(f"CSV files found: {', '.join(D.files) or 'none'}  ({len(D.rows)} rows)\n")
    R = Report()
    for name, fn in (("literature checks R1-R4", literature_checks), ("design notes", design_notes), ("I1", i1_scf), ("I2/I3", i2_i3_cc), ("I4", i4_imag),
                     ("I5/I6", i5_i6), ("I7", i7_rank), ("I8", i8_basis), ("I9", i9_v4), ("I10", i10_fc), ("I11", i11_geom_1h), ("I12", i12_scout),
                     ("I13", i13_spread), ("I14", i14_retry_mode), ("I15", i15_basis_correction)):
        try:
            fn(D, R)
        except Exception as ex:                       # a bug in one check must never stop the others
            R.add(name.split()[0], "check code", WARN, msg=f"check crashed: {repr(ex)[:150]}")
    dec = decide_1h(D, R)
    paper = paper_level_1h(dec)
    c = R.counts()
    print("\n" + "-" * 100); print("Decision per system (rule 4.4)")
    for s in ALL_SYSTEM_ORDER:
        if s in dec:
            print(f"  {SYSTEMS[s]['n']} {SYSTEMS[s]['name']:18s} -> {dec[s][0]:30s} {dec[s][1]}")
    print("\nPaper-level rule:", paper)
    print("\nCounts: " + "  ".join(f"{k} {v}" for k, v in c.items()))
    if write:
        with open(path("check_report.csv"), "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=["item", "scope", "status", "value", "reference", "difference", "tolerance", "message", "raw_value", "raw_difference"]); w.writeheader(); w.writerows(R.items)
        md = ["# Check report, Paper B1 v5 (1-hour design: six reactions)", "", f"Written {now()}.  Tolerances in kcal/mol unless stated.  Items that cannot exist in this design are NOT RUN, never FAIL.", "",
              "Counts: " + ", ".join(f"{k} {v}" for k, v in c.items()), "", "## Decision per system", "", "| # | System | Tier | Decision | Why |", "|---|---|---|---|---|"]
        for s in ALL_SYSTEM_ORDER:
            if s in dec:
                md.append(f"| {SYSTEMS[s]['n']} | {SYSTEMS[s]['name']} | {D.tiers.get(s, SYSTEMS[s]['tier'])} | {dec[s][0]} | {dec[s][1]} |")
        md += ["", f"**Paper-level rule:** {paper}", "", "## All items", "", "| status | item | scope | value | reference | difference | tolerance | message | raw CCSD(T) value | raw difference |", "|---|---|---|---|---|---|---|---|---|---|"]
        for x in R.items:
            md.append(f"| {x['status']} | {x['item']} | {x['scope']} | {x['value']} | {x['reference']} | {x['difference']} | {x['tolerance']} | {str(x['message']).replace('|', '/')} | {x.get('raw_value', '')} | {x.get('raw_difference', '')} |")
        open(path("check_report.md"), "w").write("\n".join(md) + "\n")
        print("\nWrote check_report.csv and check_report.md in", path(""))
    return dict(items=R.items, counts=c, decisions=dec, paper=paper)


# redefine the names the notebooks call (the full-size names stay available as _decide_orig, _run_check_orig)
run_check = run_check_1h
decide = decide_1h
paper_level = paper_level_1h


# ------------------------------------------------------------------ frozen criteria for the six-reaction design (figures module)
FROZEN_DATE = "2026-11-15 (1-hour design, written before any real number exists)"
F3_MIN_REACTIONS = 3        # the core has the basis check (def2-QZVPP) for HCN, NH3 and H2CO only; the full design required 4
FROZEN = {
    "F1": dict(finding="Barriers at relaxed stationary points, five functionals, headline set (one primary barrier per verified reaction, SN2 from the complex; n = 5 of the 6 design reactions while the CH3NC value L6 is unverified, 6 once it is confirmed)",
               supports="Skala-1.1 (with D3) mean absolute error is at most 2.0 kcal/mol AND at most the wB97M-V (anchor) mean absolute error + 0.5",
               negative="Skala mean absolute error exceeds the better of B3LYP-D3(BJ) and r2SCAN by more than 0.5 kcal/mol",
               inconclusive="everything else (a tie within 0.5 with B3LYP or r2SCAN, or Skala above 2.0 without losing)"),
    "F2": dict(finding="Error along the relaxed Tier A paths (HCN, NH3, H2CO, CH4) against the CCSD(T)/def2-TZVP reference (basis-corrected where the MP2 correction exists)",
               supports="range (max minus min) of the signed Skala error is at most 1.0 kcal/mol on at least 3 of the 4 paths",
               negative="maximum absolute Skala error is at least 2.0 kcal/mol on at least 2 of the 4 paths",
               inconclusive="everything else; always inconclusive when fewer than 3 paths have a usable reference (R1 failed or rows missing)"),
    "F3": dict(finding="Basis-set stability of the primary barrier, def2-TZVP(D) to def2-QZVPP(D), HCN, NH3, H2CO (the core); more after EXT-GPU-1",
               supports="Skala shift at most 0.5 kcal/mol AND Skala's rank among the five functionals unchanged, in at least 75% of the reactions with data (at least 3 in the 1-hour design)",
               negative="Skala shift above 1.0 kcal/mol in at least 2 reactions",
               inconclusive="everything else, or fewer than 3 reactions with both bases"),
    "F4": dict(finding="Scout check: barriers at xTB geometries against the barrier at our own B3LYP geometries, every functional of G3 (Skala with D3, wB97M-V, B3LYP-D3(BJ) in the core)",
               supports="geometry penalty below 1.0 kcal/mol in at least 75% of the reactions (5 of 6) for EVERY functional computed",
               negative="penalty above 3.0 kcal/mol in at least 3 reactions, or in every reaction of one class (SN2: both F- + CH3F and Cl- + CH3Cl)",
               inconclusive="everything else, or fewer than 4 reactions with xTB-geometry energies"),
    "F5": dict(finding="Robustness of Skala's rank among the five functionals (by absolute error; errors within 0.5 kcal/mol of each other tie)",
               supports="the most frequent rank (ties share a rank) occurs in at least 75% of the reactions with a verified reference (5 of 6; 4 of 5 while L6 is unverified)",
               negative="the most frequent rank occurs in fewer than 50% of the reactions",
               inconclusive="everything else, or fewer than 4 reactions"),
    "F6": dict(finding="Negative results kept: CH4 C-H stretch (and the rigid HCN map against CCSD(T)/def2-SVP when the optional CPU tail ran)",
               supports="T1 below 0.02 (and RHF stable at the CH4 points) at at least 90% of the CH4 points (and of the HCN grid cells in the 120 kcal/mol window, if computed)",
               negative="that fraction is below 50%: the reference is truncated at the distance where it fails, and this is said in the paper",
               inconclusive="everything else"),
}


def findings_outcome_1h(D):
    """findings_outcome of b1v5_gpu_figs.py for the design: same arithmetic, the SN2 clause of F4 uses the SN2 systems of the design, and F3 needs 3 reactions."""
    rows = []
    fn = _functionals(D)
    anc = fn[1]
    sn2 = [s for s in REACTIONS if SYSTEMS[s]["sn2"]]

    def add(fid, measured, n, verdict, sentence, status=None):
        f = FROZEN[fid]
        rows.append(dict(finding=fid, description=f["finding"], criterion_supports=f["supports"], criterion_negative=f["negative"], criterion_inconclusive=f["inconclusive"],
                         frozen_on=FROZEN_DATE, measured=measured, n=n, verdict=verdict, sentence_for_the_paper=sentence))
    # F1
    mae = {}
    for m in fn + [M_SKALA_NOD3]:
        h = headline_rows(D, m)
        if h:
            mae[m] = (float(np.mean([abs(r["error"]) for r in h])), len(h))
    if M_SKALA in mae and anc in mae:
        S, W = mae[M_SKALA][0], mae[anc][0]
        B = min([mae[m][0] for m in (M_B3LYP, M_R2SCAN) if m in mae] or [float("nan")])
        v = "supports" if (S <= 2.0 and S <= W + 0.5) else ("negative" if (B == B and S > B + 0.5) else "inconclusive")
        add("F1", "MAE (kcal/mol, n): " + ", ".join(f"{m} {a:.2f} ({n})" for m, (a, n) in mae.items()), mae[M_SKALA][1], v,
            f"On {mae[M_SKALA][1]} reactions with a verified reference Skala-1.1 (with D3) has a mean absolute barrier error of {S:.2f} kcal/mol against {W:.2f} for {anc}"
            + (f" and {B:.2f} for the better of B3LYP-D3(BJ) and r2SCAN" if B == B else "") + ".")
    else:
        add("F1", "Skala or anchor headline rows missing", 0, "NOT RUN (data missing)", "")
    # F2
    sk_paths, n_valid = [], 0
    for s in ("hcn", "nh3", "h2co", "ch4"):
        r = _path_errors(D, s, M_SKALA, M_CCSDT_BC) or _path_errors(D, s, M_SKALA, M_CCSDT_FC)
        if r is not None:
            n_valid += 1
            sk_paths.append((s, float(r["err"].max() - r["err"].min()), float(np.max(np.abs(r["err"])))))
    if n_valid:
        flat, big = sum(1 for _, rg, _m in sk_paths if rg <= 1.0), sum(1 for _, _r, mx in sk_paths if mx >= 2.0)
        v = "inconclusive" if n_valid < 3 else ("supports" if flat >= 3 else ("negative" if big >= 2 else "inconclusive"))
        add("F2", "; ".join(f"{s}: range {rg:.2f}, max |error| {mx:.2f}" for s, rg, mx in sk_paths), n_valid, v,
            f"Along {n_valid} relaxed paths the Skala error against CCSD(T) has a range of {min(r for _, r, _ in sk_paths):.2f} to {max(r for _, r, _ in sk_paths):.2f} kcal/mol.")
    else:
        add("F2", "no Skala and CCSD(T) path rows", 0, "NOT RUN (data missing)", "")
    # F3
    shifts, rank_same, n_b = [], 0, 0
    for r in basis_table(D):
        if r["method"] == M_SKALA and r["shift_main_to_large"] is not None:
            n_b += 1
            shifts.append((r["key"], r["shift_main_to_large"]))
            if abs(r["shift_main_to_large"]) <= 0.5 and r["rank_changed"] in ("no", None):
                rank_same += 1
    if n_b:
        big = sum(1 for _, x in shifts if abs(x) > 1.0)
        v = "inconclusive" if n_b < F3_MIN_REACTIONS else ("supports" if rank_same >= math.ceil(0.75 * n_b) else ("negative" if big >= 2 else "inconclusive"))
        add("F3", "Skala shifts main to large basis: " + ", ".join(f"{k} {x:+.2f}" for k, x in shifts), n_b, v,
            f"The Skala barrier moves by at most {max(abs(x) for _, x in shifts):.2f} kcal/mol from the triple-zeta to the quadruple-zeta basis in {n_b} reactions.")
    else:
        add("F3", "no basis-check rows (G2 not reached)", 0, "NOT RUN (data missing)", "")
    # F4
    pen, n_react = {}, set()
    for r in table3_scout_cost(D, record=False)[0]:
        if r["geometry_penalty_xtb"] is not None and r["method"] in fn:
            pen.setdefault(r["method"], {})[r["key"]] = r["geometry_penalty_xtb"]
            n_react.add(r["key"])
    if pen:
        thr = math.ceil(0.75 * len(n_react))
        small_all = all(sum(1 for x in v.values() if abs(x) < 1.0) >= thr for v in pen.values())
        allp = {k: x for v in pen.values() for k, x in v.items()}
        n_big = len({k for v in pen.values() for k, x in v.items() if abs(x) > 3.0})
        sn2_all = bool(sn2) and all(any(abs(pen[m].get(k, 0)) > 3.0 for m in pen) for k in sn2)
        v = "inconclusive" if len(n_react) < 4 else ("supports" if small_all else ("negative" if (n_big >= 3 or sn2_all) else "inconclusive"))
        add("F4", "geometry penalty (kcal/mol): " + "; ".join(f"{m}: " + ", ".join(f"{k} {x:+.2f}" for k, x in sorted(vv.items())) for m, vv in pen.items()), len(n_react), v,
            f"Barriers at GFN2-xTB geometries differ from those at B3LYP geometries by at most {max(abs(x) for x in allp.values()):.2f} kcal/mol over {len(n_react)} reactions and {len(pen)} functionals.")
    else:
        add("F4", "no xTB-geometry energies (G3 not reached)", 0, "NOT RUN (data missing)", "")
    # F5
    ranks = []
    for s in REACTIONS:
        l = lit_primary(s)
        if l is None:
            continue
        err = {}
        for m in fn:
            c = compare_to_lit(D, l, m, basis_for(s, "main"))
            if c is not None and c.get("value") is not None:
                err[m] = c["diff"]
        if len(err) == len(fn):
            ranks.append(_rank_with_ties(err)[M_SKALA])
    if ranks:
        from collections import Counter
        modal, cnt = Counter(ranks).most_common(1)[0]
        v = "inconclusive" if len(ranks) < 4 else ("supports" if cnt >= math.ceil(0.75 * len(ranks)) else ("negative" if cnt < 0.5 * len(ranks) else "inconclusive"))
        add("F5", f"Skala rank per reaction (1 = best, ties within {TIE}): {ranks}; most frequent rank {modal} in {cnt} of {len(ranks)}", len(ranks), v,
            f"Skala's rank among the five functionals is {modal} in {cnt} of {len(ranks)} reactions.")
    else:
        add("F5", "needs all five functionals at the stationary points", 0, "NOT RUN (data missing)", "")
    # F6
    fr = _t1_fraction(D)
    if fr:
        good, tot = sum(a for a, _ in fr.values()), sum(b for _, b in fr.values())
        frac = good / tot if tot else float("nan")
        v = "supports" if frac >= 0.9 else ("negative" if frac < 0.5 else "inconclusive")
        sk = _path_errors(D, "ch4", M_SKALA, M_CCSDT_BC) or _path_errors(D, "ch4", M_SKALA, M_CCSDT_FC)
        loc = "" if sk is None else f" The largest Skala error along the CH4 stretch is {np.max(np.abs(sk['err'])):.2f} kcal/mol at C-H = {sk['coord'][int(np.argmax(np.abs(sk['err'])))]:.2f} A."
        add("F6", "reference reliable (T1 < 0.02) at: " + ", ".join(f"{k} {a}/{b}" for k, (a, b) in fr.items()), tot, v,
            f"CCSD(T) is reliable (T1 below 0.02) at {good} of {tot} points of the CH4 stretch" + (" and the HCN map." if "hcn_grid" in fr else ".") + loc)
    else:
        add("F6", "no CH4 CCSD(T) rows", 0, "NOT RUN (data missing)", "")
    return rows


try:                                   # the figures module (GPU notebook only): use the design's criteria and the design's findings function
    findings_outcome = findings_outcome_1h
except Exception:
    pass


# ===== b1v5_1h_gpu.py (pasted verbatim from the file of the same name; edit the file, not this cell) =====
# ==============================================================================
# b1v5_1h_gpu.py  --  the phases of the 1-hour GPU notebook (PLAN_1hour.md section 2, "GPU notebook").
# Pasted after b1v5_gpu.py (GPU helpers, energy_of with the Skala D3 handling, G0 helpers), b1v5_gpu_figs.py, b1v5_1h.py (queue) and b1v5_1h_check.py.
# Calculations run through qrun (admission test before EVERY calculation).  Nothing of b1v5_gpu.py is edited; this file calls it.
#
#   install, weights   (cell 1 and weights_setup): pinned skala-cuda12x 2026.9, gpu4pyscf-cuda12x 1.8.1, pyscf; weights from Hugging Face, then the Drive cache
#   G0   gate: stored v4 HCN point 0 with Skala (D3 OFF as in v4), PBE, B3LYP; 1e-5 hartree.  No gradient test (G4 is not in the core).
#   G1a  20 species of the six reactions, five functionals, def2-TZVP(D); Skala with D3, "without D3" is the same SCF minus the stored D3 energy
#   G1b  path points (HCN 11, NH3 9, H2CO 11, CH4 9 = 40): Skala, wB97M-V, B3LYP-D3(BJ)
#   G3   xTB minimum and peak of the six reactions: Skala, wB97M-V, B3LYP-D3(BJ)
#   G2   def2-QZVPP(D) at the 9 stationary species of HCN, NH3, H2CO: Skala and wB97M-V
#   tail (1) F- + CH3Cl and OH- + CH3F stationary points, five functionals (2) PBE and r2SCAN on the paths (3) QZVPP(D) for CH3NC and the SN2 systems, B3LYP added
#        (4) PBE and r2SCAN at the xTB geometries (5) Skala-relaxed HCN (needs the Skala gradient test), CH3NC, F- + CH3F
#   G5   figures, tables, findings_outcome.csv, the automated check: on whatever data exists
#   EXTENSION = 'GPU1' runs tail items 1 to 3, 'GPU2' items 4 and 5, 'GEOM' rebuilds the geometry pack.  GPU class decides how much of the tail is tried by default.
# ==============================================================================
import os, json, time
import numpy as np

GPU_CAPS = dict(install=15, phase10=4, phase11=24, phase11p=12, phase13=12, phase12=8, phase15=5, check=5)       # minutes (plan section 2)
GPU_SPEED = dict(T4=1.0, L4=0.75, A100=0.35, OTHER=1.0, NONE=1.0)           # time per energy relative to the T4 (L4 and A100: SCALED GUESSES, not measured)
GPU_TAIL_BY_CLASS = dict(T4=[], OTHER=[], L4=[1, 2], A100=[1, 2, 3, 4, 5], NONE=[])                              # plan: T4 none expected; L4 (1), perhaps (2); A100 (1) to (4), perhaps (5)
GPU_PRIOR_T4 = {"gpu": 8.0, "gpu:tz": 7.0, "gpu:tz:hcn": 5.0, "gpu:tz:nh3": 5.0, "gpu:tz:h2co": 6.0, "gpu:tz:ch4": 5.0, "gpu:tz:ch3nc": 9.0,
                "gpu:tz:sn2_ff": 12.0, "gpu:tz:sn2_clcl": 16.0, "gpu:tz:sn2_fcl": 14.0, "gpu:tz:sn2_ohf": 14.0,
                "gpu:qz": 27.0, "gpu:qz:ch3nc": 60.0, "gpu:qz:sn2_ff": 90.0, "gpu:qz:sn2_clcl": 120.0, "gpu:qz:sn2_fcl": 110.0, "gpu:qz:sn2_ohf": 110.0,
                "gpu:v4": 15.0, "gpu:g4": 900.0, "gpu:align": 40.0, "gpu:probe": 20.0}
Q["prior"].update(GPU_PRIOR_T4)
CSV11, CSV12, CSV13 = PHASE_CSV["phase11"], PHASE_CSV["phase12"], PHASE_CSV["phase13"]
PATH_METHODS = [M_SKALA, "ANCHOR", M_B3LYP]               # G1b and G3 (core)
BASIS_METHODS = [M_SKALA, "ANCHOR"]                       # G2 (core)
PATH_SYSTEMS = ["hcn", "nh3", "h2co", "ch4"]
BASIS_SYSTEMS = ["hcn", "nh3", "h2co"]
SKALA_PREFETCH_FAILED = False


def anchor_name():
    return anchor_method()


def resolve_methods(names):
    out = []
    for m in names:
        m = anchor_name() if m == "ANCHOR" else m
        if route_of(m) != "none" and m not in out:
            out.append(m)
    return out


def path_ok_methods(names):
    """Methods that may run along paths (a CPU-routed functional is for stationary points only)."""
    return [m for m in resolve_methods(names) if route_of(m) != "cpu" or STANDIN]


# ------------------------------------------------------------------ install, weights
def weights_setup(timeout_min):
    """Restore the Skala weights from the Drive cache, else download them once (HuggingFace) in a SUBPROCESS with a wall-clock cap, then cache them on Drive.
    A slow or failed download costs at most `timeout_min`; the queue then simply reaches less.  Never raises."""
    import subprocess
    global SKALA_PREFETCH_FAILED
    out = dict(restore=restore_weights_cache())
    print("  Skala weights:", out["restore"])
    if STANDIN or NO_GPU:
        out["prefetch"] = "skipped (stand-in or no GPU)"
        return out
    code = ("from skala.functional import resolve_functional_artifact as r; a = r('%s'); print(a.path)" % SKALA_NAME)
    t0 = time.time()
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=max(30.0, 60.0 * timeout_min))
        out["prefetch"] = ("ok: " + r.stdout.strip()[-120:]) if r.returncode == 0 else ("failed: " + (r.stderr.strip()[-200:] or "no message"))
        SKALA_PREFETCH_FAILED = r.returncode != 0
    except Exception as ex:
        out["prefetch"] = "stopped at the cap: " + repr(ex)[:100]
        SKALA_PREFETCH_FAILED = True
    out["seconds"] = round(time.time() - t0, 1)
    if not SKALA_PREFETCH_FAILED:
        out["save"] = save_weights_cache()
    print(f"  Skala weights prefetch: {out['prefetch']} ({out['seconds']} s); cache: {out.get('save')}")
    return out


# ------------------------------------------------------------------ queue wrappers
def q_energy_row(csv_name, system, pid, level, method, basis, atoms, charge, kind, coordinate="", dm_key=None):
    """One energy through the queue; Skala also gets its 'without D3' row from the same SCF (free)."""
    row = qrun(csv_name, system, pid, level, method, basis, lambda: energy_of(method, atoms, charge, basis, dm_key=dm_key), kind,
               coordinate=coordinate, retry_failed=RETRY_FAILED, label=f"{system}:{pid}:{method}:{basis}")
    if method == M_SKALA:
        ensure_nod3(csv_name, system, pid, level, basis, coordinate)
    return row


def own_geom(system, sp):
    lv, g = best_geom(system, sp)
    return lv, g


def blocked_1h(name):
    """Reason (text) why a GPU phase must not start, or None."""
    if name == "phase10":
        return None
    if NO_GPU and not STANDIN:
        return "no GPU detected: only the G0 diagnostics run (Runtime > Change runtime type > GPU, then Run all again)"
    load_gstate()
    if GSTATE.get("gate_ok") is False and STOP_ON_G0_FAIL:
        return "G0 reproduction of the stored v4 energy failed (difference over 1e-4 hartree): old and new numbers must not be mixed. Fix the install, delete DONE_phase10 and run again (or set STOP_ON_G0_FAIL = False)"
    return None


def run_gphase_1h(name, title, fn, cap_min, final=False, force=None):
    why = blocked_1h(name)
    if why:
        skip_phase(name, title, why)
        return
    with qphase(name, title, cap_min, final=final, force=(FORCE_RERUN if force is None else force)) as ph:
        if not ph.skip:
            fn()


# ------------------------------------------------------------------ G0
def g0_1h():
    """G0 of the 1-hour design: weights cache, functional probe, Skala check, weights hash, the stored v4 HCN point 0 with Skala (D3 OFF), PBE, B3LYP."""
    info = dict(started=now(), hardware=hw(), versions=ver(), gpu_name=GPU_NAME, gpu_class=GPU_CLASS, gpu_mem_gb=round(GPU_MEM_GB, 1), standin=STANDIN, design="1h")
    if NO_GPU and not STANDIN:
        info["note"] = "no GPU: diagnostics only"
        save_gstate(**info)
        print("  ** NO GPU: G0 stops here after the diagnostics. **")
        return info
    if SKALA_PREFETCH_FAILED:
        globals()["CALC_MAX_SEC"] = 120                      # the weights are not cached: do not wait 20 minutes for a download inside the Skala check
        print("  weights prefetch failed earlier: the Skala check is limited to 120 s")
    ok, why = q_admit("gpu:probe")
    if ok:
        with item_guard("phase10", "probe functionals"):
            info["probe"] = g_probe_functionals()
            save_gstate(**info)
        with item_guard("phase10", "Skala check"):
            info["skala"] = g_skala_check()
            save_gstate(**info)
    else:
        q_refuse("G0 probe and Skala check", "gpu:probe", why)
    with item_guard("phase10", "weights cache save"):
        if GSTATE.get("skala_ok") and not STANDIN:
            info["weights_save"] = save_weights_cache()
    with item_guard("phase10", "weights hash"):
        info["skala_weights"] = skala_weights_info()
        w = info["skala_weights"]
        print("  Skala weights:", {k: w.get(k) for k in ("file", "sha256", "huggingface_revision", "hash_matches_package_table", "expected_d3_settings")})
        save_gstate(**info)
    th, r_ = NB4_HCN_GEOMS[0]
    at = hcn_geometry(th, r_)
    basis = "STO-3G" if STANDIN else "def2-TZVP"
    for m in ("v4-skala", "v4-pbe", "v4-b3lyp"):
        if m == "v4-skala" and not GSTATE.get("skala_ok"):
            continue
        with item_guard("phase10", f"v4 point 0 {m}"):
            reset_dm("nb4")
            qrun(CSV_REPRO, "hcn_nb4", "nb4_00", LV_NB4, m, basis, lambda m=m: energy_of(m, at, 0, basis, dm_key=("nb4", m, basis)), "gpu:v4",
                 coordinate=f"{th},{r_}", retry_failed=RETRY_FAILED, label=f"stored v4 HCN point 0, {m}")
    gate = g_gate()
    info["gate"] = gate
    print(f"  stored v4 HCN point 0: largest |difference| {gate['max_diff']:.1e} hartree ({gate['status']}); per method {gate['per_method']}")
    if STANDIN:
        GSTATE["gate_ok"] = True
        print("  stand-in: the gate is not applied (stand-in energies differ from the stored v4 energies by design)")
    elif gate["status"] == "no data":
        GSTATE["gate_ok"] = None
        print("  gate: no reproduced energy (all attempts failed or were refused); later phases run, but nothing was verified")
    else:
        GSTATE["gate_ok"] = gate["status"] != "FAIL"
        if gate["status"] == "WARN":
            print("  gate WARN: differences between 1e-5 and 1e-4 hartree: version drift, state it in the paper")
        if gate["status"] == "FAIL":
            print("  ** GATE FAILED: difference over 1e-4 hartree. The later phases will not start (STOP_ON_G0_FAIL). **")
    GSTATE["grad_ok"] = None                              # no gradient test in the core (tail item 5 runs it when it is reached)
    save_gstate(**info)
    return info


# ------------------------------------------------------------------ G1a, G1b, G3, G2
def g1a_stationary(systems=None, methods=None):
    """Every species of the design reactions with the ladder, own geometry, main basis."""
    meths = resolve_methods(methods or ["ANCHOR" if x == "ANCHOR" else x for x in PRIORITY])
    for system in [s for s in (systems or SYSTEMS_TO_RUN) if s != "ch4"]:
        basis = basis_for(system, "main")
        print(f"  {system}: stationary points")
        for sp in SPECIES_KIND[system]:
            lv, g = own_geom(system, sp)
            if g is None:
                log_error("phase11", f"{system}:{sp}", RuntimeError(f"skipped: missing input geometry for {sp} (geometry pack)"), note="skipped")
                continue
            for m in meths:
                with item_guard("phase11", f"{system}:{sp}:{m}"):
                    q_energy_row(CSV11, system, sp, lv, m, basis, g["atoms"], species_charge(system, sp), f"gpu:tz:{system}:{m}")


def g1a_align():
    """Review row 11: partners of a density-fitted retry are recomputed the same way (rare; admitted once)."""
    rows = read_rows(CSV11)
    if not any(extra_of(r).get("df") for r in rows):
        return
    ok, why = q_admit("gpu:align")
    if not ok:
        q_refuse("retry-mode alignment", "gpu:align", why)
        return
    g_align_retry_modes(CSV11)


def g1b_paths(systems=None, methods=None):
    """One functional at a time along the relaxed paths (HCN 11, NH3 9, H2CO 11, CH4 9 points)."""
    meths = path_ok_methods(methods or PATH_METHODS)
    for method in meths:
        for system in [s for s in PATH_SYSTEMS if s in (systems or SYSTEMS_TO_RUN)]:
            pts = g_load_path(system)
            if not pts:
                log_error("phase11p", f"{system}:path", RuntimeError(f"skipped: missing input geoms/{system}_path.json (geometry pack)"), note="skipped")
                continue
            basis, ch = basis_for(system, "main"), species_charge(system, "TS")
            reset_dm(system)
            print(f"  {system}: path, {method}")
            for q in pts:
                with item_guard("phase11p", f"{system}:p{q['index']}:{method}"):
                    q_energy_row(CSV11, system, f"p{q['index']:02d}", LV_SCAN, method, basis, q["atoms"], ch, f"gpu:tz:{system}:{method}",
                                 coordinate=f"{q['coordinate']:.4f}", dm_key=(system, method, basis))


def scout_species(system):
    S = SYSTEMS[system]
    return [S["rc"] or S["reactants"][0], "TS"]


def g3_scout_1h(methods=None, systems=None):
    """The xTB minimum and peak of every design reaction with the functionals of the list (core: Skala, wB97M-V, B3LYP-D3(BJ))."""
    meths = resolve_methods(methods or PATH_METHODS)
    for m in meths:
        for system in [s for s in (systems or SYSTEMS_TO_RUN) if s in REACTIONS]:
            basis = basis_for(system, "main")
            for sp in scout_species(system):
                with item_guard("phase13", f"{system}:{sp}:{m}"):
                    g = load_geom(geom_name(system, sp, LV_XTB))
                    if g is None:
                        log_error("phase13", f"{system}:{sp}", RuntimeError("skipped: missing input xTB geometry (geometry pack)"), note="skipped")
                        continue
                    if g.get("no_peak"):
                        continue
                    q_energy_row(CSV13, system, sp, LV_XTB, m, basis, g["atoms"], species_charge(system, sp), f"gpu:tz:{system}:{m}")
    g3_xtb_barriers()


def g3_xtb_barriers():
    with item_guard("phase13", "xTB own barriers"):
        out = {}
        e = {(r["system"], r["point_id"]): float(r["energy_hartree"]) for r in read_rows(PHASE_CSV["phase04"])
             if r["method"] == M_XTB and r["geometry_level"] == LV_XTB and r["converged"] in ("1", "True") and r["energy_hartree"] not in ("", "nan")}
        for system in [s for s in SYSTEMS_TO_RUN if s in REACTIONS]:
            rmin = scout_species(system)[0]
            out[system] = None if (system, "TS") not in e or (system, rmin) not in e else round((e[(system, "TS")] - e[(system, rmin)]) * HARTREE_KCAL, 3)
        write_json("phase13_xtb_barriers.json", dict(unit="kcal/mol", barrier_TS_minus_reactant_or_complex=out,
                                                      note="GFN2-xTB energies at GFN2-xTB geometries (from the geometry pack); null = no xTB peak or missing"))
        print("  xTB's own barriers (kcal/mol):", out)


def g2_basis_1h(systems=None, methods=None):
    """def2-QZVPP (QZVPPD for SN2) at the stationary species; system by system so that a barrier is complete (core: HCN, NH3, H2CO; Skala and wB97M-V)."""
    meths = resolve_methods(methods or BASIS_METHODS)
    for system in systems or [s for s in BASIS_SYSTEMS if s in SYSTEMS_TO_RUN]:
        b = basis_for(system, "large")
        print(f"  {system}: {b}")
        for m in meths:
            for sp in SPECIES_KIND[system]:
                with item_guard("phase12", f"{system}:{sp}:{m}"):
                    lv, g = own_geom(system, sp)
                    if g is None:
                        log_error("phase12", f"{system}:{sp}", RuntimeError("skipped: missing input geometry"), note="skipped")
                        continue
                    ch = species_charge(system, sp)
                    nbf = make_mol(g["atoms"], ch, b).nao
                    if nbf > GPU_QZ_MAX_NBF:
                        if not have(CSV12, system, sp, lv, m, b):
                            log_error("phase12", f"{system}:{sp}", RuntimeError(f"skipped: {b} has {nbf} basis functions, above the limit {GPU_QZ_MAX_NBF} for this GPU"), note="skipped")
                        continue
                    q_energy_row(CSV12, system, sp, lv, m, b, g["atoms"], ch, f"gpu:qz:{system}:{m}")


# ------------------------------------------------------------------ the tail
def ext78_present():
    return [s for s in EXT_REACTIONS if all(geometry_of(s, sp, LV_OWN) is not None or geometry_of(s, sp, LV_SCANPEAK) is not None for sp in SPECIES_KIND[s])]


def tail1_ext78():
    """(1) F- + CH3Cl and OH- + CH3F stationary points, five functionals (needs the geometries of the extended pack)."""
    ss = ext78_present()
    if not ss:
        log_error("tail1_ext78", "", RuntimeError("skipped: the geometry pack has no geometries for F- + CH3Cl and OH- + CH3F (REBUILD_GEOMS / EXT-geom builds them)"), note="skipped")
        print("  no geometries for reactions 7 and 8 in the pack: tail item 1 skipped")
        return
    g1a_stationary(systems=ss)


def tail2_paths_pbe_r2scan():
    g1b_paths(methods=[M_R2SCAN, M_PBE])


def tail3_qzvpp():
    """(3) QZVPP(D) for CH3NC and the SN2 systems (Skala, wB97M-V, B3LYP) and B3LYP for the three Tier A reactions."""
    g2_basis_1h(systems=["hcn", "nh3", "h2co"], methods=[M_B3LYP])
    g2_basis_1h(systems=[s for s in ["ch3nc", "sn2_ff", "sn2_clcl"] + ext78_present() if s in SYSTEMS_TO_RUN or s in EXT_REACTIONS], methods=[M_SKALA, "ANCHOR", M_B3LYP])


def tail4_scout_more():
    g3_scout_1h(methods=[M_R2SCAN, M_PBE])


def g_gradient_test_lazy():
    """The Skala gradient test of the full G0, run only when the tail reaches the Skala-relaxed geometries."""
    if GSTATE.get("grad_ok") is not None:
        return GSTATE["grad_ok"]
    if not GSTATE.get("skala_ok"):
        return False
    res = g_skala_gradient_check()
    save_gstate(gradient=res)
    return bool(GSTATE.get("grad_ok"))


def tail5_skala_relaxed(systems=("hcn",)):
    """(5) Skala-relaxed minima and transition state (parabola vertex) for HCN, and CH3NC and F- + CH3F in EXT-GPU-2; geomeTRIC is installed on demand."""
    if not _have_mod("geometric"):
        pip_install_capped(["geometric==1.1.1"], 4)
    if not g_gradient_test_lazy():
        log_error("tail5_skala_relaxed", "", RuntimeError("skipped: the Skala gradient test did not pass (or Skala is unusable)"), note="skipped")
        print("  Skala gradient test not passed: tail item 5 skipped")
        return
    variant = GSTATE.get("grad_variant", "plain")
    for system in systems:
        ok, why = q_admit("gpu:g4")
        if not ok:
            q_refuse(f"{system}: Skala-relaxed minima and TS", "gpu:g4", why)
            continue
        with item_guard("tail5_skala_relaxed", system):
            t0 = time.time()
            g4_system(system, variant)
            q_record("gpu:g4", time.time() - t0)
            g4_report(system)


def _have_mod(m):
    import importlib.util
    try:
        return importlib.util.find_spec(m) is not None
    except Exception:
        return False


TAIL_ITEMS = {1: ("tail1_ext78", "tail (1) F- + CH3Cl and OH- + CH3F, five functionals", tail1_ext78),
              2: ("tail2_paths", "tail (2) PBE and r2SCAN along the paths", tail2_paths_pbe_r2scan),
              3: ("tail3_qzvpp", "tail (3) QZVPP(D) for CH3NC and SN2, B3LYP added", tail3_qzvpp),
              4: ("tail4_scout", "tail (4) PBE and r2SCAN at the xTB geometries", tail4_scout_more),
              5: ("tail5_skala_relaxed", "tail (5) Skala-relaxed HCN", tail5_skala_relaxed)}


def tail_items_for(extension, gpu_class):
    """Which tail items to try: the extension session names them; otherwise the GPU class decides (T4: none, as the plan expects)."""
    if extension == "GPU1":
        return [1, 2, 3]
    if extension == "GPU2":
        return [4, 5]
    return list(GPU_TAIL_BY_CLASS.get(gpu_class, []))


def run_tail(items):
    for k in items:
        name, title, fn = TAIL_ITEMS[k]
        if k == 5 and EXTENSION == "GPU2":
            fn = lambda: tail5_skala_relaxed(("hcn", "ch3nc", "sn2_ff"))
        if q_remaining_min() < 1.0:
            q_refuse(title, "gpu:tail", f"deadline: {q_now_min():.1f} min used")
            continue
        run_gphase_1h(name, title, fn, max(1.0, q_remaining_min()))


def design_with_extension_data():
    """After the queue: reactions 7 and 8 enter the check and the figures if their stationary points were computed (tail item 1)."""
    have11 = {r["system"] for r in read_rows(CSV11) if r["system"] in EXT_REACTIONS}
    return DESIGN_REACTIONS + [s for s in EXT_REACTIONS if s in have11]


def g_summary_lines_1h():
    return g_summary_lines() + [""] + queue_lines()

**Cell 5: the geometry pack itself** (data, with its SHA-256).

In [ ]:
# ---- The geometry pack: 35 structures, 4 paths, 8 xTB sets, 57 KB JSON, 19 KB embedded; systems hcn, nh3, h2co, ch3nc, sn2_ff, sn2_clcl, ch4, sn2_fcl, sn2_ohf; built 2026-11-15 with pyscf=2.15.0.  SHA-256 of the pack (canonical JSON): this constant is checked in the settings cell. ----
GEOMPACK_SHA256 = "81e4533319f5fc593cc3e81daf93e6e4db4998df01eea275b3c813fc87dafae6"
GEOMPACK_B64 = "H4sIAM4W+2oC/+19a28kx7HlX2nMJwmX3c73Q4YWa48xnrvYHQmydrG7hjGoR9YMYQ45S1K25Av/9z0ns7q7qrq6u9ikdK+Aqwe7SCarKiPjcSIyMuLfXtU/XN+06f7VV//26mN13/69uk+vvnr1r7eP6eaL775c/e90d8vPb+/vmvTwcHe/+q8rtZHij2//cbUyq+buPj1craRb/fH3q+9+9z9eXb36/NPjx7tb3ENvJP4L+NFDc3/9+RE/yg97/yHdffpcNX/dfP5p9S+rWv7Nzv2o+fwDvsNf/y3dP1zf3T7g7z//9NB0X+P5diN+y79Jj/fXzddyg39/+1jfXD+mr8Um4JfVQ/pab1TE5e0Pnz7/hD+yG/Pbh+Ya1xgd8Aefq9u2esAw/nW+9bq9fvhcHodBHn9cZvP1di7/vHrV3KfqMbV4GyWUW0u5lvZ7ab6y7isd8bpterj+wPnLj5k8X60ern9c4Y+aR84C03v91uBr+vHRh9UXb9b5J/r1zdXqm7f9N2++xI26u/tPVaYayLHeUmgt8aub9Ld0Q4L0NPjpq9Uf3qx/r//7//l2/Qf9xe//25erL7796U+v36zadPtw/fjTqrt+fLy+/XC1+nB/3a7y36/0FV7w9m/vH+9uVjKt45dXGN6p9ff/9399u/pid/mHFd5khb+4/2n1p3dqBQo11+kBo/PTv//uX1/n+6T7D+m2SauH9Lj64+/+529Xj/cVH45Zrx4eQbOHVf3T4G/SNcb/LTWPuHt3d3Nz93e84Orv148fV48f06q6rW5+erxuVm/Bd9fV7ap6xG3S55X47aq7T//vBzwMr4Fx7eof6f5u/fnu+vZxlW7xHpjv/d2n8W2mBOpv+9vVj9///qvVH9+8U2tcrb4obPRleZFq+L63H65vE6h/e/dIGemvsAx/fvVwq943N83NV/mZD011u/6cqr+uuurmpsaqrb7/0+rv1cOqvb/7/Dm1Zdjd32/584/4+d1tWl1/qvCAClT+dNcmTnewVpZitH3I93/6aikJqtFNdrT8oueA1Yfqb2mlJs9++HL4tO9ev+zT9OHT/gK1UT1+fKAWaj4aftTVwzU5fMeGeKUGInCTSPzH+x8SxKCq0w3GvF6/BWvcp8fm4+qL3325FRD8BiR+z9WgXuLbcrH+7VX1ePeJV39+9frV1Z/FRlz1///lL1d/fvU2/8zK6KRV5mr2cjdwLTZOeeujN1fD6/3l7ND5vxo8fvamo6F/IT2K1LVbgjR3d/ctKEsSiU28elUW59VXtz/c3Fy9ur5t04/4DRY34W/bB44S/7xaSBHvtRXKX81e/gooIjGleYrIyyjiMPNg9dXM1a+CHPIIOdSF5IgqKKHV1ezlr4Ii6ghF9GUU8VZYa7W9mr38VVBEH6GIuYwiQQQVhQ5Xs5e/CoqYIxSxF1LEOaEs3/3g6ldBDnuEHO4yckSlvROGqnTm8ldBEXeEIv5CigRpojXuavbyV0ERf4QiYUIR3OnhY6Wso/tYddrb1jWmCqkKMkjvk3Rt1cRQdzFWnUrJNd4628qYqrY1OnjpTGobJ5MmhP0JDsMn3IuYEq7bR9XcPRVbvt1sNm9X8AsfKzo3x+El4P77vSY4gTZ1VNLITHpBQpKOMrii/L4py6Oidd7qyN+TH7yw3skQd+sCNRSV0eCAMoR/o3WM+H+4zEIpp43v77OW+cZWibhk1cJAtOHnmg2sVgxBB6PBe2YWTxo8Yo6xtRZCGSG3k5bB4NWEGU5aWyOiE3o36ejxrRxNWmCKcTcfTFoEIYSfTFpGLbTfT1pYo02Ui1jVHM5aC2OdcdEqL2cxo/EbNzdrzBDo2OyWWkRhFFh5OGujbJTgiO2sQwjSeSkHs9ZKqgDy7WYN7sECBD+ZtXZY7d2spXRKO2MXqSwjhv/IKQkMvB4dnPYabzeLEx3eao4EYFrvvdrqNamVCtGNCABGlk7KLQGwwio47wcEUFpAsp3aEwCiDsdjsuxRCufdjgBCWq98UIusGGD7KQJoq4IFQpHaUixmYKGUdhNmWV9oo3TQu5eHHdNC9jy7FXgnMUenM53w6gBEVvshE2B2zlkbdrfRXoGYAzrh51p6UHfP+QJMEsEWi3CNHvGAmpJA4b09dQeW1M7iQK/mxCAvl4BEDhgYSwPZcHJEAugzLLwtgyTXGEYuqKH4w5kAB9mtPEGsQbMQBiQAbIJlUoWQpIAOCho3LMK6Es8EEDUS7Bb8lAJGSe8cfu4pf7O4zxwhQFCEuCGrP2g0zIz6DW/vpR5oA7wuV1nZUAZCSwIfQxNrB1oMllpBBUiljSvcjtXCN4HzBSzwFnBpQDWJCeE5RXoEVs+RGcFRsBhQ5YvUojoghg8a8uVA/ijiLOqTym30EXJ4qD3ZzzIIn4MroEeUsGFjeuhgXehHKh+jK/QwsKFxSA8QDYume3rgTuQk6PMAfWKGTCThfindK1wBXStoTzBFEx0M6iJOgbBP6RGhobTLphyQdQ7zKXGcHFlJlEl6zLmQwzgoXT0mBxS6cmUgdFKhBSyMVwNaiADekCH0tLDFGIIWuJ9XNgyJgdf2UfqeGCaKQjblYe5cWIQTDrUljIWFLEN8LWY1h/bULAAutMD6xDJDGJxCCZhfzGlMCawwVqwMNIKudSZG9MKFITHwI6N6nSiEMxQpEgNYAsNHZsYYLIPsaQHcpCRHwvx75ZaoEBgvcQAfIGHeKXA5aLu3nXEoKGYT54khoBBFFg4yhnUZQHkIHBh1JCcKJIqg3ZXehJjWIkuJMgE+wogzMEqrXscKoi3pstbAFDXg3RB0QIvY6PuhsKyFhwwUCVCNX0KMcAAjpKXb4sHrdi8jcgggPWzxyA9QrY06mcbEKjWiUj4Bh1jT1bVpgsacAW/bxrRNlSq4C53uaueDrDVRH6Yw9AOyA0BHoLl9sh+wfr1+t6puP9wk7o18WOAGuDNB5x7vUFlBnRWefTf8XdZVUkKBjXDOzrTB0EE/LFFYQQxjn+uoYUqtF1xypwDEZ7G8wkrMcSXeF1zpMleWyzILiL2Kw1koBU2mCweVa805wWrBItkhblFCQPT6kfnalBniO1ipBTP08nCGBlg6SjzVg3fncLuKc95KASzOAWxc9ZfAPVdFl9LSDueI3xiY9vzmvAaE4BxlRqyjOToDgvtioHgNnVUAClCAWQTRLEGfncwSyyhhtZwVxs4icx2OTFKBtlCNnFm+tGqrFzzUznAlAUSoDDLYyNcwlESpFrzp3TBaIPEPuNKUkbzOvlr0dP7kEnOCNTmYI5ROIJ/Szh1B3/YIs2phA5Q3tyrypd+ZOu+EHc4RNIRJzeuTr43gH0W63Uq64RzhBhtrfRmJa1fmC9eIErAkIiIiActklhL2UgOPQEereYAt3RFDAfIA9dFq8spBGjK7ArLDvA8nCWeAc8+vnq+zgYUz7mnthpOMdHejLCNxHUJefiA+cvYCdg1hM5khPXfY3wBcYQf6347XMczO0IDXjMzQP1/qHEoHeATAG01RMZYiyjLna6m5jpgd4LbWQ1cBTiRxZAme4NoWD8VBOuGyLlhHGisZJpOEw0t4yfiNMPPA2B5ZRv6RBWy6KpdwirOSVTns4YaTVBStAlXKdbD9XgvGDtcRqFYqLG8eyes+qoR1gECpBdEB5zfBH6ykIOcHCRYbRIL8EomE02CNN/GqXMLC6aJm4aWI8Up6F+DilZXktfdlDxbrb4crCacc9sdkwJuvTdFVuD8chgUG0vjNgTgKy9cEdIwB/tgsiJVH1hEKwUUYkLwfSFUfTdY/BrgtmpE8Wm9pFtf9tYuUXA0/x9gYR2FYF2FtfQm9OoaM8upDGEC3JesIb8wdTBJeOQSaSnygcobYNLgjLi2cSMDlbBfzpc3+CxwM66xTYxMJLpG2N5GCloZcHbQe+iJkSfi0oVhIXouifSwQvNDUwGc59QACYFZwaaCwiGbnASdYGG7ZCHEalXytXJe0qLu6rWLolKxr3QFPNhWwJjhCgu6dT0DTJrWp7aBK6y51UaU0QpxAmgCctx/1UwHnu/VbuX6r1m/1qr3+mNr76mYp8DRHgCfXY602sC7wEHZBFEiLg3UdWjigAgYM1NUu1AR/PYxhqGHownrKboAnpmG4t0MdYNDs0PWxsedWdg0QA5xEXwp+EGRyqHSt28BeRwYPgEW9D7NoVk75+F2PZoHpvFYFl0k4iJDDnlUj0feQLHDwtQ5ZJ0PBAzvronHg2EQ52ooBSfGSIVPHSBF6ts5DcT0aiu+jKi8QAOe0ir0ig2rNtu0sdVTYkDYmwDcD8MQkpuRhMBG004Ie3SwUVlPDu6NP1kZ9uMQD7ahQkA9ALozqaKdKQaHkHV3GriKd4XWO/eugwnjO9GcL88COkOr9SGNDGIV1ucsFnZ4RdoDJgfdQxjJAapaEa9YSaMvRvsPYBPCimVIHcg9PGRBADzT9EEPLAwy9ow5cat1H3bnxYMviAVBzv2DEPQQtqlj04ixlG8X1CsFO5kxmVzFvBStB1NQPxTz0mNGwoFj2AkBhbGhkylgIllniOq0jKGA0oTEp7byckocoUhAuD2Vrgr5nhKuEGDDNHF4DMM1ah8TB42AQwog4QXtA1+JlSR+KPCgQBqDaTGYcJUxMJo6m3JvtUPhBcjLUMWrYEydyhcpYYiIhlwQxxIQaPnroLxp1wFgzC9PVwV7InlsgF9sAE1B3cSIAETxx+5hZwEp95Af+i/NZAJSjBzROLwK4kMJm/AevJXromn5ooJ85HuqI2WShB0CjKyMJmNQCSL+IU/h+3J6ZxfdQM+YIaTDHqPuYJd4IFiPjG+kY6LYjNWNEvwkgyDUlss0FdWGiWXXUYMGiZazsN2E4MmLwhIiKXk7RMj5iIoVVIYchhCU+3UTJaHuoZQKeq2MAVLWzjoHS0wjxnjqMVfbEwQoXjWPgmqrhpGmjvAwqFkHibknZrQd+9sqP0/SAbnXxFGmiALfcduTQT8qPsUSutlgoKGlbWIjbkrjDAgu1xEAxMghR94PdBD/2fc0RHQNyxq1QgR4F+jOkpuOINMCr3HzsIzkOyqPgbqP0cK8tk0bA+c2INnDbTcbeZTYC7z+x9FAy0cUe3AgyTNnhhh9p3YKQxzJs44Hz+SqzngiYb4xhRRBdA7SRWvCy7qAaW1WrZBxcdwMmrWVrIPSykoxawdP3uqt0a3xt6spaNcSwBK//BIh9wKs3jz/cpz5PV982v2Hm+rvfEIfW+uanz/zF2FMwIHuOR2+DndEy+pcpyN97KumtYWDI0ziDVZSjIBvVpon99qHaOEiXVqOdQaHogvg9YnUWyx1GoAOAhf6cz5vsNJVFoDA08L/ZoetjY0HsWTT/scIyZ+RZ1vA9fvB4nzIM0WqTw9+WURlwCoSAidUkFlTTJlyVr1EzY4YR943bfRipNxEf3mxs+XBXCm+F7yCPaqOuYJUEv+OHw+t9uK/a9/u4OtOBmXid3+2v4CG8+afr24k3UVYRP7v7e3p4fM/Xe998kmRSvfHZk4PU4524C/6p+vE9n0LDoJ0x3DkF/8Kyq7yir+4+P74/YP17Htrgt11188DvPz30d4FrErzi5lGEM69xq3Kbh6Y7vM2O1bHqET6aqhNcNcGNFqMbCS3cwlj51MYGfjUctbpTcCNbUQNP2Qo6tNIN/uF5iv5IA1PJyc/j1CGwOX7wj89pv5Tcw7cMnygsD3MRhOfJkK1EvHt9SiK40273HJ89EOBmqyYcL6STO6FQ4Hfr92JThkAm4lYoLEwuCGJHQgEMWfaae6HAKgUz4XQoTOmyjQy+yJjmBgtc3Nlx69mBF4qDz4E2CCq+8XtpUB7K/qp8jZasC/kHBN99QLFmoYhMx8wf9kopYyE3kAZmNkMMZBEKfMxIQ3iWNPDVuCevNEhCNDEUhrI2YA3mlMAfJbiKT5cGqAIPKcDfO7jidqE0VFE3MFG6hkUxoYkV2F6qpo2tdK0TPkCtRZG8823VVaEWCV4KZMV3MqZo01Qa3r1eJg0wI7CNgCZwb2PYScP3fzolCtkTglnLASFIbpA95LWM+Gi5k4jsm3vNDL+S9RUltzuw0I77Sm4vGFCd3GQEL2TJYmJTTruw8LIcdPpwMxQwEvzB/RqhiOpz7i0k0Wq4I8OBkD2pC1aHEIgIQNhHXvEmdnRPmbfj+/CGIlJjAgdWz8BJs5cLCmB1pCvDlJ+9oKyxmhtMIFBSAIJwHQOlIIZIAdFwXKE/YFOznMBqQFXAM6StETFmAYmKUqO02PgDOQnM/y5yIndy8viwVEzy6wEjaq4I8D2g2YGcMG4bYW+JWrkb/VQxMfT8gAAZY4KucGKRmHSwLnXTatEkKRoPKVMytVb6tq5i7V0tutYLWmkrgZZibHzb1eBW/oVV9UhMvv/TIhnBylkYTkEvzcEHzkJiYDDMKRnZ7/hOTlsoylrOoZm5HOf69r9eLx86/1dzj18fG3oBpxsGkCN0iSdAVzAue1aXUBRk9fGHhSGI2w+oPbiXYGlt+N3w44CzFU+/PMMClBegtOc9ejgBQ9Zm9hoAKRxV2koAe2Jg/2TeZm4NwyL0Ysk83pfbnObtACDUiEr4WqnW1dElnVQdYl1V3EVpgtId2FpWqW1TaFKqfMMEY2sqWAfXTkyAmWZSz7A2HDDgIRsV92GC63Otf/P6mzPafw+BpA/BDtMBRZ8KSdxBf/oy1QnUDJCioGKi1AJqfAAyFCH1AWOYZzJGvi2cBOaeaO56TxgjOCUZqhBQvOCci4ByYK40AbeODJWokD2sM2zhmia2EfDX6DoG7llI1fm6ER0ILFMFV74ztja+9bVzVXKxAYroanBwjW/UmC2+OcirOWQLYQWtqWMCluF25pYv3qpjfLHLcdlnuYJOYqR6tock9r+6hDM20jM/Dd6CjV7lzO4tZxgj7UbPqAzzLM4otwU/wznHytuRMXSbQKwKA8+8NuP0RQpD5yAHcD+TA6FFo12iMJRIpgUQ8gZPt8m1bRd0sl0XbEhKtElCdyTAl0rWla+SVJUC9brYwpeCkzXijLdqAWdI8EYIOaOD+MyFPWec1hlYNydH8QTNhNZRaiBA4jCgALwj3DgQpJkW0Ke25WAR9B6zLoemkIN8jDvniWPkNoXl6XqI5wicZBTOOAfGHRg2vCDRGfAR3xfeLz8AhQHSaEzpCUUN/phRVeJ5Niw/GaArOsPUWmnHqoqeB/CKoAGy8Bwu0lVw1BiTZvYPgKxQeomuapxPQruqdinVTqnOEYDhPayIlRGN6mIN4W8rqRiNTvCTQlcbKJoIr7/uJhy5SFspZ1V2tEQAJbT2W5487cP8B0pvv4QrR0ntAx24hk8RgbB8tPCtIytpXDHsxg84afgNY8cbecCUluffnuEy5OcqAMDId5LSjXmSuZ3ShLKlL7y7kCc1wQD4UgW6yWoJT0YNyGTaTjAJwCXVhtZXdS276Jq649mT1CUbE9xq3Dxq3dRWVi1UfQ0IIuIJl+GYluSmuXVwjIwWDvCvpK7+5u3pkOuL55Y+la+m6aUDFA9eg1orX5USmnDdMG4zY22fF6LhMzx9iKAd1O5Yt+kNDIDE/CJTx8FoF7ERI3mwI2RUbuBFZ5e4nk3XtS2sbAWk2IA9YFi4Ve5qbSqr21hFB1ezqlLXRtOp5KAAIWouNVXTWCXGum0crWS6yQwfuYyBmQ2loC/Mlo1OxynXIz7Cu0frRgGZHSM5B49W2kNGgpPlQF/hLmYj5oqA8TVzCAYAzdH5K1+VFAEqR1NRzXHR88LefAYMg1UWNlsbMdFGusRg4Xfii43+Qm3EQAN3drnnC19niTaysemUbvFaHXCY6brU6ljBqIC6STUd8D21etM2VWeSrRqqJgvoGWAebRhro7fjKN8RLoI0MxmOx1KYMbNlo/MhPsxN2v0RQDiJzqsRKymouBKEzgwVFdCGGh78BDDmPnDYMqRhekn0FzNWztKEgwb1xAD5wOzxHB8Ql3ABjqCCcJ7mqstsXH4IUJ11lsfz1MRFBMmAxgN1PlCZv9BHDOCGwEwKwM58k7NM1bamTi2MYtDwI7rWh6Qr0WrVwUtMoatSY6AJUl1xO0K1STe6g51jkKGVXThl4uZ5iu6rYDo5fB/A7ZIq95t3b/UxnvoPncP2VEacZK65oVcArx4MyKPA3PArHxqPYszWBkZ9y8cBc8pnqrz8ZBFpkuAY8JDfQdAWhjuwrIuB6SakDxdwJ+CxB47TEYDXqSWGUyQVQ+eqrjFdAw3XGVE3LY+VyNZ23jcxdsEJXUkPp76qdWzAFp1vIEQabsSIO8Fi0z3uQ/bUUEqBJ/Iw0QjE0LPncY233b0D+LFDf2C4KcedDV2yEMa/K4lfcLd8KLlNcMn6IxmzY9YHgy5kwT6lCRYIUHOgC30M3Ii2vjim+QNgypH3AjSY6j9mWDA8Sz/mBzNVg8BZKWadDfUjlhNGxOXMi2B6c/lkBrQwOADA8B+Zs2EWRVa7GIStoR2jq4NNWqeq0eA1V0FbNqHlFhv8PPilsqKbUkn8zKWKWl57eUI/HmFA2EpICZxArRWcsH/u67Ll1IubU6YXqsL5sZZk/TJbkoBe32BI3ARtdkMkjxZKBiaH/ArXg8cKdnoUkAi2eLKJDH/ZiXyUhAeF+hIhHCqCDrND18fGzjHxH85xsRDcCvRG8O2gUwapFZB+KlLYLW4Y8MNcMaqeN8ayIs0fcERA6uzXwgE1/cfMzvHzYi58G2YcO5YAgZWfhFwUUban5mUuqr4sj8LyvLkPcGxhE3niYIntl7oGlO3AvszASJWoarjaKebaDBXwY/TQu6Jq6+S1aKOqO227zljHLDATDtIobobsveXaOR73kBkglIi7MC5qRkx+sz7K4jfTkjnnOGct5/acXCnRwMNFyvgh58zoNf+sxS/Fb/brbRniAUwB2MxFS7BQ0l6yA8qYFbifh6SD5g6oPGtNa+K41BrZ6a6C3nIBHoW1scXNYFOCiLqx4Amjahj8zjTA7HUjU2dVBV+jGa/3zXrhag8X97uz/mdJjYn76iHcV5F7/VUOuXmZk7py4AIyszvIvB8iffC9OdY8s8cDU2aUKQMtG3ZRZZbMkblKzkhxKSdDr7gEDF/OVob211BhcnbokZEX8Wl0TMSE2eOJZzW0056ZYd5kJReYB+CZIwMFArc45l9y6ytnj2WPOX9QwSlFxxn04jap0vBDDzk+uzofSyHP96zumfN5nxOXyXsUEWA9MHjJiQwVIMuMwEYxJg0tGcMSBdh/O9B/LjLJNAJS8ySWXeRQQ6kF5h52UHWmbmvrTcdAXoBRqXIGpledbF1sKpoPo2vfyLpTxuL/drw99t3rxcovaB6ozCeKVHRD8TjtWCtGkDN1sPJeb434nvHXbhNlLDshcGJ12WqEvQMXRZiH/UiPG+zcKdYS8TxnNhnZC4qLxgMnMuEslodyRyY/ZiQCECHImcm125iVDi8CLyHNzEDcsqRwc6TJtSoYoC0jL5cVF4LBKnl4v2oAauFulxg288jyV/j6zDPLslO+MnUr55cJv//QfbgS2J0JM/wIc/IijsrLRVg4v69mQrrMiNceeGOw8MzzZrBTe8MKKU+Ul5KIozXtEIwx0DU3584CBudh/YNpFGxD09omhVCDX2xliNxhUQ38r8rXthE5DZMJZlIxMc3FzrcnwPBJgQEozlUH6BvI0AtM95+A+CQg5unOnwcQM5I/YXj9MgiZyW4uZ7zPImSeJDDCPBMha1Zbi4sSKHwrElgBTJ1c41zXQhGmpoq+Dl2tg+ksXEIDMZRNayp4hBlBRziCbSOaaM4j5O40QPYRMirtmOXfnMRQUNM27vi5P/gg+uM0b3YoS+m+5g5YHpYOTnX0I5ZXkSd79lkZQEtKTNPrHaBaqefK89m+HODRDI9HNTtyfWToJRwvNWRXigg9iLezytphMM3YnD0MfW/Lh9syuTSxZNbnD03NT16XLidH8mNmizO+LMvn94OfBa2p8FZy7BXaDcCpdYHbOI4hzIsCHhKwS0tjrTOw38KGRV4hnIPY5p1A1fq27oLwbQuZiQ3LWHa44t5a56sutLpunYltp1UnkpcN0NRBOvGbhSwfJZMToJsjywy4Icv/4h6hOukRupdlhamL6DfcwGQhKAZc4Wtf6CJa7qIF/L1iLpZZ5CIqKZpGy9p2sUtBpyCbrhZJxORsK2wl2qaCamNF/AR91/g25VwPvGgdqmki4aGL2J3yELnWb44u9ZsXWWlYmgg3IQqeFYL7bP89VzoycQm6kAVn4IxIiPkF6dB+w8OkptSOg5aN5S6nF1pLXfPsCCCarmWjFHBc9JVmpoM1te6qTtVGVq0COIJrlZok8XMrXKUtXnS00G+evs7fng8EsF6S223mCB4lDH0pjZ4XFBwOYcI2EIC5A/6PAwGaB5f6o4JXOteFjXJUDw9ojwfMd2dzgL6j2Z5G25ovFXkYx/aRAB7z4w4Sq7Syvt7s0CMjL+JaR2zH486QT8lox55rA0FcFLRuUuS8HZ8Nns7gThqm+uNrj/FEQXyCGE8UxAdtng/PhLnUHsLGF40FBM/zqkxB0qyQ6vXMThMwqMfqO54uYbGqJ/o2Lh8Sx0o7JmpYuWSjibU/DOCNaWpRN7r1MIHCdHUSVgH4B5ZeTJr1GZpKBOhGG1raq85WCdI7DoV++3qpyYMzFS3rt4LVhRmIx5I4GTOEdtkaua5On85YYgF5DDeU7a42bDBOs8DoUIZ45gxE6pM3LDwf7t6PkCDWQartZj63ICExZozuWBjc9OJhY/bo+6HDusbDoUdGXiweTHOAIyzgvoVB9lHkua+Yz5MZbm0ZwT3WqHmmkl/DFaZWXH4jsozgg1JheX6GR05zHAAfhzkBkDf/sgYCLwrPhJWumDjK4kcD6VAbVn9nVXQm9pRdsEsymCQrblsWzIbak4sSmAwwQD5RFjol6uRFJxosWAM/KLVsbAENI4Pt4F4BvwFqwlsyFTNVeHLGyjORsmNOkMnAnlpCRjNEhGfzMwXLkAJF5hCyy8e9WXEqnyyYuEIB3rcq535FBOgylpU0rWayXAhDYwKHgTXpyLDKQQiyP6M38Kd5zFSOgsuRp5Pz4zW8V1+KquXEzXHWlAW3WpErm7GSO+sU9JWhJCt1Ts5jahiWch6TR+hKZqlmUokJzzQteBjo7LwaBs7yMbQrz7TkQDAEy1YOZrLQJz5ktiasI1OMihrZFhoVnxO1+BFnbIs9Kj2X5djwdQVPlkqtc5G4sfgw8wC6likiLPx22U4b9AkFKAorGZpflGRTtUJ2sWJiheRRZN8q1owQXQfnqeYOh25jXYm8fVy3MDyNqz0TXk3DMNqZuNlRh8qDm2EHbZQuqK34dL94COHXE0GIP1sAwe1zpJ8bL3ARHrI0s/ECeI6a+eHPjBfko6vCLdpFqRU4F7bbVp1jHrSBIWDZia5r8L5tbSsvbGtFis4kLSpvauWr1ichkpStOR8v6E6FC3i0BoKy5+5f2oP8BTePzYb1QiQrWLLaGKzBRQ4ja1Kw0hmLTAkGh5Y4jIKnf+qEBauhG1INnwkQoGbcH4trGxvrpmXNMSYx+kY0qalZ8xPeVGtcJc45jN0Jf7FbgocVSSP3CVgsbxP7jNIt1mWcQux7JZCKO2+wHyNZIn9QYwGuNf4d11ig9d93aoEGoqcxdhihDWFiXO8wQiYC9SDD/ir4ODv0yMiLmBRAY+Pz4RPgFeWGiJj6iZ4hLbP0DmY6mJjtuVDFi+TPpHYu23PnynFyl+25cGVbLOeFQZlv7IxZf2ZklLVSmQPAciwRLy/nKi2wcEDMZ7jhRboLdgMUi6FmNMDYmF0Egyv6e5WsfCVSZ6ratPAE28BkeME4aaiTaitR8WB79IG1RrqalcpVkEl3Z2HwvKLTQMBKAbATm8a9TJwHwfQRndKliHspEY+vYWLG6YS64ZgNTAljofZgIBD5YOB6N3LYe4JxG25cUHhcHgdPiql24z3gXFVH5s1in/sLXa3V3Ci8nC47xTwnuR91uVjIHCeL3mFVzLCwgs/FRJTMX7mrC8mmDMAQUgb4AcFQksCVXStyuYX8oQGw8g6xzTV98sdMZQXzvMoKeD+wGksNs8XT9OiedoCXBrbQ5E3jcBEEgEPM8zGebRwEcPSiHMlWd1goaHmIUZtgG6LuGqUaSQjgg6pkZ2RXJwO7EZgD14WaG2ciGd2JTp+Dt7NiEVgxwDumxcJX2qLbu4//CW//XTbI3GESxMttkCmQXqg5wJvzeKW14XmAF4YkN1Bxy9Im2w6gqdG1b7vkLLtQNa22qq5Dk6RzVeMauNJaAQBJX9eq4XkdYF2g366WZwEvmPg04hWsPz/m+G/enjmSYxiIiAW0sPkMI975pI60at9wJscCACz7kqY56B5Z+N+QGbwY1n71Eh4m/Nhcuhy6PAg2p/EeK2PkKNVOsj5PibBLy+ChL6UMDXskjWIc2iuWvMntDi3DNaWyHkAAq26MWw4Dp4m+iK6PQQPIFonh6UcbLsyiIPogL3gGAoEfQxiUj4gyl2czIidTeJ9Rk88FqfDULD8+5CBi9PyhFQzCEwvQmMAY84MFKhzT6N2sjdAvK0b5ndktR+eiT+NyPGxVAz/PGsUWVaxZfknuHXE0VAwESbOTljaLIu4Mf7QdqJdi28IhtIY7yk0rlG3aZEWFH3e1BRCsKriQXZtiAEqrtNRdleyB2/jN22ViZFkSWcFjYSXRfeoxxeiX9hzDv+feo2KJf2JKAV+QjSH0hb6kzNaFRZ2Aqnz0S5xJrKg2NcC0jnXXJV2HyrO6s5CNqlpW03JJV62CGCdIOc9aq8obsEClFFjhjDM5v/KDlf7m7dGl/mZagEkAzhmlR9VItn0LHVyssETVzPID8FzQgb26Ym4FODiLz7aLM7m56qWtbHmO4kkTZkeFaUkjHscxNu9/KX3hjoNj8WXGvuHRORMX5V35znUCasFD21apa9jpSglY21Zp36QuwuSzQhJP5Wme1LUuOm9Tx5oOcpyGgMVexiAsFsRqx7iDJAIb6obT+9VlwwG4LVdpcECT3paa8Xjt7S5Cb18lj8dn2IivhpWziC9ZvHYML6Gavc+9IYCEDLQ3q9Z51r4zo269PJMHf5NPhtiI0DcWZbUhMJUeFzjxKj+QPUjZxK90Xs1NZK0dVi5n+1PL/nTrnCnPvnDFegsHLbzfhFcbpdlvxeZIh+H+BFsNMlOZZUOdu9RRoxlWPOzFnSopB6rS5x1tzy0Io3JyO202P2wGrbIUVWGhE+58mxzNMF7n73IiL0v28ofRqZnsRtaPelkh4xuzLw0AE0CL8GpSBciRDbxXubSkvsQEUwvjCfBdCWjZNnuRCYbD1tahjsG1kQ3IQxWqtqma4Lra264xFi5gUwOzWg0drGPHsqohwDiLSpkzm95H7C97c2BZNLdYVa6iupWx787JGFx2a/syP0zj0KUAHXe5oxFy6L/xqEMIff4iN+ugylgZKBvgMOqHyH3W0mWFZ7BZOqLswLFvoh9BTtYcZJvFXF2ch6DySOBn7vf5CY5lYnXphKUFfepSozofn7X74kVgjQhMr3KHR7hqLOGfm4RxR5VVKHd3BWq0nkWz85w0OwwqmwUUHA9A4+QzZM15SCubL3LHaFi7g+HBHPawPGECZcayq8bn3fNYpC5mqYu5gnGOUZTyrPgAZYh/g5ndPZcvnUbMtzVsx0WpZ2R7jHUVBSywUybAir5QzhRrP8BPZwa+2tZHOuMvproVLQ8IS1uDo2DFKt/qLgWfGi2jbYAH65wMUVd1bnto28qzI01sZevORA3nxYwF3Fkhk/QIUQ1N2dnqDZJ/0ofA4WrGbbM7Nh93Wo53z3XuCkUrBTeH6VWlJS1PFGozsHswRZAH2RdKz42o6C5G+rNRjepMssJ96RRiRT4gSIPCDgRy26NuezKF/cRLDyjFCIErAseWfVQvI4HMebYu5G7TEUYrv7Hn7vvIAYV5w9ra0i9BwuNVpbkJPYbcLu1yEWPei6YnBt4ZFXSNoQTcKVohB9gFRY6ll8yVK/IlTPY24cBlaQtZ9jQcS8WPUHL2zdxxLnEidfmyOCTfF2bZAWowY9+acZ0dcglz1CR7RMnLPEqrFPthiQjVzj4tS+SM0bjGpUrGOmaI6ERXsSaRbjp4vzQCvqkZtktStap1oQuukS1krsZP4pkw5DE5Y1yNxR+lccxXy2Xz899xcfsiGOWod1/zaVstdhfbHJx9KcU295v62wdjWR+voQo/lAZPt3fsIpDX7mv8dLUjytffgyavhk3a6UceFA96TJ+5zJ95z4fq8fru9rCD1e4pZJWvxfCmktGNY3clp/yQbvmr1b+s/pHu79a5NdWq73Bw/EFggK8ZTOBir8BRX7NmE795NWpRGo4+mtxznfCo6nH1+DGtHh7v7lO7+pDuPoHVTj16ASVp8w4L6DyXkuLEXZ9HSQelRV8rU5J1iw4pGY8++pmUfHw4QUidk0qnovZ0QsrRTe2Y0Yc3fR4dWVVnQMdwjo7DJ19ORilXpaHb1Yp93la5A8jKjabMI4/DJ3+uHj/un32fbqof8bA/vPl+1XeMewb3cwfssJbK6UXLXUROcr8aT2B01yes2uRBZdVKVfC8aiz8jm/M6NnjVRs9+gnLNnn0ee4vsPdp3D9HyBH3hxdh/lky2rzf2jN/zNGbMRnDCzD/5MlxjvfHq2fjeMrLeX9uxU7zPjvUzlUXPaOySufxE+yvJnMY3/cpWmvyqKL+vRiorVxibqS2zFiixw9/iuKaPPy8CHjzdBGYpeZIBmw4blaeTcvsLm5peQhKThif51LyHLyTh5WYn8uXbsruL0ZIXVpDFXXC+tf8ZmxLzbFH/7yElGI86ReScGaUHLnrM0nJgvBDWOIO5Vsee/QzSDmPS8bCECdg4Qm45OkLpyYisGtgc3rtihN2BpuLMUaY3PoJC3jwtCIMpWp5v4JKTiGKn7Dk+PFPWMSDxy+hK0MLc22ynk3XfJTg+J2fS1ZVUEohK7uUTQXDH6zqhT7kPFlPmj4V3dNt3zxNR8aPBbRfwPjNEtRsVBzwqdBTPs1JC8+2fpcwqRgrmrITfJqYfdTltOSPrdDwtk8g5uGTCnuy9rgsepsJ3fxm/HB/7OFPoObhw5fIvJmYqz456fkUNfbUnZ9NVNm3nMxExaPs1E1xByrnzS9HVjtxkr57/UJsmtOLjtz42TRlivTQPKmp2Mex2H/3+iUpetqHNuI44H4iReU4IDGJ8l2oR+cpyuTagTOdLZMbK1LzAor0Ih6daNK+dMN5iua49VOU6ejOTyTp9GG9dco1FHvRt/mbE/p09PwnUnX6/CUqVR6ijZsXI60wBxBxePcXIK8tTtrWzShtE8ZoSpx4g5+fwFO3f6lynaWuHt3Zm/HaPUG9rr74lxW3t35aVTd3tx/y3NOPj/fV6tNdm75cRnkXB5TPcTg7iZ7GF1LAs3Q/qYKN1eOFX6qCZ+muxvPSJwJEPz/hZYn2mH4LKnsTZkJ48UJ6evr8MpWRUeBh5eMAmx70OnviXXVzU1cNLu7u86OPFNh7mngdKBizxF8xZ3e95NG7PsldMYeLlxvsbW2snIkwHX/0k7yV0aNPGVZ5dL/yx+9/vwL5rz/98OnE7oyahFykPmHehrzB22+jLeQTkJM3OPGoxfhgtF1yZBqTQPvBNCYoQT9jGpNHLZ7GOOh9bDkmQanpRN7d3aaXm8r0acsh2zg8dGQyB172wbKMVJ0+HlZYMJeDhy0Wk+/Oz+QQ5p6eyvME5fBpLz2XqXI+Lfbq+GbHwtnMGeKVWq3/y0qON2vM8Ue9LWlFeBye8IjnQXEyxWiVM4lKWPjsMzWfOVIG3qvjHvNzn/kzhpWe6Ao9J650CFv8xoRdYAm+eTwMLLmXCixd4gaZC92gs1TN3cxfzAnqLvCB3Ekv7GcnbpjEDRbHl85RVskw1v/PCTDNAW0lBjkauWi7neBE/1Ihpic7OOpEWPupVB0HmZx6kZ3qo3og7PO++swXO47b2ZeKMj2ZVfUkvvbt6xciqhaT/apvfzZnfeZdVvLAhjFv9+gLPdWGzSyz67d8t8ssD8Oz8tjjf351r0/F25+lk7Q5deuXUEpnthLsQUDvzS9I2UvjtAuwiX6pMG03F6W1bhelZYHjwyite6kobXdBkMPNHBg8T1RmZp9m1jEwGN34iUSdPKuoABUGuES6w8xUcfTxTyTq5PHLdIC5TAecJ+tJGXw+YRdsJ4oX0wFPp2xwxwO/z6KryZ2/f5GI9wzV1TDPwJQNR3sqdfYZcHCO6KcTN6y6DA7OEX2ct6jcxXuOzyf6dmOhD3YbMYPB4wvBxadzup/okG9fitOtmeT0fvvLcrrcZ365mdy9cgb5RcDbBYrbHiCsUlzjJTS3PExqG978F+B3YwdKxuop6Sde3+T9fnbiXxh6Okv4lws9zfPzPvSkNiH8fKGnydNPEdJdElA+HYOdhvrF82PK4weemo++YD4Tck3nYydyH545m/y4v1y9+vGx5sniEv/PF7uKS1jVj3c8cfrHN+/Umvf94rG+uX7M0rudxcGpZLlRfeu7bW0Wnug3Lux6yseN6fvdlYptLF+rQ39+ftuWnh3tw762pwvCqDAp3C29E3pX1M1qyd6y41JtxrEeSi7qJrVg9e4yVJiD2p790PWxsSDWwYnWg+PDYeOsCMoHoazTbBW0r8wuah+taJOTlXAmVqnyddsZkMeqxmE2sm6TjIYF2WvDHotKd6lzKdkuRHLHq7yScwUkpfJCqlIkhESXeGftjNoRnYXIWTWmJxjE3LHbjHc7queCXMqA8n7bX02xPOGgJWBugsZa9Pvu2ZiqZEOT0clsxX6JuVNBlCristRpcKCwlGF26PrY2DmyN3d39+31bUVJcax14kNkd1jW0GaHvZllsca7aIM2in0MsS63d+8LNfsqOJSV91nkmLU2rJosU62kb6O27DVZyaZhHU02I2lMJTrlk+rwWKkagZ9aUWMpk2Q9OaFTrLhu96lqHqvbx/czm4BMrMeS3lR1PtH9ev1u/XpV3X64Sasv2vSBwta/WI7z/nNm165sRb6E9CoWqR9Jr4iBzRp39QHWamNkHIivYh8Qo9yAS6LyEqthd1zC8hwyjI75c4yJzo/HuP7g/nlJy30xnQkisMCy8HFYLiV2LXvmJF23PnY2pChY1lV72UTl6k6btm2rOjSmqVVVNZWwopMNW2U1SsrToqZF1GLYQJ4dZXNRj6tdUSrFutzB9RRgdkOIQYZhq0LIlmJFpK0Cs0Z7p8OwvoKyximv9LZtDnuy6lyCYUyinoWnNCqFxHaM3h/CJ5+zCEA+DwKOEgPCOTA7NFTVqVjVoBP+hy6yTd00tsWado0OuZulaSpWAata46SNMbAbYOOO8fp2837C6m83m83bVXv9gOENuP13p3i9323PeRAvwenrQXUwIXLjlDjQlqLnbmmkF2ZUMUbsqlILa6BxlFvIsnDZ+SCWkAls96VH7bwaG2WoVNeAn1PTgrbQHmypDHJ3nakbo2PTVY22UYiqMXXbOG+l7FLshDjJsoba1JpcTSpflsI5kG0FFtODaStJFVyKc5dr/I1nHT4h1JAzWR8MJjiPy9fllk6wxlVwC/S22+y1thRWzRJMB29ULtigWJPkjNoeEhQv4lKnY/JdaCpoiAamqtZGBN10AsQTtgZkqNsKot/FtrFN28J8GGgOIVxzjJVLVtCUk9dQ2wuVdknkyYkwz+Xjd0UfbxvM7OrgAb8YPVLIhnVw5B4seRuJd4ZsbTywR65ixsq/ApBrO9LHoOZGro8NXSQNZgOjAW6BUnOWgGtodFurTAsA0CmsYTDwBLtUxdp2hFaAQ/wTFrbXjXIVvBgnWyhw24G3G0hNmBeHTC9An74jcwDkKOWxwcTkQhmHNKOFYom0Nfds1NYoQqur4NUIPBqW5w/W52pTsVRxL0M1Sw9NhmrPErnrMlbChpWxbCMaFwCe7HfOUVM61nxj+RvW9zwjKkOtX0ORBFN538iqCaC8iHVSqoVKT1UNiyiq0JHMpoUYVqLVLDEuYEA7vHV7TFT6zLOJqLxbv5Xrt2r9VkPzf0ztfXVzVmRK0tg+R2QgNyzic5kBKO2ZWWxoZwZob8OwM5fduBLy2Vrg3O/Me3/QvQvmPWy9lUi+UXrUmkvBnQly14gA95m4H0qaUiEXID1XNszDIBjWzo48MnCR6EnNSr8mxig8q2WzkNiOG5JoLLvmmg6sy7KSFoC3BaRSse1C6gCdwAMOvlaE78KLIHOnNdeIZMMJP6X4dXBsQPG4cXqLInNJ0mDiqB1g7prGWvMWsKfU/Ge5KlZ6030HhwH9pYh99zRcS6wjloEts5Q2o1KNWF4Ph6vvpW6VZHUp3I99ht2k9TktjikF7S0rc+dmDywBJ8YibVkmqrSQwki4Bbn9NG4qhGdhrXMCrVi9GXo0Es869sKbWzIeliQatNA+rGHsn2ALkzEQ40a0FpiiwbI1rZWtpuqGzYtAE07G2DrBgoIKitR50TVggTq1jTTdEQH/btZ/Aah7ffNF9fhYNX+9vv3w5VKAN8gA28VbXkTQS88R1oDflpGHtFgYrXH/SYIhtrAo4A6OkID/u41bbNuSOG2V3so5RI4tEMcdRzR7QfttCz64nNKPHR8roUhlaWMWWLHNaPILO2h66/Xs0CMjlwq7jBJenLUeSggsNuCMpgZwB7SUQdei9caCvQAUjEk2gQ8SVH2AMotQAQHqooLKTw2cKBPITkqdhJ0QRSPL62eqwwU3EJYx1a0RbM2zp7oh2PQjqmuGGMLVlupEjaPCkIDgeG1vdlT3UMHTvofCi9D3PZTsSFJIqany7ezQIyN/JtcLsLTVuq1lU1PhpiZZXTcWfCp0hP0NNZvctU2oTecBgjx+3wTlk4wkj36qjL65UES7gYR2L2iJodfNqCcQHBM97gmEaXqt9z2BwCdBT1pkas1C9H3UUAYyytgOAyT5XLW+NARywrsw6e/AmFapZh9gY33MhfPZakiPHcHdyPmBS6WTHWc9kHBkU6FhGKOrbKNbI2olKwhk3cXYeFVp1hC2jWW/xZRYeK9r4bawlx4Am0iVaODdRNeescSelR9JSFuOj2Yu5yuosc4T7ODjismEyhMhEz+w+KMZD4TN89uBYFxLaXXs26NGTQaAAHLrz95cg5ezwcSCA9FM2spoJ/ogEWxrwHLZMlSylfVkKIvOFiNMOFTGsTXIAlANZGdZYtIz3MXCzUeWChTDGAmlabHcT4kiwhWtVNXAq68D21+lWncdUKFO0mvb+dbWriF1TV0lWGeLf0MFO227ynnzi4l392pfgPSlxBsLEoItAXVBuXaqRBjYCSe4obJXNioAs1Ltmv3TItwqhmEMW0oPTS08LsE1zzdlaWu4T7nDhMx1UEeGVBHrBFci0oCVovTu1DZa2Pw4rhwc2VZab3t3WuOzKgC7w6vyo3rAYC38dWZ4BajH/QoeFtaelb4H5YAVvOIoiv8ocw/x/NaO6DRE9wRDLsjc7LYqYBGHqL1RqVGdlwnOMhgLv2xrKJQmJHaBrfFyvmshIM5VMnWqs/CoVR29c6ITpjsR8oRoAB0CBpRgj1VeQFflUFJucOt2q4f34w6DK7Wcde6XKvNIUA6aZWiylTDSRrOvCt0vCeCAg74ZDmWHsagcYYSFONAQkyPYHQpoZsgRNvdIyZWWFYcFnxecbRPEsHNKbiLkcvc2YaEgfFaDYEW+7zAGAEfakElyH1muPi4ZDogBWN3LJdheszK6UmA7ACkd/BF3DEacnTmwrI467qRiGe4q0Z+RFVahAnB3wacI97CrWjikMHSd1rKtE4xFWzkTdNvE6GwnpagkuKXp0lP1yjcX6pW83/jPf/5/NFHWVNTkAAA="

**Cell 6: settings.** Deadline, extension sessions, dry-run flag, GPU detection. Starts the queue clock.

In [ ]:
# ---- Settings (all optional; the defaults are the plan) ----
import os, time, json
STANDIN = False              # True: dry run on the CPU with cheap functionals in tiny bases, HCN, NH3 and CH4 only, in a separate folder. NOT results.
STANDIN = STANDIN or os.environ.get("B1V5_STANDIN") == "1"
DEADLINE_MIN = float(os.environ.get("B1V5_DEADLINE_MIN", "50"))   # no calculation is started after this minute (clock: first line of cell 1); 5 more minutes are reserved for figures, check, summary
AUTO_DISCONNECT = True       # release the Colab runtime at the very end so billing stops
EXTENSION = os.environ.get("B1V5_EXTENSION") or None             # (the environment variable is for the tests) None = the one-hour core plus the tail of this GPU class.  'GPU1' = EXT-GPU-1 (tail 1 to 3).  'GPU2' = EXT-GPU-2 (tail 4 and 5)
RETRY_FAILED = True          # True: calculations that failed earlier are tried again (second Run all: the Skala Cl- + CH3Cl TS gets the new PBE-start rescue)
FORCE_RERUN = True           # True: ignore DONE markers (finished points are still skipped), so a finished phase with failed points is looked at again
STOP_ON_G0_FAIL = True       # stop the later phases if G0 cannot reproduce the stored v4 energy (1e-4 hartree)
RUN_CORE = EXTENSION is None
RUN_TAIL = True

# ---- what hardware do we have? ----
GPU_NAME, GPU_MEM_GB, GPU_CLASS = detect_gpu()
NO_GPU = GPU_CLASS == "NONE" and not STANDIN
_plan = GPU_PLAN[GPU_CLASS]
CALC_MAX_SEC, GPU_QZ_MAX_NBF = _plan["calc_max_sec"], _plan["qz_max_nbf"]
SYSTEMS_TO_RUN = list(DESIGN_REACTIONS)
if STANDIN:
    if not ROOT.endswith("-STANDIN"):
        setup_root(ROOT + "-STANDIN", mount=False)
    cfg_ = read_json("config.json", {}) or {}
    BASIS_OVERRIDE.update(cfg_.get("basis_override") or {"small": "STO-3G", "main": "3-21G", "large": "6-31G"})
    SYSTEMS_TO_RUN = ["hcn", "nh3", "ch4"]
    SCAN_STRIDE = 4
    GPU_QZ_MAX_NBF, CALC_MAX_SEC = 400, 600
os.environ["B1V5_ROOT"] = ROOT                                             # read by the very last cell
os.environ["B1V5_AUTO_DISCONNECT"] = "1" if AUTO_DISCONNECT else "0"
q_start(DEADLINE_MIN, t0=_T0)
Q["scale"] = GPU_SPEED.get(GPU_CLASS, 1.0)                                  # the priors are T4 times (plan estimates); L4 and A100 factors are scaled guesses, the running mean takes over after the first calculations
set_design(DESIGN_REACTIONS)
quiet_geometric()
write_json("config_gpu.json", dict(design="1h", standin=STANDIN, gpu=GPU_NAME, gpu_class=GPU_CLASS, basis_override=dict(BASIS_OVERRIDE), started=now(), deadline_min=DEADLINE_MIN, extension=EXTENSION,
                                   tail_items=tail_items_for(EXTENSION, GPU_CLASS)))
print("Hardware:", hw()); print("Versions:", ver()); print("GPU:", GPU_NAME or "none", "| class:", GPU_CLASS, "| tail items by class / extension:", tail_items_for(EXTENSION, GPU_CLASS))
print("Design reactions:", DESIGN["reactions"], "| systems to run:", SYSTEMS_TO_RUN)
print(f"Deadline: minute {DEADLINE_MIN:.0f} (now minute {q_now_min():.1f}); extension: {EXTENSION}")
if STANDIN:
    print("*** STAND-IN DRY RUN: cheap CPU functionals, tiny bases, folder", ROOT, ". These numbers are NOT Skala and NOT results. ***")
if NO_GPU:
    print("*" * 100); print("NO GPU DETECTED. Choose Runtime > Change runtime type > GPU, then Run all again.\nThis time only the G0 diagnostics will run."); print("*" * 100)
_cpu_cfg = read_json("config.json", {}) or {}
if _cpu_cfg.get("smoke") and not STANDIN:
    print("WARNING: the CPU notebook in this folder ran as a SMOKE TEST (small bases). Its CCSD(T) numbers are not the paper numbers; delete the folder and run the CPU notebook properly.")
if not _cpu_cfg and not STANDIN:
    print("NOTE: no CPU-notebook output found in this folder. Run the CPU notebook first (order: CPU, then GPU). The geometries below come from the pack anyway; the reference numbers will be missing.")

**Pack and weights.** Checks the SHA-256 of the geometry pack and of every structure, writes geoms/*.json and the rows of phase01 / phase02 / phase04 (idempotent: the CPU notebook did the same). Then restores the Skala weights from the Drive cache or downloads them once with a wall-clock cap (the rest of the 15 minutes of the install cap), and caches them on Drive.

In [ ]:
PACK_INFO = pack_setup(GEOMPACK_B64, GEOMPACK_SHA256, systems=None)
print(f"geometry pack: SHA-256 {PACK_INFO.get('sha256')}\n  expected      {GEOMPACK_SHA256}\n  {'OK: the hash matches and every structure verified' if PACK_INFO.get('ok') else 'PROBLEM: ' + str(PACK_INFO.get('problems'))}")
if not PACK_INFO.get("ok"):
    print("*" * 100); print("The geometry pack did not verify and was NOT installed. Phases that need geometries will be skipped and say so."); print("*" * 100)
WEIGHTS_INFO = weights_setup(max(1.0, GPU_CAPS["install"] - q_now_min()))
print(f"minute {q_now_min():.1f}: install, pack and weights done")

**G0: gate.** Functional probe (which of PBE, r2SCAN, B3LYP, wB97M-V run on the GPU), Skala check, SHA-256 of the weights in use, then the stored v4 HCN point 0 with Skala (D3 off, as in v4), PBE and B3LYP: must agree to 1e-5 hartree, else the later phases stop. Cap 4 minutes.

In [ ]:
run_gphase_1h("phase10", "G0 gate", g0_1h, GPU_CAPS["phase10"])
load_gstate()
print("G0 state:", {k: GSTATE.get(k) for k in ("routes", "anchor", "gate_ok", "skala_ok", "skala_route", "grad_ok")})

**G1a: five functionals at the stationary points.** All species of the six reactions (20), own B3LYP geometries from the pack, def2-TZVP (def2-TZVPD for SN2). Order: Skala (with its noD3 row), wB97M-V, B3LYP-D3(BJ), r2SCAN, PBE: if time runs out PBE and r2SCAN are cut first. Cap 18 minutes.

In [ ]:
if RUN_CORE:
    run_gphase_1h("phase11", "G1a stationary points", lambda: (g1a_stationary(), g1a_align()), GPU_CAPS["phase11"])

**G1b: paths.** Skala, wB97M-V and B3LYP-D3(BJ) along the stored relaxed paths of HCN (11), NH3 (9), H2CO (11) and the CH4 stretch (9): 40 points per functional, one functional at a time. Cap 12 minutes.

In [ ]:
if RUN_CORE:
    run_gphase_1h("phase11p", "G1b path points", g1b_paths, GPU_CAPS["phase11p"])

**G3: scout check.** Skala, wB97M-V and B3LYP-D3(BJ) at the xTB minimum and peak of the six reactions (geometries from the pack), and xTB's own barriers. Cap 6 minutes.

In [ ]:
if RUN_CORE:
    run_gphase_1h("phase13", "G3 scout check", g3_scout_1h, GPU_CAPS["phase13"])

**G2: basis check.** def2-QZVPP at the 9 stationary species of HCN, NH3, H2CO with Skala and wB97M-V, system by system so that a barrier is complete. Cap 8 minutes.

In [ ]:
if RUN_CORE:
    run_gphase_1h("phase12", "G2 basis check", g2_basis_1h, GPU_CAPS["phase12"])

**Tail list** (same admission test; ends at minute 50). T4: expected to be empty. L4: reactions 7 and 8 (F- + CH3Cl, OH- + CH3F), perhaps PBE and r2SCAN along the paths. A100: items 1 to 4, perhaps the Skala-relaxed HCN. `EXTENSION = 'GPU1'` runs items 1 to 3 and `'GPU2'` items 4 and 5 in a session of their own.

In [ ]:
_tail = tail_items_for(EXTENSION, GPU_CLASS) if (RUN_TAIL or EXTENSION) else []
print("tail items to try:", _tail)
run_tail(_tail)

**G5: figures and tables.** Reactions 7 and 8 enter the figures and the check only if their stationary points were computed (tail item 1). Regenerates the figures, `numbers_v5.csv`, the tables and `findings_outcome.csv` (criteria frozen on 2026-11-15, five of six reactions) from all CSV files in the Drive folder, on whatever data exists; a figure or table whose data is missing is skipped with a message. Always runs, cap 5 minutes.

In [ ]:
set_design(design_with_extension_data())
print("design for the figures and the check:", DESIGN["reactions"])
run_gphase_1h("phase15", "G5 figures and tables", lambda: g5_make_all(STANDIN), GPU_CAPS["phase15"], final=True, force=True)

**Automated check.** The same PASS / WARN / FAIL check as C7 of the CPU notebook, now with the GPU results; items that cannot exist in the one-hour design are NOT RUN. Always runs.

In [ ]:
def _run_check():
    global check_result
    check_result = run_check()
if not (NO_GPU and not STANDIN):
    with qphase("phase15x_check", "automated check", GPU_CAPS["check"], final=True, force=True) as ph:
        _run_check()

**Summary and release.** Writes `RUN_SUMMARY.md` (what ran, what failed, what was not run for time, minutes against the caps, GPU type) to the Drive folder.

In [ ]:
try:
    q_save_log()
    print(write_run_summary(extra_lines=g_summary_lines_1h() + ["", f"Geometry pack: SHA-256 {PACK_INFO.get('sha256')} (expected {GEOMPACK_SHA256}), verified: {PACK_INFO.get('ok')}, problems: {PACK_INFO.get('problems')}",
                                                                 f"Extension: {EXTENSION}; tail items tried: {_tail}; deadline {DEADLINE_MIN} min; GPU class {GPU_CLASS}"]))
except Exception as ex:
    print("could not write RUN_SUMMARY.md:", repr(ex)[:150])

**Last cell: release the runtime.** Independent of everything above: waits a few seconds so Drive can finish writing, then releases the Colab runtime (stops billing) unless `AUTO_DISCONNECT = False`. It runs even if earlier cells failed (use *Runtime > Run all*).

In [ ]:
# This cell does not use any variable from the cells above.
import os, time, json
print("Final cell: giving Drive a few seconds to finish writing ...")
time.sleep(8)
try:   # a short fallback note, in case the summary cell above did not run
    root = os.environ.get("B1V5_ROOT") or "/content/drive/MyDrive/skala-b1-v5"
    if os.path.isdir(root) and not os.path.exists(os.path.join(root, "RUN_SUMMARY.md")):
        st = json.load(open(os.path.join(root, "phase_status.json"))) if os.path.exists(os.path.join(root, "phase_status.json")) else {}
        open(os.path.join(root, "RUN_SUMMARY.md"), "w").write("# Run summary (fallback)\n\n" + "\n".join(f"- {k}: {v.get('status')}" for k, v in sorted(st.items())) + "\n")
except Exception as ex:
    print("fallback summary not written:", repr(ex)[:100])
_drive_ok = bool(os.environ.get("B1V5_ROOT")) or os.path.isdir("/content/drive/MyDrive") or not os.path.isdir("/content")
if not _drive_ok:      # Google Drive was not mounted: the results are only in the local folder of this runtime. Do NOT release the runtime.
    try:
        import shutil
        _loc = "/content/skala-b1-v5-local"
        _zip = shutil.make_archive("/content/skala-b1-v5-RESULTS", "zip", _loc) if os.path.isdir(_loc) else None
        print("*" * 90); print("GOOGLE DRIVE WAS NOT MOUNTED. The results are in", _loc, "and in", _zip)
        print("The runtime is NOT released, so the results are not lost. Download the zip (the download starts below, or use the Files panel on the left),")
        print("then disconnect the runtime yourself (Runtime > Disconnect and delete runtime)."); print("*" * 90)
        if _zip:
            from google.colab import files
            files.download(_zip)
    except Exception as ex:
        print("zip or download did not work (", repr(ex)[:100], "): use the Files panel on the left to download /content/skala-b1-v5-local.")
elif os.environ.get("B1V5_AUTO_DISCONNECT", "1") == "1":
    try:
        from google.colab import runtime
        print("Releasing the Colab runtime now (this stops billing).")
        runtime.unassign()
    except Exception as ex:
        print("Not on Colab, or the runtime could not be released (", repr(ex)[:80], "). Nothing else to do.")
else:
    print("AUTO_DISCONNECT is off: the runtime stays connected. Disconnect it yourself to stop billing.")